# From magma compressibility to surface uplift — for any volcano
### Thermodynamics (EVo) + magma and chamber compressibility + elastic deformation sources (dMODELS), in one notebook

How much does the ground rise when magma is injected into a reservoir, and how much magma does an observed uplift imply? Part of any injected volume is absorbed by **compressing the resident magma** (mostly its exsolved gas), and only the rest dilates the reservoir and lifts the surface. This notebook computes that split, $r_V = 1+\beta_m/\beta_c$ (Rivalta & Segall 2008), and the resulting uplift, for **any volcano, any number of magmas and reservoirs, any depth and source shape, any volatile budget, redox state and gas fugacities**.

The complete source code of the thermodynamic degassing model **EVo** (Liggins et al. 2020, 2022; v1.1.0) and of the deformation sources of **dMODELS** (dmodelspy 0.2) is part of the notebook (cells 2 and 3), so the whole chain — from the magma composition to the uplift — runs here. All inputs are typed into fill-in tables and a few form fields.

| Step | Quantity | How it is computed |
|---|---|---|
| 1 | Pressure $P$ | $P=\rho_c\,g\,z$, or given per reservoir |
| 2 | Vapour fraction $\varphi$, saturation pressure, speciation, fugacities | **EVo** (cell 2), C–O–H–S–N equilibrium |
| 3 | Magma compressibility | $\beta_m=\varphi\,\beta_{gas}+(1-\varphi)\,\beta_{liquid}$, $\beta_{gas}=1/P$ |
| 4 | Chamber compressibility $\beta_c$ | sphere $3/4\mu$; prolate spheroid (Amoruso & Crescentini 2009); penny crack (Fialko et al. 2001); or given |
| 5 | Partitioning factor | $r_V=1+\beta_m/\beta_c$ |
| 6 | Reservoir volume change | $\Delta V_c=V_e/r_V$ |
| 7 | Surface uplift | Mogi (1958) point sphere, McTigue (1987) finite sphere, Yang et al. (1988) spheroid, Fialko et al. (2001) crack |

**Quick start**
1. *Runtime ▸ Run all* — the worked example (La Fossa, Vulcano: the thesis case) runs in about 30 s.
2. For another volcano, choose it in **cell 4** (ready-made: Etna, Campi Flegrei, Kīlauea) or choose *Empty tables* and fill in **cell 5** (one column per magma) and **cell 6** (one row per reservoir): type into the boxes and add magmas or reservoirs with the buttons. The guide below cell 4 explains every row and column.
3. Click cell 9 and choose *Runtime ▸ Run after*. A missing or wrong entry is reported in plain words, naming the table, the magma or reservoir, and the row or column.

| Cell | What you set |
|---|---|
| 4 | start values of the tables (ready-made volcano, empty tables or a saved copy), project name, crust density, default μ and ν, observation distances |
| **5** | **magmas** (fill-in table): composition, EVo solubility class, T, volatiles (wt% or element ppm), vapour fraction, redox (FMQ / NNO / IW / absolute / Fe³⁺–Fe²⁺), fH₂ / fH₂O / fCO₂, β<sub>liquid</sub>, ρ<sub>melt</sub> |
| **6** | **reservoirs** (fill-in table): shape (Mogi, finite sphere, spheroid, penny crack), depth, volume, injected volumes, magmas, host rock, optional pressure / temperature / β<sub>c</sub> |
| 7 | EVo options shared by all magmas (run type, models, solver) |
| 8 | optional: step-2 values entered by hand for chosen reservoirs |
| 14, 16, 17 | detectability target, map of one reservoir, EVo calculator |

In [ ]:
#@title 1. Start-up — Python packages, display tools and fill-in tables  { display-mode: "form" }
#@markdown Checks the scientific Python packages that every Colab runtime provides (numpy, scipy, pandas, pydantic,
#@markdown matplotlib, ipywidgets) and defines the tools that draw the result tables, the messages and the fill-in
#@markdown input tables of cells 5, 6 and 8. EVo and the deformation sources follow in cells 2 and 3.
import sys, re, copy, html, importlib, importlib.util, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

_REQUIRED = {"numpy": "1.26", "scipy": "1.11", "pandas": "2.1", "pydantic": "2.0", "matplotlib": "3.7"}
_missing = [m for m in _REQUIRED if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError("Missing standard package(s): " + ", ".join(_missing) +
                      ". They are part of every Colab runtime; locally run: pip install " + " ".join(_missing))

# ---------------------------------------------------------------- display toolkit
# Colab renders each output in its own frame, so the style sheet travels with every block.
_CSS = """<style>
.lfw{font-family:Arial,Helvetica,sans-serif;color:#111;max-width:1180px}
.lfh{font-size:18px;font-weight:700;color:#fff;background:#1F4E79;padding:7px 12px;margin:16px 0 8px;border-radius:3px}
.lft{font-size:14px;font-weight:700;color:#1F4E79;margin:14px 0 5px}
.lfc{font-size:12px;color:#444;margin:5px 0 14px;line-height:1.45}
.lfs{overflow-x:auto;max-width:100%}
table.lf{border-collapse:collapse;font-size:12.5px;background:#fff;color:#111;margin:2px 0}
table.lf th{background:#2E5D8A;color:#fff;padding:5px 8px;border:1px solid #9fb3c8;text-align:center;font-weight:600}
table.lf td{padding:4px 8px;border:1px solid #cdd6e0;text-align:center;vertical-align:middle;background:#fff}
table.lf tr:nth-child(even) td{background:#f4f7fb}
table.lf td.l{text-align:left}
table.lf td.grp{background:#dfe8f2 !important;font-weight:700;text-align:left;color:#1F4E79}
table.lf td.bad{background:#fde3e1 !important;color:#8a1c1c;font-weight:700}
table.lf td.good{background:#e4f5e7 !important;color:#15521f}
table.lf td.hi{background:#fff4d6 !important;font-weight:700}
table.lf td.dim{color:#888}
.lfb{border-left:5px solid #2E75B6;background:#eef4fb;padding:8px 12px;margin:8px 0;font-size:13px;line-height:1.55;color:#111}
.lfb.warn{border-color:#c0392b;background:#fdf0ef}
.lfb.ok{border-color:#2e8b57;background:#edf8f1}
</style>"""

def _show(inner):
    display(HTML(_CSS + '<div class="lfw">' + inner + '</div>'))

def section(title):
    """Coloured section banner."""
    _show(f'<div class="lfh">{title}</div>')

def note(text, kind="info"):
    """Highlighted paragraph: kind = 'info', 'ok' or 'warn'."""
    cls = {"info": "lfb", "ok": "lfb ok", "warn": "lfb warn"}[kind]
    _show(f'<div class="{cls}">{text}</div>')

def table_html(header, rows, title=None, caption=None, classes=None, left_cols=(0,), group_rows=()):
    """HTML of a table. rows = list of lists of (HTML) strings;
    classes(i, j) may return 'bad', 'good', 'hi' or 'dim' to colour a cell;
    rows whose index is in group_rows are drawn as a full-width group heading."""
    head = "".join(f"<th>{c}</th>" for c in header)
    body = []
    for i, r in enumerate(rows):
        if i in group_rows:
            body.append(f'<tr><td class="grp" colspan="{len(header)}">{r[0]}</td></tr>')
            continue
        tds = []
        for j, v in enumerate(r):
            cl = ["l"] if j in left_cols else []
            extra = classes(i, j) if classes else None
            if extra:
                cl.append(extra)
            tds.append(f'<td class="{" ".join(cl)}">{v}</td>')
        body.append("<tr>" + "".join(tds) + "</tr>")
    out = f'<div class="lft">{title}</div>' if title else ""
    out += f'<div class="lfs"><table class="lf"><tr>{head}</tr>{"".join(body)}</table></div>'
    if caption:
        out += f'<div class="lfc">{caption}</div>'
    return out

def table(*args, **kw):
    """Render a table on the page (same arguments as table_html)."""
    _show(table_html(*args, **kw))

# ---------------------------------------------------------------- number formatting
def sci(v, nd=2):
    """1.23×10⁻⁴ style (HTML)."""
    if v is None or not np.isfinite(v):
        return "—"
    if v == 0:
        return "0"
    e = int(np.floor(np.log10(abs(v))))
    m = v / 10 ** e
    if abs(round(m, nd)) >= 10:
        m, e = m / 10, e + 1
    return f"{m:.{nd}f}×10<sup>{e}</sup>".replace("-", "−")

def num(v, nd=2):
    return "—" if v is None or not np.isfinite(v) else f"{v:.{nd}f}"

def gnum(v, sig=5):
    """General-purpose number for the EVo output tables."""
    if v is None:
        return "—"
    try:
        v = float(v)
    except (TypeError, ValueError):
        return html.escape(str(v))
    if not np.isfinite(v):
        return "—"
    if v == 0:
        return "0"
    a = abs(v)
    return f"{v:.{sig}g}" if 1e-3 <= a < 1e5 else sci(v, sig - 1)

def mm(v):
    """Displacement in mm with sensible precision."""
    a = abs(v)
    if a >= 100:  return f"{v:.0f} mm"
    if a >= 1:    return f"{v:.2f} mm"
    if a >= 0.01: return f"{v:.3f} mm"
    return f"{v * 1000:.1f} µm"

plt.rcParams.update({"font.family": "serif", "font.size": 9.5, "axes.linewidth": 0.8,
                     "xtick.direction": "in", "ytick.direction": "in", "figure.dpi": 110})

# ---------------------------------------------------------------- fill-in tables (cells 5, 6 and 8)
try:
    import ipywidgets as W
except ImportError:                      # plain Python without ipywidgets: tables are shown read-only
    W = None
if W is not None and int(W.__version__.split(".")[0]) >= 8:
    try:                                 # Colab draws ipywidgets 8 with its custom widget manager
        from google.colab import output as _colab_output
        _colab_output.enable_custom_widget_manager()
    except Exception:
        pass

EMPTY_TABLES = "Empty tables (your own volcano)"
SAVED_COPY = "Saved copy (pasted into cells 5, 6 and 8)"


def split_table_text(text):
    """Split a text table into (header, rows) of strings. Cells are separated by |, by tabs (a block
    copied from Excel or Google Sheets) or by commas: the first of these found in the header line.
    Text after # is a comment; empty lines and markdown rules (---) are skipped."""
    lines = []
    for raw in str(text).splitlines():
        line = raw.split("#", 1)[0].rstrip()
        if line.strip():
            lines.append(line)
    if not lines:
        return [], []
    head = lines[0]
    sep = "|" if "|" in head else ("\t" if "\t" in head else ",")
    lead = head.strip().startswith("|")

    def split(line):
        parts = [p.strip() for p in line.split(sep)]
        if lead and parts and parts[0] == "":
            parts = parts[1:]
        return parts

    header = split(head)
    while header and header[-1] == "":
        header.pop()
    rows = []
    for line in lines[1:]:
        cells = split(line)
        if all(re.fullmatch(r":?-{2,}:?", c) or c == "" for c in cells):
            continue
        while len(cells) > len(header) and cells[-1] == "":
            cells.pop()
        if len(cells) > len(header):
            raise ValueError(f"the line\n  {line.strip()[:110]}\nhas {len(cells)} cells but the header has "
                             f"{len(header)}. Check the separators '{sep}'.")
        rows.append(cells + [""] * (len(header) - len(cells)))
    return header, rows


def _numlike(s, many=False):
    """True if s is empty or reads as a number (or, with many=True, as numbers separated by ;)."""
    parts = re.split(r"[;\s]+", str(s).strip()) if many else [str(s)]
    for p in parts:
        p = (p.strip().replace("−", "-").replace("×10^", "e").replace("x10^", "e")
             .replace("·10^", "e").replace("*10^", "e").replace(" ", ""))
        if p in ("", "-", "—", "–"):
            continue
        try:
            if not np.isfinite(float(p)):
                return False
        except ValueError:
            return False
    return True


def field(key, label, unit="", kind="num", group="", hint="", help="", width=None, choices=None,
          ref=None, optional=False):
    """One row (magma table) or column (reservoir table) of a fill-in table.
    kind: num | list (numbers separated by ;) | text | long (several lines) | choice | ref (a name
    from another table)."""
    return dict(key=key, label=label, unit=unit, kind=kind, group=group, hint=hint, help=help,
                width=width, choices=list(choices or []), ref=ref, optional=optional)


_GRID_CSS = """<style>
.lfg{font-family:Arial,Helvetica,sans-serif;font-size:12.5px;color:#111;background:#fff;padding:6px 8px;
     border:1px solid #cdd6e0;border-radius:3px}
.lfg input,.lfg select,.lfg textarea{font-size:12.5px !important;color:#111 !important;background:#fff !important}
.lfg input{height:26px}
.lfg .lfg-bad input{background:#fdf0ef !important;border-color:#c0392b !important;color:#8a1c1c !important}
.lfg .lfg-name input{font-weight:700;color:#1F4E79 !important;background:#eef4fb !important}
.lfg .lfg-mono textarea{font-family:Menlo,Consolas,"DejaVu Sans Mono",monospace !important;font-size:12px !important;
                        white-space:pre;overflow-wrap:normal;overflow-x:auto}
.lfg .lfg-bar button{margin-right:5px}
.lfg-lab{padding:5px 6px 0 2px;line-height:1.2;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#111}
.lfg-lab code,.lfg-hd code{font-size:10.5px;color:#7a8794;background:none;padding:0}
.lfg-grp{background:#dfe8f2;color:#1F4E79;font-weight:700;padding:3px 8px;margin-top:5px}
.lfg-hd{background:#2E5D8A;color:#fff;font-weight:600;text-align:center;padding:3px 2px;line-height:1.2;
        height:100%;box-sizing:border-box;font-size:12px}
.lfg-hd span{font-weight:400;color:#d9e6f2}
.lfg-hd code{color:#c5d6e8}
.lfg-band{background:#dfe8f2;color:#1F4E79;font-weight:700;text-align:center;padding:2px 0;font-size:12px;
          white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.lfg-num{color:#5b6b7b;text-align:right;padding:5px 4px 0 0}
.lfg-st{color:#1F4E79;padding:5px 0 3px}
.lfg-help{font-size:12px;color:#333;line-height:1.45;padding:2px 0 4px}
</style>"""

_TABLES = globals().setdefault("_TABLES", {})      # kept when this cell is run again


class TableEditor:
    """A fill-in table on the page. orient='cols': one column per item (magmas; the properties are
    rows); orient='rows': one row per item (reservoirs). The values are kept as text in self.items and
    handed to the calculation by as_table()."""

    def __init__(self, key):
        self.key = key
        self.items, self.extra, self._xid = [], [], 0
        self.synced_choice, self.loaded_text = None, None
        self.edited, self.show_all, self.message = False, False, ""
        self._undo, self._root, self._refs, self._w = None, None, [], {}

    # ------------------------------------------------------------------ set-up
    @classmethod
    def open(cls, key, fields, orient, item, start, loader, label_width=260, item_width=130,
             min_items=1, extra_label=None, more_label="Show more rows"):
        t = _TABLES.get(key)
        if t is None:
            t = _TABLES[key] = cls(key)
        t.fields, t.orient, t.item = fields, orient, item
        t.fmap = {f["key"]: f for f in fields}
        t.lower = {f["key"].lower(): f["key"] for f in fields}
        t.label_width, t.item_width, t.min_items = label_width, item_width, min_items
        t.extra_label, t.more_label = extra_label, more_label
        t.name_key = "__name__" if orient == "cols" else fields[0]["key"]
        t.loader = loader
        t.start(start)
        return t

    @staticmethod
    def start_all(choice):
        """Refill every open table if the choice of cell 4 has changed (otherwise your entries are kept)."""
        for t in list(_TABLES.values()):
            if getattr(t, "loader", None) is not None:
                t.start(choice)

    def start(self, choice, force=False):
        text = self.loader(choice) or ""
        if not force and choice == self.synced_choice and text == self.loaded_text:
            return False
        if self.synced_choice is not None:
            self._push_undo()
        self.synced_choice, self.loaded_text, self.message = choice, text, ""
        self._fill(text)
        if not text.strip() and choice == SAVED_COPY:
            self.message = "This cell holds no saved copy yet, so the table starts empty."
        self.edited = False
        self.refresh()
        self._notify_names()
        return True

    def _fill(self, text):
        try:
            self._load(*split_table_text(text)) if str(text).strip() else self._blank()
        except ValueError as e:
            self._blank()
            self.message = f"The start values could not be read: {e}"

    # ------------------------------------------------------------------ data
    def _new_extra(self, key):
        self._xid += 1
        x = {"id": self._xid, "key": key}
        self.extra.append(x)
        return ("x", x["id"])

    def _blank(self):
        self.extra = []
        self.items = [{"__name__": ""} if self.orient == "cols" else {} for _ in range(self.min_items)]

    def _load(self, header, rows):
        self.extra, self.items = [], []
        if self.orient == "cols":
            self.items = [{"__name__": n.strip()} for n in header[1:]]
            for r in rows:
                if not r or not r[0].strip():
                    continue
                k = self.lower.get(r[0].strip().lower())
                if k is None:
                    k = next((("x", x["id"]) for x in self.extra if x["key"] == r[0].strip()), None) \
                        or self._new_extra(r[0].strip())
                for it, v in zip(self.items, r[1:]):
                    if v.strip():
                        it[k] = v.strip()
        else:
            slots = []
            for h in header:
                h = h.strip()
                slots.append(self.lower.get(h.lower()) or (self._new_extra(h) if h else None))
            for r in rows:
                it = {s: v.strip() for s, v in zip(slots, r) if s is not None and v.strip()}
                if it:
                    self.items.append(it)
        while len(self.items) < self.min_items:
            self.items.append({"__name__": ""} if self.orient == "cols" else {})

    @staticmethod
    def _filled(it):
        return any(str(v).strip() for v in it.values())

    def names(self):
        return [str(it.get(self.name_key, "")).strip() for it in self.items]

    def as_table(self):
        """(header, rows) in the layout of the text tables read by the calculation engine."""
        keys = [f["key"] for f in self.fields]
        xs = [x for x in self.extra if x["key"].strip()]
        its = [it for it in self.items if self._filled(it)]
        val = lambda it, k: str(it.get(k, "")).strip()
        if self.orient == "cols":
            header = ["property"] + [val(it, "__name__") for it in its]
            rows = [[k] + [val(it, k) for it in its] for k in keys]
            rows += [[x["key"].strip()] + [val(it, ("x", x["id"])) for it in its] for x in xs]
        else:
            header = keys + [x["key"].strip() for x in xs]
            rows = [[val(it, k) for k in keys] + [val(it, ("x", x["id"])) for x in xs] for it in its]
        return header, rows

    def to_text(self):
        """The table as aligned text (| separators): to copy, keep or paste back."""
        header, rows = self.as_table()
        if self.orient == "cols":
            rows = [r for r in rows if any(r[1:]) or not self.fmap.get(r[0], {}).get("optional", False)]
        else:
            keep = [j for j, h in enumerate(header)
                    if any(r[j] for r in rows) or (h in self.fmap and not self.fmap[h]["optional"])]
            header, rows = [header[j] for j in keep], [[r[j] for j in keep] for r in rows]
        tab = [[c.replace("|", "/") for c in r] for r in [header] + rows]
        w = [min(24, max(len(r[j]) for r in tab)) for j in range(len(header))]
        return "\n".join(" | ".join(c.ljust(w[j]) for j, c in enumerate(r)).rstrip() for r in tab)

    # ------------------------------------------------------------------ editing
    def _push_undo(self):
        self._undo = (copy.deepcopy(self.items), copy.deepcopy(self.extra), self.edited)

    def undo(self, *_):
        if self._undo is None:
            return
        now = (copy.deepcopy(self.items), copy.deepcopy(self.extra), self.edited)
        self.items, self.extra, self.edited = copy.deepcopy(self._undo[0]), copy.deepcopy(self._undo[1]), self._undo[2]
        self._undo, self.message = now, ""
        self.refresh()
        self._notify_names()

    def _index(self, it):
        return next(i for i, x in enumerate(self.items) if x is it)

    def _unique(self, name):
        used = {n.upper() for n in self.names()}
        base, i, new = name, 2, name
        while new.upper() in used:
            new = f"{base}_{i}" if self.orient == "cols" else f"{base} ({i})"
            i += 1
        return new

    def add_item(self, *_):
        self._push_undo()
        self.items.append({"__name__": ""} if self.orient == "cols" else {})
        self.edited = True
        self.refresh()

    def copy_item(self, it):
        self._push_undo()
        new = copy.deepcopy(it)
        if str(new.get(self.name_key, "")).strip():
            new[self.name_key] = self._unique(str(new[self.name_key]).strip())
        self.items.insert(self._index(it) + 1, new)
        self.edited = True
        self.refresh()
        self._notify_names()

    def remove_item(self, it):
        self._push_undo()
        del self.items[self._index(it)]
        self.edited = True
        self.refresh()
        self._notify_names()

    def add_extra(self, *_):
        self._push_undo()
        self._new_extra("")
        self.edited = True
        self.refresh()

    def remove_extra(self, x):
        self._push_undo()
        self.extra = [y for y in self.extra if y is not x]
        for it in self.items:
            it.pop(("x", x["id"]), None)
        self.edited = True
        self.refresh()

    def reset(self, *_):
        self._push_undo()
        self.message = ""
        self._fill(self.loaded_text or "")
        self.edited = False
        self.refresh()
        self._notify_names()

    def clear(self, *_):
        self._push_undo()
        self.message = ""
        self._blank()
        self.edited = True
        self.refresh()
        self._notify_names()

    def apply_text(self, text):
        header, rows = split_table_text(text)
        if not header or (self.orient == "cols" and len(header) < 2) or not rows:
            raise ValueError("no table found: the text needs a header line and at least one more line.")
        if self.orient == "rows" and not any(h.strip().lower() in self.lower for h in header):
            raise ValueError("none of the column names in the first line is known: "
                             + ", ".join(h for h in header[:6] if h) + " …")
        self._push_undo()
        self._load(header, rows)
        self.edited, self.message = True, ""
        self.refresh()
        self._notify_names()

    def _notify_names(self):
        names = tuple(n for n in self.names() if n)
        for t in _TABLES.values():
            for ref, e in t._refs:
                if ref == self.key:
                    e.options = names

    def _renamed(self, old, new):
        """A name was edited: update the suggestions of other tables and follow the renaming there."""
        o, n = old.strip().upper(), new.strip()
        follow = o and n and o not in {x.upper() for x in self.names()}
        for t in _TABLES.values():
            for ref, e in t._refs:
                if ref == self.key:
                    e.options = tuple(x for x in self.names() if x)
            if not follow:
                continue
            for it in t.items:
                for f in t.fields:
                    if f.get("ref") == self.key and str(it.get(f["key"], "")).strip().upper() == o:
                        it[f["key"]] = n
                        w = t._w.get((id(it), f["key"]))
                        if w is not None:
                            w.value = n

    # ------------------------------------------------------------------ drawing
    def _safe(self, fn):
        def run(*a):
            try:
                fn(*a)
            except Exception as e:                      # shown in the status line instead of a traceback
                self.message = str(e)
                self._show_status()
        return run

    def _lab(self, inner, width, cls="lfg-lab", title=""):
        t = f' title="{html.escape(title)}"' if title else ""
        return W.HTML(value=f'<div class="{cls}"{t}>{inner}</div>',
                      layout=W.Layout(width=f"{width}px", min_width=f"{width}px", flex="0 0 auto", margin="0 2px 0 0"))

    @staticmethod
    def _row(children, **kw):
        """One line of the table: as wide as its boxes, so that the whole table scrolls sideways together."""
        return W.HBox(children, layout=W.Layout(overflow="visible", width="max-content", min_width="max-content",
                                                flex="0 0 auto", **kw))

    def _small(self, text, tip, fn):
        b = W.Button(description=text, tooltip=tip,
                     layout=W.Layout(width="28px", height="26px", padding="0", margin="1px 2px 1px 0", flex="0 0 auto"))
        b.on_click(self._safe(fn))
        return b

    def _cell(self, f, it, width):
        k, kind = f["key"], f["kind"]
        v = str(it.get(k, ""))
        lay = W.Layout(width=f"{width}px", min_width=f"{width}px", flex="0 0 auto", margin="1px 2px 1px 0")
        if kind == "choice":
            opts = [("—", "")] + [tuple(c) if isinstance(c, (tuple, list)) else (c, c) for c in f["choices"]]
            vals = [o[1] for o in opts]
            if v not in vals:
                hit = [x for x in vals if x.lower() == v.strip().lower()]
                v = hit[0] if hit else v
                if v not in vals:
                    opts.append((v, v))
            e = W.Dropdown(options=opts, value=v, layout=lay)
        elif kind == "ref" and hasattr(W, "Combobox"):
            src = _TABLES.get(f["ref"])
            e = W.Combobox(value=v, options=tuple(n for n in (src.names() if src else []) if n),
                           ensure_option=False, placeholder=f["hint"], layout=lay)
            self._refs.append((f["ref"], e))
        elif kind == "long":
            lay.height = "64px"
            e = W.Textarea(value=v, placeholder=f["hint"], layout=lay)
        else:
            e = W.Text(value=v, placeholder=f["hint"], layout=lay)
        if kind in ("num", "list") and not _numlike(v, kind == "list"):
            e.add_class("lfg-bad")
        self._w[(id(it), k)] = e

        def changed(ch, it=it, k=k, e=e):
            it[k] = ch["new"]
            self.edited = True
            if kind in ("num", "list"):
                (e.remove_class if _numlike(ch["new"], kind == "list") else e.add_class)("lfg-bad")
            if k == self.name_key:
                self._renamed(str(ch["old"]), str(ch["new"]))
            self._show_status()
        e.observe(self._safe(changed), names="value")
        return e

    def _extra_key(self, x, width):
        e = W.Text(value=x["key"], placeholder="EVo key, e.g. H2_MODEL" if self.orient == "cols" else "column",
                   layout=W.Layout(width=f"{width}px", min_width=f"{width}px", flex="0 0 auto", margin="1px 2px 1px 0"))

        def changed(ch, x=x):
            x["key"] = ch["new"].strip()
            self.edited = True
            self._show_status()
        e.observe(self._safe(changed), names="value")
        return e

    def _visible(self, f):
        return (not f["optional"]) or self.show_all or any(str(it.get(f["key"], "")).strip() for it in self.items)

    def _grid_cols(self):
        lw, iw = self.label_width, self.item_width
        total = lw + len(self.items) * (iw + 2)
        names = [self._lab(f"<b>{self.item.capitalize()} name</b>", lw)]
        acts = [self._lab('<span style="color:#7a8794">⧉ copy · ✕ remove</span>', lw)]
        for it in self.items:
            e = self._cell(dict(key="__name__", kind="text", hint="NAME"), it, iw)
            e.add_class("lfg-name")
            names.append(e)
            acts.append(W.HBox([self._small("⧉", f"copy this {self.item}", lambda b, it=it: self.copy_item(it)),
                                self._small("✕", f"remove this {self.item}", lambda b, it=it: self.remove_item(it))],
                               layout=W.Layout(width=f"{iw}px", min_width=f"{iw}px", flex="0 0 auto",
                                               margin="0 2px 0 0", justify_content="center")))
        out, group = [self._row(names), self._row(acts)], None
        for f in self.fields:
            if not self._visible(f):
                continue
            if f["group"] != group:
                group = f["group"]
                out.append(self._lab(group, total, "lfg-grp"))
            unit = f' <span style="color:#555">({f["unit"]})</span>' if f["unit"] else ""
            lab = self._lab(f'{f["label"]}{unit} <code>{f["key"]}</code>', lw, title=f["help"] or f["label"])
            out.append(self._row([lab] + [self._cell(f, it, iw) for it in self.items]))
        if self.extra:
            out.append(self._lab("Extra rows — an EVo setting for one magma only (expert, see the guide)", total, "lfg-grp"))
            for x in self.extra:
                row = [W.HBox([self._extra_key(x, lw - 32), self._small("✕", "remove this row", lambda b, x=x: self.remove_extra(x))],
                              layout=W.Layout(width=f"{lw}px", min_width=f"{lw}px", flex="0 0 auto", margin="0 2px 0 0"))]
                row += [self._cell(dict(key=("x", x["id"]), kind="text", hint=""), it, iw) for it in self.items]
                out.append(self._row(row))
        return out

    def _grid_rows(self):
        fw = lambda f: f["width"] or self.item_width
        vis = [f for f in self.fields if self._visible(f)]
        lead = 26 + 2 * 30
        band, head = [self._lab("", lead)], [self._lab("", lead)]
        i = 0
        while i < len(vis):
            j = i
            while j < len(vis) and vis[j]["group"] == vis[i]["group"]:
                j += 1
            band.append(self._lab(vis[i]["group"], sum(fw(f) + 2 for f in vis[i:j]) - 2, "lfg-band", title=vis[i]["group"]))
            i = j
        for f in vis:
            unit = f'<br><span>{f["unit"]}</span>' if f["unit"] else ""
            head.append(self._lab(f'{f["label"]}{unit}<br><code>{f["key"]}</code>', fw(f), "lfg-hd",
                                  title=f["help"] or f["label"]))
        for x in self.extra:
            band.append(self._lab("not a known column", 112, "lfg-band"))
            head.append(W.HBox([self._extra_key(x, 80), self._small("✕", "remove this column", lambda b, x=x: self.remove_extra(x))],
                               layout=W.Layout(width="112px", min_width="112px", flex="0 0 auto", margin="0 2px 0 0")))
        out = [self._row(band), self._row(head, align_items="stretch")]
        for n, it in enumerate(self.items, start=1):
            row = [self._lab(str(n), 26, "lfg-num"),
                   self._small("⧉", f"copy this {self.item}", lambda b, it=it: self.copy_item(it)),
                   self._small("✕", f"remove this {self.item}", lambda b, it=it: self.remove_item(it))]
            row += [self._cell(f, it, fw(f)) for f in vis]
            row += [self._cell(dict(key=("x", x["id"]), kind="text", hint=""), it, 110) for x in self.extra]
            out.append(self._row(row))
        if not self.items:
            out.append(self._lab(f"<i>No {self.item}s. Press “+ Add {self.item}”.</i>", 420))
        return out

    def _show_status(self):
        if self._root is None:
            return
        n = sum(1 for it in self.items if self._filled(it))
        s = f"<b>{n} {self.item}{'' if n == 1 else 's'}</b>"
        if self.synced_choice:
            s += f" · start values: {html.escape(self.synced_choice)}"
        if self.edited:
            s += ' · <span style="color:#b35c00;font-weight:700">edited</span>'
        if self.message:
            s += f'<br><span style="color:#c0392b">{html.escape(self.message)}</span>'
        self._status.value = f'<div class="lfg-st">{s}</div>'

    @staticmethod
    def _close(w):
        for c in getattr(w, "children", ()):
            TableEditor._close(c)
        w.close()

    def refresh(self):
        if self._root is None:
            return
        old = list(self._grid.children)
        self._refs, self._w = [], {}
        self._grid.children = self._grid_cols() if self.orient == "cols" else self._grid_rows()
        for w in old:
            self._close(w)
        self._b_undo.disabled = self._undo is None
        if self._textbox.layout.display != "none":
            self._area.value = self.to_text()
        self._show_status()

    def _build(self):
        def btn(text, tip, fn, width="auto"):
            b = W.Button(description=text, tooltip=tip, layout=W.Layout(width=width, flex="0 0 auto", margin="0 5px 3px 0"))
            b.on_click(self._safe(fn))
            return b
        bar = [btn(f"+ Add {self.item}", f"add an empty {self.item}", self.add_item)]
        if self.extra_label:
            bar.append(btn(self.extra_label, "add a row for any EVo setting (expert)", self.add_extra))
        if any(f["optional"] for f in self.fields):
            more = W.ToggleButton(value=self.show_all, description=self.more_label,
                                  tooltip="show the optional rows/columns even when they are empty",
                                  layout=W.Layout(width="auto", flex="0 0 auto", margin="0 5px 3px 0"))

            def toggled(ch):
                self.show_all = ch["new"]
                self.refresh()
            more.observe(self._safe(toggled), names="value")
            bar.append(more)
        self._b_undo = btn("↶ Undo", "undo the last change of rows, columns or start values", self.undo)
        bar += [self._b_undo,
                btn("↺ Reset", "back to the start values (the choice in cell 4)", self.reset),
                btn("Empty table", "remove every value", self.clear)]
        tv = W.ToggleButton(value=False, description="Text view (copy / paste)",
                            tooltip="the whole table as text; paste a block from Excel or Google Sheets",
                            layout=W.Layout(width="auto", flex="0 0 auto", margin="0 5px 3px 0"))
        bar.append(tv)
        self._status = W.HTML()
        self._area = W.Textarea(layout=W.Layout(width="98%", height="230px"))
        self._area.add_class("lfg-mono")
        self._text_msg = W.HTML()

        def apply(*_):
            try:
                self.apply_text(self._area.value)
                self._text_msg.value = '<span style="color:#2e8b57">The table now holds the text above.</span>'
            except ValueError as e:
                self._text_msg.value = f'<span style="color:#c0392b">Not applied: {html.escape(str(e))}</span>'

        def show_text(*_):
            self._area.value = self.to_text()
            self._text_msg.value = ""
        self._textbox = W.VBox([
            W.HTML('<div class="lfg-help"><b>Copy</b> the text below to keep a copy of this table (select it, then Ctrl+C / ⌘C). '
                   "<b>Paste</b> a table here — a block copied from Excel or Google Sheets, or text with <code>|</code> "
                   "between the cells, laid out like the text shown — and press <i>Apply</i>. "
                   "To keep the table inside the notebook file, paste the text between the quotes of "
                   "<code>SAVED_…</code> in the code of this cell (double-click the cell to see it) and choose "
                   "<i>Saved copy</i> in cell 4.</div>"),
            self._area,
            W.HBox([btn("Apply the text to the table", "replace the table by the text above", apply),
                    btn("Show the current table again", "rewrite the text from the table", show_text)]),
            self._text_msg], layout=W.Layout(display="none", margin="2px 0 8px 0"))

        def text_toggled(ch):
            self._textbox.layout.display = "flex" if ch["new"] else "none"
            if ch["new"]:
                show_text()
        tv.observe(self._safe(text_toggled), names="value")
        self._grid = W.VBox(layout=W.Layout(overflow="auto", width="100%", padding="2px 0 8px 0"))
        bar_box = W.HBox(bar, layout=W.Layout(flex_flow="row wrap"))
        bar_box.add_class("lfg-bar")
        self._root = W.VBox([W.HTML(_GRID_CSS), bar_box, self._status, self._textbox, self._grid])
        self._root.add_class("lfg")

    def show(self):
        if W is None:
            note("The fill-in table needs the package <code>ipywidgets</code> (part of every Colab runtime; "
                 "elsewhere: <code>pip install ipywidgets</code>). The values below are used as they are.", "warn")
            header, rows = self.as_table()
            table([html.escape(h) for h in header], [[html.escape(c) or "—" for c in r] for r in rows])
            return
        if self._root is None:
            self._build()
        self.refresh()
        display(self._root)

section("Start-up complete")
_vers = [[m, importlib.import_module(m).__version__, f"≥ {v}"] for m, v in _REQUIRED.items()]
_vers.append(["ipywidgets", W.__version__ if W is not None else "not available", "for the fill-in tables of cells 5, 6, 8"])
table(["Package", "Version in this runtime", "Needed"],
      [["Python", platform.python_version(), "≥ 3.11"]] + _vers,
      caption="EVo and the deformation sources of dMODELS run from their source code in cells 2 and 3.")

In [ ]:
#@title 2. EVo — thermodynamic degassing model (complete source code, version 1.1.0)  { display-mode: "form" }
#@markdown This cell contains the **complete source code of the thermodynamic degassing code EVo**
#@markdown (Liggins, Shorttle & Rimmer 2020; Liggins et al. 2022; version 1.1.0, commit `2487939`, GPL-3.0):
#@markdown its 17 computational modules (everything except the command-line front end `__main__.py` and the
#@markdown file-based parameter sweep `multirun.py`) and its 11 thermochemical data tables, stored verbatim as
#@markdown text and loaded as the Python package `evo_local`, which computes step 2 of the chain.
#@markdown Double-click the cell to read the code. The adaptations (renamed imports, two changes that return
#@markdown EVo's results directly instead of writing files, one bug fix in an EVo warning message) are listed in the table printed below the
#@markdown cell; the integrity of every embedded module is checked against the SHA-256 of the original file.
import sys, io, re, types, json, hashlib, linecache, warnings, contextlib
import importlib, importlib.abc, importlib.util
import numpy as np
import pandas as pd

EVO_ORIGIN = dict(name="EVo", version="1.1.0", commit="2487939e18d98292f9b8f3a1f0dea04b707bd89f",
                  authors="P. Liggins, O. Shorttle, P. B. Rimmer", licence="GPL-3.0-or-later",
                  repository="github.com/pipliggins/EVo")


# =============================================================================
# A) In-memory package loader (generic: also used for dMODELS in cell 3)
# =============================================================================
class _MemoryPackage(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    """Serves the modules of one package from a {module_name: source_text} dictionary."""

    def __init__(self, name, sources, inject_all=None, inject=None):
        self.name, self.sources = name, sources
        self.inject_all, self.inject = inject_all or {}, inject or {}

    def find_spec(self, fullname, path=None, target=None):
        if fullname == self.name:
            return importlib.util.spec_from_loader(fullname, self, is_package=True)
        head, _, tail = fullname.partition(".")
        if head == self.name and tail in self.sources:
            return importlib.util.spec_from_loader(fullname, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        tail = module.__name__.partition(".")[2] or "__init__"
        src = self.sources[tail]
        fname = f"<{self.name}/{tail}.py>"
        linecache.cache[fname] = (len(src), None, src.splitlines(True), fname)  # readable tracebacks
        module.__dict__.update(self.inject_all)
        module.__dict__.update(self.inject.get(tail, {}))
        exec(compile(src, fname, "exec"), module.__dict__)


def restore_blank_lines(sources, blanks):
    """Put back the blanks of whitespace-only lines (stored separately because the notebook
    front end strips them before a cell runs), so each module is byte-identical to the original."""
    out = {}
    for name, src in sources.items():
        if name in blanks:
            lines = src.split("\n")
            for i, ws in blanks[name]:
                lines[i] = ws
            src = "\n".join(lines)
        out[name] = src
    return out


def install_memory_package(name, sources, inject_all=None, inject=None):
    """Register (or re-register) a package whose modules live in `sources`, and import it."""
    for k in [k for k in sys.modules if k == name or k.startswith(name + ".")]:
        del sys.modules[k]
    sys.meta_path[:] = [f for f in sys.meta_path
                        if not (isinstance(f, _MemoryPackage) and f.name == name)]
    sys.meta_path.insert(0, _MemoryPackage(name, sources, inject_all, inject))
    pkg = importlib.import_module(name)
    for mod in sources:                     # import every module now, so that errors show up here
        if mod != "__init__":
            importlib.import_module(f"{name}.{mod}")
    return pkg


# =============================================================================
# B) What changes when EVo runs inside a notebook instead of from files
# =============================================================================
class EVoStop(SystemExit):
    """Raised wherever EVo would call exit(): the run stopped, the message says why."""


def _evo_exit(msg=None):
    raise EVoStop(msg)


def _evo_input(prompt=""):
    raise EVoStop(f"EVo requested interactive input: {prompt}")


EVO_PROMPTS = []            # yes/no questions EVo asked (answered 'yes', as the command-line default would)


def _evo_yes(question, default="yes"):
    EVO_PROMPTS.append(str(question))
    return True


_IMPORT_RE = re.compile(r"^(\s*)(from|import)\s+evo(?=[\s.])", re.M)

# (module, original text, replacement, reason) - each must match exactly once
EVO_PATCHES = [
    ("dgs_classes",
     'data_file = importlib.resources.files("evo") / "data" / filename\n\n'
     '        with data_file.open("r", encoding="utf-8") as path:',
     'data_file = _EVO_DATA[filename]  # embedded table (was evo/data/<file>)\n\n'
     '        with io.StringIO(data_file) as path:',
     "The Gibbs-energy tables of the gas species are read from memory instead of the package data folder."),
    ("writeout",
     "    if write:\n        filepath = sys.run.results_folder",
     "    if False:  # embedded: results stay in memory (was: if write:)\n        filepath = sys.run.results_folder",
     "EVo never writes its CSV output (also not for a crashed run); results are returned in memory."),
    ("messages",
     "f\"Warning: The set temperature of {T:3Ng} K is outside the T-dependent \"",
     "f\"Warning: The set temperature of {T:.4g} K is outside the T-dependent \"",
     "Bug fix: an invalid format code in this EVo 1.1.0 warning crashed every run whose temperature lies outside "
     "the calibration range of the solubility law; EVo's intended behaviour (warn, continue with fixed solubility "
     "coefficients) now applies and the warning is listed on the page. Not reached with the thesis inputs."),
]


# =============================================================================
# C) The embedded EVo source code (verbatim) and its data tables
# =============================================================================
_EVO_SOURCES = {
"__init__": r'''"""
EVo
A Python model for volcanic degassing, using the equilibrium constants and mass balance
method.
"""

from evo.dgs import run_evo as run_evo

__all__ = ["run_evo"]
''',
"constants": r'''# constants.py

# Universal gas constant
R = 8.3144598

m = {}
"""Atomic and molecular masses, g/mol"""
m["h"] = 1.00794
m["o"] = 15.9994
m["c"] = 12.0000
m["s"] = 32.066
m["n"] = 14.0067
m["si"] = 28.0855
m["ti"] = 47.867
m["al"] = 26.981539
m["mn"] = 54.938044
m["mg"] = 24.305
m["ca"] = 40.078
m["na"] = 22.990
m["k"] = 39.0983
m["p"] = 30.97376
m["li"] = 6.941
m["h2"] = 2.0164
m["o2"] = 31.9988
m["h2o"] = 18.01528
m["co2"] = 44.0095
m["co"] = 28.0101
m["ch4"] = 16.0425
m["so2"] = 64.0638
m["h2s"] = 34.08088
m["s2"] = 64.13
m["ocs"] = 60.0751
m["n2"] = 28.0134
m["fe"] = 55.845
m["sio2"] = 60.0843
m["tio2"] = 79.8658
m["al2o3"] = 101.961278
m["fe2o3"] = 159.6882
m["feo"] = 71.8444
m["feot"] = 71.8444
m["mno"] = 70.937445
m["mgo"] = 40.3044
m["cao"] = 56.0774
m["na2o"] = 61.978938
m["k2o"] = 94.196
m["p2o5"] = 141.944524
m["li2o"] = 29.8814

# PL: change so reads as a_basalt, a_rhyolite, a_phonolite etc.
# These are stored in the Molecule instance for each species,
# so can specify at that stage which is relevant.
basalt = {
    "a": {  # First solubility constant
        "H2O": 6.576e-4,
        "H2": 3.400e-7,
        "S2": 0,  # this is sulfide capacity, controlled in the molecule class
        "CO2": 1.729e-6,
        "O2": 0,  # considered insoluble
        "CO": 0,
        "CH4": 0,
        "SO2": 0,
        "H2S": 0,
        "OCS": 0,
        "N2": 0.0611e-6,
    },  # physical dissolution
    "b": {  # Second solubility constant, as above.
        "H2O": 0.5698,
        "H2": 1.2800,
        "S2": 0.5,
        "CO2": 0.8540,
        "O2": 1,  # Considered insoluble
        "CO": 1,
        "CH4": 1,
        "SO2": 1,
        "H2S": 1,
        "OCS": 1,
        "N2": 5.97e-16,
    },  # chemical dissolution
}


def p_h2o_a(T=1473.15):
    return -3.166e-9 * T**2 + 7.48e-6 * T - 3.853e-3


def p_h2o_b(T=1473.15):
    return 2.555e-6 * T**2 - 5.827e-3 * T + 3.918


# T-dependent solubilities can also be replaced with fixed constants outside of the
# tested temperature limits; this is controlled in the Molecule class.
phonolite = {
    "a": {  # First solubility constant
        "H2O": p_h2o_a,
        "H2": 3.400e-7,
        "SO2": 2.019e-4,
        "H2S": 4.172e-5,
        "CO2": 4.339e-7,
        "O2": 0,  # considered insoluble
        "CO": 0,
        "CH4": 0,
        "S2": 0,
        "OCS": 0,
        "N2": 0.0611e-6,
    },
    "b": {  # Second solubility constant, as above.
        "H2O": p_h2o_b,
        "H2": 1.2800,
        "SO2": 0.4366,
        "H2S": 0.5015,
        "CO2": 0.8006,
        "O2": 1,  # Considered insoluble
        "CO": 1,
        "CH4": 1,
        "S2": 1,
        "OCS": 1,
        "N2": 5.97e-16,
    },
}


def r_h2o_a(T=1473.15):
    return 2.5973e-8 * T**2 - 4.8473e-5 * T + 2.298e-2


def r_h2o_b(T=1473.15):
    return -5.1482e-6 * T**2 + 9.4853e-3 * T - 3.7085


def r_co2_a(T=1473.15):
    return 2.8895e-9 * T - 1.9625e-6


def r_co2_b(T=1473.15):
    return -1.0764e-3 * T + 1.9639


rhyolite = {
    "a": {  # First solubility constant
        "H2O": r_h2o_a,
        "H2": 3.400e-7,
        "SO2": 5.6322e-8,
        "H2S": 2.3164e-6,
        "CO2": r_co2_a,  #
        "O2": 0,  # considered insoluble
        "CO": 0,
        "CH4": 0,
        "S2": 0,
        "OCS": 0,
        "N2": 0.0611e-6,
    },
    "b": {  # Second solubility constant, as above.
        "H2O": r_h2o_b,
        "H2": 1.2800,
        "SO2": 1.2937,
        "H2S": 0.7338,
        "CO2": r_co2_b,
        "O2": 1,  # Considered insoluble
        "CO": 1,
        "CH4": 1,
        "S2": 1,
        "OCS": 1,
        "N2": 5.97e-16,
    },
}

PTcrit = {}
PTcrit["O2"] = [154.75, 50.7638]
PTcrit["H2O"] = [647.25, 221.1925]
PTcrit["H2"] = [33.25, 12.9696]
PTcrit["CO2"] = [304.15, 73.8659]
PTcrit["CH4"] = [191.05, 46.4069]
PTcrit["CO"] = [133.15, 34.9571]
PTcrit["S2"] = [208.15, 72.954]
PTcrit["SO2"] = [430.95, 78.7295]
PTcrit["OCS"] = [377.55, 65.8612]
PTcrit["H2S"] = [373.55, 90.0779]
PTcrit["N2"] = [126.2, 33.9]  # Roskosz 2006
''',
"ferric": r'''# ferric.py
#
# methods for calculating ferric and ferrous iron from melt composition and fO2
# do one or the other, assumes only one has been specified
# plus methods for calculating the iron fraction during decompression steps,
# for use in fe_equilibrate

import numpy as np

# ------------------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------------------


def kc91_fo2(C, T, P, lnfo2):
    """
    Calculates the Fe2O3/FeO mole ratio of a melt where the fO2 is known.

    Parameters
    ----------
    C : dictionary
        Major element composition of the silicate melt as MOLE FRACTIONS
        Required species: Al2O3, FeOt, CaO,  Na2O, K2O
    T : float
        Temperature in degrees K
    P : float
        Pressure in pascals (Pa)
    lnfo2 : float
        ln(fO2)

    Returns
    -------
    F : float
        Fe2O3/FeO mole ratio

    References
    ----------
    Kress and Carmichael (1991) The compressibility of silicate liquids
    containing Fe2O3 and the effect of composition, temperature, oxygen
    fugacity and pressure on their redox states.
    Contributions to Mineralogy and Petrology.
    """

    a = 0.196
    b = 1.1492e4  # K
    c = -6.675

    dal2o3 = -2.243
    dfeo = -1.828
    dcao = 3.201
    dna2o = 5.854
    dk2o = 6.215

    e = -3.36
    f = -7.01e-7  # K/Pa
    g = -1.54e-10  # /Pa
    h = 3.85e-17  # K/Pa^2

    T0 = 1673.0  # K

    F = np.exp(
        a * lnfo2
        + b / T
        + c
        + dal2o3 * C["al2o3"]
        + dfeo * C["feo"]
        + dcao * C["cao"]
        + dna2o * C["na2o"]
        + dk2o * C["k2o"]
        + e * (1.0 - T0 / T - np.log(T / T0))
        + f * P / T
        + g * (T - T0) * P / T
        + h * P**2 / T
    )

    # returns mole fraction Fe2O3/FeO
    return F


def kc91_fe3(Cm, T, P):
    """
    Calculates the oxygen fugacity (fO2) of a melt given where the ferric/
    ferrous ratio is known.

    Parameters
    ----------
    C : dictionary
        Major element composition of the silicate melt as mole fractions
        Required species: Al2O3, FeO, Fe2O3, CaO,  Na2O, K2O
    T : float
        Temperature in degrees K
    P : float
        Pressure in pascals (Pa)

    Returns
    -------
    FO2 : float
        Absolute fO2

    References
    ----------
    Kress and Carmichael (1991) The compressibility of silicate liquids
    containing Fe2O3 and the effect of composition, temperature, oxygen
    fugacity and pressure on their redox states.
    Contributions to Mineralogy and Petrology.
    """
    a = 0.196
    b = 1.1492e4  # K
    c = -6.675

    dal2o3 = -2.243
    dfeo = -1.828
    dcao = 3.201
    dna2o = 5.854
    dk2o = 6.215

    e = -3.36
    f = -7.01e-7  # K/Pa
    g = -1.54e-10  # /Pa
    h = 3.85e-17  # K/Pa^2

    T0 = 1673.0  # K

    FO2 = np.exp(
        (
            np.log(Cm["fe2o3"] / Cm["feo"])
            - b / T
            - c
            - dal2o3 * Cm["al2o3"]
            - dfeo * (Cm["feo"] + Cm["fe2o3"] * 0.8998)
            - dcao * Cm["cao"]
            - dna2o * Cm["na2o"]
            - dk2o * Cm["k2o"]
            - e * (1.0 - T0 / T - np.log(T / T0))
            - f * (P / T)
            - g * (T - T0) * P / T
            - h * P**2 / T
        )
        / a
    )

    return FO2


def r2013_fo2(cm, T, P, lnfo2):
    """
    Calculates the Fe2O3/FeO mole ratio of an FeOt>15 wt% melt where the
    fO2 is known.

    Parameters
    ----------
    C : dictionary
        Major element composition of the silicate melt as mole fractions
        Required species: Al2O3, FeOt, CaO, Na2O, K2O, P2O5
    T : float
        Temperature in degrees K
    P : float
        Pressure in pascals (Pa)
    lnfo2 : float
        ln(fO2)

    Returns
    -------
    F : float
        Fe2O3/FeO mole ratio

    References
    ----------
    Righter et al. (2013) Redox systematics of martian magmas with
    implications for magnetite stability. American Mineralogist.
    """

    a = 0.22
    b = 3800
    c = -370
    d = -6.6  # feo
    e = 7.3  # al2o3
    f = 17.3  # cao
    g = 132.3  # na2o
    h = -147.8  # k2o
    i = 0.6  # p2o5
    j = -4.26

    Pgpa = P * 1e-9

    F = np.exp(
        a * lnfo2
        + b / T
        + c * (Pgpa / T)
        + d * cm["feo"]
        + e * cm["al2o3"]
        + f * cm["cao"]
        + g * cm["na2o"]
        + h * cm["k2o"]
        + i * cm["p2o5"]
        + j
    )

    return F


def r2013_fe3(Cm, T, P):
    """
    Calculates the oxygen fugacity (fO2) of an FeOt>15 wt% melt given
    where the ferric/ferrous ratio is known.

    Parameters
    ----------
    C : dictionary
        Major element composition of the silicate melt as mole fractions
        Required species: Al2O3, FeOt, CaO, Na2O, K2O, P2O5
    T : float
        Temperature in degrees K
    P : float
        Pressure in pascals (Pa)

    Returns
    -------
    FO2 : float
        Absolute fO2

    References
    ----------
    Righter et al. (2013) Redox systematics of martian magmas with
    implications for magnetite stability.
    """

    a = 0.22
    b = 3800
    c = -370
    d = -6.6  # feo
    e = 7.3  # al2o3
    f = 17.3  # cao
    g = 132.3  # na2o
    h = -147.8  # k2o
    i = 0.6  # p2o5
    j = -4.26

    Pgpa = P * 1e-9  # GPa

    FO2 = np.exp(
        (
            np.log(Cm["fe2o3"] / Cm["feo"])
            - b / T
            - c * (Pgpa / T)
            - d * (Cm["feo"] + Cm["fe2o3"] * 0.8998)
            - e * Cm["al2o3"]
            - f * Cm["cao"]
            - g * Cm["na2o"]
            - h * Cm["k2o"]
            - i * Cm["p2o5"]
            - j
        )
        / a
    )

    return FO2
''',
"solubility_laws": r'''"""
Module to store the solubility laws of each species present in the melt,
using the method specified in the run.yaml file.
"""

import warnings

import numpy as np
from scipy.optimize import fsolve
from scipy.special import erf

import evo.constants as cnst
import evo.conversions as cnvs

# ------------------------------------------------------------------------------
# MODEL DEFINITIONS (Melt from gas & fugacity from melt)
# ------------------------------------------------------------------------------


def ardia2013(fCH4, P):
    """
    Returns the weight fraction of CH4 in the melt.

    Parameters
    ----------
    fCH4 : float
        The fugacity of CH4 (bar)
    P : float
        Current pressure (bar)

    Returns
    -------
    float
        The weight fraction of CH4 in the melt.

    References
    ----------
    Ardia et al.(2013) Solubility of CH4 in a synthetic basaltic melt,
    with applications to atmosphere-magma ocean-core partitioning of
    volatiles and to the evolution of the Martian atmosphere. GCA.
    """
    P = P * 1e-4  # Convert to GPa
    fCH4 = fCH4 * 1e-4

    k = np.exp(4.93 - (0.000193 * P))

    return (k * fCH4) / 1e6


def ardia2013_fugacity(melt_CH4, P):
    """
    Returns the fugacity of CH4.

    Parameters
    ----------
    melt_CH4 : float
        The weight fraction of CH4 in the melt
    P : float
        Pressure (bar)

    Returns
    -------
    fCH4 : float
        Fugacity of CH4 (bar)

    References
    ----------
    Ardia et al.(2013) Solubility of CH4 in a synthetic basaltic melt,
    with applications to atmosphere-magma ocean-core partitioning of
    volatiles and to the evolution of the Martian atmosphere. GCA.
    """
    P = P * 1e-4  # Convert to GPa

    k = np.exp(4.93 - (0.000193 * P))

    fugacity = (melt_CH4 * 1e6) / k

    return fugacity * 1e4  # convert fugacity from GPa to bar


def armstrong2015(fCO, P):
    """
    Returns the weight fraction of CO in the melt.

    Parameters
    ----------
    fCO : float
        The fugacity of CO in the gas phase (bar)
    P : float
        Pressure (bars)

    Returns
    -------
    CO : float
        The weight fraction of CO in the melt.

    References
    ----------
    Armstrong et al. (2015) Speciation and solubility of reduced C-O-H-N
    volatiles in mafic melt: Implications for volcanism, atmospheric
    evolution, and deep volatile cycles in the terrestrial planets. GCA.
    """

    C = 10 ** (-0.738 + 0.876 * np.log10(fCO) - 5.44e-5 * P)
    melt_CO = C / 1e6  # ppm to weight fraction

    return melt_CO


def armstrong2015_fugacity(melt_CO, P):
    """
    Returns the fugacity of CO.

    Parameters
    ----------
    melt_CO : float
        Weight fraction of CO in the melt
    P : float
        Pressure (bars)

    Returns
    -------
    fCO : float
        Fugacity of CO (bar)

    References
    ----------
    Armstrong et al. (2015) Speciation and solubility of reduced C-O-H-N
    volatiles in mafic melt: Implications for volcanism, atmospheric
    evolution, and deep volatile cycles in the terrestrial planets. GCA.
    """

    melt_CO = melt_CO * 1e6

    fCO = 10 ** ((np.log10(melt_CO) + 0.738 + 5.44e-5 * P) / 0.876)

    return fCO


def burguisser2015_co2(fCO2, CO2):
    """
    Returns the weight fraction of CO2 in the melt using the method
    applied in D-Compress.

    Parameters
    ----------
    fCO2 : float
        The CO2 fugacity
    CO2 : Molecule class
        The CO2 instance of the Molecule class

    Returns
    -------
    float
        The weight fraction of CO2 in the melt.

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """
    return CO2.solCon["a"] * (fCO2) ** CO2.solCon["b"]


def burguisser2015_co2_fugacity(melt_co2, CO2):
    """
    Returns the fugacity of CO2 according to the solubility law applied
    in D-Compress.

    Parameters
    ----------
    melt_co2 : float
        The weight fraction of CO2 in the melt
    CO2 : Molecule class
        The CO2 instance of the Molecule class

    Returns
    -------
    float
        The fugacity of CO2 in the gas phase.

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """
    return (melt_co2 / CO2.solCon["a"]) ** (1 / CO2.solCon["b"])


# PL: for consistency, might be more sensible to just provide fH2 here.
def burguisser2015_h2(mH2, H2, P, H2Y=None):
    """
    Returns the weight fraction of H2 in the melt using the method applied
    in D-Compress.

    Parameters
    ----------
    mH2 : float
        The mol fraction of H2 in the gas phase
    H2 : Molecule class
        The H2 instance of the Molecule class
    P : float
        Pressure (bars)
    H2Y : float, optional
        The H2 fugacity coefficient (required if different to one stored in H2)

    Returns
    -------
    float
        The weight fraction of H2 in the melt.

    Warnings
    --------
    This solubility law should only be used where direct comparisons to
    DCompress are required; there is a unit error in the implementation.

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """
    if H2Y is None:
        return H2.solCon["a"] * (H2.Y * mH2 * P) ** H2.solCon["b"]
    else:
        return H2.solCon["a"] * (H2Y * mH2 * P) ** H2.solCon["b"]


def burguisser2015_h2_fugacity(melt_h2, H2):
    """
    Returns the fugacity of H2 using the solubility law applied
    in D-Compress.

    Parameters
    ----------
    melt_h2 : float
        The weight fraction of H2 in the melt
    H2 : Molecule class
        The H2 instance of the Molecule class

    Returns
    -------
    float
        The fugacity of H2 in the gas phase

    Warnings
    --------
    This solubility law should only be used where direct comparisons to
    DCompress are required; there is a unit error in the implementation.

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """
    return (melt_h2 / H2.solCon["a"]) ** (1 / H2.solCon["b"])


def gaillard2003_h2(mH2, H2, P, melt, H2Y=None):
    """
    Returns the weight fraction of H2 in the melt using the method derived in
    Gaillard et. al., 2003 (returns g/m3; melt density converts from H2 density to
    weight fraction)
    Applicable for fH2 = 0.02 - 70 bar, and 573 - 1273 K

    Args:
        mH2 (float): The mol fraction of H2 in the gas phase
        H2 (class): The H2 instance of the Molecule class
        P (float): Pressure (bars)
        melt (class): Active instance of Melt class
        H2Y (float): The H2 fugacity coefficient (if different to one stored in H2)

    Returns:
        The weight fraction of H2 in the melt.
    """

    melt_density = melt.rho(P=P * 1e5) / 1e3  # convert to units g/cm3 from kg/m3

    if H2Y is None:
        return (3.4e-7 * (H2.Y * mH2 * P) ** 1.28) / melt_density
    else:
        return (3.4e-7 * (H2Y * mH2 * P) ** 1.28) / melt_density


def gaillard2003_h2_fugacity(melt_h2, melt):
    """
    Returns the fugacity of H2 in the melt using the solubility law derived in
    Gaillard et. al., 2003.

    Args:
        melt_h2 (float): The weight fraction of H2 in the melt
        melt (class): The active instance of the Melt class

    Returns:
        The fugacity of H2 in the gas phase
    """

    melt_density = melt.rho() / 1e3

    return ((melt_h2 * melt_density) / 3.4e-7) ** (1 / 1.28)


def burguisser2015_h2o(mH2O, H2O, P, H2OY=None):
    """
    Returns the weight fraction of H2O in the melt using the solubility
    law applied in D-Compress.

    Parameters
    ----------
    mH2O : float
        The mole fraction of H2O in the gas phase
    H2O : Molecule class
        The H2O instance of the Molecule class
    P : float
        Pressure (bars)
    H2OY : float, optional
        The H2O fugacity coefficient (required if different to one stored in H2O)

    Returns
    -------
    float
        The weight fraction of H2O in the melt

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """

    if H2OY is None:
        return H2O.solCon["a"] * (H2O.Y * mH2O * P) ** H2O.solCon["b"]
    else:
        return H2O.solCon["a"] * (H2OY * mH2O * P) ** H2O.solCon["b"]


def burguisser2015_h2o_fugacity(melt_h2o, H2O):
    """
    Returns the fugacity of H2O using the solubility law applied
    in D-Compress.

    Parameters
    ----------
    melt_h2o : float
        The weight fraction of H2O in the melt
    H2O : Molecule class
        The H2O instance of the Molecule class

    Returns
    -------
    float
        The fugacity of H2O in the gas phase

    References
    ----------
    Burguisser et al., (2015) Simulating the behaviour of volatiles
    belonging to the C-O-H-S system in silicate melts under magmatic
    conditions with the software D-Compress. Computers & Geosciences.
    """

    return (melt_h2o / H2O.solCon["a"]) ** (1 / H2O.solCon["b"])


def eguchi2018(fCO2, fO2, T, P, melt):
    """
    Returns the weight fraction of CO2 dissolved in the melt, as a converted
    sum of both molecular CO2, and carbonate (CO3^2-).

    Parameters
    ----------
    fCO2 : float
        The fugacity of CO2
    fO2 : float
        The fugacity of oxygen
    T : float
        Temperature (K)
    P : float
        Pressure (bars)
    melt : Melt class
        The active instance of the Melt class

    Returns
    -------
    float
        Weight fraction of CO2 dissolved in the melt, as the converted
        sum for CO2(mol) + CO3

    References
    ----------
    Eguchi, J., and Dasgupta, R. (2018) A CO2 solubility model for silicate
    melts from fluid saturation to graphite or diamond saturation.
    Chemical Geology.
    """

    def NBO(mol_fracs):
        nsi = mol_fracs["sio2"]
        nti = mol_fracs["tio2"]
        nal = mol_fracs["al2o3"] * 2
        nfe2 = mol_fracs["feo"]
        nfe3 = mol_fracs["fe2o3"] * 2
        nmn = mol_fracs["mno"]
        nmg = mol_fracs["mgo"]
        nca = mol_fracs["cao"]
        nna = mol_fracs["na2o"] * 2
        nk = mol_fracs["k2o"] * 2
        np = mol_fracs["p2o5"] * 2
        o = (
            mol_fracs["sio2"] * 2
            + mol_fracs["tio2"] * 2
            + mol_fracs["al2o3"] * 3
            + nfe2
            + nfe3 * 1.5
            + mol_fracs["mno"]
            + mol_fracs["mgo"]
            + mol_fracs["cao"]
            + mol_fracs["na2o"]
            + mol_fracs["k2o"]
            + mol_fracs["p2o5"] * 5
        )

        NM = nmg + nca + nfe2 + nna + nk + nmn

        Al = nal - NM

        if Al > 0:
            al_tet = NM
        else:
            al_tet = nal

        Fe = nfe3 + Al

        if Al > 0:
            fe_tet = 0
        elif Al <= 0 and Fe > 0:
            fe_tet = Al * -1
        elif Al <= 0 and Fe <= 0:
            fe_tet = nfe3

        Tet = nsi + nti + np + al_tet + fe_tet
        NBO = 2 * o - 4 * Tet
        return NBO

    def xi(fCO2, T, P, nbo, oxides, name="co2"):
        cao = oxides["cao"]
        na2o = oxides["na2o"]
        k2o = oxides["k2o"]

        if name == "co2":
            DH = -90212
            DV = 0.000019244
            DS = -43.0815
            B = 1114.9
            yNBO = -7.0937
            A_CaO = 0
            A_Na2O = 0
            A_K2O = 0

        elif name == "co3":
            DH = -164480
            DV = 0.00002384
            DS = -43.6385
            B = 1473.2
            yNBO = 3.291
            A_CaO = 1.68e5
            A_Na2O = 1.759e5
            A_K2O = 2.1085e5

        # PL: This isn't great really
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            lnXi = (
                -(DV * (P * 1e5) / (8.3144 * T))
                + DH / (8.3144 * T)
                + (np.log(fCO2) * B) / T
                + DS / 8.3144
                + (yNBO * nbo)
                + (A_CaO * cao + A_Na2O * na2o + A_K2O * k2o) / (8.3144 * T)
            )

        return np.exp(lnXi)

    oxides = melt.iron_fraction(np.log(fO2), ppa=P * 1e5)[
        0
    ]  # dry melt as oxide mol fractions

    nbo = NBO(oxides)

    Xco2 = xi(fCO2, T, P, nbo, oxides, name="co2")
    Xco3 = xi(fCO2, T, P, nbo, oxides, name="co3")

    FW_one = melt.formula_weight(fO2, P)

    # This assumes that the only species present in the melt is CO2/CO3.
    # Not completely accurate here.
    CO2_CO2 = (44.01 * Xco2) / (44.01 * Xco2 + (1 - (Xco2 + Xco3)) * FW_one)
    CO2_CO3 = (44.01 * Xco3) / (44.01 * Xco3 + (1 - (Xco2 + Xco3)) * FW_one)

    return CO2_CO2 + CO2_CO3


def eguchi2018_fugacity(melt_co2, fO2, T, P, melt):
    """
    Returns the fugacity of CO2 from the total CO2 melt fraction.

    First finds the partitioning between molecular CO2 and CO3^2-,
    then calculates the corresponding fCO2.

    Parameters
    ----------
    melt_co2 : float
        The total weight fraction of CO2 in the melt
    fO2 : float
        The fugacity of oxygen
    T : float
        Temperature (K)
    P : float
        Pressure (bars)
    melt : Melt class
        The active instance of the Melt class

    Returns
    -------
    fCO2 : float
        Fugacity of CO2

    References
    ----------
    Eguchi, J., and Dasgupta, R. (2018) A CO2 solubility model for silicate
    melts from fluid saturation to graphite or diamond saturation.
    Chemical Geology.
    """

    def f(fco2, fO2, T, P, melt, melt_CO2):
        return eguchi2018(fco2, fO2, T, P, melt) - melt_CO2

    fCO2 = fsolve(f, 1.0, args=(fO2, T, P, melt, melt_co2))[0]

    # Check for graphite saturation
    logK1 = (
        40.07639 - 2.53932 * 10**-2 * T + 5.27096 * 10**-6 * T**2 + 0.0267 * (P - 1) / T
    )
    graph_fco2 = 10**logK1 * 10 ** np.log10(fO2)

    if fCO2 > graph_fco2:
        melt.graphite_sat = True
        return graph_fco2
    else:
        melt.graphite_sat = False
        return fCO2


def graphite_fco2(T, P, fO2):
    """
    Calculates the fCO2 in equilibrium with a graphite saturated melt.

    Parameters
    ----------
    T : float
        Temperature (K)
    P : float
        Pressure (bars)
    fO2 : float
        The fugacity of oxygen

    Returns
    -------
    float
        Fugacity of CO2 in a graphite saturated melt

    References
    ----------
    Holloway, J.R., Pan, V. and Gudmundsson, G. (1992) High-pressure
    fluid-absent melting experiments in the presence of graphite; oxygen
    fugacity, ferric/ferrous ratio and dissolved CO2.
    European Journal of Mineralogy.
    """

    logK1 = (
        40.07639 - 2.53932 * 10**-2 * T + 5.27096 * 10**-6 * T**2 + 0.0267 * (P - 1) / T
    )
    graph_fco2 = 10**logK1 * 10 ** np.log10(fO2)

    return graph_fco2


def libourel2003(mN2, fO2, P):
    """
    Returns the weight fraction of N in the melt.

    Parameters
    ----------
    mN2 : float
        mole fraction of N2 in the gas phase
    mO2 : float
        mole fraction of O2 in the gas phase
    O2 : Molecule class
        O2 instance of the Molecule class
    P : float
        Pressure (bars)

    Returns
    -------
    float
        The weight fraction of N2 in the melt

    References
    ----------
    Libourel, G., Marty, B., and Humbert, F. (2003) Nitrogen solubility
    in basaltic melt. Part I. Effect of oxygen fugacity. GCA.
    """
    return (
        0.0611e-6 * P * 0.986923 * mN2
        + 5.97e-16 * fO2**-0.75 * (P * 0.986923 * mN2) ** 0.5
    )


def libourel2003_fugacity(n_melt, nY, fO2, P):
    """
    Returns the fugacity of N2 in the gas phase.

    Parameters
    ----------
    n_melt : float
        Weight fraction of N2 in the melt
    nY : float
        Fugacity coefficient of N2
    fO2 : float
        Oxygen fugacity
    P : float
        Pressure (bars)

    Returns
    -------
    fn2 : float
        The fugacity of N2

    References
    ----------
    Libourel, G., Marty, B., and Humbert, F. (2003) Nitrogen solubility
    in basaltic melt. Part I. Effect of oxygen fugacity. GCA.
    """

    def n_quadratic(n_melt):
        a = 0.0611e-6
        b = 5.97e-16

        def f0(x):
            return a * x + b * ((fO2) ** (-0.75)) * (x**0.5) - n_melt

        with warnings.catch_warnings():
            warnings.filterwarnings("error")
            try:
                PN2 = fsolve(f0, 1)[0]
            except RuntimeWarning:
                try:
                    PN2 = fsolve(f0, 1e-10)[0]
                except RuntimeWarning as err:
                    raise RuntimeError("Failed to find N2 partial pressure.") from err

        return PN2

    PN2 = n_quadratic(n_melt)
    mN2 = PN2 / (P * 0.986923)  # pressure in atmospheres
    fn2 = nY * mN2 * P

    return fn2


def nash2019(fO2, P, T, melt, run):
    """
    Returns the ratio of sulfate:sulfide (S6+/S2-) in the melt.

    Parameters
    ----------
    fO2 : float
        Absolute oxygen fugacity (not relative to a buffer)
    P : float
        Pressure (bars)
    T : float
        Temperature (K)
    melt : Melt class
        Active instance of the Melt class
    run : RunDef class
        Active instance of the RunDef class

    Returns
    -------
    ratio : float
        S6+/S2- ratio of sulfur species in the melt

    References
    ----------
    Nash, W.M., Smythe, D.J., and Wood, B.J., (2019) Compositional and
    temperature effects on sulfur speciation and solubility in silicate
    melts. EPSL.
    Eq 11
    """
    F = 2 * cnvs.fo2_2F(
        melt.Cm(), T, P * 1e5, np.log(fO2), run.FO2_MODEL
    )  # F is mFe2O3/mFeO. *2 gives fe3/fe2.

    ratio = 10 ** (8 * np.log10(F) + (8.7436e6 / T**2) - (27703 / T) + 20.273)

    return ratio


def oneill2002(fO2, P, melt):
    """
    Returns the sulfide capacity of a melt.

    Parameters
    ----------
    fO2 : float
        Absolute oxygen fugacity (not relative to a buffer)
    P : float
        Pressure (bars)
    melt : Melt class
        Active instance of the Melt class

    Returns
    -------
    capacity : float
        Sulfide (S2-) capacity of melt as a weight fraction

    References
    ----------
    O'Neill, H.S.C., and Mavrogenes, J.A. (2002) The sulfide capacity
    and the sulfur content at sulfide-saturation of silicate melts at
    1400C and 1 bar. Journal of Petrology.
    """
    mol_frac = melt.iron_fraction(np.log(fO2), ppa=P * 1e5)[0]
    wts = cnvs.mol2wt(mol_frac)
    feo = wts["feo"] * 100  # wt% of feo in melt

    capacity = (
        0.0003 * (100 - feo) * np.exp(0.21 * feo)
    ) / 1000000  # Convert ppm -> wt fraction
    return capacity


def oneill2020(T, melt):
    """
    Returns the sulfide capacity of a melt.

    Parameters
    ----------
    T : float
        Temperature (K)
    melt : Melt class
        Active instance of the Melt class

    Returns
    -------
    capacity : float
        Sulfide (S2-) capacity of melt as a weight fraction

    References
    ----------
    O'Neill, H.S.C. (2021) The thermodynamic controls on sulfide saturation
    in silicate melts with application to ocean floor basalts.
    In: Magma Redox Chemistry.
    Eq 10.34
    """
    if not bool(melt.cm_dry):
        comp = melt.Cm()
    else:
        comp = melt.cm_dry

    if comp["fe2o3"] > 0:
        comp["feo"] = comp["feo"] + comp["fe2o3"] * 0.8998
        comp["fe2o3"] = 0.0

    comp = cnvs.mol2wt(comp)

    # Converts the dry mol fraction to normalized mol fraction on single cation basis.
    comp = cnvs.single_cat(comp)

    Na = comp["na2o"]
    Ti = comp["tio2"]
    K = comp["k2o"]
    Ca = comp["cao"]
    Mg = comp["mgo"]
    Si = comp["sio2"]
    Al = comp["al2o3"]
    Fe2 = comp["feo"]  # FeOt
    Mn = comp["mno"]
    FeX = Fe2 + Mn

    capacity = (
        np.exp(
            -23590 / T
            + 8.77
            + (1673 / T)
            * (
                6.7 * (Na + K)
                + 1.8 * Al
                + 4.9 * Mg
                + 8.1 * Ca
                + 5 * Ti
                + 8.9 * FeX
                - 22.2 * FeX * Ti
                + 7.2 * FeX * Si
            )
            - 2.06 * erf(-7.2 * FeX)
        )
    ) / 1000000  # Convert ppm -> wt fraction

    return capacity


# ------------------------------------------------------------------------------
# MELT CONTENT MODEL SELECTION
# ------------------------------------------------------------------------------


def ch4_melt(fCH4, P, name="ardia2013"):
    """
    Returns the number of moles of CH4 in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ---------
    fCH4 : float
        The fugacity of CH4 in the gas
    P : float
        Pressure (bar)
    name : {'ardia2013', 'None'}
        The name of the solubility law to be used. If `None`, CH4 is
        treated as insoluble; this should be done for fO2 < IW+1.

    Returns
    -------
    float
        The number of moles of CH4 in the melt
    """

    if name == "ardia2013":
        return ardia2013(fCH4, P) / cnst.m["ch4"]

    elif name == "None":
        return 0


def co_melt(fCO, P, name="armstrong2015"):
    """
    Returns the number of moles of CO in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ---------
    fCO : float
        The fugacity of CO in the gas
    P : float
        Pressure (bar)
    name : {'armstrong2015', 'None'}
        The name of the solubility law to be used. If `None`, CO is
        treated as insoluble; this should be done for fO2 < IW+1.

    Returns
    -------
    float
        The number of moles of CO in the melt
    """

    if name == "armstrong2015":
        return armstrong2015(fCO, P) / cnst.m["co"]

    elif name == "None":
        return 0


def co2_melt(fCO2, CO2, fO2, T, P, melt, name="burguisser2015"):
    """
    Returns the number of moles of CO2 in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    fCO2 : float
        CO2 fugacity
    CO2 : Molecule class
        CO2 instance of the Molecule class
    fO2 : float
        Absolute oxygen fugacity
    T : float
        Temperature (K)
    P : float
        Pressure (bars)
    melt : Melt class
        Active instance of the Melt class
    name : {'burguisser2015', 'eguchi2018'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The number of moles of CO2 in the melt.
    """

    if name == "burguisser2015":
        return burguisser2015_co2(fCO2, CO2) / cnst.m["co2"]

    elif name == "eguchi2018":
        return eguchi2018(fCO2, fO2, T, P, melt) / cnst.m["co2"]


def h2_melt(mH2, H2, P, melt, name="gaillard2003", Y=None):
    """
    Returns the number of moles of H2 in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    mH2 : float
        Mole fraction of H2 in the gas phase
    H2 : Molecule class
        H2 instance of the Molecule class
    P : float
        Pressure (bars)
    melt : Melt class
        Active instance of the Melt class
    name : {'burguisser2015', 'gaillard2003'}
        The name of the solubility law to be used.
    Y : float, optional
        The H2 fugacity coefficient, if different to version stored in H2

    Returns
    -------
    float
        The number of moles of H2 in the melt.
    """

    if name == "burguisser2015":
        return burguisser2015_h2(mH2, H2, P, H2Y=Y) / cnst.m["h2"]

    elif name == "gaillard2003":
        return gaillard2003_h2(mH2, H2, P, melt, H2Y=Y) / cnst.m["h2"]


def h2o_melt(mH2O, H2O, P, name="burguisser2015", Y=None):
    """
    Returns the number of moles of H2O in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    mH2O : float
        Mole fraction of H2O in the gas phase
    H2O : Molecule class
        H2O instance of the Molecule class
    P : float
        Pressure (bars)
    name : {'burguisser2015'}
        The name of the solubility law to be used.
    Y : float, optional
        The H2O fugacity coefficient, if different to version stored
        in H2O

    Returns
    -------
    float
        The number of moles of H2O in the melt.
    """

    if name == "burguisser2015":
        return burguisser2015_h2o(mH2O, H2O, P, H2OY=Y) / cnst.m["h2o"]


def n_melt(mN2, fO2, P, name="libourel2003"):
    """
    Returns the number of moles of N in the melt.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    mN2 : float
        Mole fraction of N2 in the gas phase
    fO2 : float
        Absolute oxygen fugacity
    P : float
        Pressure (bars)
    name : {'libourel2003'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The number of moles of N in the melt.
    """
    if name == "libourel2003":
        return libourel2003(mN2, fO2, P) / cnst.m["n"]


def sulfate_melt(fS2, fO2, P, T, melt, run, name="nash2019"):
    """
    Returns the number of moles of S6+ in the melt.

    Allows selection of the capacity law being used.

    Parameters
    ----------
    fS2 : float
        The fugacity of S2
    fO2 : float
        The absolute oxygen fugacity
    P : float
        Pressure (bars)
    T : float
        Temperature (K)
    melt : Melt class
        Active instance of the Melt class
    run : RunDef class
        Active instance of the RunDef class
    name : {'nash2019'}
        The name of the solubility law to be used

    Returns
    -------
    float
        The weight fraction of sulfate (S6+) in the melt.
    """

    if name == "nash2019":
        ratio = nash2019(fO2, P, T, melt, run)

        return ratio * (sulfide_melt(fS2, fO2, P, T, melt, name=run.SULFIDE_CAPACITY))


def sulfide_melt(fS2, fO2, P, T, melt, name="oneill2020"):
    """
    Returns the number of moles of S2- in the melt.

    Allows selection of the capacity law being used.

    Parameters
    ----------
    fS2 : float
        The fugacity of S2
    fO2 : float
        The absolute oxygen fugacity
    P : float
        Pressure (bars)
    T : float
        Temperature (K)
    melt : Melt class
        Active instance of the Melt class
    name : {'oneill2020', 'oneill2002'}
        The name of the solubility law to be used

    Returns
    -------
    float
        The weight fraction of sulfide (S2-) in the melt.
    """
    if name == "oneill2002":
        capacity = oneill2002(fO2, P, melt)

    elif name == "oneill2020":
        capacity = oneill2020(T, melt)

    return (capacity * (fS2 / fO2) ** 0.5) / cnst.m["s"]


# ------------------------------------------------------------------------------
# FUGACITY MODEL SELECTION
# ------------------------------------------------------------------------------


def ch4_fugacity(melt_ch4, P, name="ardia2013"):
    """
    Returns the fugacity of CH4.

    Allows selection of the solubility law being used.

    Parameters
    ---------
    melt_ch4 : float
        The weight fraction of CH4 in the melt
    P : float
        Pressure (bar)
    name : {'ardia2013', 'None'}
        The name of the solubility law to be used. If `None`, CH4 is
        treated as insoluble; this should be done for fO2 < IW+1.

    Returns
    -------
    float
        The fugacity of CH4 (bar)
    """
    if name == "ardia2013":
        return ardia2013_fugacity(melt_ch4, P)

    elif name == "None":
        return 0


def co_fugacity(melt_co, P, name="armstrong2015"):
    """
    Returns the fugacity of CO

    Allows selection of the solubility law being used.

    Parameters
    ---------
    melt_co : float
        The weight fraction of CO in the melt
    P : float
        Pressure (bar)
    name : {'armstrong2015', 'None'}
        The name of the solubility law to be used. If `'None'`, CO is
        treated as insoluble; this should be done for fO2 < IW+1.

    Returns
    -------
    float
        The fugacity of CO (bar)
    """
    if name == "armstrong2015":
        return armstrong2015_fugacity(melt_co, P)

    elif name == "None":
        return 0


def co2_fugacity(melt_co2, CO2, fO2, T, P, melt, name="burguisser2015"):
    """
    Returns the fugacity of CO2 in the gas.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    melt_co2 : float
        The weight fraction of CO2 in in the melt
    CO2 : Molecule class
        CO2 instance of the Molecule class
    fO2 : float
        Absolute oxygen fugacity
    T : float
        Temperature (K)
    P : float
        Pressure (bars)
    melt : Melt class
        Active instance of the Melt class
    name : {'burguisser2015', 'eguchi2018'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The fugacity of CO2 (bar)
    """

    if name == "burguisser2015":
        return burguisser2015_co2_fugacity(melt_co2, CO2)

    elif name == "eguchi2018":
        return eguchi2018_fugacity(melt_co2, fO2, T, P, melt)


def h2_fugacity(melt_h2, H2, melt, name="burguisser2015"):
    """
    Returns the fugacity of H2 in the gas.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    melt_h2 : float
        The weight fraction of H2 in the melt
    H2 : Molecule class
        H2 instance of the Molecule class
    melt : Melt class
        Active instance of the Melt class
    name : {'burguisser2015', 'gaillard2003'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The fugacity of H2 (bar)
    """

    if name == "burguisser2015":
        return burguisser2015_h2_fugacity(melt_h2, H2)

    elif name == "gaillard2003":
        return gaillard2003_h2_fugacity(melt_h2, melt)


def h2o_fugacity(melt_h2o, H2O, name="burguisser2015"):
    """
    Returns the fugacity of H2O in the gas.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    melt_h2o : float
        The weight fraction of H2O in the melt
    H2O : Molecule class
        H2O instance of the Molecule class
    name : {'burguisser2015'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The fugacity of H2O (bar)
    """

    if name == "burguisser2015":
        return burguisser2015_h2o_fugacity(melt_h2o, H2O)


def n2_fugacity(melt_n, N2Y, fO2, P, name="libourel2003"):
    """
    Returns the fugacity of N2 in the gas.

    Allows selection of the solubility law being used.

    Parameters
    ----------
    melt_n : float
        The weight fraction of N in the melt
    N2Y : float
        The fugacity coefficient of N2
    fO2 : float
        Absolute oxygen fugacity
    P : float
        Pressure (bars)
    name : {'libourel2003'}
        The name of the solubility law to be used.

    Returns
    -------
    float
        The fugacity of N2 (bar)
    """

    if name == "libourel2003":
        return libourel2003_fugacity(melt_n, N2Y, fO2, P)


def S2_fugacity(
    s_melt, fO2, P, T, melt, run, sulfidename="oneill2020", sulfatename="nash2019"
):
    """
    Returns the fugacity of S2 in the gas.

    Allows selection of the capacity laws being used.

    Parameters
    ----------
    s_melt : float
        The weight fraction of total sulfur (S2- + S6+) in the melt
    fO2 : float
        The absolute oxygen fugacity
    P : float
        Pressure (bars)
    T : float
        Temperature (K)
    melt : Melt class
        Active instance of the Melt class
    run : RunDef class
        Active instance of the RunDef class
    sulfidename : {'oneill2020', 'oneill2002'}
        The name of the sulfide capacity law to be used
    sulfatename : {'nash2019'}
        The name of the sulfate capacity law to be used

    Returns
    -------
    fS2 : float
        The fugacity of S2 (bar)
    """

    if sulfatename == "nash2019":
        # sets the Fe2/Fe3 ratio and dry melt chemistry for the sulfide capacity
        melt.cm_dry = melt.iron_fraction(np.log(fO2), ppa=P * 1e5)[0]

        # need to find the ratio of s6+/s2-, then use the amount of s2- to get mS2.
        ratio = nash2019(fO2, P, T, melt, run)  # s6+/s2-
        s2_melt = s_melt * (1 / (1 + ratio))

        if sulfidename == "oneill2002":
            capacity = oneill2002(fO2, P, melt)

        elif sulfidename == "oneill2020":
            capacity = oneill2020(T, melt)

        fS2 = (s2_melt / capacity) ** (1 / 0.5) * fO2

        return fS2
''',
"conversions": r'''# conversions.py

import numpy as np

from evo import constants as cnst
from evo import ferric
from evo import solubility_laws as sl

# ------------------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------------------


def K2C(T):
    """Convert degrees K into degrees C"""
    return T - 273.15


def round_down(n, dp):
    """Round a number according to the number of decimal places"""
    step_size = min(1, dp)
    factor = 1 / step_size
    return np.floor(n * factor) / factor


# ------------------------------------------------------------------------
# FOR EQUILIBRATION OF FO2 TO THE FE CONTENT OF THE MAGMA
# ------------------------------------------------------------------------


def norm(c, f=1.0, cons=None):
    """
    Normalises a set of values to 1, or a given value.

    Parameters
    ----------
    c : dict
        The dictionary of values to be normalised
    f : float, optional
        The value to normalise to, by default 1.
    cons : list of str, optional
        Any molecules that should have their values held constant,
        by default None

    Returns
    -------
    tmp : dict
        Dictionary `c` normalised to `f`
    """

    sm = 0
    tmp = {}
    constant = 0

    if cons:
        for x in c:
            if x in cons:
                constant += c[x]
            else:
                sm += c[x]
    else:
        for x in c:
            sm += c[x]

    if cons:
        for x in c:
            if x in cons:
                tmp[x] = c[x]
            else:
                tmp[x] = (f - constant) * c[x] / sm
    else:
        for x in c:
            tmp[x] = f * c[x] / sm

    return tmp


def wt2mol(c, *args):
    """
    Converts a composition given as weight fractions to mole fractions

    Uses EQ 16 of supp. mat.

    Parameters
    ----------
    c : dict
        The dictionary to be converted
    *args : float
        Any specific values to be returned (e.g., sio2)

    Returns
    -------
    mol : dict
        The dictionary `c` as a set of mole fractions
    """

    mol = {}
    sm = 0.0

    for x in c:
        if x == "o(fe)":
            mol[x] = c[x] / cnst.m["o"]
            sm += c[x] / cnst.m["o"]
        else:
            mol[x] = c[x] / cnst.m[x]
            sm += c[x] / cnst.m[x]

    for x in c:
        mol[x] = mol[x] / sm

    if (
        args
    ):  # should be able to specify one element from the list to return if necessary.
        for arg in args:
            return mol[arg]
    else:
        return mol


def fo2_2F(cm, t, p, lnfo2, model="kc1991"):
    """
    Generate the fe3/fe2 ratio based on the set fO2 and fO2 model.

    Parameters
    ----------
    cm : dict
        Dry silicate melt major element composition as a mole fractions.
    t : float
        Temperatue (K)
    p : float
        Pressure (Pa)
    lnfo2 : float
        Ln(fO2)
    model : {'kc1991', 'r2013'}
        Name of fO2 model

    Returns
    -------
    float
        Fe3/Fe2 mole fraction ratio
    """

    if model == "kc1991":
        return ferric.kc91_fo2(cm, t, p, lnfo2)
    elif model == "r2013":
        return ferric.r2013_fo2(cm, t, p, lnfo2)


def c2fo2(cm, t, p, model):
    """
    Calculates fO2 based on Fe3/Fe2 ratio from the melt major element
    composition and the fO2 model.

    Parameters
    ----------
    cm : dict
        Dry silicate melt major element composition as a mole fractions.
    t : float
        Temperatue (K)
    p : float
        Pressure (Pa)
    model : {'kc1991', 'r2013'}
        Name of fO2 model chosen

    Returns
    -------
    float
        Absolute fO2
    """

    if model == "kc1991":
        return ferric.kc91_fe3(cm, t, p)
    elif model == "r2013":
        return ferric.r2013_fe3(cm, t, p)


def single_cat(c):
    """Convert dry weight fraction to single cation mol fraction. All iron as FeOt"""

    mol = {}
    sm = 0

    for x in c:
        if x in ["al2o3", "na2o", "k2o"]:
            mol[x] = c[x] / (cnst.m[x] * 0.5)
            sm += c[x] / (cnst.m[x] * 0.5)
        elif x in ["mgo", "sio2", "cao", "tio2", "mno", "feo"]:
            mol[x] = c[x] / cnst.m[x]
            sm += c[x] / cnst.m[x]

    for x in c:
        if x not in mol:
            pass
        else:
            mol[x] = mol[x] / sm

    return mol


# ------------------------------------------------------------------------
# CONVERSIONS TO AND FROM THE FMQ and IW BUFFERS FOR EASY COMPARISON TO LITERATURE
# ------------------------------------------------------------------------


def fmq2fo2(dfmq, t, p, name):  # dfmq = the value relative to the fmq buffer
    """Convert FMQ to absolute fO2"""
    return dfmq + fmq(t, p, name)


def fmq_2iw(dfmq, t, p, name):
    """Convert FMQ to IW"""
    fo2 = fmq2fo2(dfmq, t, p, name)
    return fo2_2iw(fo2, t, p, name)


def fo2_2fmq(FO2, t, p, name):  # returns as fo2 relative to the FMQ buffer
    """Convert absolute fO2 to fO2 relative to FMQ"""
    return FO2 - fmq(t, p, name)


def fmq(t, p, name):
    """Calculates the absolute fO2 of the FMQ buffer"""
    if name == "frost1991":
        return (
            -25096.3 / t + 8.735 + 0.11 * (p - 1) / t
        )  # Input pressure as bar, temp in Kelvin


def iw2fo2(FO2, t, p, name):
    """Converts IW to absolute fO2"""
    return FO2 + iw(t, p, name)


def iw_2fmq(diw, t, p, name):
    """Converts IW to FMQ"""
    fo2 = iw2fo2(diw, t, p, name)
    return fo2_2iw(fo2, t, p, name)


def fo2_2iw(FO2, t, p, name):
    """Converts absolute fO2 to IW"""
    return FO2 - iw(t, p, name)


def iw(t, p, name):
    """Calculates the absolute fO2 of the IW buffer"""
    if name == "frost1991":
        return -27489 / t + 6.702 + 0.055 * (p - 1) / t


def nno2fo2(FO2, t, p, name):
    """Converts NNO to absolute fO2"""
    return FO2 + nno(t, p, name)


def fo2_2nno(FO2, t, p, name):
    """Converts absolute fO2 to the NNO buffer"""
    return FO2 - nno(t, p, name)


def nno(t, p, name):
    """Calculates the absolute fO2 of the NNO buffer"""
    if name == "frost1991":
        return -24930 / t + 9.36 + 0.046 * (p - 1) / t


def generate_fo2(sys, dfo2, buffer, P):
    """
    Returns ln(fO2) when given fO2 relative to a buffer

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    dfo2 : float
        fO2 relative to a mineral buffer
    buffer : {'FMQ', 'IW', 'NNO'}
        Mineral buffer fO2 is given relative to
    P : float
        Current pressure (bar)

    Returns
    -------
    fo2 : float
        Ln(fO2)
    """
    # PL: Edit to allow setting from an fe2/fe3 ratio somehow.

    if buffer == "FMQ":
        fo2 = np.log(10 ** fmq2fo2(dfo2, sys.T, P, sys.run.FMQ_MODEL))  # ln fO2

    elif buffer == "IW":
        fo2 = np.log(10 ** iw2fo2(dfo2, sys.T, P, sys.run.FMQ_MODEL))

    elif buffer == "NNO":
        fo2 = np.log(10 ** nno2fo2(dfo2, sys.T, P, sys.run.FMQ_MODEL))

    return fo2


def generate_fo2_buffer(sys, fo2, P, buffer_choice=None):
    """
    Returns fO2 relative to a chosen rock buffer

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    fo2 : float
        Absolute fO2
    P : float
        Pressure (bar)
    buffer_choice : float, optional
        The mineral buffer to give fO2 relative to, by default None.
        This will then give the fO2 relative to the buffer set in the
        environment file.

    Returns
    -------
    float
        fO2 relative to a mineral buffer
    """

    if buffer_choice is not None:
        buffer = buffer_choice
    elif sys.run.FO2_buffer_SET is True:
        buffer = sys.run.FO2_buffer
    else:
        return fo2_2fmq(np.log10(fo2), sys.T, P, sys.run.FMQ_MODEL)

    if buffer == "FMQ":
        return fo2_2fmq(np.log10(fo2), sys.T, P, sys.run.FMQ_MODEL)

    elif buffer == "IW":
        return fo2_2iw(np.log10(fo2), sys.T, P, sys.run.FMQ_MODEL)

    elif buffer == "NNO":
        return fo2_2nno(np.log10(fo2), sys.T, P, sys.run.FMQ_MODEL)


# ------------------------------------------------------------------------
# CONVERSIONS FOR THE SOLVER SCRIPT
# ------------------------------------------------------------------------


def mols2wts(**kwargs):  # input as "H2O" = gas.mH2O[-1], "O2"...etc
    """Converts a gas composition from mole fraction to weight fraction.

    Enables species to be entered in any order, input should take the form
    "H2O" = gas.mH2O[-1], "O2" = gas.mO2[-1]...

    Parameters
    ----------
    **kwargs : string float pairs
        gas species as mole fractions

    Returns
    -------
    x : dict
        Dictionary of gas phase species given as weight fractions
    """
    x = {}
    sum = 0

    for key, value in kwargs.items():
        sum += value * cnst.m[str(key.lower())]

    for key, value in kwargs.items():
        x[str(key)] = (value * cnst.m[str(key.lower())]) / sum

    return x


def mol2wt(c):
    """Converts a composition dict from mole fraction to weight fractions"""
    mol = {}
    sm = 0

    for x in c:
        if x == "ofe" or x == "ogas":
            mol[x] = c[x] * cnst.m["o"]
            sm += c[x] * cnst.m["o"]
        else:
            mol[x] = c[x] * cnst.m[x]
            sm += c[x] * cnst.m[x]

    for x in c:
        mol[x] = mol[x] / sm

    return mol


def mean_mol_wt(**kwargs):  # input as "H2O" = gas.mH2O[-1], "O2"...etc
    """
    Calculates the mean molecular weight of the gas phase

    Enables species to be entered in any order, input should take the form
    "H2O" = gas.mH2O[-1], "O2" = gas.mO2[-1]...

    Parameters
    ----------
    **kwargs : str float pairs
        gas species as mole fractions

    Returns
    -------
    sum : float
        mean molecular weight (kg/mol)
    """
    sum = 0

    for key, value in kwargs.items():
        sum += value * (cnst.m[str(key.lower())] / 1000)  # in kg/mol

    return sum


def frac2perc(x):
    """Convert fraction to percentage"""
    X = []
    for i in x:
        X.append(i * 100)
    return X


def atomicM_calc(sys, melt, gas, element, i, WgT=None):
    """
    Calculates the mass of a volatile element given gas and melt comp

    Calculates the weight fraction of a volatile element given the
    gas composition and the dissolved volatile content of the melt. For
    comparison to the stored element masses to assert conservation of mass.

    `i` allows this to be calculated for any model step.

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    melt : Melt class
        Active instance of the Melt class
    gas : Gas class
        Active instance of the Gas class
    element : {'o', 'h', 'c', 's', n'}
        The volatile element required
    i : int
        index required to find the gas and melt composition
    WgT : float, optional
        The total gas mass fraction, by default None

    Returns
    -------
    float
        element mass fraction
    """

    if WgT is None:
        WgT = sys.WgT[i + 1]

    if sys.run.GAS_SYS == "OH":
        if element == "o":
            return cnst.m["o"] * (
                ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
            )

        elif element == "h":
            return (
                2
                * cnst.m["h"]
                * (
                    ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                    + ((WgT * gas.Wt["H2"][i]) + (melt.h2[i])) / cnst.m["h2"]
                )
            )

        elif element == "c" or element == "n" or element == "s":
            return 0

        elif element == "o_tot":
            if sys.run.FE_SYSTEM is False:
                return 0
            else:
                return (
                    cnst.m["o"]
                    * (
                        ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                        + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                    )
                ) + melt.ofe[i]

    elif sys.run.GAS_SYS == "COH":
        if element == "o":
            return cnst.m["o"] * (
                ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                + (WgT * gas.Wt["CO"][i] + melt.co[i]) / cnst.m["co"]
                + (2 * ((WgT * gas.Wt["CO2"][i]) + (melt.co2[i]))) / cnst.m["co2"]
            )

        elif element == "h":
            return (
                2
                * cnst.m["h"]
                * (
                    ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                    + ((WgT * gas.Wt["H2"][i]) + (melt.h2[i])) / cnst.m["h2"]
                    + 2 * (WgT * gas.Wt["CH4"][i] + melt.ch4[i]) / cnst.m["ch4"]
                )
            )

        elif element == "c":
            return cnst.m["c"] * (
                ((gas.Wt["CO2"][i] * WgT) + melt.co2[i]) / cnst.m["co2"]
                + (gas.Wt["CO"][i] * WgT + melt.co[i]) / cnst.m["co"]
                + (gas.Wt["CH4"][i] * WgT + melt.ch4[i]) / cnst.m["ch4"]
                + melt.graphite[i] / cnst.m["c"]
            )

        elif element == "s" or element == "n":
            return 0

        elif element == "o_tot":
            if sys.run.FE_SYSTEM is False:
                return 0
            else:
                return (
                    cnst.m["o"]
                    * (
                        ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                        + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                        + (WgT * gas.Wt["CO"][i] + melt.co[i]) / cnst.m["co"]
                        + (2 * ((WgT * gas.Wt["CO2"][i]) + (melt.co2[i])))
                        / cnst.m["co2"]
                    )
                ) + melt.ofe[i]

    elif sys.run.GAS_SYS == "SOH":
        if element == "o":
            return cnst.m["o"] * (
                ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                + (2 * WgT * gas.Wt["SO2"][i]) / cnst.m["so2"]
            )

        elif element == "h":
            return (
                2
                * cnst.m["h"]
                * (
                    ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                    + ((WgT * gas.Wt["H2"][i]) + (melt.h2[i])) / cnst.m["h2"]
                    + (WgT * gas.Wt["H2S"][i]) / cnst.m["h2s"]
                )
            )

        elif element == "c" or element == "n":
            return 0

        elif element == "s":
            return cnst.m["s"] * (
                (2 * gas.Wt["S2"][i] * WgT) / cnst.m["s2"]
                + (gas.Wt["H2S"][i] * WgT) / cnst.m["h2s"]
                + (gas.Wt["SO2"][i] * WgT) / cnst.m["so2"]
                + melt.s[i] / cnst.m["s"]
            )

        elif element == "o_tot":
            if sys.run.FE_SYSTEM is False:
                return 0
            else:
                return (
                    cnst.m["o"]
                    * (
                        ((WgT * gas.Wt["H2O"][i]) + (melt.h2o[i])) / cnst.m["h2o"]
                        + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                        + (2 * WgT * gas.Wt["SO2"][i]) / cnst.m["so2"]
                    )
                ) + melt.ofe[i]

    elif sys.run.GAS_SYS == "COHS" or sys.run.GAS_SYS == "COHSN":
        if element == "o":
            return cnst.m["o"] * (
                ((WgT * gas.Wt["H2O"][i]) + melt.h2o[i]) / cnst.m["h2o"]
                + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                + (WgT * gas.Wt["CO"][i] + melt.co[i]) / cnst.m["co"]
                + (2 * ((WgT * gas.Wt["CO2"][i]) + melt.co2[i])) / cnst.m["co2"]
                + (2 * (WgT * gas.Wt["SO2"][i])) / cnst.m["so2"]
            )

        elif element == "o_tot":
            if sys.run.FE_SYSTEM is False:
                return 0
            else:
                return (
                    cnst.m["o"]
                    * (
                        ((WgT * gas.Wt["H2O"][i]) + melt.h2o[i]) / cnst.m["h2o"]
                        + (2 * WgT * gas.Wt["O2"][i]) / cnst.m["o2"]
                        + (WgT * gas.Wt["CO"][i] + melt.co[i]) / cnst.m["co"]
                        + (2 * ((WgT * gas.Wt["CO2"][i]) + melt.co2[i])) / cnst.m["co2"]
                        + (2 * (WgT * gas.Wt["SO2"][i])) / cnst.m["so2"]
                    )
                ) + melt.ofe[i]

        elif element == "h":
            return (
                2
                * cnst.m["h"]
                * (
                    ((WgT * gas.Wt["H2O"][i]) + melt.h2o[i]) / cnst.m["h2o"]
                    + ((WgT * gas.Wt["H2"][i]) + melt.h2[i]) / cnst.m["h2"]
                    + (gas.Wt["H2S"][i] * WgT) / cnst.m["h2s"]
                    + 2 * ((WgT * gas.Wt["CH4"][i]) + melt.ch4[i]) / cnst.m["ch4"]
                )
            )

        elif element == "c":
            return cnst.m["c"] * (
                ((gas.Wt["CO2"][i] * WgT) + melt.co2[i]) / cnst.m["co2"]
                + (gas.Wt["CO"][i] * WgT + melt.co[i]) / cnst.m["co"]
                + (gas.Wt["CH4"][i] * WgT + melt.ch4[i]) / cnst.m["ch4"]
                + melt.graphite[i] / cnst.m["c"]
            )

        elif element == "s":
            return cnst.m["s"] * (
                2 * (gas.Wt["S2"][i] * WgT) / cnst.m["s2"]
                + (gas.Wt["H2S"][i] * WgT) / cnst.m["h2s"]
                + (gas.Wt["SO2"][i] * WgT) / cnst.m["so2"]
                + melt.s[i] / cnst.m["s"]
            )

        elif element == "n":
            if sys.run.GAS_SYS == "COHSN":
                return melt.n[i] + cnst.m["n"] * (
                    (2 * gas.Wt["N2"][i] * WgT) / cnst.m["n2"]
                )
            else:
                return 0


def get_graphite(sys, melt, P, CO2, mCO, mCO2, mCH4, mO2, co2Y, coY, ch4Y, o2Y, N):
    """Returns the mass of graphite in the melt."""

    return (
        (sys.atomicM["c"] / cnst.m["c"])
        - (
            N * (mCO + mCO2 + mCH4)
            + sl.co2_melt(
                (co2Y * mCO2 * P),
                CO2,
                (o2Y * mO2 * P),
                sys.T,
                P,
                melt,
                name=sys.run.C_MODEL,
            )
            + sl.co_melt((coY * mCO * P), P, name=sys.run.CO_MODEL)
            + sl.ch4_melt((ch4Y * mCH4 * P), P, name=sys.run.CH4_MODEL)
        )
    ) * cnst.m["c"]
''',
"density": r'''# density.py
"""
Calculates density of melt according to changing temp, pressure and compositions.
From Sperra(2000) Physical properties of magmas, Encyclopedia of Volcanology
"""

from evo import constants as cnst

# ------------------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------------------


def den_calc_spera2000(Cm, T, P):
    """
    Calculates the density of the melt according to composition, T and P.

    Parameters
    ----------
    Cm : dict
        Composition of the melt, optionally including H2O and CO2, as mole
        fractions.
    T : float
        Temperature (K)
    P : float
        Pressure, pascals (Pa)

    Returns
    -------
    float
        melt density, kg/m3

    References
    ----------
    Lesher, E.C., & Spera, F.J. (2015) Thermodynamic and transport
    properties of silicate melts and magma. The Encyclopedia of
    Volcanoes.
    """

    T0 = 1673  # Ref temp, Kelvin (1400 celsius)
    P0 = 1.0e-4  # Ref pressure, GPa (1 bar)

    # reference volumes: 10^-6 m3/mol, 1673 K (mm^3 per mol) Table S5.1
    vols = {}
    vols["sio2"] = 26.86 / 10.0**6
    vols["tio2"] = 23.16 / 10.0**6
    vols["al2o3"] = 37.42 / 10.0**6
    vols["fe2o3"] = 42.13 / 10.0**6
    vols["feo"] = 13.65 / 10.0**6
    vols["mgo"] = 11.69 / 10.0**6
    vols["cao"] = 16.53 / 10.0**6
    vols["na2o"] = 28.88 / 10.0**6
    vols["k2o"] = 45.07 / 10.0**6
    vols["li2o"] = 16.85 / 10.0**6
    vols["h2o"] = 26.27 / 10.0**6
    vols["co2"] = 33.00 / 10.0**6

    # dV/dT|P: 10^-9 m3/mol.K       #Thermal expansion per K
    dvdt = {}
    dvdt["sio2"] = 0.0 / 10.0**9
    dvdt["tio2"] = 7.24 / 10.0**9
    dvdt["al2o3"] = 0.0 / 10.0**9
    dvdt["fe2o3"] = 9.09 / 10.0**9
    dvdt["feo"] = 2.92 / 10.0**9
    dvdt["mgo"] = 3.27 / 10.0**9
    dvdt["cao"] = 3.74 / 10.0**9
    dvdt["na2o"] = 7.68 / 10.0**9
    dvdt["k2o"] = 12.08 / 10.0**9
    dvdt["li2o"] = 5.25 / 10.0**9
    dvdt["h2o"] = 9.46 / 10.0**9
    dvdt["co2"] = 0.0 / 10.0**9

    # dV/dP: 10^-6 m3/mol.GPa       #Compressability per GPa of pressure
    dvdp = {}
    dvdp["sio2"] = -1.89 / 10.0**6
    dvdp["tio2"] = -2.31 / 10.0**6
    dvdp["al2o3"] = -2.26 / 10.0**6
    dvdp["fe2o3"] = -2.53 / 10.0**6
    dvdp["feo"] = -0.45 / 10.0**6
    dvdp["mgo"] = 0.27 / 10.0**6
    dvdp["cao"] = 0.34 / 10.0**6
    dvdp["na2o"] = -2.40 / 10.0**6
    dvdp["k2o"] = -6.75 / 10.0**6
    dvdp["li2o"] = -1.02 / 10.0**6
    dvdp["h2o"] = -3.15 / 10.0**6
    dvdp["co2"] = 0.0 / 10.0**6

    # normalise to available oxides
    sm = 0
    tmp_Cm = {}  # Composition in mole fractions
    for e in vols:
        if e in Cm.keys():
            sm += Cm[e]
    for e in vols:
        if e in Cm.keys():
            tmp_Cm[e] = Cm[e] / sm

    V = 0.0
    for x in tmp_Cm:
        V += tmp_Cm[x] * (vols[x] + dvdt[x] * (T - T0) + dvdp[x] * (P / 10.0**9 - P0))
    M = sumM(tmp_Cm)

    return (M / 1000.0) / V


def sumM(Cm):
    """Sums the mole frac*molecular mass for the silicate melt composition"""
    M = 0.0
    for e in Cm:
        M += cnst.m[e] * Cm[e]
    return M
''',
"init_cons": r'''"""
Finds the atomic mass fraction of the system if they haven't already been set
(ATOMIC_MASS_SET = True)
and the saturation pressure hasn't been asked for (FIND_SATURATION=TRUE).
"""

import numpy as np
from numpy import sqrt

import evo.constants as cnst
import evo.messages as msgs
import evo.solubility_laws as sl


def get_inputs(sys, run, melt, gas, mols) -> tuple[float, ...]:
    """
    Calculates a subset of gas mole fractions based on the provided
    input parameters.

    Calculates the gas mole fractions corresponding to the initial
    provided parameters. E.g., calculates the mole fraction of H2O if the
    weight fraction of H2O in the melt is provided as an input parameter.

    If the fO2 is calculated at this point, the ferric/ferrous ratio in
    the melt is also set here.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    tuple of floats
        mole fractions of H2O, O2, H2, or CO2, H2O, O2 and H2 is the system is COHS(N).
    """

    if run.GAS_SYS == "COH":
        H2O, O2, H2, _, CO2, _ = mols
    elif run.GAS_SYS == "SOH":
        H2O, O2, H2, S2 = mols[:4]
    elif run.GAS_SYS == "COHS":
        H2O, O2, H2, _, CO2, _, S2 = mols[:7]
    elif run.GAS_SYS == "COHSN":
        H2O, O2, H2, _, CO2, _, S2 = mols[:7]

    if run.FH2_SET is True:
        mH2 = run.FH2_START / (H2.Y * sys.P)
        gas.mH2.append(mH2)

        if sys.FO2 is not None:
            mO2 = np.exp(sys.FO2) / (O2.Y * sys.P)
            gas.mO2.append(mO2)

            mH2O = (sys.K["K1"] * H2.Y * mH2 * (O2.Y * mO2 * sys.P) ** 0.5) / H2O.Y
            gas.mH2O.append(mH2O)

        elif run.FH2O_SET is True:
            mH2O = run.FH2O_START / (H2O.Y * sys.P)
            gas.mH2O.append(mH2O)

            mO2 = (
                ((H2O.Y * mH2O * sys.P) / ((H2.Y * mH2 * sys.P) * sys.K["K1"])) ** 2
            ) / (O2.Y * sys.P)
            gas.mO2.append(mO2)
            sys.FO2 = np.log(O2.Y * mO2 * sys.P)

        elif run.WTH2O_SET is True:
            mH2O = sl.h2o_fugacity(run.WTH2O_START, H2O, name=run.H2O_MODEL) / (
                H2O.Y * sys.P
            )
            assert H2O.Y * mH2O * sys.P > 0, "H2O fugacity is negative!"
            gas.mH2O.append(mH2O)

            mO2 = (
                ((H2O.Y * mH2O * sys.P) / ((H2.Y * mH2 * sys.P) * sys.K["K1"])) ** 2
            ) / (O2.Y * sys.P)
            gas.mO2.append(mO2)
            sys.FO2 = np.log(O2.Y * mO2 * sys.P)

    elif sys.FO2:
        mO2 = np.exp(sys.FO2) / (O2.Y * sys.P)
        gas.mO2.append(mO2)

        if run.FH2O_SET is True:
            mH2O = run.FH2O_START / (H2O.Y * sys.P)
            gas.mH2O.append(mH2O)

        elif run.WTH2O_SET is True:
            mH2O = sl.h2o_fugacity(run.WTH2O_START, H2O, name=run.H2O_MODEL) / (
                H2O.Y * sys.P
            )
            assert H2O.Y * mH2O * sys.P > 0, "fH2O is negative!"
            gas.mH2O.append(mH2O)

        mH2 = (H2O.Y * mH2O * sys.P) / (
            sys.K["K1"] * H2.Y * sys.P * np.exp(sys.FO2) ** 0.5
        )
        assert H2.Y * mH2 * sys.P > 0, "fH2 is negative!"
        gas.mH2.append(mH2)

    if (
        not run.FCO2_SET
        and not run.WTCO2_SET
        and not run.SULFUR_SET
        and not run.GRAPHITE_SATURATED
    ):
        # Set the Fe2/Fe3 ratio and dry melt chemistry ready for sulfide capacity
        melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        return mH2O, mO2, mH2

    elif run.SULFUR_SET:  # This comes first
        # Set the Fe2/Fe3 ratio and dry melt chemistry ready for sulfide capacity
        melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        mS2 = sl.S2_fugacity(
            run.SULFUR_START,
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            run,
            sulfidename=run.SULFIDE_CAPACITY,
            sulfatename=run.SULFATE_CAPACITY,
        ) / (S2.Y * sys.P)
        gas.mS2.append(mS2)

        return mS2, mH2O, mO2, mH2

    elif run.FCO2_SET:
        mCO2 = run.FCO2_START / (CO2.Y * sys.P)
        gas.mCO2.append(mCO2)

        # Set the Fe2/Fe3 ratio and dry melt chemistry ready for sulfide capacity
        melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        return mCO2, mH2O, mO2, mH2

    elif run.WTCO2_SET:
        if run.C_MODEL == "eguchi2018":
            fCO2 = sl.co2_fugacity(
                run.WTCO2_START,
                CO2,
                np.exp(sys.FO2),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            if melt.graphite_sat is True:
                found = msgs.graphite_warn(melt)
                if found is False:
                    mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P)) / (
                        CO2.Y * sys.P
                    )
                    gas.mCO2.append(mCO2)
                    melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

                    co2_melt = (
                        sl.co2_melt(
                            (CO2.Y * mCO2 * sys.P),
                            CO2,
                            (O2.Y * mO2 * sys.P),
                            sys.T,
                            sys.P,
                            melt,
                            name=run.C_MODEL,
                        )
                        * cnst.m["co2"]
                    )
                    graph_melt = (
                        (run.WTCO2_START - co2_melt) / cnst.m["co2"]
                    ) * cnst.m["c"]  # wt frac graphite in melt
                    melt.graphite_sat = True
                    melt.graph_current = graph_melt / cnst.m["c"]
                    run.WTCO2_START = co2_melt

                elif found is True:
                    mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P)) / (
                        CO2.Y * sys.P
                    )
                    gas.mCO2.append(mCO2)
                    melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

                    co2_melt = (
                        sl.co2_melt(
                            (CO2.Y * mCO2 * sys.P),
                            CO2,
                            (O2.Y * mO2 * sys.P),
                            sys.T,
                            sys.P,
                            melt,
                            name=run.C_MODEL,
                        )
                        * cnst.m["co2"]
                    )
                    run.WTCO2_START = co2_melt

            elif melt.graphite_sat is False:
                mCO2 = fCO2 / (CO2.Y * sys.P)
                gas.mCO2.append(mCO2)

                melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        else:
            mCO2 = sl.co2_fugacity(
                run.WTCO2_START,
                CO2,
                np.exp(sys.FO2),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            ) / (CO2.Y * sys.P)
            gas.mCO2.append(mCO2)

            # Set the Fe2/Fe3 ratio and dry melt chemistry ready for sulfide capacity
            melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        return mCO2, mH2O, mO2, mH2

    elif run.GRAPHITE_SATURATED is True:
        melt.graphite_sat = True
        melt.graph_current = run.GRAPHITE_START / cnst.m["c"]

        mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P)) / (CO2.Y * sys.P)
        gas.mCO2.append(mCO2)
        melt.cm_dry = melt.iron_fraction(sys.FO2)[0]

        return mCO2, mH2O, mO2, mH2


def oh(sys, run, melt, gas, mols):
    """
    Calculates the initial conditions for the OH system.

    Based on the subset of parameters provided at input, where the starting
    pressure is known, the speciation of the gas phase as mole fractions is
    calculated. Based on this data, the volatile content of the melt is
    found and the masses of each volatile element (O, C, H etc) is set,
    ready for use in the equilibrium constant and mass-balance solution.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    sys.atomicM : dict
        A dictionary containing the mass fraction of each volatile element
        in the system.
    """
    H2O, O2, H2 = mols

    if run.FH2_SET is True:
        fH2 = run.FH2_START
        assert fH2 > 0, "fH2 is negative!"
        mH2 = fH2 / (H2.Y * sys.P)
        gas.mH2.append(mH2)

        def quadratic():
            a = 1 / (O2.Y * sys.P)
            b = ((sys.K["K1"] * H2.Y * mH2) / H2O.Y) ** 2
            c = -((1 - mH2**2) / (O2.Y * sys.P))

            mo2 = [x for x in np.roots([a, b, c]) if x > 0]
            return mo2[0]

        mO2 = quadratic()

        gas.mO2.append(mO2)
        sys.FO2 = np.log(O2.Y * mO2 * sys.P)

    # assumes FO2 has either been set by the user,
    # or calculated from the iron ratio in an earlier step
    else:
        fO2 = np.exp(sys.FO2)
        mO2 = fO2 / (O2.Y * sys.P)
        gas.mO2.append(mO2)

        mH2 = ((1.0 - mO2) * H2O.Y) / (sys.K["K1"] * H2.Y * (fO2**0.5) + H2O.Y)
        gas.mH2.append(mH2)

    mH2O = 1 - mH2 - mO2
    gas.mH2O.append(mH2O)

    empty_lists = [gas.mCO2, gas.mCO, gas.mCH4, gas.mSO2, gas.mS2, gas.mH2S, gas.mN2]
    for alist in empty_lists:
        alist.append(np.float64(0))

    lst = {"o2": mO2, "h2": mH2, "h2o": mH2O}
    mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

    # WtO
    sys.atomicM["o"] = cnst.m["o"] * (
        (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2)
        + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
    )

    # WtH(total)
    sys.atomicM["h"] = (
        2
        * cnst.m["h"]
        * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
        )
    )
    return sys.atomicM


def coh(sys, run, melt, gas, mols):
    """
    Calculates the initial conditions for the COH system.

    Based on the subset of parameters provided at input, where the starting
    pressure is known, the speciation of the gas phase as mole fractions is
    calculated. Based on this data, the volatile content of the melt is
    found and the masses of each volatile element (O, C, H etc) is set,
    ready for use in the equilibrium constant and mass-balance solution.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    sys.atomicM : dict
        A dictionary containing the mass fraction of each volatile element
        in the system.
    """
    H2O, O2, H2, CO, CO2, CH4 = mols

    # uses get_inputs to find which parameters have been set and get the water system.
    mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

    # Solve for CH4

    assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"

    mCH4 = (1 - mH2 - mO2 - mH2O) / (
        1
        + ((sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2) / (CO2.Y * (H2O.Y * mH2O) ** 2))
        + (
            (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
            / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5 * (H2O.Y * mH2O) ** 2)
        )
    )
    gas.mCH4.append(mCH4)

    mCO2 = (
        sys.K["K2"] * (O2.Y * mO2 * sys.P) ** 0.5 * CO.Y * (1 - mH2 - mO2 - mH2O - mCH4)
    ) / (CO2.Y + sys.K["K2"] * (O2.Y * mO2 * sys.P) ** 0.5 * CO.Y)
    gas.mCO2.append(mCO2)

    # Check graphite saturation
    if run.C_MODEL == "eguchi2018":
        graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P))
        if (CO2.Y * mCO2 * sys.P) > graph_fCO2:
            exit(
                "Error: Melt is graphite saturated. "
                "System is overconstrained; please rerun by setting the melt "
                "graphite content and finding the saturation pressure, "
                "or stop using eguchi2018 in C_MODEL."
            )

    mCO = 1 - mH2O - mH2 - mO2 - mCH4 - mCO2
    gas.mCO.append(mCO)

    empty_lists = [gas.mSO2, gas.mS2, gas.mH2S, gas.mN2]
    for list in empty_lists:
        list.append(np.float64(0))

    lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "co": mCO, "co2": mCO2, "ch4": mCH4}
    mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

    # WtO
    sys.atomicM["o"] = cnst.m["o"] * (
        (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO)
        + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        + 2
        * sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
    )

    # WtH(total)
    sys.atomicM["h"] = (
        2
        * cnst.m["h"]
        * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        )
    )

    # WtC
    sys.atomicM["c"] = cnst.m["c"] * (
        (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
        + sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        + melt.graph_current
    )
    return sys.atomicM


def soh(sys, run, melt, gas, mols):
    """
    Calculates the initial conditions for the SOH system.

    Based on the subset of parameters provided at input, where the starting
    pressure is known, the speciation of the gas phase as mole fractions is
    calculated. Based on this data, the volatile content of the melt is
    found and the masses of each volatile element (O, C, H etc) is set,
    ready for use in the equilibrium constant and mass-balance solution.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    sys.atomicM : dict
        A dictionary containing the mass fraction of each volatile element
        in the system.
    """
    H2O, O2, H2, S2, SO2, H2S = mols

    # uses get_inputs to find which parameters have been set and get the water system.
    mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

    assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"

    # Quadratic solve for SO2

    def quadratic():
        a = SO2.Y**2 / (((sys.K["K5"] * O2.Y * mO2) ** 2) * S2.Y * sys.P)
        b = 1 + (
            (sys.K["K4"] * H2O.Y * mH2O * SO2.Y)
            / (sys.K["K5"] * H2S.Y * ((O2.Y * mO2) ** 1.5) * sqrt(sys.P))
        )
        c = -(1 - mH2 - mO2 - mH2O)

        mSO2 = [x for x in np.roots([a, b, c]) if x > 0 and x < 1]
        return mSO2[0]

    mSO2 = quadratic()
    gas.mSO2.append(mSO2)

    mS2 = (SO2.Y * mSO2) ** 2 / ((sys.K["K5"] * O2.Y * mO2) ** 2 * S2.Y * sys.P)
    gas.mS2.append(mS2)

    mH2S = (1 - mO2 - mS2 - mSO2) / (
        1
        + (H2S.Y / (sys.K["K1"] * sys.K["K4"] * H2.Y * (S2.Y * mS2 * sys.P) ** 0.5))
        + ((H2S.Y * (O2.Y * mO2) ** 0.5) / (H2O.Y * sys.K["K4"] * (S2.Y * mS2) ** 0.5))
    )
    gas.mH2S.append(mH2S)

    empty_lists = [gas.mCO2, gas.mCO, gas.mCH4, gas.mN2]
    for lst in empty_lists:
        lst.append(np.float64(0))

    lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "so2": mSO2, "s2": mS2, "h2s": mH2S}
    mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

    # WtO
    sys.atomicM["o"] = cnst.m["o"] * (
        (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mSO2)
        + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
    )

    # WtH
    sys.atomicM["h"] = (
        2
        * cnst.m["h"]
        * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + mH2S)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
        )
    )

    # WtS
    sys.atomicM["s"] = cnst.m["s"] * (
        (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
        + sl.sulfide_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            name=run.SULFIDE_CAPACITY,
        )
        + sl.sulfate_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            run,
            name=run.SULFATE_CAPACITY,
        )
    )

    return sys.atomicM


def cohs(sys, run, melt, gas, mols):
    """
    Calculates the initial conditions for the COHS system.

    Based on the subset of parameters provided at input, where the starting
    pressure is known, the speciation of the gas phase as mole fractions is
    calculated. Based on this data, the volatile content of the melt is
    found and the masses of each volatile element (O, C, H etc) is set,
    ready for use in the equilibrium constant and mass-balance solution.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    sys.atomicM : dict
        A dictionary containing the mass fraction of each volatile element
        in the system.
    """
    H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S = mols

    if not run.SULFUR_SET:
        mCO2, mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

        assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"
        assert 1 > mCO2 > 0, "CO2 fraction is too high for specified gas fraction"

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)
        gas.mCO.append(mCO)

        mCH4 = (
            CO2.Y
            * mCO2
            * (H2O.Y * mH2O * sys.P) ** 2
            / (sys.K["K3"] * CH4.Y * (O2.Y * mO2 * sys.P) ** 2)
        )
        gas.mCH4.append(mCH4)

        # Quadratic solve for SO2

        def quadratic():
            a = (SO2.Y / (sys.K["K5"] * O2.Y * mO2)) ** 2 / (S2.Y * sys.P)
            b = 1 + (
                sys.K["K4"]
                * H2O.Y
                * mH2O
                * SO2.Y
                / (sys.K["K5"] * H2S.Y * (O2.Y * mO2) ** 1.5 * sys.P**0.5)
            )
            c = -(1 - mH2 - mO2 - mH2O - mCO - mCO2 - mCH4)

            mSO2 = [x for x in np.roots([a, b, c]) if x > 0 and x < 1]
            if mSO2:
                return mSO2[0]
            else:
                exit(
                    "fSO2 is negative. Gas fraction or melt volatile content too high."
                )

        mSO2 = quadratic()
        gas.mSO2.append(mSO2)

        mS2 = (SO2.Y * mSO2) ** 2 / ((sys.K["K5"] * (O2.Y * mO2)) ** 2 * S2.Y * sys.P)
        gas.mS2.append(mS2)

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * SO2.Y * mSO2) / (
            sys.K["K5"] * H2S.Y * (O2.Y * mO2) ** 1.5 * sys.P**0.5
        )
        gas.mH2S.append(mH2S)

    else:  # sulfur has been used as an input
        mS2, mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

        assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"
        assert 1 > mS2 > 0, "S2 fraction wrong"

        mSO2 = (sys.K["K5"] * O2.Y * mO2 * (S2.Y * mS2 * sys.P) ** 0.5) / SO2.Y
        gas.mSO2.append(mSO2)

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * (S2.Y * mS2) ** 0.5) / (
            (O2.Y * mO2) ** 0.5 * H2S.Y
        )
        gas.mH2S.append(mH2S)

        mCH4 = (1 - mH2 - mO2 - mH2O - mS2 - mSO2 - mH2S) / (
            1
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (CO2.Y * (H2O.Y * mH2O) ** 2)
            )
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (
                    sys.K["K2"]
                    * CO.Y
                    * (O2.Y * mO2 * sys.P) ** 0.5
                    * (H2O.Y * mH2O) ** 2
                )
            )
        )
        gas.mCH4.append(mCH4)

        mCO2 = (sys.K["K3"] * CH4.Y * mCH4 * (O2.Y * mO2) ** 2) / (
            (H2O.Y * mH2O) ** 2 * CO2.Y
        )
        gas.mCO2.append(mCO2)

        # Check graphite saturation
        if run.C_MODEL == "eguchi2018":
            graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P))
            if (CO2.Y * mCO2 * sys.P) > graph_fCO2:
                exit(
                    "Error: Melt is graphite saturated. "
                    "System is overconstrained; please rerun by setting the melt "
                    "graphite content and finding the saturation pressure, "
                    "or stop using eguchi2018 in C_MODEL."
                )

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)
        gas.mCO.append(mCO)

    # Empty list
    gas.mN2.append(0.0)

    lst = {
        "o2": mO2,
        "h2": mH2,
        "h2o": mH2O,
        "co": mCO,
        "co2": mCO2,
        "ch4": mCH4,
        "so2": mSO2,
        "s2": mS2,
        "h2s": mH2S,
    }
    mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

    # WtO
    sys.atomicM["o"] = cnst.m["o"] * (
        (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
        + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        + 2
        * sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
    )

    # WtH
    sys.atomicM["h"] = (
        2
        * cnst.m["h"]
        * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4 + mH2S)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        )
    )

    # WtC
    sys.atomicM["c"] = cnst.m["c"] * (
        (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
        + sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        + melt.graph_current
    )

    # WtS
    sys.atomicM["s"] = cnst.m["s"] * (
        (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
        + sl.sulfide_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            name=run.SULFIDE_CAPACITY,
        )
        + sl.sulfate_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            run,
            name=run.SULFATE_CAPACITY,
        )
    )

    return sys.atomicM


def cohsn(sys, run, melt, gas, mols):
    """
    Calculates the initial conditions for the COHSN system.

    Based on the subset of parameters provided at input, where the starting
    pressure is known, the speciation of the gas phase as mole fractions is
    calculated. Based on this data, the volatile content of the melt is
    found and the masses of each volatile element (O, C, H etc) is set,
    ready for use in the equilibrium constant and mass-balance solution.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    run : RunDef class
        The active instance of the RunDef class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    mols : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.

    Returns
    -------
    sys.atomicM : dict
        A dictionary containing the mass fraction of each volatile element
        in the system.
    """
    H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2 = mols

    if not run.SULFUR_SET:
        mCO2, mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

        assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"
        assert 1 > mCO2 > 0, "CO2 fraction is too high for specified gas fraction"

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)
        gas.mCO.append(mCO)

        mCH4 = (
            CO2.Y
            * mCO2
            * (H2O.Y * mH2O * sys.P) ** 2
            / (sys.K["K3"] * CH4.Y * (O2.Y * mO2 * sys.P) ** 2)
        )
        gas.mCH4.append(mCH4)

        mN2 = sl.n2_fugacity(
            run.NITROGEN_START, N2.Y, (O2.Y * mO2 * sys.P), sys.P, name=run.N_MODEL
        ) / (N2.Y * sys.P)

        if mN2 > 1:
            exit("Warning: Too high a nitrogen content for the pressure.")
        gas.mN2.append(mN2)

        # Quadratic solve for SO2

        def quadratic():
            a = (SO2.Y / (sys.K["K5"] * O2.Y * mO2)) ** 2 / (S2.Y * sys.P)
            b = 1 + (
                sys.K["K4"]
                * H2O.Y
                * mH2O
                * SO2.Y
                / (sys.K["K5"] * H2S.Y * (O2.Y * mO2) ** 1.5 * sys.P**0.5)
            )
            c = -(1 - mH2 - mO2 - mH2O - mCO - mCO2 - mCH4 - mN2)

            mSO2 = [x for x in np.roots([a, b, c]) if x > 0 and x < 1]
            if mSO2:
                return mSO2[0]
            else:
                exit(
                    "fSO2 is negative. Gas fraction or melt volatile content too high."
                )

        mSO2 = quadratic()
        gas.mSO2.append(mSO2)

        mS2 = (SO2.Y * mSO2) ** 2 / ((sys.K["K5"] * (O2.Y * mO2)) ** 2 * S2.Y * sys.P)
        gas.mS2.append(mS2)

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * SO2.Y * mSO2) / (
            sys.K["K5"] * H2S.Y * (O2.Y * mO2) ** 1.5 * sys.P**0.5
        )
        gas.mH2S.append(mH2S)

    else:  # sulfur has been used as an input
        mS2, mH2O, mO2, mH2 = get_inputs(sys, run, melt, gas, mols)

        assert 1 > (1 - mH2O - mH2 - mO2) > 0, "Some fugacities are negative!"
        assert 1 > mS2 > 0, "S2 fraction wrong"

        mSO2 = (sys.K["K5"] * O2.Y * mO2 * (S2.Y * mS2 * sys.P) ** 0.5) / SO2.Y
        gas.mSO2.append(mSO2)

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * (S2.Y * mS2) ** 0.5) / (
            (O2.Y * mO2) ** 0.5 * H2S.Y
        )
        gas.mH2S.append(mH2S)

        mN2 = sl.n2_fugacity(
            run.NITROGEN_START, N2.Y, (O2.Y * mO2 * sys.P), sys.P, name=run.N_MODEL
        ) / (N2.Y * sys.P)

        if mN2 > 1:
            exit("Warning: Too high a nitrogen content for the pressure.")
        gas.mN2.append(mN2)

        mCH4 = (1 - mH2 - mO2 - mH2O - mS2 - mSO2 - mH2S - mN2) / (
            1
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (CO2.Y * (H2O.Y * mH2O) ** 2)
            )
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (
                    sys.K["K2"]
                    * CO.Y
                    * (O2.Y * mO2 * sys.P) ** 0.5
                    * (H2O.Y * mH2O) ** 2
                )
            )
        )
        gas.mCH4.append(mCH4)

        mCO2 = (sys.K["K3"] * CH4.Y * mCH4 * (O2.Y * mO2) ** 2) / (
            (H2O.Y * mH2O) ** 2 * CO2.Y
        )
        gas.mCO2.append(mCO2)

        # Check graphite saturation
        if run.C_MODEL == "eguchi2018":
            graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * mO2 * sys.P))
            if (CO2.Y * mCO2 * sys.P) > graph_fCO2:
                exit(
                    "Error: Melt is graphite saturated. "
                    "System is overconstrained; please rerun by setting the melt "
                    "graphite content and finding the saturation pressure, "
                    "or stop using eguchi2018 in C_MODEL."
                )

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)
        gas.mCO.append(mCO)

    lst = {
        "o2": mO2,
        "h2": mH2,
        "h2o": mH2O,
        "co": mCO,
        "co2": mCO2,
        "ch4": mCH4,
        "so2": mSO2,
        "s2": mS2,
        "h2s": mH2S,
        "n2": mN2,
    }
    mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

    # WtO
    sys.atomicM["o"] = cnst.m["o"] * (
        (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
        + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        + 2
        * sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
    )

    # WtH
    sys.atomicM["h"] = (
        2
        * cnst.m["h"]
        * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4 + mH2S)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        )
    )

    # WtC
    sys.atomicM["c"] = cnst.m["c"] * (
        (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
        + sl.co2_melt(
            (CO2.Y * mCO2 * sys.P),
            CO2,
            (O2.Y * mO2 * sys.P),
            sys.T,
            sys.P,
            melt,
            name=run.C_MODEL,
        )
        + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        + melt.graph_current
    )

    # WtS
    sys.atomicM["s"] = cnst.m["s"] * (
        (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
        + sl.sulfide_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            name=run.SULFIDE_CAPACITY,
        )
        + sl.sulfate_melt(
            (S2.Y * mS2 * sys.P),
            (O2.Y * mO2 * sys.P),
            sys.P,
            sys.T,
            melt,
            run,
            name=run.SULFATE_CAPACITY,
        )
    )

    # WtN
    sys.atomicM["n"] = cnst.m["n"] * (
        (sys.WgT[0] / sum(mjMj)) * (2 * mN2)
        + sl.n_melt(mN2, (O2.Y * mO2 * sys.P), sys.P, name=run.N_MODEL)
    )
    return sys.atomicM
''',
"solvgas": r'''# ------------------------------------------------------------------------
# solvgas.py

from math import exp

import numpy as np
from scipy.optimize import newton

import evo.constants as cnst

# ------------------------------------------------------------------------
# FUGACITY COEFFICIENTS
# ------------------------------------------------------------------------


def find_Y(P, T, species_list):
    """
    Calculates the fugacity coefficients for all relevant species.

    Parameters
    ----------
    P : float
        Pressure (bar)
    T : float
        Temperature (K)
    species_list : [strings]
        A list of each molecule in the system given as capitalised strings.

    Returns
    -------
    tuple
        a tuple of all the fugacity coefficients, as floats.

    References
    ----------
    Shaw, H.R., & Wones, D.R. (1964). Fugacity coefficients for hydrogen
    gas between 0 degrees and 1000C, for pressures to 3000 atm.
    American Journal of Science.

    Holland, T., & Powell, R. (1991). A Compensated-Redlich-Kwong (CORK)
    equation for volumes and fugacities of CO2 and H2O in the range 1 bar
    to 50 kbar and 100-1600C. Contributions to Mineralogy and Petrology.

    Shi, P., & Saxena, S. K. (1992). Thermodynamic modeling of the C-H-O-S
    fluid system. American Mineralogist.
    """

    h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, ocs_y, n2_y = (
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
    )

    for species in species_list:
        if species == "H2":
            # Shaw and Wones 1964
            logyH2 = (
                P * np.exp((-3.8402 * (T ** (1 / 8))) + 0.541)
                - P**2 * np.exp(-0.1263 * (T**0.5) - 15.98)
                + 300 * np.exp(-0.11901 * T - 5.941) * np.exp(-(P / 300) - 1)
            )

            h2_y = np.exp(logyH2)

        elif species == "CO2":
            # Holland and Powell 1991
            # eq 6, 8, Table 1

            P0 = 5.00  # kbar
            R = cnst.R / 1000  # kJ/mol.K
            a = 741.2 + (-0.10891 * T) + (-3.4203e-4 * T**2)
            b = 3.057  # kJ/kbar/mol
            c = -2.26924e-1 + 7.73793e-5 * T
            d = 1.33790e-2 + (-1.01740e-5 * T)
            P_kb = P / 1000.0  # kbar

            lnf = (
                R * T * np.log(1000.0 * P_kb)
                + b * P_kb
                + (a / (b * (T) ** 0.5))
                * (np.log(R * T + b * P_kb) - np.log(R * T + 2.0 * b * P_kb))
                + (2 / 3) * c * P_kb**1.5
                + (d / 2.0) * P_kb**2.0
            ) / (R * T)

            co2_y = np.exp(lnf) / (P_kb * 1000.0)

        elif species == "N2":
            # Holland and Powell 1991 extension to other species

            Tc = cnst.PTcrit[species][0]
            Pc = cnst.PTcrit[species][1]

            R = cnst.R / 1000  # kJ/mol.K
            a = 5.45963e-5 * (Tc ** (5 / 2) / Pc) - 8.6392e-6 * (Tc ** (3 / 2) / Pc) * T
            b = 9.18301e-4 * (Tc / Pc)  # kJ/kbar/mol
            c = -3.30558e-5 * (Tc / Pc ** (3 / 2)) + T * (2.30524e-6 / Pc ** (3 / 2))
            d = 6.93054e-7 * (Tc / Pc**2) + (-8.38293e-8 / Pc**2) * T
            P_kb = P / 1000.0  # kbar

            lnf = (
                R * T * np.log(1000.0 * P_kb)
                + b * P_kb
                + (a / (b * np.sqrt(T)))
                * (np.log(R * T + b * P_kb) - np.log(R * T + 2.0 * b * P_kb))
                + (2 / 3) * c * P_kb**1.5
                + (d / 2.0) * P_kb**2.0
            ) / (R * T)

            n2_y = np.exp(lnf) / (P_kb * 1000.0)

        elif species == "H2O":
            # Holland and Powell 1991 - above 673 K ONLY

            P0 = 2.0
            P_kb = P / 1000.0  # kbar
            R = cnst.R / 1000  # KJ/ mol.K
            a = (
                1113.4
                + (-0.22291 * (T - 673))
                + (-3.8022e-4 * (T - 673) ** 2)
                + 1.7791e-7 * (T - 673) ** 3
            )
            b = 1.465  # kJ/kbar/mol
            c = -3.025650e-2 + (-5.343144e-6 * T)
            d = -3.2297554e-3 + (2.2215221e-6 * T)

            def find_V(T, P_kb, P0, R, a, b, c, d, guess):
                def volMRK(
                    T, P_kb, R, a, b, guess
                ):  # Temp, Pressure, H2O/CO2, a or agas, initial guess for NR
                    # PL: having trimmed down, there may just be one real root for this,
                    # negating need to newton algoritham, just use a solve.
                    def f(x):
                        return (
                            P_kb * (x**3)
                            - R * T * x**2
                            - (b * R * T + (b**2) * P_kb - a / T**0.5) * x
                            - (a * b / T**0.5)
                        )

                    # where Vmrk is designated by x

                    def df(x):  # Defines the differentiated eq of Vmrk
                        return (
                            3 * P_kb * x**2
                            - 2 * R * T * x
                            - (b * R * T + b**2 * P_kb - a / T**0.5)
                        )

                    # Carries out NR on eq f, with initial guess 'guess'
                    # and to an allowable error of tol
                    return newton(f, guess, fprime=df)

                V_mrk = volMRK(T, P_kb, R, a, b, guess)

                if V_mrk < 0:
                    raise ValueError("find_Y(H2O): MRK Volume is negative")

                if P_kb > P0:
                    V = V_mrk + c * ((P_kb - P0) ** 0.5) + d * (P_kb - P0)
                else:
                    V = V_mrk

                if V < 0:
                    raise ValueError("find_Y(H2O): Total volume is negative")

                return V

            guess = (R * T / P_kb) + b
            Z = (P_kb * find_V(T, P_kb, P0, R, a, b, c, d, guess)) / (R * T)

            if P_kb > P0:
                lnYvirial = (1 / (R * T)) * (
                    ((2 / 3) * c * (P_kb - P0) ** 1.5) + ((d / 2) * (P_kb - P0) ** 2)
                )  # EQ. A.3
            else:
                lnYvirial = 0

            MRKa = a / (b * R * T**1.5)
            MRKb = (b * P_kb) / (R * T)

            h2o_y = np.exp(
                Z - 1 - np.log(Z - MRKb) - MRKa * np.log(1 + (MRKb / Z)) + lnYvirial
            )  # EQ. A.2

        elif species == "SO2":
            # Shi and Saxena 1992

            Pr = P / cnst.PTcrit[species][1]
            P0r = 1.0 / cnst.PTcrit[species][1]
            Tr = T / cnst.PTcrit[species][0]
            A = (
                0.92854
                + 0.43269e-1 * Tr
                + -0.24671 * Tr**-1
                + 0.24999 * Tr**-2
                + -0.53182 * Tr**-3
                + -0.16461e-1 * np.log(Tr)
            )
            B = (
                0.84866e-3
                + -0.18379e-2 * Tr
                + 0.66787e-1 * Tr**-1
                + -0.29427e-1 * Tr**-2
                + 0.29003e-1 * Tr**-3
                + 0.54808e-2 * np.log(Tr)
            )
            C = (
                -0.35456e-3
                + 0.23316e-4 * Tr
                + 0.94159e-3 * Tr**-1
                + -0.81653e-3 * Tr**-2
                + 0.23154e-3 * Tr**-3
                + 0.55542e-4 * np.log(Tr)
            )
            D = 0.0
            integral = (
                A * np.log(Pr / P0r)
                + B * (Pr - P0r)
                + (C / 2.0) * (Pr**2 - P0r**2)
                + (D / 3.0) * (Pr**3 - P0r**3)
            )

            so2_y = (np.exp(integral) + 0) / P

        else:  # For O2, CO, CH4, H2S, S2 and OCS
            # Shi and Saxena 1992

            def q_less(a, b, c, d, e, Tr):
                # when P < 1000, except H2S
                return a + b * Tr**-1 + c * Tr**-1.5 + d * Tr**-3 + e * Tr**-4

            def q_geq(a, b, c, d, e, f, g, h, Tr):
                # when P>= 1000, except H2S
                return (
                    a
                    + b * Tr
                    + c * Tr**-1
                    + d * Tr**2
                    + e * Tr**-2
                    + f * Tr**3
                    + g * Tr**-3
                    + h * np.log(Tr)
                )

            def Z(a, b, c, d, pr, p0r):
                """Returns the SS(1991) integral Z(p, t)/P, eq. 11"""
                return (
                    a * np.log(pr / p0r)
                    + b * (pr - p0r)
                    + (c / 2.0) * (pr**2 - p0r**2)
                    + (d / 3.0) * (pr**3 - p0r**3)
                )

            def Z1000(pcrit, Tr):
                # z at 1000 bar, except H2S
                Pr = 1000 / pcrit
                P0r = 1 / pcrit

                A = 1
                B = (0.09827 / Tr) + (-0.2709 / Tr**3)
                C = (-0.00103 / Tr**1.5) + (0.01427 / Tr**4)
                D = 0.0

                return Z(A, B, C, D, Pr, P0r)  # noqa: B023

            def Z5000(pcrit, Tr):
                # z at 5000 bar, except H2S
                Pr = 5000 / pcrit
                P0r = 1000 / pcrit

                A = 1 + (-0.5917 / Tr**2)
                B = 0.09122 / Tr
                C = (-0.0001416 / Tr**2) + (-0.000002835 * np.log(Tr))
                D = 0.0

                return Z(A, B, C, D, Pr, P0r)  # noqa: B023

            def H2S500(pcrit, Tr):
                # z for H2S at 500 bar
                Pr = 500.0 / pcrit
                P0r = 1.0 / pcrit

                A = q_geq(1.4721, 1.1177, 3.9657, 0, -10.028, 0, 4.5484, -3.8200, Tr)
                B = q_geq(
                    0.16066, 0.10887, 0.29014, 0, -0.99593, 0, -0.18627, -0.45515, Tr
                )
                C = q_geq(
                    -0.28933,
                    -7.0522e-02,
                    0.39828,
                    0,
                    -5.0533e-02,
                    0,
                    0.1176,
                    0.33972,
                    Tr,
                )
                D = 0.0

                return Z(A, B, C, D, Pr, P0r)  # noqa: B023

            def find_Z0(Tr, species):
                # Returns the A, B, C and D parameters for Z, and Z0

                if species != "H2S":
                    if P < 1000.0:
                        P0r = 1.0 / cnst.PTcrit[species][1]

                        A = q_less(1, 0, 0, 0, 0, Tr)
                        B = q_less(0, 0.09827, 0, -0.2709, 0, Tr)
                        C = q_less(0, 0, -0.00103, 0, 0.01427, Tr)
                        D = 0
                        Z0 = np.log(1.0)

                    elif P == 1000.0:
                        P0r = 1.0 / cnst.PTcrit[species][1]

                        A = 0
                        B = 0
                        C = 0
                        D = 0
                        Z0 = Z1000(cnst.PTcrit[species][1], Tr)

                    elif P > 1000.0 and P < 5000.0:
                        P0r = 1000.0 / cnst.PTcrit[species][1]

                        A = q_geq(1, 0, 0, 0, -0.5917, 0, 0, 0, Tr)
                        B = q_geq(0, 0, 0.09122, 0, 0, 0, 0, 0, Tr)
                        C = q_geq(0, 0, 0, 0, -0.0001416, 0, 0, -2.835e-06, Tr)
                        D = 0
                        Z0 = Z1000(cnst.PTcrit[species][1], Tr)

                    elif P == 5000.0:
                        P0r = 1000.0 / cnst.PTcrit[species][1]

                        A = 0
                        B = 0
                        C = 0
                        D = 0
                        Z0 = Z5000(cnst.PTcrit[species][1], Tr) + Z1000(
                            cnst.PTcrit[species][1], Tr
                        )

                    else:  # P > 5000 bar
                        P0r = 5000.0 / cnst.PTcrit[species][1]

                        A = q_geq(2.0614, 0, 0, 0, -2.235, 0, 0, -0.3941, Tr)
                        B = q_geq(0, 0, 0.05513, 0, 0.03934, 0, 0, 0, Tr)
                        C = q_geq(0, 0, -1.894e-06, 0, -1.109e-05, 0, -2.189e-05, 0, Tr)
                        D = q_geq(0, 0, 5.053e-11, 0, 0, -6.303e-21, 0, 0, Tr)
                        Z0 = Z5000(cnst.PTcrit[species][1], Tr) + Z1000(
                            cnst.PTcrit[species][1], Tr
                        )

                elif species == "H2S":
                    if P < 500.0:
                        P0r = 1.0 / cnst.PTcrit[species][1]

                        A = q_geq(
                            1.4721, 1.1177, 3.9657, 0, -10.028, 0, 4.5484, -3.8200, Tr
                        )
                        B = q_geq(
                            0.16066,
                            0.10887,
                            0.29014,
                            0,
                            -0.99593,
                            0,
                            -0.18627,
                            -0.45515,
                            Tr,
                        )
                        C = q_geq(
                            -0.28933,
                            -7.0522e-02,
                            0.39828,
                            0,
                            -5.0533e-02,
                            0,
                            0.1176,
                            0.33972,
                            Tr,
                        )
                        D = 0.0
                        Z0 = np.log(1.0)

                    elif P == 500.0:
                        P0r = 1.0 / cnst.PTcrit[species][1]

                        A = 0.0
                        B = 0.0
                        C = 0.0
                        D = 0.0
                        Z0 = H2S500(cnst.PTcrit["H2S"][1], Tr)

                    elif P > 500.0:
                        P0r = 500.0 / cnst.PTcrit["H2S"][1]

                        A = q_geq(
                            0.59941,
                            -1.557e-03,
                            0.04525,
                            0,
                            0.36687,
                            0,
                            -0.79248,
                            0.26058,
                            Tr,
                        )
                        B = q_geq(
                            2.2545e-02,
                            1.7473e-03,
                            0.048253,
                            0,
                            -0.01989,
                            0,
                            0.032794,
                            -0.010985,
                            Tr,
                        )
                        C = q_geq(
                            5.7375e-04,
                            -2.0944e-06,
                            -0.0011894,
                            0,
                            0.0014661,
                            0,
                            -0.00075605,
                            -0.00027985,
                            Tr,
                        )
                        D = 0.0
                        Z0 = H2S500(cnst.PTcrit["H2S"][1], Tr)

                return A, B, C, D, P0r, Z0

            def y(P, T, species):
                """
                Main equation for calculating fugacity coefficients for a
                single `species` using Shi & Saxena 1992.

                Parameters
                ----------
                P : float
                    Pressure (bar)
                T : float
                    Temperature (K)
                species : string
                    The species name

                Returns
                -------
                float
                    fugacity coefficient for `species`.
                """

                # only calibrated to 1 bar, they start increasing gamma again < 1 bar
                # which should happen - gases become more ideal at low pressure.
                if P < 1.0:
                    return 1

                Tr = T / cnst.PTcrit[species][0]
                Pr = P / cnst.PTcrit[species][1]

                A, B, C, D, P0r, Z0 = find_Z0(Tr, species)

                intZ = Z(A, B, C, D, Pr, P0r)  # noqa: B023

                return np.exp(intZ + Z0) / P

            if species == "O2":
                o2_y = y(P, T, "O2")
            elif species == "CO":
                co_y = y(P, T, "CO")
            elif species == "CH4":
                ch4_y = y(P, T, "CH4")
            elif species == "S2":
                s2_y = y(P, T, "S2")
            elif species == "H2S":
                h2s_y = y(P, T, "H2S")
            elif species == "OCS":
                ocs_y = y(P, T, "OCS")
            # elif species == 'N2':
            #     n2_y = y(P, T, 'N2')

    return h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, n2_y, ocs_y


def set_Y(sys, molecules):
    """
    Sets the fugacity coefficient for each Molecule instance in the system

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    molecules : list of Molecule classes
        A list of the active Molecule class instances. Order-sensitive.
    """
    if sys.run.GAS_SYS == "OH":
        H2O, O2, H2 = molecules
        H2O.Y, O2.Y, H2.Y = find_Y(sys.P, sys.T, sys.SC)[:3]

    elif sys.run.GAS_SYS == "COH":
        H2O, O2, H2, CO, CO2, CH4 = molecules
        H2O.Y, O2.Y, H2.Y, CO.Y, CO2.Y, CH4.Y = find_Y(sys.P, sys.T, sys.SC)[:6]

    elif sys.run.GAS_SYS == "SOH":
        H2O, O2, H2, S2, SO2, H2S = molecules
        H2O.Y, O2.Y, H2.Y, S2.Y, SO2.Y, H2S.Y = (
            find_Y(sys.P, sys.T, sys.SC)[:3] + find_Y(sys.P, sys.T, sys.SC)[6:9]
        )

    elif sys.run.GAS_SYS == "COHS" and sys.OCS is False:
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S = molecules
        H2O.Y, O2.Y, H2.Y, CO.Y, CO2.Y, CH4.Y, S2.Y, SO2.Y, H2S.Y = find_Y(
            sys.P, sys.T, sys.SC
        )[:9]

    elif sys.run.GAS_SYS == "COHS" and sys.OCS is True:
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, OCS = molecules
        H2O.Y, O2.Y, H2.Y, CO.Y, CO2.Y, CH4.Y, S2.Y, SO2.Y, H2S.Y, OCS.Y = (
            find_Y(sys.P, sys.T, sys.SC)[:9] + find_Y(sys.P, sys.T, sys.SC)[-1:]
        )

    elif sys.run.GAS_SYS == "COHSN":
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2 = molecules
        H2O.Y, O2.Y, H2.Y, CO.Y, CO2.Y, CH4.Y, S2.Y, SO2.Y, H2S.Y, N2.Y = find_Y(
            sys.P, sys.T, sys.SC
        )[:10]


# ------------------------------------------------------------------------
# EQUILIBRIUM CONSTANTS
# ------------------------------------------------------------------------


def get_K(sys, molecules):
    """
    Calculates equilibrium constants using Gibbs Free Energy minimisation.

    Pressure independent, calculated on startup for a fixed temperature,
    based on the methods used in Unterborn (2016).

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    molecules : list of Molecule classes
        A list of the active Molecule class instances. Order-sensitive.

    Returns
    -------
    K : dict
        A dictionary of equilibrium constants.

    References
    ----------
    Unterborn, C.T., Dismukes, E.E. & Panero, W.R. (2016) Scaling the Earth:
    A sensitivity analysis of terrestrial exoplanetary interior models. ApJ.
    """

    K = {}  # Temporary store for K values
    del_Gf_dict = {}  # Stores dG of formation values with molecules as key value pairs
    R = cnst.R

    for mol in molecules:
        # Stores the Gibbs energy in a dictionary with names not object identifiers.
        del_Gf_dict[mol.Mol] = mol.get_G(sys.T)

    # K1: H2 + 1/2 O2 -> H2O
    K1_Gf = (
        del_Gf_dict["H2O"] - del_Gf_dict["H2"] - (0.5 * (del_Gf_dict["O2"]))
    ) * 1000  # so in J rather than kJ
    K["K1"] = exp(-K1_Gf / (R * sys.T))
    # K['K1'] = 10**(12510/T - 0.979*log10(T) + 0.483)

    if "CO2" in sys.SC:
        # K2: CO + 1/2 O2 -> CO2
        K2_Gf = (
            (del_Gf_dict["CO2"] - 0.5 * del_Gf_dict["O2"]) - del_Gf_dict["CO"]
        ) * 1000
        K["K2"] = np.exp(-K2_Gf / (R * sys.T))

        # K3: CH4 + 2 O2 -> CO2 + 2 H2O
        K3_Gf = (
            del_Gf_dict["CO2"]
            + 2 * del_Gf_dict["H2O"]
            - del_Gf_dict["CH4"]
            - 2 * del_Gf_dict["O2"]
        ) * 1000
        K["K3"] = np.exp(-K3_Gf / (R * sys.T))

    if "S2" in sys.SC:
        # K4: 0.5 S2 + H2O -> H2S + 0.5 O2
        K4_Gf = (
            del_Gf_dict["H2S"]
            + 0.5 * del_Gf_dict["O2"]
            - 0.5 * del_Gf_dict["S2"]
            - del_Gf_dict["H2O"]
        ) * 1000
        K["K4"] = exp(-K4_Gf / (R * sys.T))

        # K['K4'] = 10**(-8117/T + 0.188*np.log10(T) - 0.352)

        # K5: 0.5 S2 + O2 -> SO2
        K5_Gf = (
            del_Gf_dict["SO2"] - 0.5 * del_Gf_dict["S2"] - del_Gf_dict["O2"]
        ) * 1000
        K["K5"] = np.exp(-K5_Gf / (R * sys.T))

    if sys.OCS is True:
        # K6: OCS + H2O -> CO2 + H2S
        K6_Gf = (
            del_Gf_dict["CO2"]
            + del_Gf_dict["H2S"]
            - del_Gf_dict["OCS"]
            - del_Gf_dict["H2O"]
        ) * 1000
        K["K6"] = np.exp(-K6_Gf / (R * sys.T))

        # k6 = np.exp(0.482 + 16.166e2*(1/T) +
        #  0.081e-3*T - 5.715e3*(1/T**2) - 2.224e-1*np.log(T))

    return K
''',
"plots": r'''import matplotlib.pyplot as plt
import numpy as np


# fO2 and dFMQ
def plot_fo2FMQ(df):
    """Plots the fO2 relative to FMQ against pressure"""
    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
    ax1.plot(df["FMQ"], df["P"])
    ax1.set_yscale("log")
    ax1.set_ylabel("Pressure (bars)")
    ax1.invert_yaxis()
    ax1.set_xlabel(r"$\Delta$ FMQ")

    fo2 = []
    for x in df["fo2"]:
        fo2.append(np.log10(np.exp(x)))

    ax2.plot(fo2, df["P"])
    ax2.set_xlabel("log(10) fO2")

    return fig


# Gas speciation mol
def plot_gasspecies_mol(df):
    """Plots the gas speciation (mole fraction) vs pressure"""

    fig, ax = plt.subplots()
    ax.plot(df["mH2O"], df["P"], c="darkblue", label="H2O")
    ax.plot(df["mH2"], df["P"], c="steelblue", label="H2")
    ax.plot(df["mO2"], df["P"], c="firebrick", label="O2")
    ax.annotate("H2O", xy=[df["mH2O"].iloc[-1], df["P"].iloc[-1]], color="darkblue")
    ax.annotate("H2", xy=[df["mH2"].iloc[-1], df["P"].iloc[-1]], color="steelblue")
    ax.annotate("O2", xy=[df["mO2"].iloc[-1], df["P"].iloc[-1]], color="firebrick")

    ax.plot(df["mCO2"], df["P"], c="saddlebrown", label="CO2")
    ax.plot(df["mCO2"], df["P"], c="darkgreen", label="CO")
    ax.plot(df["mCH4"], df["P"], c="forestgreen", label="CH4")
    ax.annotate("CO2", xy=[df["mCO2"].iloc[-1], df["P"].iloc[-1]], color="saddlebrown")
    ax.annotate("CO", xy=[df["mCO2"].iloc[-1], df["P"].iloc[-1]], color="darkgreen")
    ax.annotate("CH4", xy=[df["mCH4"].iloc[-1], df["P"].iloc[-1]], color="forestgreen")

    ax.plot(df["mSO2"], df["P"], c="maroon", label="SO2")
    ax.plot(df["mS2"], df["P"], c="goldenrod", label="S2")
    ax.plot(df["mH2S"], df["P"], c="pink", label="H2S")
    ax.annotate("SO2", xy=[df["mSO2"].iloc[-1], df["P"].iloc[-1]], color="maroon")
    ax.annotate("S2", xy=[df["mS2"].iloc[-1], df["P"].iloc[-1]], color="goldenrod")
    ax.annotate("H2S", xy=[df["mH2S"].iloc[-1], df["P"].iloc[-1]], color="pink")

    ax.plot(df["mN2"], df["P"], c="grey", label="N2")
    ax.annotate("N2", xy=[df["mN2"].iloc[-1], df["P"].iloc[-1]], color="grey")

    ax.set_xscale("log")
    ax.invert_yaxis()
    ax.set_xlabel("Gas speciation (mol frac)")
    ax.set_ylabel("Pressure (bar)")
    return fig


# Gas speciation wt%


def plot_gasspecies_wt(df):
    """Plots the gas speciation (weight fraction) vs pressure"""

    fig, ax = plt.subplots()

    for g, color in zip(
        ["wH2O", "wH2", "wO2", "wCO2", "wCO", "wCH4", "wSO2", "wS2", "wH2S", "wN2"],
        [
            "darkblue",
            "steelblue",
            "firebrick",
            "saddlebrown",
            "darkgreen",
            "mediumseagreen",
            "maroon",
            "goldenrod",
            "pink",
            "grey",
        ],
        strict=True,
    ):
        ax.plot(df[g], df["P"], c=color, label=g[1:])
        ax.annotate(g[1:], xy=[df[g].iloc[-1], df["P"].iloc[-1]], color=color)

    ax.set_xscale("log")
    ax.invert_yaxis()
    ax.set_xlabel("Gas phase (wt %)")
    ax.set_ylabel("Pressure (bar)")
    return fig


# Melt volatile wt%


def plot_meltspecies(df):
    """Plots the volatile content of the melt vs pressure"""

    fig, ax = plt.subplots()
    for s, color in zip(
        [
            "H2O_melt",
            "H2_melt",
            "CO2_melt",
            "CO_melt",
            "CH4_melt",
            "S2-_melt",
            "S6+_melt",
            "N_melt",
        ],
        [
            "darkblue",
            "steelblue",
            "saddlebrown",
            "darkgreen",
            "mediumseagreen",
            "maroon",
            "pink",
            "grey",
        ],
        strict=True,
    ):
        ax.plot(df[s], df["P"], c=color, label=s.split("_")[0])
        ax.annotate(s.split("_")[0], xy=[df[s].iloc[-1], df["P"].iloc[-1]], color=color)

    ax.set_xscale("log")
    ax.invert_yaxis()
    ax.set_xlabel("Melt volatile content (wt%)")
    ax.set_ylabel("Pressure (bar)")
    return fig


# Exsolved gas mass and volume


def plot_gasfraction(df):
    """Plots the gas volume fraction vs pressure"""

    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
    ax1.plot(df["Gas_wt"], df["P"])
    ax1.invert_yaxis()
    ax1.set_xlabel("Exsolved gas wt%")
    ax1.set_ylabel("Pressure (bar)")
    ax2.plot(df["Exsol_vol%"], df["P"])
    ax2.set_xlabel("Exsolved gas volume %")
    return fig
''',
"sat_pressure": r'''"""
Calculates the pressure of volatile saturation for use either alone,
or for initializing an EVo run.
Using system fO2, and melt H2O, CO2 and S contents, gas partial pressures are
calculated for the initial guess pressure,
then the pressure is iterated until the sum of the partial pressures equals the
overburden pressure.
Once the saturation pressure is calculated, WgT is set to 1e-8 and the atomic masses
are calculated. P is then rounded down to the nearest 1 bar,
and the solver is allowed to find the starting conditions at that pressure.

Using this method, the first and second values in mol fraction lists etc WILL NOT MATCH
(recalculate) as they are for different pressures.
"""

import re
import warnings

import numpy as np
from scipy.optimize import fsolve

import evo.constants as cnst
import evo.conversions as cnvs
import evo.messages as msgs
import evo.solubility_laws as sl
import evo.solvgas as sg


def get_molfrac(P, fugacities, gamma):
    """
    Converts fugacities into mole fractions.

    Parameters
    ----------
    P : float
        Pressure (bar)
    fugacities : tuple of floats
        tuple of all 10 fugacities
    gamma : tuple of floats
        tuple of all 10 fugacity coefficients

    Returns
    -------
    tuple of floats
        tuple of all the relevant mole fractions.
    """

    fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 = fugacities
    h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, n2_y = gamma

    if fco2 == 0 and fs2 == 0:
        return fh2o / (h2o_y * P), fo2 / (o2_y * P), fh2 / (h2_y * P)
    elif fco2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
        )
    elif fs2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
        )
    elif fn2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
        )
    else:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
            fn2 / (n2_y * P),
        )


def p_tot(P, fugacities, gamma):
    """
    Returns the difference between the sum of the gas phase partial pressures,
    and the total system pressure.

    Parameters
    ----------
    P : float
        Total system pressure (bar)
    fugacities : tuple of floats
        tuple of all 10 fugacities
    gamma : tuple of floats
        tuple of all 10 fugacity coefficients

    Returns
    -------
    float
        The difference between the sum of the gas phase partial pressures,
        and the total system pressure.
    """

    fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 = fugacities
    h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, n2_y = gamma

    if fco2 == 0 and fs2 == 0:
        return fh2o / h2o_y + fo2 / o2_y + fh2 / h2_y - P
    elif fco2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            - P
        )
    elif fs2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            - P
        )
    elif fn2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            - P
        )
    else:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            + fn2 / n2_y
            - P
        )


def sat_pressure(run, sys, gas, melt, mols):
    """
    Calculates the volatile saturation pressure.

    The volatile saturation pressure is calculated for a given melt
    volatile content. Atomic masses for the system are then calculated
    and fixed, and the pressure is rounded down for the first pressure
    step using `decompress()`.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class
    mols : {[H2O, O2, H2],
            [H2O, O2, H2, CO, CO2, CH4],
            [H2O, O2, H2, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
        A list of the active Molecule class instances.

    Returns
    -------
    P_sat : float
        The volatile saturation pressure (bar)
    values : list of floats
        The composition of the gas phase at `P_sat`, as mole fractions
    gamma : list of floats
        a list of all calculated fugacity coefficients at `P_sat`, as floats.
    mols : {[H2O, O2, H2],
            [H2O, O2, H2, CO, CO2, CH4],
            [H2O, O2, H2, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
        An updated list of the active Molecule class instances.
    melt.graphite_sat : bool
        If the melt is graphite saturated at `P_sat`, returns True.
    """

    def get_f(P, sys, melt, gamma):
        """
        Returns gas fugacities at the current pressure according to the melt

        Parameters
        ----------
        P : float
            Current pressure
        sys : ThermoSystem class
            Active instance of the ThermoSystem class
        melt : Melt class
            Active instance of the Melt class
        gamma : list of floats
            list of fugacity coefficients

        Returns
        -------
        fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 : tuple of floats
            tuple of all the fugacity coefficients
        """

        O2.Y = gamma[1]

        if run.FO2_buffer_SET:
            fo2 = np.exp(
                cnvs.generate_fo2(sys, sys.run.FO2_buffer_START, sys.run.FO2_buffer, P)
            )

        else:
            fo2 = cnvs.c2fo2(melt.Cm(), sys.T, P * 1e5, sys.run.FO2_MODEL)

        fh2o = sl.h2o_fugacity(run.WTH2O_START, H2O, name=run.H2O_MODEL)
        fh2 = fh2o / (sys.K["K1"] * fo2**0.5)

        if run.WTCO2_SET:
            fco2 = sl.co2_fugacity(
                run.WTCO2_START, CO2, fo2, sys.T, P, melt, name=run.C_MODEL
            )  # This returns graph_fCO2 if graphite saturated.
            fco = fco2 / (sys.K["K2"] * fo2**0.5)
            fch4 = (fco2 * fh2o**2) / (sys.K["K3"] * fo2**2)

        elif run.GRAPHITE_SATURATED:
            melt.graphite_sat = True
            fco2 = sl.graphite_fco2(sys.T, P, fo2)
            fco = fco2 / (sys.K["K2"] * fo2**0.5)
            fch4 = (fco2 * fh2o**2) / (sys.K["K3"] * fo2**2)

        else:
            fco2 = 0
            fco = 0
            fch4 = 0

        if run.SULFUR_SET:
            fs2 = sl.S2_fugacity(
                run.SULFUR_START,
                fo2,
                P,
                sys.T,
                melt,
                run,
                sulfidename=run.SULFIDE_CAPACITY,
                sulfatename=run.SULFATE_CAPACITY,
            )
            fso2 = sys.K["K5"] * fo2 * fs2**0.5
            fh2s = (sys.K["K4"] * fh2o * fs2**0.5) / (fo2**0.5)

        else:
            fs2 = 0
            fso2 = 0
            fh2s = 0

        if run.NITROGEN_SET:
            n2_y = gamma[-1]
            fn2 = sl.n2_fugacity(run.NITROGEN_START, n2_y, fo2, P, name=run.N_MODEL)
        else:
            fn2 = 0

        return fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2

    # start of function ---------------------------------------------------------------

    if run.GAS_SYS == "OH":
        H2O, O2, H2 = mols
    elif run.GAS_SYS == "COH":
        H2O, O2, H2, CO, CO2, CH4 = mols
    elif run.GAS_SYS == "SOH":
        H2O, O2, H2, S2, SO2, H2S = mols
    elif run.GAS_SYS == "COHS":
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S = mols
    elif run.GAS_SYS == "COHSN":
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, _ = mols

    def find_p(P, sys, melt):
        """
        Function to solve iteratively to find P_sat.

        Parameters
        ----------
        P : float
            Pressure (bar)
        sys : ThermoSystem class
            Active instance of the ThermoSystem class
        melt : Melt class
            Active instance of the Melt class

        Returns
        -------
        dP : float
            The difference between the sum of the gas phase partial
            pressures, and the total system pressure.
        """
        gamma = sg.find_Y(P, sys.T, sys.SC)[:10]
        fugacity = get_f(P, sys, melt, gamma)

        dP = p_tot(P, fugacity, gamma)

        return dP

    with warnings.catch_warnings():
        warnings.filterwarnings("error")
        try:
            P_sat = fsolve(find_p, 1.0, args=(sys, melt))
        except RuntimeWarning as err:
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            ) from err

    # store the saturation pressure
    P_sat = float(re.sub(r"[\[\]]", "", np.array_str(P_sat)))
    sys.P = P_sat
    gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
    fugacities = get_f(P_sat, sys, melt, gamma)

    # Reset the system fO2 to match that at the saturation pressure
    if run.FO2_buffer_SET:
        sys.FO2 = cnvs.generate_fo2(
            sys, sys.run.FO2_buffer_START, sys.run.FO2_buffer, sys.P
        )

    else:
        sys.FO2 = np.log(cnvs.c2fo2(melt.Cm(), sys.T, sys.Ppa, sys.run.FO2_MODEL))

    # Updates the fugacity coefficients stored in the molecule classes
    sg.set_Y(sys, mols)

    # set the gas fraction at saturation to 1e-6 wt%
    sys.WgT[0] = 1e-8

    # Find the atomic masses at saturation, enforcing sum(mol fracs) = 1

    if run.GAS_SYS == "OH":
        mO2 = get_molfrac(sys.P, fugacities, gamma)[1]

        mH2 = ((1.0 - mO2) * H2O.Y) / (
            sys.K["K1"] * H2.Y * ((O2.Y * mO2 * sys.P) ** 0.5) + H2O.Y
        )

        mH2O = 1 - mO2 - mH2

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        )

        # WtH(total)
        sys.atomicM["h"] = (
            2
            * cnst.m["h"]
            * (
                (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            )
        )

        sys.atomicM["c"], sys.atomicM["s"], sys.atomicM["n"] = 0.0, 0.0, 0.0

        # add gas speciation to lists to act as initial guess for first 'proper' P step
        lists = [gas.mH2O, gas.mO2, gas.mH2]
        empty_list = [gas.mCO, gas.mCO2, gas.mCH4, gas.mS2, gas.mSO2, gas.mH2S, gas.mN2]
        values = [mH2O, mO2, mH2]

    elif run.GAS_SYS == "COH":
        mH2O, mO2, mH2 = get_molfrac(sys.P, fugacities, gamma)[:3]

        mCH4 = (1 - mH2 - mO2 - mH2O) / (
            1
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (CO2.Y * (H2O.Y * mH2O) ** 2)
            )
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (
                    sys.K["K2"]
                    * CO.Y
                    * (O2.Y * mO2 * sys.P) ** 0.5
                    * (H2O.Y * mH2O) ** 2
                )
            )
        )

        mCO2 = (
            sys.K["K2"]
            * (O2.Y * mO2 * sys.P) ** 0.5
            * CO.Y
            * (1 - mH2 - mO2 - mH2O - mCH4)
        ) / (CO2.Y + sys.K["K2"] * (O2.Y * mO2 * sys.P) ** 0.5 * CO.Y)

        mCO = 1 - mH2O - mH2 - mO2 - mCH4 - mCO2

        if run.GRAPHITE_SATURATED is True:
            melt.graph_current = run.GRAPHITE_START / cnst.m["c"]

        elif run.GRAPHITE_SATURATED is False and melt.graphite_sat is True:
            msgs.graphite_warn_saturation(
                melt, (CO2.Y * mCO2 * sys.P), (O2.Y * mO2 * sys.P), CO2
            )

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "co": mCO, "co2": mCO2, "ch4": mCH4}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        # WtH(total)
        sys.atomicM["h"] = (
            2
            * cnst.m["h"]
            * (
                (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
                + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            )
        )

        # WtC
        sys.atomicM["c"] = cnst.m["c"] * (
            (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
            + sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
            + melt.graph_current
        )

        sys.atomicM["s"] = 0.0

        sys.atomicM["n"] = 0.0

        lists = [gas.mH2O, gas.mO2, gas.mH2, gas.mCO, gas.mCO2, gas.mCH4]
        empty_list = [gas.mS2, gas.mSO2, gas.mH2S, gas.mN2]
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4]

    elif run.GAS_SYS == "SOH":
        mH2O, mO2, mH2 = get_molfrac(sys.P, fugacities, gamma)[:3]

        # Quadratic solve for SO2

        def quadratic():
            a = SO2.Y**2 / (((sys.K["K5"] * O2.Y * mO2) ** 2) * S2.Y * sys.P)
            b = 1 + (
                (sys.K["K4"] * H2O.Y * mH2O * SO2.Y)
                / (sys.K["K5"] * H2S.Y * ((O2.Y * mO2) ** 1.5) * np.sqrt(sys.P))
            )
            c = -(1 - mH2 - mO2 - mH2O)

            mSO2 = [x for x in np.roots([a, b, c]) if x > 0 and x < 1]
            return mSO2[0]

        mSO2 = quadratic()

        mS2 = (SO2.Y * mSO2) ** 2 / ((sys.K["K5"] * O2.Y * mO2) ** 2 * S2.Y * sys.P)

        mH2S = (1 - mO2 - mS2 - mSO2) / (
            1
            + (H2S.Y / (sys.K["K1"] * sys.K["K4"] * H2.Y * (S2.Y * mS2 * sys.P) ** 0.5))
            + (
                (H2S.Y * (O2.Y * mO2) ** 0.5)
                / (H2O.Y * sys.K["K4"] * (S2.Y * mS2) ** 0.5)
            )
        )

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "so2": mSO2, "s2": mS2, "h2s": mH2S}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        )

        # WtH
        sys.atomicM["h"] = (
            2
            * cnst.m["h"]
            * (
                (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + mH2S)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            )
        )

        # WtS
        sys.atomicM["s"] = cnst.m["s"] * (
            (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
            + sl.sulfide_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                name=run.SULFIDE_CAPACITY,
            )
            + sl.sulfate_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                run,
                name=run.SULFATE_CAPACITY,
            )
        )

        sys.atomicM["c"] = 0.0

        sys.atomicM["n"] = 0.0

        lists = [gas.mH2O, gas.mO2, gas.mH2, gas.mS2, gas.mSO2, gas.mH2S]
        empty_list = [gas.mCO, gas.mCO2, gas.mCH4, gas.mN2]
        values = [mH2O, mO2, mH2, mS2, mSO2, mH2S]

    elif run.GAS_SYS == "COHS":
        mH2O, mO2, mH2, mS2, mSO2, mH2S = (
            get_molfrac(sys.P, fugacities, gamma)[:3]
            + get_molfrac(sys.P, fugacities, gamma)[6:]
        )

        mCH4 = (1 - mH2 - mO2 - mH2O - mS2 - mSO2 - mH2S) / (
            1
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (CO2.Y * (H2O.Y * mH2O) ** 2)
            )
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (
                    sys.K["K2"]
                    * CO.Y
                    * (O2.Y * mO2 * sys.P) ** 0.5
                    * (H2O.Y * mH2O) ** 2
                )
            )
        )

        mCO2 = (sys.K["K3"] * CH4.Y * mCH4 * (O2.Y * mO2) ** 2) / (
            (H2O.Y * mH2O) ** 2 * CO2.Y
        )

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)

        if run.GRAPHITE_SATURATED is True:
            melt.graph_current = run.GRAPHITE_START / cnst.m["c"]

        elif run.GRAPHITE_SATURATED is False and melt.graphite_sat is True:
            msgs.graphite_warn_saturation(
                melt, (CO2.Y * mCO2 * sys.P), (O2.Y * mO2 * sys.P), CO2
            )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "so2": mSO2,
            "s2": mS2,
            "h2s": mH2S,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        # WtH
        sys.atomicM["h"] = (
            2
            * cnst.m["h"]
            * (
                (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4 + mH2S)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
                + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            )
        )

        # WtC
        sys.atomicM["c"] = cnst.m["c"] * (
            (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
            + sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
            + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            + melt.graph_current
        )

        # WtS
        sys.atomicM["s"] = cnst.m["s"] * (
            (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
            + sl.sulfide_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                name=run.SULFIDE_CAPACITY,
            )
            + sl.sulfate_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                run,
                name=run.SULFATE_CAPACITY,
            )
        )

        sys.atomicM["n"] = 0.0

        lists = [
            gas.mH2O,
            gas.mO2,
            gas.mH2,
            gas.mCO,
            gas.mCO2,
            gas.mCH4,
            gas.mS2,
            gas.mSO2,
            gas.mH2S,
        ]
        empty_list = [gas.mN2]
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S]

    elif run.GAS_SYS == "COHSN":
        mH2O, mO2, mH2, mS2, mSO2, mH2S, mN2 = (
            get_molfrac(sys.P, fugacities, gamma)[:3]
            + get_molfrac(sys.P, fugacities, gamma)[6:]
        )

        mCH4 = (1 - mH2 - mO2 - mH2O - mS2 - mSO2 - mH2S - mN2) / (
            1
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (CO2.Y * (H2O.Y * mH2O) ** 2)
            )
            + (
                (sys.K["K3"] * CH4.Y * (O2.Y * mO2) ** 2)
                / (
                    sys.K["K2"]
                    * CO.Y
                    * (O2.Y * mO2 * sys.P) ** 0.5
                    * (H2O.Y * mH2O) ** 2
                )
            )
        )

        mCO2 = (sys.K["K3"] * CH4.Y * mCH4 * (O2.Y * mO2) ** 2) / (
            (H2O.Y * mH2O) ** 2 * CO2.Y
        )

        mCO = CO2.Y * mCO2 / (sys.K["K2"] * CO.Y * (O2.Y * mO2 * sys.P) ** 0.5)

        if run.GRAPHITE_SATURATED is True:
            melt.graph_current = run.GRAPHITE_START / cnst.m["c"]

        elif run.GRAPHITE_SATURATED is False and melt.graphite_sat is True:
            msgs.graphite_warn_saturation(
                melt, (CO2.Y * mCO2 * sys.P), (O2.Y * mO2 * sys.P), CO2
            )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "so2": mSO2,
            "s2": mS2,
            "h2s": mH2S,
            "n2": mN2,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        # WtH
        sys.atomicM["h"] = (
            2
            * cnst.m["h"]
            * (
                (sys.WgT[0] / sum(mjMj)) * (mH2O + mH2 + 2 * mCH4 + mH2S)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
                + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            )
        )

        # WtC
        sys.atomicM["c"] = cnst.m["c"] * (
            (sys.WgT[0] / sum(mjMj)) * (mCO + mCO2 + mCH4)
            + sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
            + melt.graph_current
        )

        # WtS
        sys.atomicM["s"] = cnst.m["s"] * (
            (sys.WgT[0] / sum(mjMj)) * (2 * mS2 + mH2S + mSO2)
            + sl.sulfide_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                name=run.SULFIDE_CAPACITY,
            )
            + sl.sulfate_melt(
                (S2.Y * mS2 * sys.P),
                (O2.Y * mO2 * sys.P),
                sys.P,
                sys.T,
                melt,
                run,
                name=run.SULFATE_CAPACITY,
            )
        )

        # WtN
        sys.atomicM["n"] = cnst.m["n"] * (
            (sys.WgT[0] / sum(mjMj)) * (2 * mN2)
            + sl.n_melt(mN2, (O2.Y * mO2 * sys.P), sys.P, name=run.N_MODEL)
        )

        lists = [
            gas.mH2O,
            gas.mO2,
            gas.mH2,
            gas.mCO,
            gas.mCO2,
            gas.mCH4,
            gas.mS2,
            gas.mSO2,
            gas.mH2S,
            gas.mN2,
        ]
        empty_list = []
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S, mN2]

    # add gas speciation to lists to act as initial guess for first 'proper' P step
    for ls, val in zip(lists, values, strict=True):
        ls.append(val)

    for ls in empty_list:
        ls.append(np.float64(0))

    # Round P down to nearest bar ready for solver to find WgT at 'starting' pressure.
    sys.P = cnvs.round_down(sys.P, sys.P_step)
    run.P_START = sys.P

    return P_sat, values, gamma, mols, melt.graphite_sat


def satp_writeout(sys, melt, gas, P, values, gamma, mols, graph_sat=False):
    """
    Returns a string in the same style as the output csv file for the
    saturation pressure.

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    melt : Melt class
        Active instance of the Melt class
    gas : Gas class
        Active instance of the Gas class
    P : float
        The volatile saturation pressure
    values : list of floats
        The composition of the gas phase in mole fractions
    gamma : list of floats
        list of the gas phase fugacity coefficients
    mols : list of Molecule classes
        A list of the active Molecule classes
    graph_sat : bool, optional
        If True, indicates that the melt was graphite saturated at the
        point of volatile saturation, by default False

    Returns
    -------
    string
        A single string to be put at the top of the output csv file.
    """

    if sys.run.GAS_SYS == "OH":
        H2O, _, H2 = mols
        mH2O, mO2, mH2 = tuple(values)
        h2oy, o2y, h2y = gamma[:3]

        wts = cnvs.mols2wts(H2O=mH2O, O2=mO2, H2=mH2)
        empty = ["CO2", "CO", "CH4", "SO2", "S2", "H2S", "N2"]
        for i in empty:
            wts[i] = 0.0
        M = cnvs.mean_mol_wt(H2O=mH2O, O2=mO2, H2=mH2)

    elif sys.run.GAS_SYS == "COH":
        H2O, _, H2, _, CO2, _ = mols
        mH2O, mO2, mH2, mCO, mCO2, mCH4 = tuple(values)
        h2oy, o2y, h2y, coy, co2y, ch4y = gamma[:6]

        wts = cnvs.mols2wts(H2O=mH2O, O2=mO2, H2=mH2, CO=mCO, CO2=mCO2, CH4=mCH4)
        empty = ["SO2", "S2", "H2S", "N2"]
        for i in empty:
            wts[i] = 0.0
        M = cnvs.mean_mol_wt(H2O=mH2O, O2=mO2, H2=mH2, CO=mCO, CO2=mCO2, CH4=mCH4)

    elif sys.run.GAS_SYS == "SOH":
        H2O, _, H2 = mols[:3]
        mH2O, mO2, mH2, mS2, mSO2, mH2S = tuple(values)
        h2oy, o2y, h2y, s2y, so2y, h2sy = gamma[:3] + gamma[6:9]

        wts = cnvs.mols2wts(H2O=mH2O, O2=mO2, H2=mH2, S2=mS2, SO2=mSO2, H2S=mH2S)
        empty = ["CO2", "CO", "CH4", "N2"]
        for i in empty:
            wts[i] = 0.0
        M = cnvs.mean_mol_wt(H2O=mH2O, O2=mO2, H2=mH2, S2=mS2, SO2=mSO2, H2S=mH2S)

    elif sys.run.GAS_SYS == "COHS":
        H2O, _, H2, _, CO2 = mols[:5]
        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S = tuple(values)
        h2oy, o2y, h2y, coy, co2y, ch4y, s2y, so2y, h2sy = gamma[:9]

        wts = cnvs.mols2wts(
            H2O=mH2O,
            O2=mO2,
            H2=mH2,
            CO=mCO,
            CO2=mCO2,
            CH4=mCH4,
            S2=mS2,
            SO2=mSO2,
            H2S=mH2S,
        )
        empty = ["N2"]
        for i in empty:
            wts[i] = 0.0
        M = cnvs.mean_mol_wt(
            H2O=mH2O,
            O2=mO2,
            H2=mH2,
            CO=mCO,
            CO2=mCO2,
            CH4=mCH4,
            S2=mS2,
            SO2=mSO2,
            H2S=mH2S,
        )

    elif sys.run.GAS_SYS == "COHSN":
        H2O, _, H2, _, CO2 = mols[:5]
        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S, mN2 = tuple(values)
        h2oy, o2y, h2y, coy, co2y, ch4y, s2y, so2y, h2sy, n2y = gamma

        wts = cnvs.mols2wts(
            H2O=mH2O,
            O2=mO2,
            H2=mH2,
            CO=mCO,
            CO2=mCO2,
            CH4=mCH4,
            S2=mS2,
            SO2=mSO2,
            H2S=mH2S,
            N2=mN2,
        )
        M = cnvs.mean_mol_wt(
            H2O=mH2O,
            O2=mO2,
            H2=mH2,
            CO=mCO,
            CO2=mCO2,
            CH4=mCH4,
            S2=mS2,
            SO2=mSO2,
            H2S=mH2S,
            N2=mN2,
        )

    rho_melt = melt.rho(P=(P * 1e5))

    GvF = 1 / (
        1
        + ((M * P * 1e5 * (1 - sys.WgT[0])) / (cnst.R * sys.T * rho_melt * sys.WgT[0]))
    )

    rho_gas = ((M / 1000) * P * 1e5) / (cnst.R * sys.T)
    rho_bulk = (rho_gas * GvF) + (rho_melt * (1 - GvF))

    if sys.run.ATOMIC_MASS_SET is False:
        melt_h2o = sys.run.WTH2O_START
    else:
        melt_h2o = (
            sl.h2o_melt(mH2O, H2O, P, name=sys.run.H2O_MODEL, Y=h2oy) * cnst.m["h2o"]
        )
    melt_h2 = sl.h2_melt(mH2, H2, P, melt, name=sys.run.H2_MODEL, Y=h2y) * cnst.m["h2"]

    if (
        sys.run.GAS_SYS == "COHS"
        or sys.run.GAS_SYS == "COH"
        or sys.run.GAS_SYS == "COHSN"
    ):
        if sys.run.ATOMIC_MASS_SET is False and graph_sat is False:
            melt_co2 = sys.run.WTCO2_START
        else:
            melt_co2 = (
                sl.co2_melt(
                    (co2y * mCO2 * P),
                    CO2,
                    (o2y * mO2 * P),
                    sys.T,
                    P,
                    melt,
                    name=sys.run.C_MODEL,
                )
                * cnst.m["co2"]
            )

        if sys.run.CO_MODEL != "None":
            melt_co = (
                sl.co_melt((coy * mCO * P), P, name=sys.run.CO_MODEL) * cnst.m["co"]
            )
        else:
            melt_co = 0.0

        if sys.run.CH4_MODEL != "None":
            melt_ch4 = (
                sl.ch4_melt((ch4y * mCH4 * P), P, name=sys.run.CH4_MODEL)
                * cnst.m["ch4"]
            )
        else:
            melt_ch4 = 0.0

        if graph_sat is True:
            graphite = (
                (sys.atomicM["c"] / cnst.m["c"])
                - (
                    ((wts["CO2"] * sys.WgT[0]) + melt_co2) / cnst.m["co2"]
                    + (wts["CO"] * sys.WgT[0] + melt_co) / cnst.m["co"]
                    + (wts["CH4"] * sys.WgT[0] + melt_ch4) / cnst.m["ch4"]
                )
            ) * cnst.m["c"]
        else:
            graphite = 0.0

    else:
        melt_co2 = 0.0
        melt_co = 0.0
        melt_ch4 = 0.0
        graphite = 0.0

        mCO, mCO2, mCH4 = 0.0, 0.0, 0.0
        coy, co2y, ch4y = 0.0, 0.0, 0.0

    if (
        sys.run.GAS_SYS == "COHS"
        or sys.run.GAS_SYS == "SOH"
        or sys.run.GAS_SYS == "COHSN"
    ):
        fo2 = o2y * mO2 * P
        melt.cm_dry, F = melt.iron_fraction(
            np.log(fo2), ppa=P * 1e5
        )  # Set the Fe2/Fe3 ratio and dry melt chemistry for the sulfide capacity

        # use mS2 to work back and find amount of S2- in melt
        s2_melt = (
            sl.sulfide_melt(
                (s2y * mS2 * P),
                (o2y * mO2 * P),
                P,
                sys.T,
                melt,
                name=sys.run.SULFIDE_CAPACITY,
            )
            * cnst.m["s"]
        )  # sulfide_melt returns *number of moles* not mass
        s6_melt = (
            sl.sulfate_melt(
                (s2y * mS2 * P),
                (o2y * mO2 * P),
                P,
                sys.T,
                melt,
                sys.run,
                name=sys.run.SULFATE_CAPACITY,
            )
            * cnst.m["s"]
        )
        melt_s = s2_melt + s6_melt

    else:
        mS2, mSO2, mH2S = 0, 0, 0
        s2y, so2y, h2sy = 0, 0, 0

        fo2 = o2y * mO2 * P
        melt.cm_dry, F = melt.iron_fraction(np.log(fo2), ppa=P * 1e5)

        melt_s = 0
        s2_melt = 0
        s6_melt = 0

    if sys.run.GAS_SYS == "COHSN":
        if sys.run.ATOMIC_MASS_SET is False:
            melt_n = sys.run.NITROGEN_START
        else:
            melt_n = (
                sl.n_melt(mN2, (o2y * mO2 * P), P, name=sys.run.N_MODEL) * cnst.m["n"]
            )

    else:
        melt_n, mN2, n2y = 0.0, 0.0, 0.0

    atomico = cnst.m["o"] * (
        ((sys.WgT[0] * wts["H2O"]) + melt_h2o) / cnst.m["h2o"]
        + (2 * sys.WgT[0] * wts["O2"]) / cnst.m["o2"]
        + (sys.WgT[0] * wts["CO"] + melt_co) / cnst.m["co"]
        + (2 * ((sys.WgT[0] * wts["CO2"]) + melt_co2)) / cnst.m["co2"]
        + (2 * (sys.WgT[0] * wts["SO2"])) / cnst.m["so2"]
    )

    fo2_buffer = sys.run.FO2_buffer

    data = {
        "P": P,
        fo2_buffer: cnvs.generate_fo2_buffer(sys, (o2y * mO2 * P), P),
        "fo2": fo2,
        "F": F,
        "rho_bulk": rho_bulk,
        "rho_melt": rho_melt,
        "Exsol_vol%": GvF * 100,
        "Gas_wt": sys.WgT[0] * 100,
        "mol_mass": M,
        "mH2O": gas.mH2O[0],
        "mH2": gas.mH2[0],
        "mO2": gas.mO2[0],
        "mCO2": gas.mCO2[0],
        "mCO": gas.mCO[0],
        "mCH4": gas.mCH4[0],
        "mSO2": gas.mSO2[0],
        "mH2S": gas.mH2S[0],
        "mS2": gas.mS2[0],
        "mN2": gas.mN2[0],
        "wH2O": wts["H2O"],
        "wH2": wts["H2"],
        "wO2": wts["O2"],
        "wCO2": wts["CO2"],
        "wCO": wts["CO"],
        "wCH4": wts["CH4"],
        "wSO2": wts["SO2"],
        "wH2S": wts["H2S"],
        "wS2": wts["S2"],
        "wN2": wts["N2"],
        "fH2O": (h2oy * mH2O * P),
        "fH2": (h2y * mH2 * P),
        "fCO2": (co2y * mCO2 * P),
        "fCO": (coy * mCO * P),
        "fCH4": (ch4y * mCH4 * P),
        "fSO2": (so2y * mSO2 * P),
        "fH2S": (h2sy * mH2S * P),
        "fS2": (s2y * mS2 * P),
        "fN2": (n2y * mN2 * P),
        "mCO2/CO": gas.mCO2[0] / gas.mCO[0] if "C" in sys.run.GAS_SYS else np.nan,
        "mCO2/H2O": gas.mCO2[0] / gas.mH2O[0] if "C" in sys.run.GAS_SYS else np.nan,
        "mCO2/SO2": gas.mCO2[0] / gas.mSO2[0] if "S" in sys.run.GAS_SYS else np.nan,
        "mH2S/SO2": gas.mH2S[0] / gas.mSO2[0] if "S" in sys.run.GAS_SYS else np.nan,
        "H2O_melt": melt_h2o * 100,
        "H2_melt": melt_h2 * 100,
        "CO2_melt": melt_co2 * 100,
        "CO_melt": melt_co * 100,
        "CH4_melt": melt_ch4 * 100,
        "graph_melt": graphite * 100,
        "S2-_melt": s2_melt * 100,
        "S6+_melt": s6_melt * 100,
        "Stot_melt": melt_s * 100,
        "N_melt": melt_n * 100,
        "tot_H": sys.atomicM["h"] * 1000000,
        "tot_C": sys.atomicM["c"] * 1000000,
        "tot_O_gas": atomico * 1000000,
        "tot_O": cnvs.atomicM_calc(sys, melt, gas, "o_tot", 0) * 1000000,
        "tot_S": sys.atomicM["s"] * 1000000,
        "tot_N": sys.atomicM["n"] * 1000000,
    }

    return data
''',
"writeout": r'''"""
Writes the results of the run to a file and
contains options to produce a graph of the results.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import evo.conversions as cnvt
import evo.plots as plots
import evo.sat_pressure as sat


def get_data(sys, gas, melt, P):
    """
    Creates the dictionary structure containing output data

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    gas : Gas class
        Active instance of the Gas class
    melt : Melt class
        Active instance of the Melt class
    P : [float]
        Pressure path (bar)
    """

    fo2_buffer = sys.run.FO2_buffer

    data = {
        "P": P,
        fo2_buffer: [
            cnvt.generate_fo2_buffer(sys, np.exp(gas.fo2[i]), P[i])
            for i in range(len(P))
        ],
        "fo2": [np.exp(gas.fo2[i]) for i in range(len(P))],
        "F": melt.F[1:] if not sys.run.SINGLE_STEP else [melt.F[-1]],
        "rho_bulk": sys.rho,
        "rho_melt": melt.rho_store,
        "Exsol_vol%": [(sys.GvF[i] * 100) for i in range(len(P))],
        "Gas_wt": [(sys.WgT[i + 1] * 100) for i in range(len(P))],
        "mol_mass": gas.M,
        "mH2O": gas.mH2O[1:] if not sys.run.SINGLE_STEP else [gas.mH2O[-1]],
        "mH2": gas.mH2[1:] if not sys.run.SINGLE_STEP else [gas.mH2[-1]],
        "mO2": gas.mO2[1:] if not sys.run.SINGLE_STEP else [gas.mO2[-1]],
        "mCO2": gas.mCO2[1:] if not sys.run.SINGLE_STEP else [gas.mCO2[-1]],
        "mCO": gas.mCO[1:] if not sys.run.SINGLE_STEP else [gas.mCO[-1]],
        "mCH4": gas.mCH4[1:] if not sys.run.SINGLE_STEP else [gas.mCH4[-1]],
        "mSO2": gas.mSO2[1:] if not sys.run.SINGLE_STEP else [gas.mSO2[-1]],
        "mH2S": gas.mH2S[1:] if not sys.run.SINGLE_STEP else [gas.mH2S[-1]],
        "mS2": gas.mS2[1:] if not sys.run.SINGLE_STEP else [gas.mS2[-1]],
        "mN2": gas.mN2[1:] if not sys.run.SINGLE_STEP else [gas.mN2[-1]],
        "wH2O": gas.Wt["H2O"],
        "wH2": gas.Wt["H2"],
        "wO2": gas.Wt["O2"],
        "wCO2": gas.Wt["CO2"],
        "wCO": gas.Wt["CO"],
        "wCH4": gas.Wt["CH4"],
        "wSO2": gas.Wt["SO2"],
        "wH2S": gas.Wt["H2S"],
        "wS2": gas.Wt["S2"],
        "wN2": gas.Wt["N2"],
        "fH2O": gas.f["H2O"],
        "fH2": gas.f["H2"],
        "fCO2": gas.f["CO2"],
        "fCO": gas.f["CO"],
        "fCH4": gas.f["CH4"],
        "fSO2": gas.f["SO2"],
        "fH2S": gas.f["H2S"],
        "fS2": gas.f["S2"],
        "fN2": gas.f["N2"],
        "mCO2/CO": (
            np.nan
            if "C" not in sys.run.GAS_SYS
            else (
                [gas.mCO2[i + 1] / gas.mCO[i + 1] for i in range(len(P))]
                if not sys.run.SINGLE_STEP
                else [gas.mCO2[-1] / gas.mCO[-1]]
            )
        ),
        "mCO2/H2O": (
            np.nan
            if "C" not in sys.run.GAS_SYS
            else (
                [gas.mCO2[i + 1] / gas.mH2O[i + 1] for i in range(len(P))]
                if not sys.run.SINGLE_STEP
                else [gas.mCO2[-1] / gas.mH2O[-1]]
            )
        ),
        "mCO2/SO2": (
            np.nan
            if "S" not in sys.run.GAS_SYS
            else (
                [gas.mCO2[i + 1] / gas.mSO2[i + 1] for i in range(len(P))]
                if not sys.run.SINGLE_STEP
                else [gas.mCO2[-1] / gas.mSO2[-1]]
            )
        ),
        "mH2S/SO2": (
            np.nan
            if "S" not in sys.run.GAS_SYS
            else (
                [gas.mH2S[i + 1] / gas.mSO2[i + 1] for i in range(len(P))]
                if not sys.run.SINGLE_STEP
                else [gas.mH2S[-1] / gas.mSO2[-1]]
            )
        ),
        "H2O_melt": [melt.h2o[i] * 100 for i in range(len(P))],
        "H2_melt": [melt.h2[i] * 100 for i in range(len(P))],
        "CO2_melt": [melt.co2[i] * 100 for i in range(len(P))],
        "CO_melt": [melt.co[i] * 100 for i in range(len(P))],
        "CH4_melt": [melt.ch4[i] * 100 for i in range(len(P))],
        "graph_melt": [melt.graphite[i] * 100 for i in range(len(P))],
        "S2-_melt": [melt.sulfide[i] * 100 for i in range(len(P))],
        "S6+_melt": [melt.sulfate[i] * 100 for i in range(len(P))],
        "Stot_melt": [melt.s[i] * 100 for i in range(len(P))],
        "N_melt": [melt.n[i] * 100 for i in range(len(P))],
        "tot_H": [
            cnvt.atomicM_calc(sys, melt, gas, "h", i) * 1000000 for i in range(len(P))
        ],
        "tot_C": [
            cnvt.atomicM_calc(sys, melt, gas, "c", i) * 1000000 for i in range(len(P))
        ],
        "tot_O_gas": [
            cnvt.atomicM_calc(sys, melt, gas, "o", i) * 1000000 for i in range(len(P))
        ],
        "tot_O": [
            cnvt.atomicM_calc(sys, melt, gas, "o_tot", i) * 1000000
            for i in range(len(P))
        ],
        "tot_S": [
            cnvt.atomicM_calc(sys, melt, gas, "s", i) * 1000000 for i in range(len(P))
        ],
        "tot_N": [
            cnvt.atomicM_calc(sys, melt, gas, "n", i) * 1000000 for i in range(len(P))
        ],
    }

    return data


def writeout_file(sys, gas, melt, P, crashed=False, write=True):
    """
    Assembles the results DataFrame and optionally writes it to a CSV file.

    Parameters
    ----------
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    gas : Gas class
        Active instance of the Gas class
    melt : Melt class
        Active instance of the Melt class
    P : list
        Pressure path (bar)
    crashed : bool, False
        If EVo has crashed before reaching its final pressure,
        crashed=True will append 'CRASHED' to the datafile name.
    write : bool, True
        If True, write the results to a CSV file in sys.run.results_folder.
        Set to False when calling from library code to suppress file I/O.
    """

    data = get_data(sys, gas, melt, P)

    df = pd.DataFrame(data)

    if sys.run.FIND_SATURATION is True or sys.run.ATOMIC_MASS_SET is True:
        df_sat = pd.DataFrame(
            sat.satp_writeout(
                sys,
                melt,
                gas,
                sys.sat_conditions[0],
                sys.sat_conditions[1],
                sys.sat_conditions[2],
                sys.sat_conditions[3],
                graph_sat=sys.sat_conditions[4],
            ),
            index=[0],
        )

        df = pd.concat([df_sat, df]).reset_index(drop=True)

    if write:
        filepath = sys.run.results_folder
        if not filepath.exists():
            filepath.mkdir()

        if not crashed:
            file_name = (
                f"dgs_output_{sys.run.COMPOSITION}_{sys.run.GAS_SYS}_{sys.run.RUN_TYPE}"
                f"_{sys.T:.0f}K.csv"
            )
        else:
            file_name = (
                f"dgs_output_CRASHED_{sys.run.COMPOSITION}_{sys.run.GAS_SYS}"
                f"_{sys.run.RUN_TYPE}_{sys.T:.0f}K.csv"
            )

        df.to_csv(filepath / file_name, index=False)

    return df


def writeout_figs(df, filepath, out=None):
    """
    Selects the figures to produce at the end of a successful run.

    If an output file is not used to select the required figures, all
    options are plotted.

    Parameters
    ----------
    df:
        DataFrame containing the output data
    filepath : Path
        Path to the desired output folder
    out : Output class or None
        Active instance of the Output class, or None
    """

    filelist = list(filepath.glob("*.png"))
    # Removes previous files so if output specification is changed
    # there is no confusion as to up to date files.

    for file in filelist:
        file.unlink()

    plot_map = {
        "plot_melt_species": ("melt_species.png", plots.plot_meltspecies),
        "plot_gas_species_wt": ("speciation(wt).png", plots.plot_gasspecies_wt),
        "plot_gas_species_mol": ("speciation(mol).png", plots.plot_gasspecies_mol),
        "plot_gas_fraction": ("exsolved_gas.png", plots.plot_gasfraction),
        "plot_fo2_dFMQ": ("fo2_fmq.png", plots.plot_fo2FMQ),
    }

    chose_plots = {
        k: getattr(out, k) if out is not None else True for k in plot_map.keys()
    }

    for key, (filename, plot_func) in plot_map.items():
        if not chose_plots[key]:
            continue
        fig = plot_func(df)  # Call the plotting function
        fig.savefig(filepath / filename)
        plt.close(fig)  # Ensure the figure is closed to free memory
''',
"messages": r'''"""
Stores warning messages used thought the system to alert the user of an input error
or failure to run to completion, and an explanation.
"""

import sys

import numpy as np

import evo.constants as cnst
import evo.conversions as cnvs
import evo.solubility_laws as sl
import evo.writeout as wt


def open_earlyexit(sys, gas, melt):
    """
    Saves the current run information and prints an error to be used in other functions

    Suitable when open system degassing is being used.

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class
    """
    wt.writeout_file(
        sys,
        gas,
        melt,
        np.arange(sys.run.P_START, sys.P, sys.run.DP_MAX * -1.0, crashed=True),
    )
    print(
        "Error: This run has finished before the assigned P_STOP, "
        "as one or more volatile element is now present in too low "
        "an amount to solve the system with the specified pressure step.\n"
        "Please try reducing the stepsize for a complete run.\n"
        "The run so far has been written out in Output/dgs_output_CRASHED*.csv"
    )


def closed_earlyexit(sys, gas, melt):
    """
    Saves the current run information and prints an error to be used in other functions

    Parameters
    ----------
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class
    """
    wt.writeout_file(sys, gas, melt, sys.P_track, crashed=True)

    print(
        "Error: This run has finished before the assigned P_STOP. \n"
        "The run so far has been written out in Output/dgs_output_CRASHED_*.csv"
    )


def earlyexit(sys, gas, melt, msg):
    """Calls the early exit function and raises an assertion error."""
    if sys.run.RUN_TYPE == "open":
        open_earlyexit(sys, gas, melt)
    elif sys.run.RUN_TYPE == "closed":
        closed_earlyexit(sys, gas, melt)
    raise AssertionError(
        msg
        + " Try reducing the maximum pressure stepsize (DP_MAX) for model convergence."
    )


def runtime_error(sys, gas, melt, msg):
    """Calls the early exit function and raises a runtime error."""
    if sys.run.RUN_TYPE == "open":
        open_earlyexit(sys, gas, melt)
    elif sys.run.RUN_TYPE == "closed":
        closed_earlyexit(sys, gas, melt)
    raise RuntimeError(msg)


def query_yes_no(question, default="yes"):
    """
    Ask a yes/no question via raw_input() and return their answer.

    Parameters
    ----------
    question : string
        A string that is presented to the user.
    default : bool, optional
        The presumed answer if the user just hits <Enter>.
        It must be "yes" (the default), "no" or None (meaning
        an answer is required of the user).

    Returns
    -------
    answer : bool
        True for "yes" or False for "no".
    """
    valid = {"yes": True, "y": True, "ye": True, "no": False, "n": False}
    if default is None:
        prompt = " [y/n] "
    elif default == "yes":
        prompt = " [Y]/n "
    elif default == "no":
        prompt = " y/[N] "
    else:
        raise ValueError("invalid default answer: '%s'" % default)

    while True:
        sys.stdout.write(question + prompt)
        choice = input().lower()
        if default is not None and choice == "":
            return valid[default]
        elif choice in valid:
            return valid[choice]
        else:
            sys.stdout.write("Please respond with 'yes' or 'no' (or 'y' or 'n').\n")


def run_chem_mismatch(mtype, lims, sio2):
    """Checks the SiO2 content and melt descriptor are correct

    If the melt descriptor (e.g. 'basalt') does not match the SiO2 content of the melt,
    function is called to either force the run to continue,
    or allow it to be terminated.

    Parameters
    ----------
    mtype : {'basalt', 'phonolite', 'rhyolite'}
        The melt descriptor.
    lims : List[float, float]
        upper and lower limits on the SiO2 weight fraction associated with
        `mtype`.
    sio2 : float
        Weight fraction of SiO2 in the melt as set in the chemistry input
        file.
    """
    if sio2 < lims[0]:
        print(
            f"Warning: You specified {mtype.upper()} in the env.yaml file, "
            "but the SiO2 content of your melt specified in the chem.yaml "
            f"file is LESS THAN {lims[0]} wt%.\n"
        )
        answer = query_yes_no("Are you sure you want to continue?", default="no")
    elif sio2 > lims[1]:
        print(
            f"Warning: You specified {mtype.upper()} in the env.yaml file, "
            "but the SiO2 content of your melt specified in the chem.yaml file "
            f"is GREATER THAN {lims[1]} wt%.\n"
        )
        answer = query_yes_no("Are you sure you want to continue?", default="no")

    if answer is False:
        exit("Exiting...")


def solubility_temp(T, limits):
    """
    Allows the user to force a run to continue is the T is outside the
    tested limit of a solubility law.

    Parameters
    ----------
    T : float
        Temperature (K)
    limits : list of float
        Upper and lower temperature limits
    """
    print(
        f"Warning: The set temperature of {T:3Ng} K is outside the T-dependent "
        f"solubility limits of {limits[0]} to {limits[1]} K.\n"
    )
    answer = query_yes_no(
        "Would you like to proceed with fixed solubility coefficients?", default="yes"
    )

    if answer is False:
        exit("Exiting...")


def sulfate_warn(s6, sys, gas, melt):
    """
    Finishes a pressure step and ends a run.

    Called if sulfate makes up >10% of the dissolved sulfur content.

    Parameters
    ----------
    s6 : float
        The S6+/S_total ratio
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class

    Raises
    ------
    ValueError
    """
    # Finish off this run
    sys.GvF.append(gas.get_vol_frac(melt))
    sys.rho.append(sys.rho_bulk(melt, gas))
    gas.M.append(
        cnvs.mean_mol_wt(
            H2O=gas.mH2O[-1],
            O2=gas.mO2[-1],
            H2=gas.mH2[-1],
            CO=gas.mCO[-1],
            CO2=gas.mCO2[-1],
            CH4=gas.mCH4[-1],
            S2=gas.mS2[-1],
            SO2=gas.mSO2[-1],
            H2S=gas.mH2S[-1],
        )
    )

    # If sys.P hasn't been initiated yet, fill with the starting pressure
    if not sys.P_track:
        sys.P_track.append(sys.P)

    # Writeout results
    closed_earlyexit(sys, gas, melt)

    raise ValueError(
        f"The sulfate (S6+ speciated as SO4) content of the melt is {s6 * 100:.1f}% "
        "of the total melt S content.\n"
        "As sulfate currently does not exsolve, gas phase sulphur-bearing species "
        "will now be significantly underestimated.\n"
        "Run is terminated, results up to this point have been exported to CSV."
    )


def scss_warn(scss, s, sys, gas, melt):
    """
    Finishes a pressure step and ends a run.

    Called if the melt is sulfide saturated and therefore EVo is no longer valid.

    Parameters
    ----------
    scss : float
        The sulfide content at sulfur saturation (ppm)
    s : float
        The sulfide content of the melt (ppm)
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class
    """
    # Finish off this run
    sys.GvF.append(gas.get_vol_frac(melt))
    sys.rho.append(sys.rho_bulk(melt, gas))
    gas.M.append(
        cnvs.mean_mol_wt(
            H2O=gas.mH2O[-1],
            O2=gas.mO2[-1],
            H2=gas.mH2[-1],
            CO=gas.mCO[-1],
            CO2=gas.mCO2[-1],
            CH4=gas.mCH4[-1],
            S2=gas.mS2[-1],
            SO2=gas.mSO2[-1],
            H2S=gas.mH2S[-1],
        )
    )

    # If sys.P hasn't been initiated yet, fill with the starting pressure
    if not sys.P_track:
        sys.P_track.append(sys.P)

    # Writeout results

    closed_earlyexit(sys, gas, melt)

    exit(
        "You have reached sulphide saturation, this code is no longer valid. "
        f"SCSS: {scss:.1g} ppm, Sulfide in melt: {s:.1g} ppm\n"
        "Run is terminated, results up to this point have been exported to CSV."
    )


def vol_setup_standard(run, sys):
    """
    Checks that the conditions set in the environment file match the run
    type specified.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    """

    # check some volatile species aren't being set twice as wt% and fugacity
    if (run.FH2O_SET is True and run.WTH2O_SET is True) or (
        run.FCO2_SET is True and run.WTCO2_SET is True
    ):
        exit(
            "Warning: Please only use one option to set H2O or CO2. "
            "Set either the Wt% or fugacity value to False."
        )

    if (run.WTCO2_SET is True and run.GRAPHITE_SATURATED is True) or (
        run.FCO2_SET is True and run.GRAPHITE_SATURATED is True
    ):
        exit(
            "Warning: Please only use one setting for CO2. "
            "Set two from fCO2, WTCO2 and graphite_saturated to False."
        )

    if run.ATOMIC_MASS_SET is True:
        lst = {
            "FH2_SET": run.FH2_SET,
            "FH2O_SET": run.FH2O_SET,
            "FCO2_SET": run.FCO2_SET,
            "WTH2O_SET": run.WTH2O_SET,
            "WTCO2_SET": run.WTCO2_SET,
            "SULFUR_SET": run.SULFUR_SET,
            "NITROGEN_SET": run.NITROGEN_SET,
            "GRAPHITE_SATURATED": run.GRAPHITE_SATURATED,
        }
        if any(setting is True for setting in lst.values()):
            print(
                "Warning: Volatile contents are being set as fugacities/melt "
                "contents while ATOMIC_MASS_SET is true.\n"
            )

            answer = query_yes_no(
                "Would you like to proceed with the atomic mass set, "
                "setting all fugacities and melt contents to False?",
                default="yes",
            )

            if answer is True:
                for setting in lst.keys():
                    setattr(run, setting, False)

            if answer is False:
                answer = query_yes_no(
                    "Would you like to set ATOMIC_MASS_SET to False and continue "
                    "with the fugacities/melt contents already set?",
                    default="yes",
                )

                if answer is True:
                    run.ATOMIC_MASS_SET = False

                if answer is False:
                    exit("Exiting...")

    elif run.GAS_SYS == "OH":
        lst = [
            run.FH2O_SET,
            run.WTH2O_SET,
            run.WTCO2_SET,
            run.FCO2_SET,
            run.SULFUR_SET,
        ]  # PL: Make this a yes/no option to change setup to sat point finder with H2O.
        for opt in lst:
            if opt is True:
                exit(
                    "Error: This OH system setup cannot run with H2O, CO2 or "
                    "sulphur contents. To run with H2O and find the saturation "
                    "point of the system, set run.FIND_SATURATION to True."
                )

        if run.FH2_SET is True and sys.FO2:
            exit(
                "Warning: Please only set one of either fH2, fO2, or the "
                "properties of FeO and Fe2O3 for the OH system."
            )

        elif run.FH2_SET is False and sys.FO2 is None:
            exit(
                "Error: For the OH system, please set either the initial fH2 or "
                "fO2 in the env.yaml file, or provide proportions of FeO and Fe2O3 "
                "via the chem.yaml file."
            )

    elif run.GAS_SYS == "COH" or run.GAS_SYS == "SOH":
        if run.FCO2_SET is True or run.WTCO2_SET is True or run.SULFUR_SET is True:
            exit(
                "Warning: CO2 and S cannot be used as an initial condition for "
                f"the {run.GAS_SYS} system. \n"
                "Please set 2 initial conditions from either fH2, H2O (as fH2O or "
                "wt% H2O) or fO2 (either directly or via FeO/Fe2O3 proportions).\n"
                "To run with the complete melt volatile content and find the "
                "saturation point of the system, set run.FIND_SATURATION to True."
            )

        lst = [run.FH2O_SET, run.WTH2O_SET, run.FH2_SET]  # Allowable setup conditions
        true = 0
        if sys.FO2:  # final allowable condition, but present as a boolean value
            true += 1
        for ele in lst:
            if ele is True:
                true += 1
        if true > 2 or true <= 1:
            exit(
                "Error: Please set 2 initial conditions from either fH2, H2O "
                "(as fH2O or wt% H2O) or fO2 (either directly or via FeO/Fe2O3 "
                "proportions).\n"
                f"There are currently {true} values set."
            )

    elif run.GAS_SYS == "COHS" or run.GAS_SYS == "COHSN":
        lst = [
            run.FH2O_SET,
            run.WTH2O_SET,
            run.FH2_SET,
            run.FCO2_SET,
            run.WTCO2_SET,
            run.SULFUR_SET,
            run.GRAPHITE_SATURATED,
        ]
        true = 0
        if sys.FO2:
            true += 1
        for ele in lst:
            if ele is True:
                true += 1
        if true > 3 or true <= 2:
            exit(
                "Error: Please set 3 initial conditions from either fH2, H2O (as fH2O "
                "or wt% H2O), CO2 (as fCO2, wt% CO2 or graphite), S (as wt% S) or fO2 "
                "(either directly or via FeO/Fe2O3 proportions).\n"
                f"There are currently {true} values set."
            )

        if run.GAS_SYS == "COHSN":
            if run.NITROGEN_SET is False:
                exit(
                    "Error: Please set a value for the melt nitrogen content using "
                    "NITROGEN_SET/START."
                )

        elif run.NITROGEN_SET is True:  # and system is COHS
            print(
                "Warning: Nitrogen is being set in the melt, but the system is running "
                "with COHS (i.e. no nitrogen).\n"
            )

            answer = query_yes_no(
                "Would you like to proceed with the COHS system, "
                "ignoring the melt nitrogen content?",
                default="yes",
            )

            if answer is False:
                exit("Exiting...")


def vol_setup_saturation(run, sys):
    """
    Checks that the conditions set in the environment file match the
    requirements for finding the volatile saturation point.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    """

    # check some volatile species aren't being set twice as wt% and fugacity
    if (run.FH2O_SET is True and run.WTH2O_SET is True) or (
        run.FCO2_SET is True and run.WTCO2_SET is True
    ):
        exit(
            "Warning: Please only use one option to set H2O or CO2. "
            "Set either the Wt% or fugacity value to False."
        )

    if (run.WTCO2_SET is True and run.GRAPHITE_SATURATED is True) or (
        run.FCO2_SET is True and run.GRAPHITE_SATURATED is True
    ):
        exit(
            "Warning: Please only use one setting for CO2. Set two from fCO2, "
            "WTCO2 and graphite_saturated to False."
        )

    if run.GAS_SYS == "OH":
        lst = [
            run.WTCO2_SET,
            run.FCO2_SET,
            run.SULFUR_SET,
        ]  # PL: Make this a yes/no option to change setup to sat point finder with H2O.
        for opt in lst:
            if opt is True:
                exit(
                    "Error: This OH system setup cannot run with CO2 or sulphur "
                    "contents. Set all CO2 and S values to False."
                )

        if run.FH2_SET is True or run.FH2O_SET is True:
            exit(
                "Error: Please only set fO2 and melt volatile contents to find the "
                "volatile saturation point. "
                "Turn all fugacities to False in the env.yaml file."
            )

        if not (run.WTH2O_SET is True and sys.FO2):
            exit(
                "Error: To find the saturation point for the OH system, "
                "please set both the melt water content, "
                "and either fO2 in the env.yaml file, "
                "or provide proportions of FeO and Fe2O3 via the chem.yaml file."
            )

    elif run.GAS_SYS == "COH":
        lst = [run.FH2O_SET, run.FCO2_SET, run.FH2_SET, run.SULFUR_SET]
        for opt in lst:
            if opt is True:
                exit(
                    "Error: Please only set fO2 and melt H2O & CO2 contents to find "
                    "the volatile saturation point. "
                    "Turn all fugacities and sulfur contents to False."
                )

        if run.WTCO2_SET is True or run.GRAPHITE_SATURATED is True:
            C = True
        else:
            C = False

        if not (run.WTH2O_SET is True and C is True and sys.FO2):
            exit(
                "Error: To find the saturation point for the COH system, "
                "please set both water and CO2 contents in the melt, "
                "and either fO2 in the env.yaml file, "
                "or provide proportions of FeO and Fe2O3 via the chem.yaml file."
            )

    elif run.GAS_SYS == "SOH":
        lst = [run.FH2O_SET, run.FCO2_SET, run.WTCO2_SET, run.FH2_SET]
        for opt in lst:
            if opt is True:
                exit(
                    "Error: Please only set fO2 and melt H2O & sulphur contents to "
                    "find the volatile saturation point. Turn all fugacities and CO2 "
                    "contents to False."
                )

        if not (run.WTH2O_SET is True and run.SULFUR_SET is True and sys.FO2):
            exit(
                "Error: To find the saturation point for the SOH system, "
                "please set both water and sulphur contents in the melt, "
                "and either fO2 in the env.yaml file, "
                "or provide proportions of FeO and Fe2O3 via the chem.yaml file."
            )

    elif run.GAS_SYS == "COHS" or run.GAS_SYS == "COHSN":
        if run.WTCO2_SET is True or run.GRAPHITE_SATURATED is True:
            C = True

        lst = [run.WTH2O_SET, C, run.SULFUR_SET, sys.FO2]

        if not all(lst):  # if any are false
            exit(
                "Error: To find the volatile saturation point please make sure the melt"
                " H2O, CO2 and sulfur contents are set and 'True' in env.yaml."
            )

        if any([run.FH2O_SET, run.FH2_SET, run.FCO2_SET]):  # if any are true
            exit(
                "Error: Please only set melt weight fractions, not gas phase "
                "fugacities, when searching for the saturation point."
            )

        if run.GAS_SYS == "COHSN":
            if run.NITROGEN_SET is False:
                exit(
                    "Error: Please set a value for the melt nitrogen content "
                    "using NITROGEN_SET/START."
                )

        elif run.NITROGEN_SET is True:  # and system is COHS
            print(
                "Warning: Nitrogen is being set in the melt, but the system is running "
                "with COHS (i.e. no nitrogen).\n"
            )

            answer = query_yes_no(
                "Would you like to proceed with the COHS system, ignoring the melt "
                "nitrogen content?",
                default="yes",
            )

            if answer is False:
                exit("Exiting...")
            else:
                run.NITROGEN_SET = False


def graphite_warn(melt):
    """
    Warns the melt will be graphite saturated under the initial conditions.

    Warns of graphite saturation without knowledge of the size of the
    graphite reservoir if melt_co2 is set but melt is graphite saturated.

    Parameters
    ----------
    melt : Melt class
        Active instance of the Melt class

    Returns
    -------
    bool
        Returns True if the graphite content has now been set.
    """
    print("Warning: The melt is graphite saturated.\n")

    answer = query_yes_no(
        "Does the CO2 content set include the graphite content "
        "(melt_co2 + melt_co3 + melt_graphite)?",
        default="yes",
    )

    if answer is True:
        print("Continuing to find melt carbonate/graphite split.")
        return False

    elif answer is False:
        answer = query_yes_no(
            "Do you know the graphite content of the melt?", default="no"
        )

        if answer is True:
            while True:
                try:
                    graph = float(
                        input(
                            "Please enter the graphite content of the melt as a weight "
                            "fraction: "
                        )
                    )
                except ValueError:
                    print("Re-enter value as a float: ")
                    continue

                if graph < 0:
                    print("Sorry, your response must not be negative.")
                    continue
                elif graph > 1:
                    print("Please enter as a weight fraction < 1")
                else:
                    break

            melt.graphite_sat = True
            melt.graph_current = graph / cnst.m["c"]

            print(
                "By continuing, the melt CO2 content will be over_written based on the "
                "melt CO2 content at graphite saturation."
            )
            answer = query_yes_no(
                f"Continue with a melt graphite content of {graph * 100} wt%?",
                default="yes",
            )

            if answer is True:
                return True
            else:
                exit("Exiting...")

        elif answer is False:
            exit(
                "Error: Cannot continue with this setup. Please either set up the melt "
                "so that the total amount of C in the system is known "
                "(using atomic mass set),\nor use "
                "a different solubility law for CO2 which neglects graphite saturation."
            )


def graphite_warn_saturation(melt, fCO2, fO2, CO2):
    """
    Warns of graphite saturation & provides options to continue

    Parameters
    ----------
    melt : Melt class
        Active instance of the Melt class
    fCO2 : float
        CO2 fugacity
    fO2 : float
        Absolute oxygen fugacity
    CO2 : Molecule class
        CO2 instance of the Molecule class
    """

    print(
        "Warning: Melt is graphite saturated at the volatile saturation pressure.\n"
        "Run cannot proceed without knowing the size of the graphite reservoir at "
        "volatile saturation.\n"
    )

    answer = query_yes_no(
        (
            "Does the melt CO2 content at saturation include the graphite content "
            "(melt_co2 + melt_co3 + melt_graphite)?"
        ),
        default="yes",
    )

    if answer is True:
        co2_melt = (
            sl.co2_melt(
                fCO2, CO2, fO2, melt.sys.T, melt.sys.P, melt, name=melt.sys.run.C_MODEL
            )
            * cnst.m["co2"]
        )
        graph_melt = ((melt.sys.run.WTCO2_START - co2_melt) / cnst.m["co2"]) * cnst.m[
            "c"
        ]  # wt frac graphite in melt
        melt.graph_current = graph_melt / cnst.m["c"]

        print(
            f"The original melt CO2 content was {melt.sys.run.WTCO2_START * 100} wt%."
        )
        print(
            "At graphite-present volatile saturation, the melt CO2 content is "
            f"{co2_melt * 100} wt% and the graphite content is {graph_melt * 100} wt%."
        )

        answer = query_yes_no("Do you wish to continue?", default="yes")

        if answer is True:
            print("Continuing with calculation")

        elif answer is False:
            exit("Exiting...")

    elif answer is False:
        exit(
            "Error: Run cannot proceed without knowing the size of the graphite "
            "reservoir at volatile saturation.\nExiting..."
        )


def fo2_model_mismatch(feot, run):
    """
    Warns that the model selected for Ferric/Ferrous -> fO2 is out of
    bounds for the amount of Fe in the melt.

    Parameters
    ----------
    feot : float
        total weight fraction of Fe in the melt
    run : RunDef class
        Active instance of the RunDef class
    """

    names = {"r2013": "Righter et al. (2013)", "kc1991": "Kress & Carmicheal (1991)"}

    if feot <= 15.0:
        print(
            f"Warning: The approximate FeO(t) content of the melt is {feot:.4g} wt%. "
            f"This is out of the {names[run.FO2_MODEL]} model bounds."
        )
        answer = query_yes_no(
            "Would you like to switch to the Kress & Carmicheal (1991) fO2 model?",
            default="yes",
        )

        if answer is True:
            run.FO2_MODEL = "kc1991"
        elif answer is False:
            print(
                f"Warning: Continuing with the {names[run.FO2_MODEL]} fO2 model outside"
                " of calibration range."
            )

    elif feot > 15.0:
        print(
            f"Warning: The approximate FeO(t) content of the melt is {feot:.4g} wt%."
            "The Righter et al. (2013) fO2 model is more appropriate for high FeO melts"
        )
        answer = query_yes_no(
            "Would you like to switch to the Righter et al. (2013) fO2 model?",
            default="yes",
        )

        if answer is True:
            print("\n")
            run.FO2_MODEL = "r2013"
        elif answer is False:
            print(
                f"Warning: Continuing with the {names[run.FO2_MODEL]} fO2 model outside"
                " of calibration range.\n"
            )
''',
"fixed_weights": r'''"""
Code to find the saturation point of a melt based on it's volatile content given
as atomic weights of species in the melt. so not quite total atomic mass in the system..

Sets WgT to 1e-6 wt% and solves to find the speciation of both the gas phase and
the speciation within the melt.
"""

import warnings

import numpy as np
from scipy.optimize import fsolve, root

import evo.constants as cnst
import evo.conversions as cnvs
import evo.solubility_laws as sl
import evo.solvgas as sg


def get_molfrac(P, fugacities, gamma):
    """
    Converts fugacities into mole fractions.

    Parameters
    ----------
    P : float
        Pressure (bar)
    fugacities : tuple of floats
        tuple of all 10 fugacities
    gamma : tuple of floats
        tuple of all 10 fugacity coefficients

    Returns
    -------
    tuple of floats
        tuple of all the relevant mole fractions.
    """

    fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 = fugacities
    h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, n2_y = gamma

    if fco2 == 0 and fs2 == 0:
        return fh2o / (h2o_y * P), fo2 / (o2_y * P), fh2 / (h2_y * P)
    elif fco2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
        )
    elif fs2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
        )
    elif fn2 == 0:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
        )
    else:
        return (
            fh2o / (h2o_y * P),
            fo2 / (o2_y * P),
            fh2 / (h2_y * P),
            fco / (co_y * P),
            fco2 / (co2_y * P),
            fch4 / (ch4_y * P),
            fs2 / (s2_y * P),
            fso2 / (so2_y * P),
            fh2s / (h2s_y * P),
            fn2 / (n2_y * P),
        )


def p_tot(P, fugacities, gamma):
    """
    Returns the difference between the sum of the gas phase partial pressures,
    and the total system pressure.

    Parameters
    ----------
    P : float
        Total system pressure (bar)
    fugacities : tuple of floats
        tuple of all 10 fugacities
    gamma : tuple of floats
        tuple of all 10 fugacity coefficients

    Returns
    -------
    float
        The difference between the sum of the gas phase partial pressures,
        and the total system pressure.
    """

    fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 = fugacities
    h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y, so2_y, h2s_y, n2_y = gamma

    if fco2 == 0 and fs2 == 0:
        return fh2o / h2o_y + fo2 / o2_y + fh2 / h2_y - P
    elif fco2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            - P
        )
    elif fs2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            - P
        )
    elif fn2 == 0:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            - P
        )
    else:
        return (
            fh2o / h2o_y
            + fo2 / o2_y
            + fh2 / h2_y
            + fco / co_y
            + fco2 / co2_y
            + fch4 / ch4_y
            + fs2 / s2_y
            + fso2 / so2_y
            + fh2s / h2s_y
            + fn2 / n2_y
            - P
        )


def sat_pressure(run, sys, gas, melt, mols):
    """
    Calculates the volatile saturation pressure.

    The volatile saturation pressure is calculated where the volatile
    content has been given as a set of elemental weight fractions.
    Initial guesses are made for the masses of each species (H2O etc)
    dissolved in the melt, then the saturation pressure for those guesses is
    calculated, and elemental masses implied at that pressure are calculated.
    This is iterated to find the volatile content of the melt and volatile
    saturation pressure which satisfies the provided element mass data.

    The volatile saturation pressure is stored, and the pressure is rounded
    down for the first pressure step using `decompress()`.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    gas : Gas class
        The active instance of the Gas class
    melt : Melt class
        The active instance of the Melt class
    mols : {[H2O, O2, H2], [H2O, O2, H2, CO, CO2, CH4], [H2O, O2, H2, S2, SO2, H2S],
    [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
    [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
        A list of the active Molecule class instances.

    Returns
    -------
    P_sat : float
        The volatile saturation pressure (bar)
    values : list of floats
        The composition of the gas phase at `P_sat`, as mole fractions
    gamma : list of floats
        a list of all calculated fugacity coefficients at `P_sat`, as floats.
    mols : {[H2O, O2, H2], [H2O, O2, H2, CO, CO2, CH4], [H2O, O2, H2, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
            [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
        An updated list of the active Molecule class instances.
    melt.graphite_sat : bool
        If the melt is graphite saturated at `P_sat`, returns True.
    """

    if run.GAS_SYS == "OH":
        H2O, O2, H2 = mols
        sys.atomicM["h"] = run.ATOMIC_H / 1e6

    elif run.GAS_SYS == "COH":
        H2O, O2, H2, CO, CO2, _ = mols
        sys.atomicM["c"] = run.ATOMIC_C / 1e6
        sys.atomicM["h"] = run.ATOMIC_H / 1e6

    elif run.GAS_SYS == "SOH":
        H2O, O2, H2 = mols[:3]
        sys.atomicM["h"] = run.ATOMIC_H / 1e6
        sys.atomicM["s"] = run.ATOMIC_S / 1e6

    elif run.GAS_SYS == "COHS":
        H2O, O2, H2, CO, CO2 = mols[:5]
        sys.atomicM["c"] = run.ATOMIC_C / 1e6
        sys.atomicM["h"] = run.ATOMIC_H / 1e6
        sys.atomicM["s"] = run.ATOMIC_S / 1e6

    elif run.GAS_SYS == "COHSN":
        H2O, O2, H2, CO, CO2 = mols[:5]
        sys.atomicM["c"] = run.ATOMIC_C / 1e6
        sys.atomicM["h"] = run.ATOMIC_H / 1e6
        sys.atomicM["n"] = run.ATOMIC_N / 1e6
        sys.atomicM["s"] = run.ATOMIC_S / 1e6

    def get_f(P, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma):
        """
        Returns gas fugacities at the current pressure according to the melt

        Parameters
        ----------
        P : float
            Current pressure
        melt_h2o : float
            Current guess for the weight fraction of water in the melt
        melt_co2 : float
            Current guess for the weight fraction of CO2 in the melt
        melt_s : float
            Current guess for the weight fraction of sulfur in the melt
        melt_n : float
            Current guess for the weight fraction of nitrogen in the melt
        sys : ThermoSystem class
            Active instance of the ThermoSystem class
        melt : Melt class
            Active instance of the Melt class
        gamma : list of floats
            list of fugacity coefficients

        Returns
        -------
        fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2 : tuple of floats
            tuple of all the fugacity coefficients
        """

        O2.Y = gamma[1]

        if run.FO2_buffer_SET:
            fo2 = np.exp(
                cnvs.generate_fo2(sys, sys.run.FO2_buffer_START, sys.run.FO2_buffer, P)
            )

        else:
            fo2 = cnvs.c2fo2(melt.Cm(), sys.T, P * 1e5, sys.run.FO2_MODEL)

        fh2o = sl.h2o_fugacity(melt_h2o, H2O, name=run.H2O_MODEL)
        fh2 = fh2o / (sys.K["K1"] * fo2**0.5)

        if melt_co2 != 0.0:
            fco2 = sl.co2_fugacity(melt_co2, CO2, fo2, sys.T, P, melt, name=run.C_MODEL)
            fco = fco2 / (sys.K["K2"] * fo2**0.5)
            fch4 = (fco2 * fh2o**2) / (sys.K["K3"] * fo2**2)
        else:
            fco2 = 0
            fco = 0
            fch4 = 0

        if melt_s != 0.0:
            fs2 = sl.S2_fugacity(
                melt_s,
                fo2,
                P,
                sys.T,
                melt,
                run,
                sulfidename=run.SULFIDE_CAPACITY,
                sulfatename=run.SULFATE_CAPACITY,
            )
            fso2 = sys.K["K5"] * fo2 * fs2**0.5
            fh2s = (sys.K["K4"] * fh2o * fs2**0.5) / (fo2**0.5)

        else:
            fs2 = 0
            fso2 = 0
            fh2s = 0

        if melt_n != 0.0:
            n2_y = gamma[-1]
            fn2 = sl.n2_fugacity(melt_n, n2_y, fo2, P, name=run.N_MODEL)
        else:
            fn2 = 0

        return fh2o, fo2, fh2, fco, fco2, fch4, fs2, fso2, fh2s, fn2

    def find_p(P, melt_h2o, melt_co2, melt_s, melt_n, sys, melt):
        """
        Function to solve iteratively to find P_sat.

        Parameters
        ----------
        P : float
            Pressure (bar)
        melt_h2o : float
            Current guess for the weight fraction of water in the melt
        melt_co2 : float
            Current guess for the weight fraction of CO2 in the melt
        melt_s : float
            Current guess for the weight fraction of sulfur in the melt
        melt_n : float
            Current guess for the weight fraction of nitrogen in the melt
        sys : ThermoSystem class
            Active instance of the ThermoSystem class
        melt : Melt class
            Active instance of the Melt class

        Returns
        -------
        dP : float
            The difference between the sum of the gas phase partial
            pressures, and the total system pressure.
        """

        gamma = sg.find_Y(P, sys.T, sys.SC)[:10]
        fugacity = get_f(P, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        dP = p_tot(P, fugacity, gamma)
        return dP

    def fixed_weights_oh(guesses: list[float]):
        """
        Returns the difference between calc'd element masses and fixed values

        func(x) = 0, as required for use with a non-linear solver.

        Parameters
        ----------
        guesses : list of 1 float
            initial guess for the melt water content

        Returns
        -------
        list of floats
            The difference between the total mass of H in the system
            fixed by the initial conditions, and that calculated using the
            `guess` value for the mass fraction of water in the melt.
        """
        melt_h2o = guesses[0]
        melt_co2, melt_n, melt_s = 0.0, 0.0, 0.0

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore")
                P_sat = fsolve(
                    find_p, 1.0, args=(melt_h2o, melt_co2, melt_s, melt_n, sys, melt)
                )
        except RuntimeWarning as err:
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            ) from err

        # Check p_sat hasn't returned 1.0
        if P_sat == 1.0:
            exit("Saturation solver has failed :(. Exiting...")

        P_sat = P_sat[0]
        sys.P = P_sat

        gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
        fugacities = get_f(P_sat, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        h2o_y, o2_y, h2_y = gamma[:3]
        O2.Y = o2_y

        mH2O, mO2, mH2 = get_molfrac(P_sat, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]
        N = 1e-8 / sum(mjMj)

        return [
            (
                N * (mH2O + mH2)
                + sl.h2_melt(mH2, H2, P_sat, melt, name=run.H2_MODEL, Y=h2_y)
                + sl.h2o_melt(mH2O, H2O, P_sat, name=run.H2O_MODEL, Y=h2o_y)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"]))
        ]

    def fixed_weights_coh(guesses: list[float]):
        """
        Returns the difference between calc'd element masses and fixed values

        func(x) = 0, as required for use with a non-linear solver.

        Parameters
        ----------
        guesses : list of 2 floats
            initial guess for the melt water, CO2 content

        Returns
        -------
        list of floats
            The difference between the total mass of each element in the
            system fixed by the initial conditions, and that calculated
            using the `guess` values. (dH, dC)
        """
        melt_h2o, melt_co2 = guesses[0], guesses[1]
        melt_n, melt_s = 0.0, 0.0

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore")
                P_sat = fsolve(
                    find_p, 1.0, args=(melt_h2o, melt_co2, melt_s, melt_n, sys, melt)
                )
        except RuntimeWarning as err:
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            ) from err

        # Check p_sat hasn't returned 1.0
        if P_sat == 1.0:
            exit("Saturation solver has failed :(. Exiting...")

        P_sat = P_sat[0]
        sys.P = P_sat

        gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
        fugacities = get_f(P_sat, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y = gamma[:6]
        O2.Y = o2_y

        mH2O, mO2, mH2, mCO, mCO2, mCH4 = get_molfrac(P_sat, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "co": mCO, "co2": mCO2, "ch4": mCH4}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]
        N = 1e-8 / sum(mjMj)

        if melt.graphite_sat is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys,
                    melt,
                    P_sat,
                    CO2,
                    mCO,
                    mCO2,
                    mCH4,
                    mO2,
                    co2_y,
                    co_y,
                    ch4_y,
                    o2_y,
                    N,
                )
                / cnst.m["c"]
            )

        elif melt.graphite_sat is False:
            melt.graph_current = 0

        return [
            (
                N * (mH2O + mH2 + 2 * mCH4)
                + sl.h2_melt(mH2, H2, P_sat, melt, name=run.H2_MODEL, Y=h2_y)
                + sl.h2o_melt(mH2O, H2O, P_sat, name=run.H2O_MODEL, Y=h2o_y)
                + 2 * sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"])),
            (
                N * (mCO + mCO2 + mCH4)
                + sl.co2_melt(
                    (co2_y * mCO2 * P_sat),
                    CO2,
                    (o2_y * mO2 * P_sat),
                    sys.T,
                    P_sat,
                    melt,
                    name=run.C_MODEL,
                )
                + sl.co_melt((co_y * mCO * P_sat), P_sat, name=run.CO_MODEL)
                + sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
                + melt.graph_current
            )
            - (sys.atomicM["c"] / cnst.m["c"]),
        ]

    def fixed_weights_soh(guesses: list[float]):
        """
        Returns the difference between calc'd element masses and fixed values

        func(x) = 0, as required for use with a non-linear solver.

        Parameters
        ----------
        guesses : list of 2 floats
            initial guess for the melt water, sulfur content

        Returns
        -------
        list of floats
            The difference between the total mass of each element in the
            system fixed by the initial conditions, and that calculated
            using the `guess` values. (dH, dS)
        """
        melt_h2o, melt_s = guesses[0], guesses[1]
        melt_n, melt_co2 = 0.0, 0.0

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore")
                P_sat = fsolve(
                    find_p, 1.0, args=(melt_h2o, melt_co2, melt_s, melt_n, sys, melt)
                )
        except RuntimeWarning as err:
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            ) from err

        # Check p_sat hasn't returned 1.0
        if P_sat == 1.0:
            exit("Saturation solver has failed :(. Exiting...")

        P_sat = P_sat[0]
        sys.P = P_sat

        gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
        fugacities = get_f(P_sat, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        h2o_y, o2_y, h2_y, _, _, _, s2_y = gamma[:7]
        O2.Y = o2_y

        mH2O, mO2, mH2, mS2, mSO2, mH2S = get_molfrac(P_sat, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "s2": mS2, "so2": mSO2, "h2s": mH2S}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]
        N = 1e-8 / sum(mjMj)

        return [
            (
                N * (mH2O + mH2 + mH2S)
                + sl.h2_melt(mH2, H2, P_sat, melt, name=run.H2_MODEL, Y=h2_y)
                + sl.h2o_melt(mH2O, H2O, P_sat, name=run.H2O_MODEL, Y=h2o_y)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"])),
            (
                N * (mSO2 + mH2S + 2 * mS2)
                + sl.sulfide_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    name=run.SULFIDE_CAPACITY,
                )
                + sl.sulfate_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    run,
                    name=run.SULFATE_CAPACITY,
                )
            )
            - (sys.atomicM["s"] / cnst.m["s"]),
        ]

    def fixed_weights_cohs(guesses: list[float]):
        """
        Returns the difference between calc'd element masses and fixed values

        func(x) = 0, as required for use with a non-linear solver.

        Parameters
        ----------
        guesses : list of 3 floats
            initial guess for the melt water, CO2, sulfur content

        Returns
        -------
        list of floats
            The difference between the total mass of each element in the
            system fixed by the initial conditions, and that calculated
            using the `guess` values. (dH, dS, dC)
        """
        melt_h2o, melt_co2, melt_s = guesses[0], guesses[1], guesses[2]
        melt_n = 0.0

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore")
                P_sat = fsolve(
                    find_p, 1.0, args=(melt_h2o, melt_co2, melt_s, melt_n, sys, melt)
                )
        except RuntimeWarning as err:
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            ) from err

        # Check p_sat hasn't returned 1.0
        if P_sat == 1.0:
            exit("Saturation solver has failed :(. Exiting...")

        P_sat = P_sat[0]
        sys.P = P_sat

        gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
        fugacities = get_f(P_sat, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y = gamma[:7]
        O2.Y = gamma[1]

        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S = get_molfrac(
            P_sat, fugacities, gamma
        )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "s2": mS2,
            "so2": mSO2,
            "h2s": mH2S,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]
        N = 1e-8 / sum(mjMj)

        if melt.graphite_sat is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys,
                    melt,
                    P_sat,
                    CO2,
                    mCO,
                    mCO2,
                    mCH4,
                    mO2,
                    co2_y,
                    co_y,
                    ch4_y,
                    o2_y,
                    N,
                )
                / cnst.m["c"]
            )

        elif melt.graphite_sat is False:
            melt.graph_current = 0

        return [
            (
                N * (mH2O + mH2 + mH2S + 2 * mCH4)
                + sl.h2_melt(mH2, H2, P_sat, melt, name=run.H2_MODEL, Y=h2_y)
                + sl.h2o_melt(mH2O, H2O, P_sat, name=run.H2O_MODEL, Y=h2o_y)
                + 2 * sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"])),
            (
                N * (mSO2 + mH2S + 2 * mS2)
                + sl.sulfide_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    name=run.SULFIDE_CAPACITY,
                )
                + sl.sulfate_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    run,
                    name=run.SULFATE_CAPACITY,
                )
            )
            - (sys.atomicM["s"] / cnst.m["s"]),
            (
                N * (mCO + mCO2 + mCH4)
                + sl.co2_melt(
                    (co2_y * mCO2 * P_sat),
                    CO2,
                    (o2_y * mO2 * P_sat),
                    sys.T,
                    P_sat,
                    melt,
                    name=run.C_MODEL,
                )
                + sl.co_melt((co_y * mCO * P_sat), P_sat, name=run.CO_MODEL)
                + sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
                + melt.graph_current
            )
            - (sys.atomicM["c"] / cnst.m["c"]),
        ]

    def fixed_weights_cohsn(guesses: list[float]):
        """
        Returns the difference between calc'd element masses and fixed values

        func(x) = 0, as required for use with a non-linear solver.

        Parameters
        ----------
        guesses : list of 4 floats
            initial guess for the melt water, CO2, sulfur, nitrogen content

        Returns
        -------
        list of floats
            The difference between the total mass of each element in the
            system fixed by the initial conditions, and that calculated
            using the `guess` values. (dN, dH, dS, dC)
        """
        melt_h2o, melt_co2, melt_s, melt_n = (
            guesses[0],
            guesses[1],
            guesses[2],
            guesses[3],
        )

        P_sat, msg, ier, _ = fsolve(
            find_p,
            1.0,
            args=(melt_h2o, melt_co2, melt_s, melt_n, sys, melt),
            full_output=True,
        )

        if ier != 1:
            print(msg)
            raise RuntimeError(
                "Failed to find saturation pressure; solver not converging."
            )

        # Check p_sat hasn't returned 1.0
        if P_sat == 1.0:
            exit("Saturation solver has failed :(. Exiting...")

        P_sat = P_sat[0]
        sys.P = P_sat

        gamma = sg.find_Y(P_sat, sys.T, sys.SC)[:10]
        fugacities = get_f(P_sat, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

        h2o_y, o2_y, h2_y, co_y, co2_y, ch4_y, s2_y = gamma[:7]
        O2.Y = o2_y

        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S, mN2 = get_molfrac(
            P_sat, fugacities, gamma
        )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "s2": mS2,
            "so2": mSO2,
            "h2s": mH2S,
            "n2": mN2,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]
        N = 1e-8 / sum(mjMj)

        if melt.graphite_sat is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys,
                    melt,
                    P_sat,
                    CO2,
                    mCO,
                    mCO2,
                    mCH4,
                    mO2,
                    co2_y,
                    co_y,
                    ch4_y,
                    o2_y,
                    N,
                )
                / cnst.m["c"]
            )

        elif melt.graphite_sat is False:
            melt.graph_current = 0

        return [
            (
                N * (2 * mN2)
                + sl.n_melt(mN2, (o2_y * mO2 * P_sat), P_sat, name=run.N_MODEL)
            )
            - (sys.atomicM["n"] / cnst.m["n"]),
            (
                N * (mH2O + mH2 + mH2S + 2 * mCH4)
                + sl.h2_melt(mH2, H2, P_sat, melt, name=run.H2_MODEL, Y=h2_y)
                + sl.h2o_melt(mH2O, H2O, P_sat, name=run.H2O_MODEL, Y=h2o_y)
                + 2 * sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"])),
            (
                N * (mSO2 + mH2S + 2 * mS2)
                + sl.sulfide_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    name=run.SULFIDE_CAPACITY,
                )
                + sl.sulfate_melt(
                    (s2_y * mS2 * P_sat),
                    (o2_y * mO2 * P_sat),
                    P_sat,
                    sys.T,
                    melt,
                    run,
                    name=run.SULFATE_CAPACITY,
                )
            )
            - (sys.atomicM["s"] / cnst.m["s"]),
            (
                N * (mCO + mCO2 + mCH4)
                + sl.co2_melt(
                    (co2_y * mCO2 * P_sat),
                    CO2,
                    (o2_y * mO2 * P_sat),
                    sys.T,
                    P_sat,
                    melt,
                    name=run.C_MODEL,
                )
                + sl.co_melt((co_y * mCO * P_sat), P_sat, name=run.CO_MODEL)
                + sl.ch4_melt((ch4_y * mCH4 * P_sat), P_sat, name=run.CH4_MODEL)
                + melt.graph_current
            )
            - (sys.atomicM["c"] / cnst.m["c"]),
        ]

    # start of function -----------------------------------------------------------

    if (
        run.FO2_buffer_SET is True
        and cnvs.generate_fo2_buffer(
            sys,
            np.exp(cnvs.generate_fo2(sys, run.FO2_buffer_START, run.FO2_buffer, 300)),
            300,
            buffer_choice="IW",
        )
        < -1.5
    ):
        guess_h2o = ((run.ATOMIC_H / 1e6) / cnst.m["h"]) * 0.2 * cnst.m["h2o"]
    else:
        guess_h2o = (((run.ATOMIC_H / 1e6) / cnst.m["h"]) * cnst.m["h2o"]) / 2

    if run.GAS_SYS == "COH" or run.GAS_SYS == "COHS" or run.GAS_SYS == "COHSN":
        if (
            run.C_MODEL == "eguchi2018"
            and cnvs.generate_fo2_buffer(
                sys,
                np.exp(
                    cnvs.generate_fo2(sys, run.FO2_buffer_START, run.FO2_buffer, 300)
                ),
                300,
                buffer_choice="IW",
            )
            < -1.5
        ):
            guess_co2 = 1e-5
            melt.graphite_sat = True
        elif run.CO_MODEL != "None" and run.CH4_MODEL != "None":
            guess_co2 = ((run.ATOMIC_C / 1e6) / cnst.m["c"]) * cnst.m["co2"] * 0.2
        else:
            guess_co2 = ((run.ATOMIC_C / 1e6) / cnst.m["c"]) * cnst.m["co2"]
    else:
        melt_co2 = 0

    if run.GAS_SYS == "SOH" or run.GAS_SYS == "COHS" or run.GAS_SYS == "COHSN":
        guess_s = run.ATOMIC_S / 1e6
    else:
        melt_s = 0

    if run.GAS_SYS == "COHSN":
        guess_n = run.ATOMIC_N / 1e6
    else:
        melt_n = 0

    try:
        if run.GAS_SYS == "OH":
            melt_h2o = fsolve(fixed_weights_oh, [guess_h2o])[0]
        elif run.GAS_SYS == "COH":
            melt_h2o, melt_co2 = fsolve(fixed_weights_coh, [guess_h2o, guess_co2])
        elif run.GAS_SYS == "SOH":
            melt_h2o, melt_s = fsolve(fixed_weights_soh, [guess_h2o, guess_s])
        elif run.GAS_SYS == "COHS":
            melt_h2o, melt_co2, melt_s = fsolve(
                fixed_weights_cohs, [guess_h2o, guess_co2, guess_s]
            )
        elif run.GAS_SYS == "COHSN":
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("error")
                    melt_h2o, melt_co2, melt_s, melt_n = fsolve(
                        fixed_weights_cohsn, [guess_h2o, guess_co2, guess_s, guess_n]
                    )
            except RuntimeWarning:
                print("Convergence struggling; trying Levenberg-Marquardt method")
                sol = root(
                    fixed_weights_cohsn,
                    [guess_h2o, guess_co2, guess_s, guess_n],
                    method="lm",
                )
                melt_h2o, melt_co2, melt_s, melt_n = sol["x"]

    except Exception as err:
        raise RuntimeError(
            "Failed to find saturation pressure; solver not converging."
        ) from err

    P_sat = sys.P
    gamma = sg.find_Y(sys.P, sys.T, sys.SC)[:10]
    fugacities = get_f(sys.P, melt_h2o, melt_co2, melt_s, melt_n, sys, melt, gamma)

    # Reset the system fO2 to match that at the saturation pressure
    if run.FO2_buffer_SET:
        sys.FO2 = cnvs.generate_fo2(
            sys, sys.run.FO2_buffer_START, sys.run.FO2_buffer, sys.P
        )

    else:
        sys.FO2 = np.log(cnvs.c2fo2(melt.Cm(), sys.T, sys.Ppa, sys.run.FO2_MODEL))

    # Updates the fugacity coefficients stored in the molecule classes
    sg.set_Y(sys, mols)

    # set the gas fraction at saturation to 1e-6 wt%
    sys.WgT[0] = 1e-8

    # Find the atomic masses at saturation sum(mol fracs) = 1 is enforced

    if run.GAS_SYS == "OH":
        mH2O, mO2, mH2 = get_molfrac(sys.P, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        )

        sys.atomicM["c"], sys.atomicM["s"], sys.atomicM["n"] = 0.0, 0.0, 0.0

        # add gas speciation to lists to act as initial guess for first 'proper' P step
        lists = [gas.mH2O, gas.mO2, gas.mH2]
        empty_list = [gas.mCO, gas.mCO2, gas.mCH4, gas.mS2, gas.mSO2, gas.mH2S, gas.mN2]
        values = [mH2O, mO2, mH2]

    elif run.GAS_SYS == "COH":
        mH2O, mO2, mH2, mCO, mCO2, mCH4 = get_molfrac(sys.P, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "co": mCO, "co2": mCO2, "ch4": mCH4}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        sys.atomicM["s"] = 0.0

        sys.atomicM["n"] = 0.0

        lists = [gas.mH2O, gas.mO2, gas.mH2, gas.mCO, gas.mCO2, gas.mCH4]
        empty_list = [gas.mS2, gas.mSO2, gas.mH2S, gas.mN2]
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4]

    elif run.GAS_SYS == "SOH":
        mH2O, mO2, mH2, mS2, mSO2, mH2S = get_molfrac(sys.P, fugacities, gamma)

        lst = {"o2": mO2, "h2": mH2, "h2o": mH2O, "so2": mSO2, "s2": mS2, "h2s": mH2S}
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
        )

        sys.atomicM["c"], sys.atomicM["n"] = 0.0, 0.0

        lists = [gas.mH2O, gas.mO2, gas.mH2, gas.mS2, gas.mSO2, gas.mH2S]
        empty_list = [gas.mCO, gas.mCO2, gas.mCH4, gas.mN2]
        values = [mH2O, mO2, mH2, mS2, mSO2, mH2S]

    elif run.GAS_SYS == "COHS":
        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S = get_molfrac(
            sys.P, fugacities, gamma
        )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "so2": mSO2,
            "s2": mS2,
            "h2s": mH2S,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        sys.atomicM["n"] = 0.0

        lists = [
            gas.mH2O,
            gas.mO2,
            gas.mH2,
            gas.mCO,
            gas.mCO2,
            gas.mCH4,
            gas.mS2,
            gas.mSO2,
            gas.mH2S,
        ]
        empty_list = [gas.mN2]
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S]

    elif run.GAS_SYS == "COHSN":
        mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S, mN2 = get_molfrac(
            sys.P, fugacities, gamma
        )

        lst = {
            "o2": mO2,
            "h2": mH2,
            "h2o": mH2O,
            "co": mCO,
            "co2": mCO2,
            "ch4": mCH4,
            "so2": mSO2,
            "s2": mS2,
            "h2s": mH2S,
            "n2": mN2,
        }
        mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

        # WtO
        sys.atomicM["o"] = cnst.m["o"] * (
            (sys.WgT[0] / sum(mjMj)) * (mH2O + 2 * mO2 + 2 * mCO2 + mCO + 2 * mSO2)
            + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            + 2
            * sl.co2_melt(
                (CO2.Y * mCO2 * sys.P),
                CO2,
                (O2.Y * mO2 * sys.P),
                sys.T,
                sys.P,
                melt,
                name=run.C_MODEL,
            )
            + sl.co_melt((CO.Y * mCO * sys.P), sys.P, name=run.CO_MODEL)
        )

        lists = [
            gas.mH2O,
            gas.mO2,
            gas.mH2,
            gas.mCO,
            gas.mCO2,
            gas.mCH4,
            gas.mS2,
            gas.mSO2,
            gas.mH2S,
            gas.mN2,
        ]
        empty_list = []
        values = [mH2O, mO2, mH2, mCO, mCO2, mCH4, mS2, mSO2, mH2S, mN2]

    # add gas speciation to lists to act as initial guess for first 'proper' P step
    for ls, val in zip(lists, values, strict=True):
        ls.append(val)

    for ls in empty_list:
        ls.append(np.float64(0))

    # Round P down to nearest bar ready for solver to find WgT at 'starting' pressure.
    sys.P = cnvs.round_down(sys.P, sys.P_step)
    run.P_START = sys.P

    return P_sat, values, gamma, mols, melt.graphite_sat
''',
"dgs_classes": r'''# dgs_classes

import copy
import importlib.resources
import inspect
import itertools
import re
from pathlib import PosixPath
from typing import Literal, Optional

import numpy as np
from pydantic import BaseModel, ConfigDict, field_validator, model_validator

import evo.constants as cnst
import evo.conversions as cnvs
import evo.density as density
import evo.init_cons as ic
import evo.messages as msgs
import evo.solubility_laws as sl
import evo.solvgas as sg


class RunDef(BaseModel):
    """
    Stores the parameters of the degassing run

    Attributes
    ----------
    COMPOSITION : {'basalt', 'phonolite', 'rhyolite'}
        Name of magma composition
    RUN_TYPE : {'closed', 'open'}
        Run type - closed keeps gas and melt in contact, open removes an
        aliquot of the gas after every pressure step
    SINGLE_STEP : bool, default = False
        Choose whether to calculate the composition at a single pressure
    FIND_SATURATION : bool, default = False
        Choose to calculate the volatile saturation point rather than set
        a starting pressure
    GAS_SYS : {'OH', 'COH', 'SOH', 'COHS', 'COHSN'}
        Sets the volatile elements of interest
    FE_SYSTEM : bool, default = True
        If True, oxygen is exchanged between iron in the melt and the gas
    S_SAT_WARN : bool, default = True
        If true, stops a run is the melt becomes sulfide saturated.
    T_START : float
        System temperature, K
    P_START : float
        Starting pressure, bar
    DP_MIN : float
        Minimum pressure step size, bar. Must be equal to `DP_MAX` if
        `RUN_TYPE` = 'open'
    DP_MAX : float
        Maximum pressure step size, bar. Must be equal to `DP_MIN` if
        `RUN_TYPE` = 'open'
    MASS : float
        System mass, grams
    WgT : float
        Initial gas mass fraction (not percent). Not required if either
        `FIND_SATURATION` or `ATOMIC_MASS_SET` = True.
    LOSS_FRAC : float, <1
        The fraction of the total gas in the system to be removed if
        `RUN_TYPE` = 'open'.
    DENSITY_MODEL : {'spera2000'}
        Select a model to calculate the magma density with.
    FO2_MODEL : {'kc1991', 'r2013'}
        Select the model to calculate the relationship between the ferric/
        ferrous ratio and the melt fO2.
    FMQ_MODEL : {'frost1991'}
        Select the model to calculate fO2 mineral buffers with
    H2O_MODEL : {'burguisser2015'}
        Select the H2O solubility model
    H2_MODEL : {'gaillard2003', 'burguisser2015'}
        Select the H2 solubility model
    C_MODEL : {'burguisser2015', 'eguchi2018'}
        Select the CO2 solubility model
    CO_MODEL : {None, 'armstrong2015'}
        Select the CO solubility model
    CH4_MODEL : {None, 'ardia2013'}
        Select the CH4 solubility model
    SULFIDE_CAPACITY : {'oneill2020', 'oneill2002'}
        Select the sulfide capacity model
    SULFATE_CAPACITY : {'nash2019'}
        Select the sulfate capacity model
    SCSS : {'liu2007'}
        Select the model to calculate the sulfide capacity at sulfide
        saturation
    N_MODEL : {'libourel2003'}
        Select the N2 solubility model
    FO2_buffer : {'FMQ', 'IW', 'NNO'}
        Mineral buffer to cite fO2 relative to
    FO2_buffer_START : float
        System fO2 relative to `FO2_buffer`.
    FO2_START : float
        Set the absolute fO2. Do not use in conjunction with `FO2_buffer_START`.
    FH2_START : float
        Set the H2 fugacity as a starting condition (bar)
    FH2O_START : float
        Set the H2O fugacity as a starting condition (bar)
    FCO2_START : float
        Set the CO2 fugacity as a starting condition (bar)
    ATOMIC_MASS_SET : bool, default False
        Choose to set the mass of each volatile element in the system and calculate the
        volatile saturation point, rather than set a starting pressure
    ATOMIC_H, ATOMIC_C, ATOMIC_S, ATOMIC_N : float
        Mass fraction (parts per million) of the corresponding element in
        the system. Only use alongside `ATOMIC_MASS_START`.
    WTH2O_START, WTCO2_START, SULFUR_START, NITROGEN_START : float
        Initial weight FRACTION (not %) of the species dissolved in the
        melt at the starting pressure.
    GRAPHITE_SATURATED  : bool, default = False
        If True, the melt starts graphite saturated, with a mass of
        graphite specified in `GRAPHITE_START`.
    GRAPHITE_START : float
        Initial melt graphite content as a weight fraction, use if
        `GRAPHITE_SATURATED` is True
    """

    model_config = ConfigDict(extra="forbid")

    COMPOSITION: Literal["basalt", "phonolite", "rhyolite"] = "basalt"
    RUN_TYPE: Literal["closed", "open"] = "closed"
    SINGLE_STEP: bool = False
    FIND_SATURATION: bool = False
    GAS_SYS: str = "OH"
    FE_SYSTEM: bool = True
    OCS: bool = False
    S_SAT_WARN: bool = False

    # Physical parameters
    T_START: float = 1473.15
    P_START: float = 3000.0
    P_STOP: float = 1.0
    DP_MIN: float = 1.0
    DP_MAX: float = 100.0
    MASS: float = 100.0
    WgT: float = 0.001
    LOSS_FRAC: float = 0.99

    # Model options
    DENSITY_MODEL: Literal["spera2000"] = "spera2000"
    FO2_MODEL: Literal["kc1991", "r2013"] = "kc1991"
    FMQ_MODEL: Literal["frost1991"] = "frost1991"
    H2O_MODEL: Literal["burguisser2015"] = "burguisser2015"
    H2_MODEL: Literal["gaillard2003", "burguisser2015"] = "gaillard2003"
    C_MODEL: Literal["burguisser2015", "eguchi2018"] = "burguisser2015"
    CO_MODEL: Literal["armstrong2015", "None"] = "None"
    CH4_MODEL: Literal["ardia2013", "None"] = "None"
    SULFIDE_CAPACITY: Literal["oneill2020", "oneill2002"] = "oneill2020"
    SULFATE_CAPACITY: Literal["nash2019"] = "nash2019"
    SCSS: Literal["liu2007"] = "liu2007"
    N_MODEL: Literal["libourel2003"] = "libourel2003"

    # fO2 and fugacity controls
    FO2_buffer_SET: bool = False
    FO2_buffer: Literal["FMQ", "IW", "NNO"] = "FMQ"
    FO2_buffer_START: float = 0.0
    FO2_SET: bool = False
    FO2_START: float = 0.0
    FH2_SET: bool = False
    FH2_START: float = 0.24
    FH2O_SET: bool = False
    FH2O_START: float = 1000.0
    FCO2_SET: bool = False
    FCO2_START: float = 0.01

    # Atomic masses - ppm
    ATOMIC_MASS_SET: bool = False
    ATOMIC_H: float = 550.0
    ATOMIC_C: float = 200.0
    ATOMIC_S: float = 4000.0
    ATOMIC_N: float = 10.0

    # Initial melt volatile concentrations
    WTH2O_SET: bool = False
    WTH2O_START: float = 0.03
    WTCO2_SET: bool = False
    WTCO2_START: float = 0.01
    SULFUR_SET: bool = False
    SULFUR_START: float = 0.001
    NITROGEN_SET: bool = False
    NITROGEN_START: float = 0.01
    GRAPHITE_SATURATED: bool = False
    GRAPHITE_START: float = 0.0

    # Optional output parameter
    results_folder: Optional[PosixPath] = None

    @model_validator(mode="after")
    def validate_config(self) -> "RunDef":
        if self.OCS:
            raise ValueError(
                "Error: OCS functionality has not been released yet. Sorry!"
            )

        if self.MASS <= 0:
            raise ValueError("The system has no mass value")

        if self.WgT <= 0 and not self.FIND_SATURATION and not self.ATOMIC_MASS_SET:
            raise ValueError(
                "A gas weight fraction of greater than 0 must be entered "
                "if the saturation pressure is not being calculated."
            )

        return self

    @field_validator("GAS_SYS")
    @classmethod
    def validate_gas_sys(cls, v: str) -> str:
        ALLOWED_VARIANTS = ["OH", "COH", "SOH", "COHS", "COHSN"]
        # Normalize: uppercase, sort letters
        normalized = "".join(sorted(v.upper()))
        for allowed in ALLOWED_VARIANTS:
            if normalized == "".join(sorted(allowed)):
                return allowed  # Return canonical form (e.g. "COHS")
        raise ValueError(
            f"Invalid GAS_SYS: {v}. Must be a permutation of one of {ALLOWED_VARIANTS}"
        )

    def print_conditions(self):
        print("".join(f"{k} = {v}\n" for k, v in self.model_dump().items()))


#  -------------------------------------------------------------------------
class ThermoSystem:
    """
    Thermodynamic properties of the whole system (gas and melt).

    Attributes
    ----------

    run : RunDef class
        Active instance of the RunDef class
    T : float
        System temperature (K), inherited from RunDef
    P : float
        Current pressure (bar)
    P_step : float
        The current size of the pressure step to take (bar)
    P_track : list of floats
        Stores the pressures at which solutions have been found
    M : float
        The system mass, inherited from RunDef
    rho : list of floats
        The current bulk density of the melt+gas at every step
    sat_conditions : tuple
        Stores the saturation conditions (P_sat, gas composition, fugacity
        coefficients, melt content)
    graph_unsat_rerun : bool, default = False
        Records whether a re-run of a pressure is being performed because
        the melt became graphite under-saturated.
    FO2 : float
        current absolute fO2 STORED AS LN(FO2)
    FO2_buffer : float
        Stores the fO2 of the system relative to the mineral buffer stored
        in RunDef.
    SC : list of str
        Stores the system chemistry - the gas species being modelled.
    K : dict
        Stores the the equilibrium constants for gas phase equilibria
    atomicM : dict
        Masses of each volatile element - weight fractions.
    WgT : list of floats
        stores the total gas weight fraction at every pressure step.
    GvF : list of floats
        Stores the gas volume fraction at every pressure step
    Cw : dict
        Stores the SYSTEM mass fractions of oxides and volatile elements,
        normalised to 1. Iron stored as Fe so oxygen isn't counted twice.
    """

    def __init__(self, run):
        self.run = run
        self.T = self.run.T_START  # K
        self.P = self.run.P_START  # bar
        self.P_step = self.run.DP_MAX  # bar
        self.P_track = []  # stores the pressure steps run at
        self.M = self.run.MASS  # g
        self.rho = []  # bulk density
        self.sat_conditions = 0  # store saturation pressure + gas speciation
        self.graph_unsat_rerun = False

        # intialise for subsequent setting
        self.FO2 = None  # fo2 # STORED AS A LN(FO2) SO DO e**SELF.FO2 WHERE IT'S NEEDED
        self.FO2_buffer = None
        self.SC = []  # System chemistry - i.e. gas species present.
        self.K = {}  # Stores K values (equilibrium constants)
        self.atomicM = {}  # Atomic masses for mass balance - weight fraction
        self.WgT = [self.run.WgT]  # Total gas wt%, initially set by user.
        self.GvF = []  # Stores the gas volume fraction at each step
        self.Cw = {}
        self.OCS = False  # Can be flipped after the last pressure step for a final run.

    @property
    def Ppa(self):
        """Converts system pressure in bar, to pressure in pascal (Pa)"""
        return self.P * 1e5

    # Selects the system chemistry from the system the user wants to examine.
    def gas_system(self, run):
        """
        Selects the system chemistry given the elements the user wishes to track.

        Parameters
        ----------
        run : RunDef class
            Active instance of the RunDef class

        Returns
        -------
        self.SC : list of str
            List of all the gas phase species.
        """

        run.GAS_SYS = run.GAS_SYS.upper()

        def perm(string, length):
            return list(map("".join, itertools.permutations(string, length)))

        OH = perm("OH", 2)
        COH = perm("COH", 3)
        SOH = perm("SOH", 3)
        COHS = perm("COHS", 4)
        COHSN = perm("COHSN", 5)

        if run.GAS_SYS in OH:
            run.GAS_SYS = "OH"
            self.SC = ["H2O", "O2", "H2"]
        elif run.GAS_SYS in COH:
            run.GAS_SYS = "COH"
            self.SC = ["H2O", "O2", "H2", "CO2", "CO", "CH4"]
        elif run.GAS_SYS in SOH:
            run.GAS_SYS = "SOH"
            self.SC = ["H2O", "O2", "H2", "S2", "H2S", "SO2"]
        elif run.GAS_SYS in COHS:
            run.GAS_SYS = "COHS"
            self.SC = ["H2O", "O2", "H2", "CO2", "CO", "CH4", "S2", "H2S", "SO2"]
        elif run.GAS_SYS in COHSN:
            run.GAS_SYS = "COHSN"
            self.SC = ["H2O", "O2", "H2", "CO2", "CO", "CH4", "S2", "H2S", "SO2", "N2"]
        return self.SC

    def get_atomic_mass(self, run, gas, melt, mols):
        """Finds the masses of volatile elements in the system."""

        if run.GAS_SYS == "OH":
            ic.oh(self, run, melt, gas, mols)
        elif run.GAS_SYS == "COH":
            ic.coh(self, run, melt, gas, mols)
        elif run.GAS_SYS == "SOH":
            ic.soh(self, run, melt, gas, mols)
        elif run.GAS_SYS == "COHS":
            ic.cohs(self, run, melt, gas, mols)
        elif run.GAS_SYS == "COHSN":
            ic.cohsn(self, run, melt, gas, mols)

    def norm_with_gas(self, melt):  # for iron equilibration.
        """
        Normalises the melt with the gas phase ready to equilibrate the oxygen.

        Updates the Cw attribute of ThermoSystem to be oxides + gases
        normalised to 1. Finds the ferric/ferrous ratio and adds this
        to melt.F

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class
        """

        cons = 0  # holds constant volatile species and iron content
        C_s = {}  # Holds the values to be normalised
        C_w = melt.Cw()  # wt %

        for key in self.atomicM:
            self.Cw[key] = self.atomicM[key]
            cons += self.atomicM[key]

        # The oxide composition of the melt partitioned as wt %
        # PL: This doesn't convert fe2o3 to feo at the moment, it probably should?
        # Need to catch scenario where only fe2o3 is provided?
        for ele in C_w:
            if ele == "feo" or ele == "fe2o3":
                cons += C_w[ele] / 100
                self.Cw[ele] = C_w[ele] / 100
            else:
                # oxide weights only, without volatiles or iron, ready to be scaled.
                C_s[ele] = C_w[ele] / 100

        # returns the 9 major oxides as weight %, with feoT and volatiles held constant.
        oxides = cnvs.norm(C_s, (1 - cons))

        self.Cw.update(
            oxides
        )  # adds the normalised oxide values to the total system mass frac dictionary

        melt.cm_dry, F = melt.iron_fraction(
            self.FO2
        )  # this is now a duplicate of the one inside init cons, hopefully.
        melt.F.append(F)

    def get_wto_tot(self, melt):
        """
        Calculates the total amount of O: volatile system + iron.

        Finds the mass fraction of elemental iron, O stored in Fe within
        the melt and total O in the system. Updates self.Cw so it has
        feo/fe2o3 ratio.

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class

        Returns
        -------
        self.atomicM : dict
            Dict of the volatile element masses, updated with oxygen values
        """

        Cm = cnvs.wt2mol(self.Cw)

        for k, v in melt.cm_dry.items():
            Cm[k] = v

        Cm = cnvs.norm(Cm, cons=["o", "h", "c", "s", "n"])

        Cm["fe"] = Cm["feo"] + 2 * Cm["fe2o3"]
        m_ofe = (
            Cm["feo"] + 3 * Cm["fe2o3"]
        )  # The mole frac of oxygen tied up in the melt.

        Cm.pop("feo", None)
        Cm.pop("fe2o3", None)

        Cm["ogas"] = Cm["o"]
        Cm["ofe"] = m_ofe
        Cm.pop("o", None)

        Cm = cnvs.norm(Cm, cons=["fe", "ogas", "ofe", "h", "c", "s", "n"])
        self.Cw = dict(cnvs.mol2wt(Cm))

        self.atomicM["fe"] = self.Cw["fe"]
        self.atomicM["ofe"] = self.Cw["ofe"]
        self.atomicM["o_tot"] = self.atomicM["o"] + self.Cw["ofe"]

        return self.atomicM

    def fe_equil(self, melt, mo2, O2):
        """
        Calculate the amount of oxygen in the melt for use in solver.

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class
        mo2 : float
            Current mol fraction of O2 in the gas phase
        O2 : Molecule class
            O2 instance of the Molecule class

        Returns
        -------
        ofe : float
            Mass of volatile elemental O in the melt.
        """

        fo2 = np.log(O2.Y * mo2 * self.P)

        # returns anhydrous silicate melt comp mol fraction, Fe2/Fe3.
        F = melt.iron_fraction(fo2)[1]

        ofe = (
            cnst.m["o"]
            * (self.atomicM["fe"] / cnst.m["fe"])
            * ((1 + 3 * F) / (1 + 2 * F))
        )  # THIS STEP total weight fraction of atomic O in iron

        return ofe

    def fe_save(self, melt, mo2, O2):
        """
        Saveout the amount of O in the gas and melt.

        After the melt is in redox equilibrium with the gas phase, the
        amount of oxygen and the ferric/ferrous ratio is recorded.

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class
        mo2 : float
            mole fraction of O2 in the gas phase
        O2 : Molecule class
            O2 instance of the Molecule class
        """

        fo2 = np.log(O2.Y * mo2 * self.P)

        # returns anhydrous silicate melt comp mol fraction, Fe2/Fe3.
        F = melt.iron_fraction(fo2)[1]

        ofe = (
            cnst.m["o"]
            * (self.atomicM["fe"] / cnst.m["fe"])
            * ((1 + 3 * F) / (1 + 2 * F))
        )  # THIS STEP total weight fraction of atomic O in iron

        self.atomicM["ofe"] = ofe
        self.atomicM["o"] = self.atomicM["o_tot"] - ofe
        melt.ofe.append(ofe)
        melt.F.append(F)

    def mass_conservation_reset(self, melt, gas):
        """
        Deletes results of current run to redo at a higher pressure.

        Called if the element masses are no longer being conserved. Deletes
        the results of the last run and reduces the step size to help
        convergence towards the correct solution, ready to run again with
        the same initial conditions at a higher pressure.

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class
        gas : Gas class
            Active instance of the Gas class
        """

        sys_lsts = [self.WgT, self.GvF, self.rho]
        gas_lsts = [
            gas.fo2,
            gas.mH2O,
            gas.mO2,
            gas.mH2,
            gas.mCO,
            gas.mCO2,
            gas.mCH4,
            gas.mS2,
            gas.mSO2,
            gas.mH2S,
            gas.mN2,
            gas.M,
        ]
        melt_lsts = [
            melt.fmq,
            melt.rho_store,
            melt.h2o,
            melt.h2,
            melt.co2,
            melt.co,
            melt.ch4,
            melt.graphite,
            melt.sulfide,
            melt.sulfate,
            melt.s,
            melt.n,
            melt.F,
            melt.ofe,
        ]
        gas_wts_f = ["H2O", "O2", "H2", "CO", "CO2", "CH4", "S2", "SO2", "H2S", "N2"]

        for x in sys_lsts + gas_lsts + melt_lsts:
            del x[-1]

        for x in gas_wts_f:
            del gas.Wt[x][-1]
            if x != "O2":
                del gas.f[x][-1]

        if melt.graphite_sat is True:
            melt.graph_current = melt.graphite[-1]

        if self.P_step / 10 >= self.run.DP_MIN:
            del self.P_track[-1]
            self.P = self.P + self.P_step
            self.P_step = self.P_step / 10
            print("mass conservation reset")
        else:
            del self.P_track[-1]
            msgs.closed_earlyexit(self, gas, melt)
            exit(
                "Error: Mass is no longer being conserved. "
                "Please reduce the minimum pressure stepsize and try again."
                "\nExiting..."
            )

    def pressure_step(self):
        """Reduces the pressure step size by x10 as it approaches `P_STOP`"""

        # If taking another pressure step won't take you below P_STOP, continue.
        if self.P - self.P_step > self.run.P_STOP:
            self.P -= self.P_step

        # If taking another step puts you below the min pressure and step/10 is less
        # than the minimum stepsize, set stepsize to min.
        elif (
            self.P - self.P_step / 10 > self.run.P_STOP
            and self.P_step / 10 <= self.run.DP_MIN
        ):
            self.P_step = self.run.DP_MIN
            self.P -= self.P_step

        else:
            while (
                self.P_step > self.run.DP_MIN
            ):  # Loop to get to the next possible pressure step
                if (
                    self.P - self.P_step / 10 > self.run.P_STOP
                ):  # If step/10 doesn't put you below min pressure
                    self.P_step = self.P_step / 10
                    self.P -= self.P_step
                    break
                else:
                    self.P_step = self.P_step / 10  # otherwise /10 and loop back around
            else:
                self.P -= (
                    self.P_step
                )  # This will trigger the outer loop when it sees self.P < P_STOP.

    def variable_step(self):
        """Reduces the step size by x10 when called during decompression"""

        if self.P_step / 10 >= self.run.DP_MIN:
            self.P = self.P + self.P_step - (self.P_step / 10)
            self.P_step = self.P_step / 10
            print(f"Convergence issue; reducing stepsize to {self.P_step}")
            return None
        else:
            return (
                "Minimum pressure step reached without achieving convergence "
                f"at {self.P} bar."
            )

    def rho_bulk(self, melt, gas):
        """Returns the density of the bulk magma (gas+melt combined)"""

        return (gas.rho() * self.GvF[-1]) + (
            melt.rho(P=self.P * 1e5) * (1 - self.GvF[-1])
        )


#  -------------------------------------------------------------------------
class Molecule:
    """
    Defines the properties of each molecular species in the gas phase

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class
    mol : str
        The name of the molecule being instantated

    Attributes
    ----------
    Mol : str
        Molecule name
    M : float
        Molecular mass (g/mol)
    PTcrit : dict
        Critical T's (K) and P's (bar) for calculating fugacity coefficients
        using the method from Shi and Saxena (1992)
    solCon : dict
        Contains the 'a' and 'b' solubility constants for use in solubility
        laws from Burguisser & Scaillet (2015).
    sys : ThermoSystem class
        Active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class
    delG : float
        Gibbs free energy of formation
    Y : float
        Current fugacity coefficient
    y_constants : dict
        Stores any constants required for calculating the fugacity
        coefficient.
    """

    def __init__(self, run, sys, melt, mol):
        self.Mol = str(mol)  # Molecule name

        # initialise for subsequent settings
        self.M = cnst.m[self.Mol.lower()]  # Molecular mass

        # Critical T's (K) and P's (bar) respectively, for fugacity coefficients.
        self.PTcrit = {"T": cnst.PTcrit[self.Mol][0], "P": cnst.PTcrit[self.Mol][1]}

        if run.COMPOSITION == "basalt":
            if self.Mol == "S2":
                pass
            else:
                self.solCon = {
                    "a": cnst.basalt["a"][str(self.Mol)],
                    "b": cnst.basalt["b"][str(self.Mol)],
                }  # Solubility constant a and b

        elif run.COMPOSITION == "phonolite":
            lim = [1073.15, 1523.15]  # Tested temperature dependent solubility limits

            if self.Mol == "H2O" and lim[0] < sys.T < lim[1]:
                self.solCon = {
                    "a": cnst.phonolite["a"][str(self.Mol)](cnvs.K2C(sys.T)),
                    "b": cnst.phonolite["b"][str(self.Mol)](cnvs.K2C(sys.T)),
                }  # T-dependent solubility constant a and b

            elif self.Mol == "H2O" and (sys.T < lim[0] or sys.T > lim[1]):
                msgs.solubility_temp(sys.T, lim)
                self.solCon = {
                    "a": 4.69e-4,
                    "b": 0.6441,
                }  # T-independent solubility constant a and b
            else:
                self.solCon = {
                    "a": cnst.phonolite["a"][str(self.Mol)],
                    "b": cnst.phonolite["b"][str(self.Mol)],
                }  # Solubility constant a and b

        elif run.COMPOSITION == "rhyolite":
            lim = [1063.15, 1283.15]

            if (self.Mol == "H2O" or self.Mol == "CO2") and lim[0] < sys.T < lim[1]:
                self.solCon = {
                    "a": cnst.rhyolite["a"][str(self.Mol)](cnvs.K2C(sys.T)),
                    "b": cnst.rhyolite["b"][str(self.Mol)](cnvs.K2C(sys.T)),
                }  # Solubility constant a and b

            elif self.Mol == "H2O" and (sys.T < lim[0] or sys.T > lim[1]):
                msgs.solubility_temp(sys.T, lim)
                self.solCon = {
                    "a": 5.4344e-4,
                    "b": 0.6344,
                }  # T-independent solubility constant a and b

            elif self.Mol == "CO2" and (sys.T < lim[0] or sys.T > lim[1]):
                self.solCon = {
                    "a": 4.9358e-7,
                    "b": 1.04896,
                }  # T-independent solubility constant a and b

            else:
                self.solCon = {
                    "a": cnst.rhyolite["a"][str(self.Mol)],
                    "b": cnst.rhyolite["b"][str(self.Mol)],
                }  # Solubility constant a and b

        self.sys = sys
        self.melt = melt
        self.delG = 0  # Gibbs Free energy of formation
        self.Y = 0  # Fugacity coefficient
        self.y_constants = {}  # Stores constants for activity coefficient calculation

    def get_G(self, T):
        """
        Gets the Gibbs free energy of formation.

        Calculates the GfF for a given temperature, to use in calculations
        of equilibrium constants. Reads in critical data for the species
        from the 'data' folder, linearly interpolating between temperatures.

        Parameters
        ----------
        T : float
            Current temperature (K)

        Returns
        -------
        self.delG : float
            Gibbs free energy of formation
        """

        filename = f"{self.Mol}.txt"

        data_file = importlib.resources.files("evo") / "data" / filename

        with data_file.open("r", encoding="utf-8") as path:
            Temp_ref = []  # To store temperatures for interpolation
            del_G_ref = []  # To store values of G for interpolation
            inter_count = 0

            # Iterate through file to find correct values according to T provided.
            for aRow in path:
                values = aRow.split("\t")
                if not aRow.startswith("#"):  # Skips descriptive headers
                    if T - float(values[0]) == 0:
                        # Exact match found, no interpolation needed
                        self.delG = float(values[6])
                        return self.delG
                    elif inter_count == 1:
                        # Adds in the upper table values for interpolation
                        Temp_ref.append(float(values[0]))
                        del_G_ref.append(float(values[6]))
                        break
                    elif T - float(values[0]) <= 99:
                        Temp_ref.append(float(values[0]))
                        del_G_ref.append(float(values[6]))
                        inter_count += 1

        # Perform interpolation if needed
        if len(Temp_ref) == 2:
            self.delG = np.interp(T, Temp_ref, del_G_ref)

        return self.delG


#  -------------------------------------------------------------------------
class Melt:
    """
    Creates an object for the silicate melt, stores mass and chemical comp

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class

    Attributes
    ----------
    graphite_sat : bool, default = False
        Records whether the melt is currently graphite saturated
    Cs : dict
        Stores the major oxide chemistry as real mass values (g).
        All iron as FeO.
    cm_dry : dict
        Dry major oxide composition as mole fractions, with feo+fe2o3
    cm_density : dict
        Stores the normalised major oxide comp plus H2O and CO2 as mole
        fractions, for use calculating the melt density.
    cm_volatiles : dict
        Stores the full melt composition, major oxides plus volatiles, as
        mole fractions.
    fmq : list of floats
        Stores the fO2 of the melt at every pressure step relative to the
        FMQ buffer
    h2o, h2, co2, co, ch4, sulfide, sulfate, n : list of floats
        Stores the melt weight fraction of each species after every
        pressure step
    s : list of floats
        Stores the total melt S weight fraction after every P step
    F : list of floats
        Stores the ferric/ferrous ratio of the melt after every P step
    ofe : list of floats
        Store the mass of O in iron oxides after every P step
    graph_current : float
        The number of moles in the system in the current timestep
    graphite : list of floats
        Stores the weight fraction of graphite in the melt after every P step
    rho_store : list of floats
        Stores the density of the volatile free melt after every P step
    """

    def __init__(self, run, sys):
        self.run = run
        self.sys = sys
        self.graphite_sat = False
        self.Cs = {}
        self.cm_dry = {}
        self.cm_density = {}
        self.cm_volatiles = {}
        self.fmq = []
        self.h2o = []
        self.h2 = []
        self.co2 = []
        self.co = []
        self.ch4 = []
        self.sulfide = []  # stores s2-
        self.sulfate = []  # stores s6+
        self.s = []
        self.n = []  # stores the melt n content, independent of speciation.
        self.F = []  # stores the mfe2o3/mFeO ratio at each step.
        self.ofe = []
        self.graph_current = (
            0  # number of moles of graphite in the system at current timestep
        )
        self.graphite = []  # stores graphite mass after each timesetep
        self.rho_store = []

    def chem_init(self, eles, chem):
        """
        Loads in major oxide composition from chemistry file.

        Gets the weight percent and actual masses of the oxide species
        in the silicate melt, adds in fe2o3 as a required species if
        missing, checks the ferric/ferrous model is appropriate for
        the amount of iron in the melt and calculates the fO2 if the
        ferric/ferrous ratio has been given.

        Parameters
        ----------
        eles : list of str
            The oxide names present in the silicate melt
        chem : list of floats
            The proportion of each species in the melt as weight PERCENT
        """

        # intialise constant masses of oxides vectors
        # as record of conserved system property
        self.C_s = {}  # actual oxide masses (g) NOT MASS FRACTION

        tmp_Cw = {}  # weight percent (the value which is read in by the file)

        length = 0  # counts through the values stored for the oxide composition
        sm = 0  # sums for average calculations

        # readin raw composition data
        for e in eles:
            tmp_Cw.update({e: chem[length]})
            sm += tmp_Cw[e]
            length += 1

        # normalise to 100 and define initial system element mass
        for e in eles:
            tmp_Cw[e] = tmp_Cw[e] * 100.0 / sm
            self.C_s[e] = tmp_Cw[e] / 100.0 * self.run.MASS  # actual oxide mass

        # add Fe+++ in if component has not been specified
        if "fe2o3" not in tmp_Cw:
            tmp_Cw["fe2o3"] = 0.0
            self.C_s["fe2o3"] = 0.0

        self.Cs = copy.copy(self.C_s)  # These are actual masses in g

        # This won't be completely accurate if fe2o3 is high
        # because they're not normalised; but close enough.
        if (
            self.run.FO2_MODEL == "r2013"
            and (tmp_Cw["feo"] + (tmp_Cw["fe2o3"] / cnst.m["fe2o3"]) * cnst.m["feo"])
            <= 15.0
        ):
            msgs.fo2_model_mismatch(
                (tmp_Cw["feo"] + (tmp_Cw["fe2o3"] / cnst.m["fe2o3"]) * cnst.m["feo"]),
                self.run,
            )
        elif (
            self.run.FO2_MODEL == "kc1991"
            and (tmp_Cw["feo"] + (tmp_Cw["fe2o3"] / cnst.m["fe2o3"]) * cnst.m["feo"])
            > 15.0
        ):
            msgs.fo2_model_mismatch(
                (tmp_Cw["feo"] + (tmp_Cw["fe2o3"] / cnst.m["fe2o3"]) * cnst.m["feo"]),
                self.run,
            )

        # if Fe3 and Fe2 exist, then need to determine system fO2;
        # also updates the Fe mass in system.
        if "feo" in eles and "fe2o3" in eles and self.sys.FO2 is None:
            self.sys.FO2 = np.log(
                cnvs.c2fo2(self.Cm(), self.sys.T, self.sys.Ppa, self.run.FO2_MODEL)
            )

        # store element names comprising liquid
        self.eles = self.Cs.keys()

    def iron_fraction(self, fo2, ppa=None):
        """
        Returns the melt composition and ferric/ferrous ratio for a given fO2

        Parameters
        ----------
        fo2 : float
            Ln(fO2)
        ppa : float, optional
            Current pressure in pascal, by default None. If None, pressure
            is taken from the ThermoSystem object

        Returns
        -------
        composition : dict
            major oxide composition as mole fractions,
              updated with new ferric/ferrous ratio
        F : float
            ferric/ferrous ratio
        """

        if ppa is None:
            ppa = self.sys.Ppa

        composition = self.Cm()  # molar composition just silicate melt

        # Outputs MOL fe2o3 / feo. Runs without dissolved volatiles to get the fraction,
        # then normalised with below.
        F = cnvs.fo2_2F(composition, self.sys.T, ppa, fo2, self.sys.run.FO2_MODEL)

        mFeoT = composition["feo"]

        mfe2o3 = F * mFeoT / (1 + 0.8998 * F)

        mfeo = mfe2o3 / F

        composition["feo"] = mfeo

        composition["fe2o3"] = mfe2o3

        composition = cnvs.norm(composition)
        return composition, F

    def Cw(self):
        "Converts ACTUAL MASS to WEIGHT PERCENT."
        return cnvs.norm(self.Cs, 100.0)

    def Cm(self):
        "Converts to ACTUAL MASS to MOLAR FRACTION"

        Cw = cnvs.norm(
            self.Cs, 1.0
        )  # converts real mass to mass fraction for the wt2mol conversion.
        return cnvs.wt2mol(Cw)

    def cm_singleO(self, mol_oxides):
        """
        Returns the melt composition as mol fracs on a single oxygen basis

        Parameters
        ----------
        mol_oxides : dict
            Melt composition as mole fractions

        Returns
        -------
        cation_formula : dict
            Melt composition as single oxygen mole fractions, with names
            as cations only.
        """

        def num_oxygen(x):
            """Returns the number of oxygen atoms in a species"""
            if x in ["na2o", "k2o", "mgo", "cao", "mno", "feo", "nio"]:
                return 1

            elif x in ["sio2", "tio2"]:
                return 2

            elif x in ["al2o3", "cr2o3", "fe2o3"]:
                return 3

            elif x == "p2o5":
                return 5

        def num_anion(x):
            """Returns the number of anion atoms in a species"""
            if x in ["sio2", "tio2", "mgo", "cao", "mno", "feo", "nio"]:
                return 1

            elif x in ["al2o3", "cr2o3", "fe2o3", "p2o5", "na2o", "k2o"]:
                return 2

        total_o = 0
        mol_cations = {}

        for name, x in mol_oxides.items():
            mol_cations[name] = x * num_anion(name)
            total_o += x * num_oxygen(name)

        formula = {}

        for name, x in mol_cations.items():
            formula[name] = x / total_o

        cation_formula = {}

        for x in formula.keys():
            if x == "feo":
                cation_formula["fe2"] = formula[x]
            elif x == "fe2o3":
                cation_formula["fe3"] = formula[x]
            else:
                no_num = re.sub(r"\d", "", x)
                new_name = re.sub("o", "", no_num)
                cation_formula[new_name] = formula[x]

        return cation_formula

    def formula_weight(self, fo2, P):
        """
        Calculates the single-O formula weight of the volatile-free melt.

        Finds the fe2/fe3 ratio first.

        Parameters
        ----------
        fO2 : float
            Absolute oxygen fugacity (bar)
        P : float
            Pressure (bar)

        Returns
        -------
        fwm : float
            The mean formula weight (g/mol)
        """

        mol_oxides = self.iron_fraction(np.log(fo2), ppa=P * 1e5)[0]

        formula = self.cm_singleO(mol_oxides)

        fwm = 0

        # strip down to cation name
        for x in formula.keys():
            no_num = re.sub(r"\d", "", x)
            new_name = re.sub("o", "", no_num)

            fwm += cnst.m[new_name] * formula[x]

        fwm += cnst.m["o"]

        return fwm

    def rho(self, T=None, P=None):
        """calculates the density of the volatile-free silicate melt"""
        if T is None:
            T = self.sys.T
        if P is None:
            P = self.sys.Ppa

        if self.cm_density:
            composition = self.cm_density
        else:
            composition = self.Cm()

        # K, Pa
        if self.run.DENSITY_MODEL == "spera2000":
            return density.den_calc_spera2000(composition, T, P)

    def sulfide_sat(self, comp):
        """
        Calculates the sulfur content at sulfide saturation of the melt

        Parameters
        ----------
        comp : dict
            The composition of the melt as mole fractions, including the
            dissolved volatile content.

        Returns
        -------
        float
            The sulfur content at sulfide saturation, in parts per million

        References
        ----------
        Liu, Y., Samaha, N-T., Baker, D.R. (2007) Sulfur concentration at
        sulfide saturation (SCSS) in magmatic silicate melts. GCA.
        """

        if self.run.SCSS == "liu2007":
            "Liu et al 2007"

            def mfm():
                Na = comp["na2o"] * 2
                K = comp["k2o"] * 2
                Ca = comp["cao"]
                Mg = comp["mgo"]
                Fe2 = comp["feo"]
                Si = comp["sio2"]
                Al = comp["al2o3"] * 2
                Fe3 = comp["fe2o3"] * 2

                return (Na + K + 2 * (Ca + Mg + Fe2)) / (Si * (Al + Fe3))

            MFM = mfm()

            mH2O = comp["h2o"]  # need to get mol frac of H2O IN MELT
            mFeO = comp["feo"]

            return np.exp(
                11.35251
                - 4454.6 / self.sys.T
                - 0.0319 * (self.sys.P / self.sys.T)
                + 0.71006 * np.log(MFM)
                - 1.98063 * (MFM * mH2O)
                + 0.21867 * np.log(mH2O)
                + 0.36192 * np.log(mFeO)
            )

    def melt_dicts(self):
        """
        Generates dicts containing the mole fraction composition of
        melt+volatiles, and one to calculate the density with.

        Returns
        -------
        melt_wet_mol : dict
            Melt + volatile composition as mole fractions,
             for melt.cm_volatiles
        density_mol : dict
            Melt + CO2 + H2O composition as mole fractions,
             for melt.cm_density
        """
        dry_wt = cnvs.mol2wt(self.cm_dry)
        dry_wt["wgt"] = self.sys.WgT[-1]

        dry_wt["h2o"] = self.h2o[-1]
        dry_wt["h2"] = self.h2[-1]
        dry_wt["co2"] = self.co2[-1]
        dry_wt["co"] = self.co[-1]
        dry_wt["ch4"] = self.ch4[-1]
        dry_wt["s"] = self.s[-1]
        dry_wt["n"] = self.n[-1]

        sys_wet_wt = cnvs.norm(
            dry_wt
        )  # PL: check this at a later date, seems superfluous?

        melt_wet_wt = dict(sys_wet_wt)
        melt_wet_wt.pop("wgt", None)
        melt_wet_wt = cnvs.norm(melt_wet_wt)
        melt_wet_mol = cnvs.wt2mol(
            melt_wet_wt
        )  # mol fraction comp of the melt, including all dissolved volatile species

        # Double check density isn't like sulphur where the melt is anhydrous,
        # plus wt water on top. PL: check co/ch4 don't need to go in?
        density = dict(sys_wet_wt)
        density.pop("wgt", None)
        density.pop("h2", None)
        density.pop("co", None)
        density.pop("ch4", None)
        density.pop("s", None)
        density.pop("n", None)
        density = cnvs.norm(density)
        density_mol = cnvs.wt2mol(
            density
        )  # includes all silicate elements plus dissolved H2O and CO2.

        return melt_wet_mol, density_mol

    def melt_composition(self, gas, mols):
        """
        Major function to run at end of P step, updates all results stores.

        Runs after a successful pressure step. Calculates and stores the
        new volatile content of the melt, saves the fO2, ferric/ferrous
        ratio and new major oxide fractions, and checks for both sulfide
        saturation and whether the sulfate fraction exceeds 10% (at which
        point the run ends).

        Parameters
        ----------
        gas : Gas class
            Active instance of the Gas class
        mols : list of Molecule classes
            A list of the active Molecule classes
        """

        # set the molecule names, and calculate the melt volatile weight percents.
        if self.run.GAS_SYS == "OH":
            H2O, O2, H2 = mols
        elif self.run.GAS_SYS == "COH":
            H2O, O2, H2, CO, CO2, CH4 = mols
        elif self.run.GAS_SYS == "SOH":
            H2O, O2, H2, S2 = mols[:4]
        elif self.run.GAS_SYS == "COHS":
            H2O, O2, H2, CO, CO2, CH4, S2 = mols[:7]
        elif self.run.GAS_SYS == "COHSN":
            H2O, O2, H2, CO, CO2, CH4, S2 = mols[:7]

        self.h2o.append(
            sl.h2o_melt(gas.mH2O[-1], H2O, self.sys.P, name=self.sys.run.H2O_MODEL)
            * cnst.m["h2o"]
        )
        self.h2.append(
            sl.h2_melt(gas.mH2[-1], H2, self.sys.P, self, name=self.sys.run.H2_MODEL)
            * cnst.m["h2"]
        )

        if (
            self.sys.run.GAS_SYS == "COHS"
            or self.sys.run.GAS_SYS == "COH"
            or self.sys.run.GAS_SYS == "COHSN"
        ):
            self.co2.append(
                sl.co2_melt(
                    (CO2.Y * gas.mCO2[-1] * self.sys.P),
                    CO2,
                    (O2.Y * gas.mO2[-1] * self.sys.P),
                    self.sys.T,
                    self.sys.P,
                    self,
                    name=self.sys.run.C_MODEL,
                )
                * cnst.m["co2"]
            )

            if self.sys.run.CO_MODEL != "None":
                self.co.append(
                    sl.co_melt(
                        (CO.Y * gas.mCO[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CO_MODEL,
                    )
                    * cnst.m["co"]
                )
            else:
                self.co.append(0.0)

            if self.sys.run.CH4_MODEL != "None":
                self.ch4.append(
                    sl.ch4_melt(
                        (CH4.Y * gas.mCH4[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CH4_MODEL,
                    )
                    * cnst.m["ch4"]
                )
            else:
                self.ch4.append(0.0)

            if self.graphite_sat is True:
                self.graphite.append(self.graph_current * cnst.m["c"])
            else:
                self.graphite.append(0.0)

        else:
            self.co2.append(0.0)
            self.co.append(0.0)
            self.ch4.append(0.0)
            self.graphite.append(0.0)

        if (
            self.sys.run.GAS_SYS == "COHS"
            or self.sys.run.GAS_SYS == "SOH"
            or self.sys.run.GAS_SYS == "COHSN"
        ):
            fo2 = O2.Y * gas.mO2[-1] * self.sys.P
            self.cm_dry, F = self.iron_fraction(np.log(fo2))

            if self.run.FE_SYSTEM is False:
                # otherwise this is done in fe_save
                self.F.append(F)

            self.sulfide.append(
                sl.sulfide_melt(
                    (S2.Y * gas.mS2[-1] * self.sys.P),
                    fo2,
                    self.sys.P,
                    self.sys.T,
                    self,
                    name=self.sys.run.SULFIDE_CAPACITY,
                )
                * cnst.m["s"]
            )
            self.sulfate.append(
                sl.sulfate_melt(
                    (S2.Y * gas.mS2[-1] * self.sys.P),
                    fo2,
                    self.sys.P,
                    self.sys.T,
                    self,
                    self.sys.run,
                    name=self.sys.run.SULFATE_CAPACITY,
                )
                * cnst.m["s"]
            )
            self.s.append(self.sulfide[-1] + self.sulfate[-1])

        else:
            if self.run.FE_SYSTEM is False:
                # otherwise this is done in self.fe_save
                fo2 = O2.Y * gas.mO2[-1] * self.sys.P
                self.cm_dry, F = self.iron_fraction(np.log(fo2))
                self.F.append(F)

            self.sulfide.append(0.0)
            self.sulfate.append(0.0)
            self.s.append(0.0)

        if self.sys.run.GAS_SYS == "COHSN":
            self.n.append(
                sl.n_melt(
                    gas.mN2[-1],
                    (O2.Y * gas.mO2[-1] * self.sys.P),
                    self.sys.P,
                    name=self.sys.run.N_MODEL,
                )
                * cnst.m["n"]
            )
        else:
            self.n.append(0.0)

        self.cm_volatiles, self.cm_density = self.melt_dicts()

        self.rho_store.append(self.rho())  # Saves the melt density at each step

        if self.sys.run.S_SAT_WARN is True and self.run.GAS_SYS in [
            "SOH",
            "COHS",
            "COHSN",
        ]:
            melt_mols = (
                self.cm_volatiles
            )  # The melt phase normalised to 1, including dissolved volatiles

            # Sulfide + sulfate content of the melt in ppm
            S = (
                sl.sulfide_melt(
                    (S2.Y * gas.mS2[-1] * self.sys.P),
                    (O2.Y * gas.mO2[-1] * self.sys.P),
                    self.sys.P,
                    self.sys.T,
                    self,
                    name=self.sys.run.SULFIDE_CAPACITY,
                )
                * cnst.m["s"]
                * 1e6
            )
            S6 = (
                sl.sulfate_melt(
                    (S2.Y * gas.mS2[-1] * self.sys.P),
                    (O2.Y * gas.mO2[-1] * self.sys.P),
                    self.sys.P,
                    self.sys.T,
                    self,
                    self.sys.run,
                    name=self.sys.run.SULFATE_CAPACITY,
                )
                * cnst.m["s"]
                * 1e6
            )

            SCSS = self.sulfide_sat(
                melt_mols
            )  # gives the sulphur content in ppm for sulphide saturation

            if S > SCSS:
                msgs.scss_warn(
                    SCSS, S, self.sys, gas, self
                )  # If sulfide content is above SCSS, writeout and exit.
            elif S6 / (S + S6) >= 0.1:
                msgs.sulfate_warn(
                    S6 / (S + S6), self.sys, gas, self
                )  # If the S6 content is >10%, writes out current data and closes
                # as so4 currently doesn't degass.


#  -------------------------------------------------------------------------


class Gas:
    """Stores the properties and results for the gas as it decompresses.
    Holds the results of each decompression step.

    Attributes
    ----------
    sys : ThermoSystem object
        Active instance of the ThermoSystem class
    fo2 : string of floats
        ln(fo2) after every pressure step
    mO2, mH2O, mH2, mCO, mCO2, mCH4, mSO2, mH2S, mS2, mN2 : list of floats
        Stores the mole fractions of each species in the gas phase after
        every pressure step
    atomicM : dict
        Stores the mass fraction of each element in the gas phase
    wt : dict
        Temporary store for the results of converting mole frac to wt frac
    Wt : dict
        Stores the gas weight fraction after every pressure step
    f : dict
        Stores the fugacity of each species in the gas phase after every
        pressure step
    M : list of floats
        Stores the mean molecular mass of the gas phase after every P step
    """

    def __init__(self, sys):
        self.sys = sys
        self.fo2 = []
        self.mO2 = []  # Stores mO2 value after each pressure step
        self.mH2O = []
        self.mH2 = []
        self.mCO = []
        self.mCO2 = []
        self.mCH4 = []
        self.mSO2 = []
        self.mH2S = []
        self.mS2 = []
        self.mN2 = []
        self.atomicM = {}
        self.wt = {}  # Temp store for results of converting mole frac to weight frac
        self.Wt = {}  # Store of weight fractions at each pressure step
        self.f = {}  # Store the fugacity of a species at each step.
        self.M = []  # Stores the mean molecular weight of the gas phase

    def get_WgT(self, melt, mols):
        """Calculates the gas weight fraction. NOT PERCENT."""
        if self.sys.run.GAS_SYS == "OH":
            H2O = mols
            wgt = (
                (self.sys.atomicM["o"] / cnst.m["o"])
                - sl.h2o_melt(
                    self.mH2O[-1], H2O, self.sys.P, name=self.sys.run.H2O_MODEL
                )
            ) / (
                (self.mH2O[-1] + 2 * self.mO2[-1])
                / (
                    self.mH2[-1] * cnst.m["h2"]
                    + self.mH2O[-1] * cnst.m["h2o"]
                    + self.mO2[-1] * cnst.m["o2"]
                )
            )

        elif self.sys.run.GAS_SYS == "COH":
            H2O, O2, H2, CO, CO2, CH4 = mols
            lst = {
                "o2": self.mO2[-1],
                "h2": self.mH2[-1],
                "h2o": self.mH2O[-1],
                "co": self.mCO[-1],
                "co2": self.mCO2[-1],
                "ch4": self.mCH4[-1],
            }

            mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

            wgt = (
                (
                    self.sys.atomicM["c"] / cnst.m["c"]
                    - sl.co2_melt(
                        (CO2.Y * self.mCO2[-1] * self.sys.P),
                        CO2,
                        (O2.Y * self.mO2[-1] * self.sys.P),
                        self.sys.T,
                        self.sys.P,
                        melt,
                        name=self.sys.run.C_MODEL,
                    )
                    - sl.co_melt(
                        (CO.Y * self.mCO[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CO_MODEL,
                    )
                    - sl.ch4_melt(
                        (CH4.Y * self.mCH4[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CH4_MODEL,
                    )
                    - melt.graph_current
                )
                / (self.mCO[-1] + self.mCO2[-1] + self.mCH4[-1])
            ) * sum(mjMj)

        elif self.sys.run.GAS_SYS == "SOH":
            H2O, O2, H2 = mols[:3]
            lst = {
                "o2": self.mO2[-1],
                "h2": self.mH2[-1],
                "h2o": self.mH2O[-1],
                "s2": self.mS2[-1],
                "so2": self.mSO2[-1],
                "h2s": self.mH2S[-1],
            }

            mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

            wgt = (
                (
                    self.sys.atomicM["h"] / (2 * cnst.m["h"])
                    - (
                        sl.h2o_melt(
                            self.mH2O[-1], H2O, self.sys.P, name=self.sys.run.H2O_MODEL
                        )
                        + sl.h2_melt(
                            self.mH2[-1],
                            H2,
                            self.sys.P,
                            melt,
                            name=self.sys.run.H2_MODEL,
                        )
                    )
                )
                / (self.mH2O[-1] + self.mH2[-1] + self.mH2S[-1])
            ) * sum(mjMj)

        elif self.sys.run.GAS_SYS == "COHS":
            H2O, O2, H2, CO, CO2, CH4 = mols[:6]
            lst = {
                "o2": self.mO2[-1],
                "h2": self.mH2[-1],
                "h2o": self.mH2O[-1],
                "co": self.mCO[-1],
                "co2": self.mCO2[-1],
                "ch4": self.mCH4[-1],
                "s2": self.mS2[-1],
                "so2": self.mSO2[-1],
                "h2s": self.mH2S[-1],
            }

            mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

            wgt = (
                (
                    self.sys.atomicM["c"] / cnst.m["c"]
                    - sl.co2_melt(
                        (CO2.Y * self.mCO2[-1] * self.sys.P),
                        CO2,
                        (O2.Y * self.mO2[-1] * self.sys.P),
                        self.sys.T,
                        self.sys.P,
                        melt,
                        name=self.sys.run.C_MODEL,
                    )
                    - sl.co_melt(
                        (CO.Y * self.mCO[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CO_MODEL,
                    )
                    - sl.ch4_melt(
                        (CH4.Y * self.mCH4[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CH4_MODEL,
                    )
                    - melt.graph_current
                )
                / (self.mCO[-1] + self.mCO2[-1] + self.mCH4[-1])
            ) * sum(mjMj)

        elif self.sys.run.GAS_SYS == "COHSN":
            H2O, O2, H2, CO, CO2, CH4 = mols[:6]
            lst = {
                "o2": self.mO2[-1],
                "h2": self.mH2[-1],
                "h2o": self.mH2O[-1],
                "co": self.mCO[-1],
                "co2": self.mCO2[-1],
                "ch4": self.mCH4[-1],
                "s2": self.mS2[-1],
                "so2": self.mSO2[-1],
                "h2s": self.mH2S[-1],
                "n2": self.mN2[-1],
            }

            mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

            wgt = (
                (
                    self.sys.atomicM["c"] / cnst.m["c"]
                    - sl.co2_melt(
                        (CO2.Y * self.mCO2[-1] * self.sys.P),
                        CO2,
                        (O2.Y * self.mO2[-1] * self.sys.P),
                        self.sys.T,
                        self.sys.P,
                        melt,
                        name=self.sys.run.C_MODEL,
                    )
                    - sl.co_melt(
                        (CO.Y * self.mCO[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CO_MODEL,
                    )
                    - sl.ch4_melt(
                        (CH4.Y * self.mCH4[-1] * self.sys.P),
                        self.sys.P,
                        name=self.sys.run.CH4_MODEL,
                    )
                    - melt.graph_current
                )
                / (self.mCO[-1] + self.mCO2[-1] + self.mCH4[-1])
            ) * sum(mjMj)

        if wgt < 0.0:
            raise ValueError("Gas weight fraction is negative.")
        else:
            return wgt

    def get_vol_frac(self, melt):
        """
        Returns the gas volume fraction of the system.

        Eq 15 of Burguisser & Scaillet (2015).

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class

        Returns
        -------
        GvF : float
            The volume fraction of gas in the system.
        """

        if self.sys.run.GAS_SYS == "OH":
            self.wt = cnvs.mols2wts(H2O=self.mH2O[-1], O2=self.mO2[-1], H2=self.mH2[-1])
            empty_lists = {
                "CO2": 0.0,
                "CO": 0.0,
                "CH4": 0.0,
                "S2": 0.0,
                "SO2": 0.0,
                "H2S": 0.0,
                "N2": 0.0,
            }
            M = cnvs.mean_mol_wt(H2O=self.mH2O[-1], O2=self.mO2[-1], H2=self.mH2[-1])
        elif self.sys.run.GAS_SYS == "COH":
            empty_lists = {"S2": 0.0, "SO2": 0.0, "H2S": 0.0, "N2": 0.0}
            self.wt = cnvs.mols2wts(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
            )
            M = cnvs.mean_mol_wt(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
            )
        elif self.sys.run.GAS_SYS == "SOH":
            empty_lists = {"CO2": 0.0, "CO": 0.0, "CH4": 0.0, "N2": 0.0}
            self.wt = cnvs.mols2wts(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
            )
            M = cnvs.mean_mol_wt(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
            )
        elif self.sys.run.GAS_SYS == "COHS":
            empty_lists = {"N2": 0.0}
            self.wt = cnvs.mols2wts(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
            )
            M = cnvs.mean_mol_wt(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
            )
        elif self.sys.run.GAS_SYS == "COHSN":
            empty_lists = None
            self.wt = cnvs.mols2wts(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
                N2=self.mN2[-1],
            )
            M = cnvs.mean_mol_wt(
                H2O=self.mH2O[-1],
                O2=self.mO2[-1],
                H2=self.mH2[-1],
                CO=self.mCO[-1],
                CO2=self.mCO2[-1],
                CH4=self.mCH4[-1],
                S2=self.mS2[-1],
                SO2=self.mSO2[-1],
                H2S=self.mH2S[-1],
                N2=self.mN2[-1],
            )

        for key in self.wt:
            if key in self.Wt:
                self.Wt[key].append(self.wt[key])
            else:
                self.Wt[key] = [self.wt[key]]

        if empty_lists is not None:
            for key in empty_lists:
                if key in self.Wt:
                    self.Wt[key].append(empty_lists[key])
                else:
                    self.Wt[key] = [empty_lists[key]]

        GvF = 1 / (
            1
            + (
                (M * self.sys.P * 1e5 * (1 - self.sys.WgT[-1]))
                / (
                    cnst.R
                    * self.sys.T
                    * melt.rho(P=(self.sys.P * 1e5))
                    * self.sys.WgT[-1]
                )
            )
        )

        return GvF

    def get_fugacity(self, molecules, gas_phase):
        """Returns the fugacity of each species in the gas phase.

        Saves the fugacities directly to self.f. Inputs as [H2O, O2,...],
        [gas.mH2O[-1], gas.mO2[-1],...].

        Parameters
        ----------
        molecules : list of Molecule instances
            List of all the Molecule instances, corresponding to all
            species in the gas phase.
        gas_phase : list of floats
            List of gas phase mole fractions, in same order as `molecules`
        """

        for m, g in zip(molecules, gas_phase, strict=True):
            if isinstance(m, str):
                if m in self.f:
                    self.f[m].append(0.0)
                else:
                    self.f[m] = [0.0]

            else:
                if m.Mol in self.f:
                    self.f[m.Mol].append(
                        m.Y * g * self.sys.P
                    )  # appends to the f dictionary for storage.
                else:
                    self.f[m.Mol] = [m.Y * g * self.sys.P]

    def rho(self):
        """Returns the density of the gas phase"""
        M = 0
        for mol in self.sys.SC:
            M += self.wt[mol] * cnst.m[mol.lower()]

        return ((M / 1000) * self.sys.P * 1e5) / (cnst.R * self.sys.T)

    def get_ys(self, system):
        """
        Returns the fugacity coefficients of all the gas phase species

        Calls the solvgas function `set_Y()`.

        Parameters
        ----------
        system : tuple of Molecule instances
            Tuple containing all the Molecule class instances,
            corresponding to all the species in the gas phase.
        """

        return sg.set_Y(self.sys, system)

    def get_atomic_mass(self, run, sys):
        """
        Calculates the mass fraction of each element in the gas phase only

        Parameters
        ----------
        run : RunDef class
            Active instance of the RunDef class
        sys : ThermoSystem class
            Active instance of the ThermoSystem class

        Warnings
        --------
        Currently unused, only valid for 'gas only' runs currently not
        implemented.
        """

        if run.GAS_SYS == "COHS" and sys.OCS is True:
            mO2 = self.mO2[-1]
            mH2O = self.mH2O[-1]
            mH2 = self.mH2[-1]
            mCO2 = self.mCO2[-1]
            mCO = self.mCO[-1]
            mCH4 = self.mCH4[-1]
            mSO2 = self.mSO2[-1]
            mS2 = self.mS2[-1]
            mH2S = self.mH2S[-1]

            lst = {
                "o2": mO2,
                "h2": mH2,
                "h2o": mH2O,
                "co": mCO,
                "co2": mCO2,
                "ch4": mCH4,
                "so2": mSO2,
                "s2": mS2,
                "h2s": mH2S,
            }
            mjMj = [lst[ele] * cnst.m[ele] for ele in lst]

            # WtO
            self.atomicM["o"] = cnst.m["o"] * (
                (sys.WgT[-1] * mH2O) / sum(mjMj)
                + (2 * (sys.WgT[-1] * mO2) / sum(mjMj))
                + 2 * (sys.WgT[-1] * mSO2) / sum(mjMj)
                + (sys.WgT[-1] * mCO) / sum(mjMj)
                + 2 * (sys.WgT[-1] * mCO2) / sum(mjMj)
            )

            # WtH
            self.atomicM["h"] = (
                2
                * cnst.m["h"]
                * (
                    (sys.WgT[-1] * mH2O) / sum(mjMj)
                    + (sys.WgT[-1] * mH2) / sum(mjMj)
                    + (sys.WgT[-1] * mH2S) / sum(mjMj)
                    + 2 * ((sys.WgT[-1] * mCH4) / sum(mjMj))
                )
            )

            # WtC
            self.atomicM["c"] = cnst.m["c"] * (
                (sys.WgT[-1] * mCO) / sum(mjMj)
                + (sys.WgT[-1] * mCO2) / sum(mjMj)
                + (sys.WgT[-1] * mCH4) / sum(mjMj)
            )

            # WtS
            self.atomicM["s"] = cnst.m["s"] * (
                (2 * sys.WgT[-1] * mS2) / sum(mjMj)
                + (sys.WgT[-1] * mH2S) / sum(mjMj)
                + (sys.WgT[-1] * mSO2) / sum(mjMj)
            )

    def open_system(self, melt, fraction):
        """
        Reduces the gas fraction by removing an aliquot.

        Removes a gas fraction from the system in accordance with open
        system degassing. Fraction to be removed is controlled by the user
        in the environment file. If fraction = 0.9, 10% 0f the gas phase
        will remain after a pressure step.

        Then recalculates the masses of each volatile element in the
        system now some has been removed.

        Parameters
        ----------
        melt : Melt class
            Active instance of the Melt class
        fraction : float
            Fraction of the gas phase to be removed after every pressure step.
        """

        keep = 1 - fraction
        self.sys.WgT[-1] = self.sys.WgT[-1] * keep

        self.sys.atomicM["h"] = cnvs.atomicM_calc(
            self.sys, melt, self, "h", -1, WgT=self.sys.WgT[-1]
        )
        self.sys.atomicM["o"] = cnvs.atomicM_calc(
            self.sys, melt, self, "o", -1, WgT=self.sys.WgT[-1]
        )
        self.sys.atomicM["o_tot"] = cnvs.atomicM_calc(
            self.sys, melt, self, "o_tot", -1, WgT=self.sys.WgT[-1]
        )
        self.sys.atomicM["c"] = cnvs.atomicM_calc(
            self.sys, melt, self, "c", -1, WgT=self.sys.WgT[-1]
        )
        self.sys.atomicM["s"] = cnvs.atomicM_calc(
            self.sys, melt, self, "s", -1, WgT=self.sys.WgT[-1]
        )
        self.sys.atomicM["n"] = cnvs.atomicM_calc(
            self.sys, melt, self, "n", -1, WgT=self.sys.WgT[-1]
        )


#  -------------------------------------------------------------------------
class Output:
    """
    To request the desired outputs through the output control file

    Attributes
    ----------
    plot_melt_species : bool
        If True, the melt volatile content will be plotted against
        pressure at the end of the run.
    plot_gas_species_wt : bool
        If True, the composition of the gas phase as weight fractions
        will be plotted against pressure at the end of the run.
    plot_gas_species_mol : bool
        If True, the composition of the gas phase as mole fractions
        will be plotted against pressure at the end of the run.
    plot_gas_fraction : bool
        If True, the gas volume fraction will be plotted against pressure
    plot_fo2_dFMQ : bool
        If True, fO2 relative to FMQ will be plotted against pressure
    """

    n_par = 5

    def __init__(self):
        # Graphical outputs
        self.plot_melt_species = True
        self.plot_gas_species_wt = True
        self.plot_gas_species_mol = True
        self.plot_gas_fraction = True
        self.plot_fo2_dFMQ = False

    def set_outputs(self, params):
        """
        Sets class up with parameters from the file.

        Checks for invalid strings and values from the file.

        Parameters
        ----------
        params : dict
            dictionary of attribute:value pairs read in from yaml file.
        """

        for par, val in params.items():
            if self.zTest_Params(par):
                setattr(self, par, val)

            else:
                raise ValueError("Invalid output choice: %s" % par)

    # helper methods
    def zTest_Params(self, par):
        """Checks `par` is a valid attribute of the Output class"""

        for item in inspect.getmembers(self):
            if item[0] == par:
                return True
        return 0

    def print_conditions(self):
        """Prints the chosen outputs to the terminal after setup with file."""

        length = 0
        for item in inspect.getmembers(self):
            if not inspect.ismethod(self):
                if length < self.n_par:
                    print(item[0], ":", item[1])
                    length += 1
                else:
                    break
        print("\n")
''',
"solver": r'''# solver.py
"""STORES THE SOLVING EQUATIONS AND SOLVE ROUTINE"""

from __future__ import annotations

import warnings

import numpy as np
from scipy import optimize

import evo.constants as cnst
import evo.conversions as cnvs
import evo.messages as msgs
import evo.solubility_laws as sl
import evo.solvgas as sg


def decompress(run, sys, melt, gas, system):
    """
    Main model function to calculate the gas+melt chemistry at a single PT condition.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    system : {[H2O, O2, H2],
              [H2O, O2, H2, CO, CO2, CH4],
              [H2O, O2, H2, S2, SO2, H2S],
              [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
              [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
        A list of the Molecule classes active.

    Returns
    -------
    None
    """

    # ----------------------------------------------------------------------------------
    # FUNCTION DEFINITIONS
    # ----------------------------------------------------------------------------------

    # OH equations
    def oh_eqs(guesses: list[float], fe=True):
        """
        Defines the non-linear equation set to be solved for the OH system.

        Parameters
        ----------
        guesses : list of 1 float
            The value of mO2 to be iterated.
        fe : bool, True
            If true, redox equilibrium between melt and the gas is enforced

        Returns
        -------
        float
            The difference between the total atomic mass of O in the system
            fixed by the initial conditions, and that calculated using the
            `guess` value for mO2.
        """

        x = guesses[0]  # x = mO2

        mH2O = (1 - x) / (
            1 + (H2O.Y / (sys.K["K1"] * H2.Y * (x * O2.Y * sys.P) ** 0.5))
        )

        mH2 = 1 - mH2O - x

        H = (
            (sys.atomicM["h"] / (2 * cnst.m["h"]))
            - sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            - sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
        ) / (mH2O + mH2)

        if fe is True:
            ofe = sys.fe_equil(melt, x, O2) / cnst.m["o"]
            atomico = sys.atomicM["o_tot"]
        elif fe is False:
            ofe = 0
            atomico = sys.atomicM["o"]

        return [
            (
                (H * (mH2O + 2 * x))
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + ofe
            )
            - atomico / cnst.m["o"]
        ]

    # COH system -----------------------------------------------------------------------
    def coh_eqs(guesses: list[float], fe=True, graph=False):
        """
        Defines the non-linear equation set to be solved for the COH system.

        Parameters
        ----------
        guesses : list of 2 floats
            The values of mO2, mCO to be iterated.
        fe : bool
            If true, redox equilibrium between melt and the gas is enforced, by default
            True.
        graph : bool
            If true, the melt is graphite saturated. fCO2 is then controlled by the CCO
            buffer not gas phase equilibrium, and guesses[1] is overridden. By default
            False.

        Returns
        -------
        list of floats
            The difference between the total atomic masses of elements in the system
            fixed by the initial conditions, and that calculated using the
            `guess` value for mO2 and mCO. Returned as [dO, dC].

        Warns
        ------
        RuntimeWarning
            If a NaN appears in the guesses generated by the scipy solver.
        """
        # x = mO2  y = mCO

        for g in guesses:
            if np.isnan(g):
                raise RuntimeWarning("NaN in guesses")
            elif g <= 0.0:
                guesses[np.where(guesses == g)] = 1e-20

        if graph is False:
            x, y = guesses[0], guesses[1]

            mCO2 = (sys.K["K2"] * CO.Y * y * (O2.Y * x * sys.P) ** 0.5) / CO2.Y

        elif graph is True:
            x = guesses[0]

            mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * x * sys.P)) / (CO2.Y * sys.P)

            y = (CO2.Y * mCO2) / (
                sys.K["K2"] * CO.Y * (O2.Y * x * sys.P) ** 0.5
            )  # mCO, dependent on CO2 from graphite saturation.

        a = (
            CO2.Y
            * mCO2
            * (H2O.Y * sys.P) ** 2
            / (sys.K["K3"] * CH4.Y * (x * O2.Y * sys.P) ** 2)
        )
        b = 1 + (
            (H2O.Y * sys.P) / (sys.K["K1"] * H2.Y * sys.P * (O2.Y * x * sys.P) ** 0.5)
        )
        c = -(1 - x - y - mCO2)

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        try:
            mH2O = mh2o[0]
        except IndexError:
            mH2O = float("nan")

        mH2 = H2O.Y * mH2O / (sys.K["K1"] * H2.Y * (O2.Y * x * sys.P) ** 0.5)

        mCH4 = (CO2.Y * mCO2 * (H2O.Y * mH2O) ** 2) / (
            sys.K["K3"] * CH4.Y * (O2.Y * x) ** 2
        )

        H = (
            (sys.atomicM["h"] / (2 * cnst.m["h"]))
            - sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            - sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            - 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        ) / (mH2O + mH2 + 2 * mCH4)

        if fe is True:
            ofe = sys.fe_equil(melt, x, O2) / cnst.m["o"]
            atomico = sys.atomicM["o_tot"]
        elif fe is False:
            ofe = 0
            atomico = sys.atomicM["o"]

        if graph is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys, melt, sys.P, CO2, y, mCO2, mCH4, x, CO2.Y, CO.Y, CH4.Y, O2.Y, H
                )
                / cnst.m["c"]
            )

            return [
                (
                    H * (mH2O + 2 * x + y + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * x * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * y * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"])
            ]

        elif graph is False:
            melt.graph_current = 0.0

            return [
                (
                    H * (mH2O + 2 * x + y + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * x * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * y * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"]),
                (
                    H * (y + mCO2 + mCH4)
                    + sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * x * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * y * sys.P), sys.P, name=run.CO_MODEL)
                    + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
                )
                - (sys.atomicM["c"] / (cnst.m["c"])),
            ]

    def soh_eqs(guesses: list[float], fe=True):
        """
        Defines the non-linear equation set to be solved for the SOH system.

        Parameters
        ----------
        guesses : list of 2 floats
            The values of mO2, mS2 to be iterated.
        fe : bool
            If true, redox equilibrium between melt and the gas is enforced, by default
            True.

        Returns
        -------
        list of floats
            The difference between the total atomic masses of elements in the system
            fixed by the initial conditions, and that calculated using the
            `guess` value for mO2 and mCO. Returned as [dO, dS].

        Warns
        ------
        RuntimeWarning
            If a NaN appears in the guesses generated by the scipy solver.
        """
        # x = mO2, y = mS2

        for g in guesses:
            if np.isnan(g):
                raise RuntimeWarning("NaN in guesses")
            elif g <= 0.0:
                guesses[np.where(guesses == g)] = 1e-20

        x, y = guesses[0], guesses[1]

        mSO2 = (sys.K["K5"] * O2.Y * x * np.sqrt(S2.Y * y * sys.P)) / SO2.Y

        mH2S = (1 - x - y - mSO2) / (
            1
            + (H2S.Y / (sys.K["K1"] * sys.K["K4"] * H2.Y * np.sqrt(S2.Y * y * sys.P)))
            + ((H2S.Y * np.sqrt(O2.Y * x)) / (H2O.Y * sys.K["K4"] * np.sqrt(S2.Y * y)))
        )

        mH2 = (
            H2S.Y
            * mH2S
            / (sys.K["K1"] * sys.K["K4"] * H2.Y * (S2.Y * y * sys.P) ** 0.5)
        )

        mH2O = (H2S.Y * mH2S * np.sqrt(O2.Y * x)) / (
            H2O.Y * sys.K["K4"] * np.sqrt(S2.Y * y)
        )

        mH2 = H2O.Y * mH2O / (H2.Y * sys.K["K1"] * np.sqrt(O2.Y * x * sys.P))

        S = (
            (sys.atomicM["s"] / cnst.m["s"])
            - sl.sulfide_melt(
                (S2.Y * y * sys.P),
                (O2.Y * x * sys.P),
                sys.P,
                sys.T,
                melt,
                name=run.SULFIDE_CAPACITY,
            )
            - sl.sulfate_melt(
                (S2.Y * y * sys.P),
                (O2.Y * x * sys.P),
                sys.P,
                sys.T,
                melt,
                run,
                name=run.SULFATE_CAPACITY,
            )
        ) / (2 * y + mH2S + mSO2)

        if fe is True:
            ofe = sys.fe_equil(melt, x, O2) / cnst.m["o"]
            atomico = sys.atomicM["o_tot"]
        elif fe is False:
            ofe = 0
            atomico = sys.atomicM["o"]

        return [
            (
                S * (mH2O + 2 * x + 2 * mSO2)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + ofe
            )
            - (atomico / cnst.m["o"]),
            (
                S * (mH2 + mH2O + mH2S)
                + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            )
            - (sys.atomicM["h"] / (2 * cnst.m["h"])),
        ]

    def cohs_eqs(guesses: list[float], fe=True, graph=False):
        """
        Defines the non-linear equation set to be solved for the COHS system.

        Parameters
        ----------
        guesses : list of floats
            Values of [(mCO) mS2, mO2] to be iterated. mCO left out if `fe` = False.
        fe : bool
            If true, redox equilibrium between melt and the gas is enforced, by default
            True.
        graph : bool
            If true, the melt is graphite saturated. fCO2 is then controlled by the CCO
            buffer not gas phase equilibrium, and guesses[0] is overridden. By default
            False.

        Returns
        -------
        list of floats
            The difference between the total atomic masses of elements in the system
            fixed by the initial conditions, and that calculated using the `guess`
            values. Returned as [dO, dC, dS] if `graph` = False, else [dO, dS].

        Warns
        ------
        RuntimeWarning
            If a NaN appears in the guesses generated by the scipy solver.
        """
        # x = mCO, y = mS2, z = mO2

        for g in guesses:
            if np.isnan(g):
                raise RuntimeWarning("NaN in guesses")
            elif g <= 0.0:
                guesses[np.where(guesses == g)] = 1e-20

        if graph is False:
            x, y, z = guesses[0], guesses[1], guesses[2]

            mCO2 = (sys.K["K2"] * CO.Y * x * (O2.Y * z * sys.P) ** 0.5) / CO2.Y

        elif graph is True:
            y, z = guesses[0], guesses[1]

            mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * z * sys.P)) / (CO2.Y * sys.P)

            x = (CO2.Y * mCO2) / (
                sys.K["K2"] * CO.Y * (O2.Y * z * sys.P) ** 0.5
            )  # mCO, dependent on CO2 from graphite saturation.

        mSO2 = (sys.K["K5"] * O2.Y * z * (S2.Y * y * sys.P) ** 0.5) / SO2.Y

        a = (CO2.Y * mCO2 * H2O.Y**2) / (sys.K["K3"] * CH4.Y * (O2.Y * z) ** 2)
        b = (
            (H2O.Y / (sys.K["K1"] * H2.Y * (O2.Y * z * sys.P) ** 0.5))
            + (sys.K["K4"] * H2O.Y * (S2.Y * y) ** 0.5 / (H2S.Y * (O2.Y * z) ** 0.5))
            + 1
        )
        c = -(1 - z - mCO2 - x - mSO2 - y)

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        try:
            mH2O = mh2o[0]
        except IndexError:
            mH2O = float("nan")

        mH2 = (H2O.Y * mH2O) / (sys.K["K1"] * H2.Y * (O2.Y * z * sys.P) ** 0.5)

        mCH4 = (CO2.Y * mCO2 * (H2O.Y * mH2O) ** 2) / (
            sys.K["K3"] * CH4.Y * (O2.Y * z) ** 2
        )

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * (S2.Y * y) ** 0.5) / (
            H2S.Y * (O2.Y * z) ** 0.5
        )

        H = (
            (sys.atomicM["h"] / (2 * cnst.m["h"]))
            - sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
            - sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
            - 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
        ) / (mH2O + mH2 + mH2S + 2 * mCH4)

        if fe is True:
            ofe = sys.fe_equil(melt, z, O2) / cnst.m["o"]
            atomico = sys.atomicM["o_tot"]
        elif fe is False:
            ofe = 0
            atomico = sys.atomicM["o"]

        if graph is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys, melt, sys.P, CO2, x, mCO2, mCH4, z, CO2.Y, CO.Y, CH4.Y, O2.Y, H
                )
                / cnst.m["c"]
            )

            return [
                (
                    H * (2 * z + 2 * mSO2 + mH2O + x + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"]),
                (
                    H * (mSO2 + mH2S + 2 * y)
                    + sl.sulfide_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        name=run.SULFIDE_CAPACITY,
                    )
                    + sl.sulfate_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        run,
                        name=run.SULFATE_CAPACITY,
                    )
                )
                - (sys.atomicM["s"] / cnst.m["s"]),
            ]

        elif graph is False:
            melt.graph_current = 0.0

            return [
                (
                    H * (2 * z + 2 * mSO2 + mH2O + x + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"]),
                (
                    H * (x + mCO2 + mCH4)
                    + sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
                )
                - (sys.atomicM["c"] / (cnst.m["c"])),
                (
                    H * (mSO2 + mH2S + 2 * y)
                    + sl.sulfide_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        name=run.SULFIDE_CAPACITY,
                    )
                    + sl.sulfate_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        run,
                        name=run.SULFATE_CAPACITY,
                    )
                )
                - (sys.atomicM["s"] / cnst.m["s"]),
            ]

    def cohsn_eqs(guesses: list[float], fe=True, graph=False):
        """
        Defines the non-linear equation set to be solved for the COHSN system.

        Parameters
        ----------
        guesses : list of floats
            Values of [mN2, (mCO), mS2, mO2] to be iterated. mCO left out if
            `fe` = False.
        fe : bool
            If true, redox equilibrium between melt and the gas is enforced, by default
            True.
        graph : bool
            If true, the melt is graphite saturated. fCO2 is then controlled by the CCO
            buffer not gas phase equilibrium, and mCO is not required. By default False.

        Returns
        -------
        list of floats
            The difference between the total atomic masses of elements in the system
            fixed by the initial conditions, and that calculated using the `guess`
            values. Returned as [dO, dH, dS, dC] if `graph` = False, else [dO, dH, dS].

        Warns
        ------
        RuntimeWarning
            If a NaN appears in the guesses generated by the scipy solver.
        """
        # w = mN2 x = mCO, y = mS2, z = mO2

        for g in guesses:
            if np.isnan(g):
                raise RuntimeWarning("NaN in guesses")
            elif g <= 0.0:
                guesses[np.where(guesses == g)] = 1e-20

        if graph is False:
            w, x, y, z = guesses[0], guesses[1], guesses[2], guesses[3]

            mCO2 = (sys.K["K2"] * CO.Y * x * (O2.Y * z * sys.P) ** 0.5) / CO2.Y

        elif graph is True:
            w, y, z = guesses[0], guesses[1], guesses[2]

            mCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * z * sys.P)) / (CO2.Y * sys.P)

            x = (CO2.Y * mCO2) / (
                sys.K["K2"] * CO.Y * (O2.Y * z * sys.P) ** 0.5
            )  # mCO, dependent on CO2 from graphite saturation.

        mSO2 = (sys.K["K5"] * O2.Y * z * (S2.Y * y * sys.P) ** 0.5) / SO2.Y

        a = (CO2.Y * mCO2 * H2O.Y**2) / (sys.K["K3"] * CH4.Y * (O2.Y * z) ** 2)
        b = (
            (H2O.Y / (sys.K["K1"] * H2.Y * (O2.Y * z * sys.P) ** 0.5))
            + (sys.K["K4"] * H2O.Y * (S2.Y * y) ** 0.5 / (H2S.Y * (O2.Y * z) ** 0.5))
            + 1
        )
        c = -(1 - z - mCO2 - x - mSO2 - y - w)

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        try:
            mH2O = mh2o[0]
        except IndexError:
            mH2O = float("nan")

        mH2 = (H2O.Y * mH2O) / (sys.K["K1"] * H2.Y * (O2.Y * z * sys.P) ** 0.5)

        mCH4 = (CO2.Y * mCO2 * (H2O.Y * mH2O) ** 2) / (
            sys.K["K3"] * CH4.Y * (O2.Y * z) ** 2
        )

        mH2S = (sys.K["K4"] * H2O.Y * mH2O * (S2.Y * y) ** 0.5) / (
            H2S.Y * (O2.Y * z) ** 0.5
        )

        N = (
            (sys.atomicM["n"] / cnst.m["n"])
            - sl.n_melt(w, (O2.Y * z * sys.P), sys.P, name=run.N_MODEL)
        ) / (2 * w)

        if fe is True:
            ofe = sys.fe_equil(melt, z, O2) / cnst.m["o"]
            atomico = sys.atomicM["o_tot"]
        elif fe is False:
            ofe = 0
            atomico = sys.atomicM["o"]

        if graph is True:
            melt.graph_current = (
                cnvs.get_graphite(
                    sys, melt, sys.P, CO2, x, mCO2, mCH4, z, CO2.Y, CO.Y, CH4.Y, O2.Y, N
                )
                / cnst.m["c"]
            )

            return [
                (
                    N * (2 * z + 2 * mSO2 + mH2O + x + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"]),
                (
                    N * (mH2O + mH2 + mH2S + 2 * mCH4)
                    + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
                )
                - (sys.atomicM["h"] / (2 * cnst.m["h"])),
                (
                    N * (mSO2 + mH2S + 2 * y)
                    + sl.sulfide_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        name=run.SULFIDE_CAPACITY,
                    )
                    + sl.sulfate_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        run,
                        name=run.SULFATE_CAPACITY,
                    )
                )
                - (sys.atomicM["s"] / cnst.m["s"]),
            ]

        elif graph is False:
            melt.graph_current = 0.0

            return [
                (
                    N * (2 * z + 2 * mSO2 + mH2O + x + 2 * mCO2)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2
                    * sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + ofe
                )
                - (atomico / cnst.m["o"]),
                (
                    N * (mH2O + mH2 + mH2S + 2 * mCH4)
                    + sl.h2_melt(mH2, H2, sys.P, melt, name=run.H2_MODEL)
                    + sl.h2o_melt(mH2O, H2O, sys.P, name=run.H2O_MODEL)
                    + 2 * sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
                )
                - (sys.atomicM["h"] / (2 * cnst.m["h"])),
                (
                    N * (mSO2 + mH2S + 2 * y)
                    + sl.sulfide_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        name=run.SULFIDE_CAPACITY,
                    )
                    + sl.sulfate_melt(
                        (S2.Y * y * sys.P),
                        (O2.Y * z * sys.P),
                        sys.P,
                        sys.T,
                        melt,
                        run,
                        name=run.SULFATE_CAPACITY,
                    )
                )
                - (sys.atomicM["s"] / cnst.m["s"]),
                (
                    N * (x + mCO2 + mCH4)
                    + sl.co2_melt(
                        (CO2.Y * mCO2 * sys.P),
                        CO2,
                        (O2.Y * z * sys.P),
                        sys.T,
                        sys.P,
                        melt,
                        name=run.C_MODEL,
                    )
                    + sl.co_melt((CO.Y * x * sys.P), sys.P, name=run.CO_MODEL)
                    + sl.ch4_melt((CH4.Y * mCH4 * sys.P), sys.P, name=run.CH4_MODEL)
                )
                - (sys.atomicM["c"] / cnst.m["c"]),
            ]

    # Jacobian Newton-Rapheson ---------------------------------------------------------
    def jac_newton(run, gas, mols):
        """
        Solves for the key variables in the next pressure step using newton raphson.

        Takes the result from the previous pressure step as an initial guess,
        and runs scipy fsolve to optimize for the result. If the system does not
        converge, the pressure step is decreased by a factor of 10 and the system
        re-runs the calculation at a higher pressure.

        Parameters
        ----------
        run : RunDef class
            Active instance of RunDef class
        gas : Gas class
            Active instance of Gas class
        mols : {[H2O, O2, H2], [H2O, O2, H2, CO, CO2, CH4], [H2O, O2, H2, S2, SO2, H2S],
                [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S],
                [H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2]}
            All active instances of the Molecule class

        Returns
        -------
        tuple
            Appropriate number of variable solutions (1 for OH, 2 for COH etc).
        """

        # get initial guesses based on which volatile system is being run

        if run.GAS_SYS == "OH" or run.GAS_SYS == "COH" or run.GAS_SYS == "SOH":
            guessx = gas.mO2[-1]

            if run.GAS_SYS == "COH":
                guessy = gas.mCO[-1]
            if run.GAS_SYS == "SOH":
                guessy = gas.mS2[-1]

        elif run.GAS_SYS == "COHS" or run.GAS_SYS == "COHSN":
            guessx = gas.mCO[-1]
            guessy = gas.mS2[-1]
            guessz = gas.mO2[-1]

            if run.GAS_SYS == "COHSN":
                guessw = gas.mN2[-1]

        # Solve using scipy fsolve

        with warnings.catch_warnings():
            warnings.filterwarnings("error")
            initial_pstep = sys.P_step
            while sys.P_step >= run.DP_MIN:
                try:
                    if run.GAS_SYS == "OH":
                        newguess = optimize.fsolve(
                            oh_eqs, [guessx], args=(run.FE_SYSTEM)
                        )
                        return newguess[0]

                    elif run.GAS_SYS == "COH":
                        if melt.graphite_sat is True:
                            newguess = optimize.fsolve(
                                coh_eqs,
                                [guessx],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )

                            mCO2 = sl.graphite_fco2(
                                sys.T, sys.P, (O2.Y * newguess[0] * sys.P)
                            ) / (CO2.Y * sys.P)

                            mCO = (CO2.Y * mCO2) / (
                                sys.K["K2"] * CO.Y * (O2.Y * newguess[0] * sys.P) ** 0.5
                            )

                            return newguess[0], mCO

                        elif melt.graphite_sat is False:
                            newguess = optimize.fsolve(
                                coh_eqs,
                                [guessx, guessy],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )
                            return newguess[0], newguess[1]

                    elif run.GAS_SYS == "SOH":
                        newguess = optimize.fsolve(
                            soh_eqs, [guessx, guessy], args=(run.FE_SYSTEM)
                        )
                        return newguess[0], newguess[1]

                    elif run.GAS_SYS == "COHS":
                        if melt.graphite_sat is True:
                            newguess = optimize.fsolve(
                                cohs_eqs,
                                [guessy, guessz],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )

                            mCO2 = sl.graphite_fco2(
                                sys.T, sys.P, (O2.Y * newguess[1] * sys.P)
                            ) / (CO2.Y * sys.P)

                            mCO = (CO2.Y * mCO2) / (
                                sys.K["K2"] * CO.Y * (O2.Y * newguess[1] * sys.P) ** 0.5
                            )

                            return mCO, newguess[0], newguess[1]

                        elif melt.graphite_sat is False:
                            newguess = optimize.fsolve(
                                cohs_eqs,
                                [guessx, guessy, guessz],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )
                            return newguess[0], newguess[1], newguess[2]

                    elif run.GAS_SYS == "COHSN":
                        if melt.graphite_sat is True:
                            newguess = optimize.fsolve(
                                cohsn_eqs,
                                [guessw, guessy, guessz],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )

                            mCO2 = sl.graphite_fco2(
                                sys.T, sys.P, (O2.Y * newguess[2] * sys.P)
                            ) / (CO2.Y * sys.P)

                            mCO = (CO2.Y * mCO2) / (
                                sys.K["K2"] * CO.Y * (O2.Y * newguess[2] * sys.P) ** 0.5
                            )

                            return newguess[0], mCO, newguess[1], newguess[2]

                        elif melt.graphite_sat is False:
                            newguess = optimize.fsolve(
                                cohsn_eqs,
                                [guessw, guessx, guessy, guessz],
                                args=(run.FE_SYSTEM, melt.graphite_sat),
                            )
                            return newguess[0], newguess[1], newguess[2], newguess[3]

                except RuntimeWarning as err:
                    message = sys.variable_step()
                    if message is None:
                        sg.set_Y(sys, mols)
                        sys.P_track[-1] = sys.P
                        if sys.graph_unsat_rerun is True:
                            # make sure if previous step was graphite saturated,
                            # this is the condition we start with.
                            melt.graphite_sat = True
                    else:
                        # resets pstep for the iron equilibration scenario
                        sys.P_step = initial_pstep  # PL: is this still necessary?
                        print(message)
                        raise RuntimeError(message) from err

    # ----------------------------------------------------------------------------------
    # CALCULATION AND PRINT LOOPS
    # ----------------------------------------------------------------------------------

    if run.GAS_SYS == "OH":
        H2O, O2, H2 = system

        # Recalculate the activity coefficients at the new pressure
        if sys.P < run.P_START:
            gas.get_ys(system)

        try:
            guessx = jac_newton(run, gas, system)
        except RuntimeError:
            del sys.P_track[-1]  # delete failed pressure step
            msgs.earlyexit(
                sys,
                gas,
                melt,
                (
                    "Model failed to converge at lowest pressure step."
                    "\nData has been written out.\nExiting EVo."
                ),
            )

        if run.FE_SYSTEM is True:
            sys.fe_save(melt, guessx, O2)

        gas.mO2.append(guessx)

        print(gas.mO2[-1], sys.P, "End of a pressure step!!!!!!!!!!!!!!!!!!!!!!")

        gas.mH2O.append(
            (1 - gas.mO2[-1])
            / (1 + (H2O.Y / (sys.K["K1"] * H2.Y * (gas.mO2[-1] * O2.Y * sys.P) ** 0.5)))
        )
        gas.mH2.append(1 - gas.mO2[-1] - gas.mH2O[-1])
        empty_lists = [
            gas.mCO2,
            gas.mCO,
            gas.mCH4,
            gas.mSO2,
            gas.mS2,
            gas.mH2S,
            gas.mN2,
        ]
        for list in empty_lists:
            list.append(np.float64(0))

        gas.fo2.append(np.log(O2.Y * gas.mO2[-1] * sys.P))
        melt.fmq.append(cnvs.fo2_2fmq(gas.fo2[-1], sys.T, sys.P, sys.run.FMQ_MODEL))
        gas.get_fugacity(
            [H2O, H2, "CO2", "CO", "CH4", "S2", "SO2", "H2S", "N2"],
            [
                gas.mH2O[-1],
                gas.mH2[-1],
                gas.mCO2[-1],
                gas.mCO[-1],
                gas.mCH4[-1],
                gas.mS2[-1],
                gas.mSO2[-1],
                gas.mH2S[-1],
                gas.mN2[-1],
            ],
        )

        sys.WgT.append(gas.get_WgT(melt, H2O))
        melt.melt_composition(gas, system)
        sys.GvF.append(gas.get_vol_frac(melt))
        sys.rho.append(sys.rho_bulk(melt, gas))
        gas.M.append(cnvs.mean_mol_wt(H2O=gas.mH2O[-1], O2=gas.mO2[-1], H2=gas.mH2[-1]))

        # Check mass is being conserved

        if run.FE_SYSTEM is True:
            o = "o_tot"
        elif run.FE_SYSTEM is False:
            o = "o"

        for x in ["h", o]:
            if (
                abs(
                    cnvs.atomicM_calc(sys, melt, gas, x, -1, WgT=sys.WgT[-1])
                    - sys.atomicM[x]
                )
                / sys.atomicM[x]
            ) * 100 > 1e-5 and cnvs.atomicM_calc(
                sys, melt, gas, x, -1, WgT=sys.WgT[-1]
            ) != 0.0:
                sys.mass_conservation_reset(melt, gas)
                break

        if run.RUN_TYPE == "open":
            gas.open_system(melt, run.LOSS_FRAC)

    # COH system -----------------------------------------------------------------------
    elif run.GAS_SYS == "COH":
        H2O, O2, H2, CO, CO2, CH4 = system

        # Recalculate the activity coefficients at the new pressure
        if sys.P < run.P_START:
            gas.get_ys(system)

        try:
            guessx, guessy = jac_newton(run, gas, system)
        except RuntimeError:
            del sys.P_track[-1]  # delete failed pressure step
            msgs.earlyexit(
                sys,
                gas,
                melt,
                (
                    "Model failed to converge at lowest pressure step."
                    "\nData has been written out.\nExiting EVo."
                ),
            )

        if melt.graphite_sat is True and melt.graph_current <= 0:
            melt.graphite_sat = False
            melt.graph_current = 0
            guessx, guessy = jac_newton(run, gas, system)

        if run.C_MODEL == "eguchi2018":
            # Check graphite saturation
            graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * guessx * sys.P))

            fCO2 = sys.K["K2"] * CO.Y * guessy * sys.P * (O2.Y * guessx * sys.P) ** 0.5

            if fCO2 > graph_fCO2:
                print(
                    "oh no, graphite saturated... gas_fCO2:",
                    fCO2,
                    "graphite_fco2:",
                    graph_fCO2,
                )
                melt.graphite_sat = True

        if run.FE_SYSTEM is True:
            sys.fe_save(melt, guessx, O2)

        gas.mO2.append(guessx)
        gas.mCO.append(guessy)

        print(
            gas.mO2[-1],
            gas.mCO[-1],
            sys.P,
            "End of a pressure step!!!!!!!!!!!!!!!!!!!!!!",
        )

        # Append the speciation for the pressure step

        gas.mCO2.append(
            sys.K["K2"]
            * CO.Y
            * gas.mCO[-1]
            * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5
            / CO2.Y
        )

        a = (
            CO2.Y
            * gas.mCO2[-1]
            * (H2O.Y * sys.P) ** 2
            / (sys.K["K3"] * CH4.Y * (guessx * O2.Y * sys.P) ** 2)
        )
        b = 1 + (
            (H2O.Y * sys.P)
            / (sys.K["K1"] * H2.Y * sys.P * (O2.Y * guessx * sys.P) ** 0.5)
        )
        c = -(1 - guessx - guessy - gas.mCO2[-1])

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        gas.mH2O.append(mh2o[0])

        gas.mH2.append(
            H2O.Y
            * gas.mH2O[-1]
            / (sys.K["K1"] * H2.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5)
        )

        gas.mCH4.append(
            CO2.Y
            * gas.mCO2[-1]
            * (H2O.Y * gas.mH2O[-1] * sys.P) ** 2
            / (sys.K["K3"] * CH4.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 2)
        )

        empty_lists = [gas.mSO2, gas.mS2, gas.mH2S, gas.mN2]
        for list in empty_lists:
            list.append(np.float64(0))

        # Append all extraneous data for the pressure step

        gas.fo2.append(np.log(O2.Y * gas.mO2[-1] * sys.P))
        melt.fmq.append(cnvs.fo2_2fmq(gas.fo2[-1], sys.T, sys.P, sys.run.FMQ_MODEL))
        gas.get_fugacity(
            [H2O, H2, CO2, CO, CH4, "S2", "SO2", "H2S", "N2"],
            [
                gas.mH2O[-1],
                gas.mH2[-1],
                gas.mCO2[-1],
                gas.mCO[-1],
                gas.mCH4[-1],
                gas.mS2[-1],
                gas.mSO2[-1],
                gas.mH2S[-1],
                gas.mN2[-1],
            ],
        )

        sys.WgT.append(gas.get_WgT(melt, system))
        melt.melt_composition(gas, system)
        sys.GvF.append(gas.get_vol_frac(melt))
        sys.rho.append(sys.rho_bulk(melt, gas))
        gas.M.append(
            cnvs.mean_mol_wt(
                H2O=gas.mH2O[-1],
                O2=gas.mO2[-1],
                H2=gas.mH2[-1],
                CO=gas.mCO[-1],
                CO2=gas.mCO2[-1],
                CH4=gas.mCH4[-1],
            )
        )

        # Check mass is being conserved

        if run.FE_SYSTEM is True:
            o = "o_tot"
        elif run.FE_SYSTEM is False:
            o = "o"

        for x in ["c", o, "h"]:
            if (
                abs(
                    cnvs.atomicM_calc(sys, melt, gas, x, -1, WgT=sys.WgT[-1])
                    - sys.atomicM[x]
                )
                / sys.atomicM[x]
            ) * 100 > 1e-5 and cnvs.atomicM_calc(
                sys, melt, gas, x, -1, WgT=sys.WgT[-1]
            ) != 0.0:
                sys.mass_conservation_reset(melt, gas)
                break

        if run.RUN_TYPE == "open":
            gas.open_system(melt, run.LOSS_FRAC)

    elif run.GAS_SYS == "SOH":
        H2O, O2, H2, S2, SO2, H2S = system

        # Recalculate the activity coefficients at the new pressure

        if sys.P < run.P_START:
            gas.get_ys(system)

        try:
            guessx, guessy = jac_newton(run, gas, system)
        except RuntimeError:
            del sys.P_track[-1]  # delete failed pressure step
            msgs.earlyexit(
                sys,
                gas,
                melt,
                (
                    "Model failed to converge at lowest pressure step."
                    "\nData has been written out.\nExiting EVo."
                ),
            )

        if run.FE_SYSTEM is True:
            sys.fe_save(melt, guessx, O2)

        gas.mO2.append(guessx)
        gas.mS2.append(guessy)

        print(
            gas.mO2[-1],
            gas.mS2[-1],
            sys.P,
            "End of a pressure step!!!!!!!!!!!!!!!!!!!!!!",
        )

        gas.mSO2.append(
            (sys.K["K5"] * O2.Y * gas.mO2[-1] * (S2.Y * gas.mS2[-1] * sys.P) ** 0.5)
            / SO2.Y
        )

        gas.mH2S.append(
            (1 - gas.mO2[-1] - gas.mS2[-1] - gas.mSO2[-1])
            / (
                1
                + (
                    H2S.Y
                    / (
                        sys.K["K1"]
                        * sys.K["K4"]
                        * H2.Y
                        * (S2.Y * gas.mS2[-1] * sys.P) ** 0.5
                    )
                    + (
                        (H2S.Y * (O2.Y * gas.mO2[-1]) ** 0.5)
                        / (H2O.Y * sys.K["K4"] * (S2.Y * gas.mS2[-1]) ** 0.5)
                    )
                )
            )
        )

        gas.mH2.append(
            H2S.Y
            * gas.mH2S[-1]
            / (sys.K["K1"] * sys.K["K4"] * H2.Y * (S2.Y * gas.mS2[-1] * sys.P) ** 0.5)
        )

        gas.mH2O.append(
            1 - gas.mO2[-1] - gas.mH2[-1] - gas.mS2[-1] - gas.mSO2[-1] - gas.mH2S[-1]
        )

        empty_lists = [gas.mCO2, gas.mCO, gas.mCH4, gas.mN2]
        for list in empty_lists:
            list.append(
                np.float64(0.0)
            )  # to get inf rather than an error when printing

        gas.fo2.append(np.log(O2.Y * gas.mO2[-1] * sys.P))
        melt.fmq.append(cnvs.fo2_2fmq(gas.fo2[-1], sys.T, sys.P, sys.run.FMQ_MODEL))
        gas.get_fugacity(
            [H2O, H2, "CO2", "CO", "CH4", S2, SO2, H2S, "N2"],
            [
                gas.mH2O[-1],
                gas.mH2[-1],
                gas.mCO2[-1],
                gas.mCO[-1],
                gas.mCH4[-1],
                gas.mS2[-1],
                gas.mSO2[-1],
                gas.mH2S[-1],
                gas.mN2[-1],
            ],
        )

        # if FeS is saturated as pyrrhotite, solid phase so doesn't generate error,
        # but will mess up the S dissolved in magma.

        sys.WgT.append(gas.get_WgT(melt, system))
        melt.melt_composition(gas, system)
        sys.GvF.append(gas.get_vol_frac(melt))
        sys.rho.append(sys.rho_bulk(melt, gas))
        gas.M.append(
            cnvs.mean_mol_wt(
                H2O=gas.mH2O[-1],
                O2=gas.mO2[-1],
                H2=gas.mH2[-1],
                S2=gas.mS2[-1],
                SO2=gas.mSO2[-1],
                H2S=gas.mH2S[-1],
            )
        )

        # Check mass is being conserved

        if run.FE_SYSTEM is True:
            o = "o_tot"
        elif run.FE_SYSTEM is False:
            o = "o"

        for x in ["s", o, "h"]:
            if (
                abs(
                    cnvs.atomicM_calc(sys, melt, gas, x, -1, WgT=sys.WgT[-1])
                    - sys.atomicM[x]
                )
                / sys.atomicM[x]
            ) * 100 > 1e-5 and cnvs.atomicM_calc(
                sys, melt, gas, x, -1, WgT=sys.WgT[-1]
            ) != 0.0:
                sys.mass_conservation_reset(melt, gas)
                break

        if run.RUN_TYPE == "open":
            gas.open_system(melt, run.LOSS_FRAC)

    elif run.GAS_SYS == "COHS":
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S = system

        # Recalculate the activity coefficients at the new pressure

        if sys.P < run.P_START:
            gas.get_ys(system)

        try:
            guessx, guessy, guessz = jac_newton(run, gas, system)
        except RuntimeError:
            del sys.P_track[-1]  # delete failed pressure step
            msgs.earlyexit(
                sys,
                gas,
                melt,
                (
                    "Model failed to converge at lowest pressure step."
                    "\nData has been written out.\nExiting EVo."
                ),
            )

        if melt.graphite_sat is True and melt.graph_current <= 0:
            melt.graphite_sat = False
            melt.graph_current = 0
            guessx, guessy, guessz = jac_newton(run, gas, system)

        if run.C_MODEL == "eguchi2018":
            # Check graphite saturation
            graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * guessz * sys.P))

            fCO2 = sys.K["K2"] * CO.Y * guessx * sys.P * (O2.Y * guessz * sys.P) ** 0.5

            if fCO2 > graph_fCO2:
                print(
                    "oh no, graphite saturated... gas_fCO2:",
                    fCO2,
                    "graphite_fco2:",
                    graph_fCO2,
                )

        if run.FE_SYSTEM is True:
            sys.fe_save(melt, guessz, O2)

        gas.mCO.append(guessx)
        gas.mS2.append(guessy)
        gas.mO2.append(guessz)

        print(
            gas.mCO[-1],
            gas.mS2[-1],
            gas.mO2[-1],
            sys.P,
            "End of a pressure step!!!!!!!!!!!!!!!!!!!!!!",
        )

        gas.mCO2.append(
            (sys.K["K2"] * CO.Y * gas.mCO[-1] * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5)
            / CO2.Y
        )

        gas.mSO2.append(
            (sys.K["K5"] * O2.Y * gas.mO2[-1] * (S2.Y * gas.mS2[-1] * sys.P) ** 0.5)
            / SO2.Y
        )

        a = (CO2.Y * gas.mCO2[-1] * H2O.Y**2) / (
            sys.K["K3"] * CH4.Y * (O2.Y * gas.mO2[-1]) ** 2
        )
        b = (
            (H2O.Y / (sys.K["K1"] * H2.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5))
            + (
                sys.K["K4"]
                * H2O.Y
                * (S2.Y * gas.mS2[-1]) ** 0.5
                / (H2S.Y * (O2.Y * gas.mO2[-1]) ** 0.5)
            )
            + 1
        )
        c = -(1 - gas.mO2[-1] - gas.mCO2[-1] - gas.mCO[-1] - gas.mSO2[-1] - gas.mS2[-1])

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        gas.mH2O.append(mh2o[0])

        gas.mH2.append(
            (H2O.Y * gas.mH2O[-1])
            / (sys.K["K1"] * H2.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5)
        )

        gas.mCH4.append(
            (CO2.Y * gas.mCO2[-1] * (H2O.Y * gas.mH2O[-1]) ** 2)
            / (sys.K["K3"] * CH4.Y * (O2.Y * gas.mO2[-1]) ** 2)
        )

        gas.mH2S.append(
            (sys.K["K4"] * H2O.Y * gas.mH2O[-1] * (S2.Y * gas.mS2[-1]) ** 0.5)
            / (H2S.Y * (O2.Y * gas.mO2[-1]) ** 0.5)
        )

        empty_lists = [gas.mN2]
        for list in empty_lists:
            list.append(np.float64(0.0))

        gas.fo2.append(np.log(O2.Y * gas.mO2[-1] * sys.P))
        melt.fmq.append(cnvs.fo2_2fmq(gas.fo2[-1], sys.T, sys.P, sys.run.FMQ_MODEL))
        gas.get_fugacity(
            [H2O, H2, CO2, CO, CH4, S2, SO2, H2S, "N2"],
            [
                gas.mH2O[-1],
                gas.mH2[-1],
                gas.mCO2[-1],
                gas.mCO[-1],
                gas.mCH4[-1],
                gas.mS2[-1],
                gas.mSO2[-1],
                gas.mH2S[-1],
                gas.mN2[-1],
            ],
        )

        # if FeS is saturated as pyrrhotite, solid phase so doesn't generate error,
        # but will mess up the S dissolved in magma.(?)

        sys.WgT.append(gas.get_WgT(melt, system))
        melt.melt_composition(gas, system)
        sys.GvF.append(gas.get_vol_frac(melt))
        sys.rho.append(sys.rho_bulk(melt, gas))
        gas.M.append(
            cnvs.mean_mol_wt(
                H2O=gas.mH2O[-1],
                O2=gas.mO2[-1],
                H2=gas.mH2[-1],
                CO=gas.mCO[-1],
                CO2=gas.mCO2[-1],
                CH4=gas.mCH4[-1],
                S2=gas.mS2[-1],
                SO2=gas.mSO2[-1],
                H2S=gas.mH2S[-1],
            )
        )

        # Check mass is being conserved

        if run.FE_SYSTEM is True:
            o = "o_tot"
        elif run.FE_SYSTEM is False:
            o = "o"

        for x in ["h", "c", o, "s"]:
            if (
                abs(
                    cnvs.atomicM_calc(sys, melt, gas, x, -1, WgT=sys.WgT[-1])
                    - sys.atomicM[x]
                )
                / sys.atomicM[x]
            ) * 100 > 1e-5 and cnvs.atomicM_calc(
                sys, melt, gas, x, -1, WgT=sys.WgT[-1]
            ) != 0.0:
                sys.mass_conservation_reset(melt, gas)
                break

        # When running an open system,
        # remove a fraction of the gas phase for the next step.
        if run.RUN_TYPE == "open":
            gas.open_system(melt, run.LOSS_FRAC)

    elif run.GAS_SYS == "COHSN":
        H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2 = system

        # Recalculate the activity coefficients at the new pressure

        if sys.P < run.P_START:
            gas.get_ys(system)

        try:
            guessw, guessx, guessy, guessz = jac_newton(run, gas, system)
        except RuntimeError:
            del sys.P_track[-1]  # delete failed pressure step
            msgs.earlyexit(
                sys,
                gas,
                melt,
                (
                    "Model failed to converge at lowest pressure step."
                    "\nData has been written out.\nExiting EVo."
                ),
            )

        if melt.graphite_sat is True and melt.graph_current <= 0:
            melt.graphite_sat = False
            melt.graph_current = 0
            sys.graph_unsat_rerun = True
            guessw, guessx, guessy, guessz = jac_newton(run, gas, system)

        if run.C_MODEL == "eguchi2018":
            # Check graphite saturation
            graph_fCO2 = sl.graphite_fco2(sys.T, sys.P, (O2.Y * guessz * sys.P))

            fCO2 = sys.K["K2"] * CO.Y * guessx * sys.P * (O2.Y * guessz * sys.P) ** 0.5

            if fCO2 > graph_fCO2:
                print(
                    "oh no, graphite saturated... gas_fCO2:",
                    fCO2,
                    "graphite_fco2:",
                    graph_fCO2,
                )

            if sys.graph_unsat_rerun is True:
                # reset now the run is complete
                sys.graph_unsat_rerun = False

        if run.FE_SYSTEM is True:
            sys.fe_save(melt, guessz, O2)

        gas.mN2.append(guessw)
        gas.mCO.append(guessx)
        gas.mS2.append(guessy)
        gas.mO2.append(guessz)

        print(
            gas.mN2[-1],
            gas.mCO[-1],
            gas.mS2[-1],
            gas.mO2[-1],
            sys.P,
            "End of a pressure step!!!!!!!!!!!!!!!!!!!!!!",
        )

        gas.mCO2.append(
            (sys.K["K2"] * CO.Y * gas.mCO[-1] * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5)
            / CO2.Y
        )

        gas.mSO2.append(
            (sys.K["K5"] * O2.Y * gas.mO2[-1] * (S2.Y * gas.mS2[-1] * sys.P) ** 0.5)
            / SO2.Y
        )

        a = (CO2.Y * gas.mCO2[-1] * H2O.Y**2) / (
            sys.K["K3"] * CH4.Y * (O2.Y * gas.mO2[-1]) ** 2
        )
        b = (
            (H2O.Y / (sys.K["K1"] * H2.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5))
            + (
                sys.K["K4"]
                * H2O.Y
                * (S2.Y * gas.mS2[-1]) ** 0.5
                / (H2S.Y * (O2.Y * gas.mO2[-1]) ** 0.5)
            )
            + 1
        )
        c = -(
            1
            - gas.mO2[-1]
            - gas.mCO2[-1]
            - gas.mCO[-1]
            - gas.mSO2[-1]
            - gas.mS2[-1]
            - gas.mN2[-1]
        )

        mh2o = [x for x in np.roots([a, b, c]) if x > 0]
        gas.mH2O.append(mh2o[0])

        gas.mH2.append(
            (H2O.Y * gas.mH2O[-1])
            / (sys.K["K1"] * H2.Y * (O2.Y * gas.mO2[-1] * sys.P) ** 0.5)
        )  # and so should this

        gas.mCH4.append(
            (CO2.Y * gas.mCO2[-1] * (H2O.Y * gas.mH2O[-1]) ** 2)
            / (sys.K["K3"] * CH4.Y * (O2.Y * gas.mO2[-1]) ** 2)
        )

        gas.mH2S.append(
            (sys.K["K4"] * H2O.Y * gas.mH2O[-1] * (S2.Y * gas.mS2[-1]) ** 0.5)
            / (H2S.Y * (O2.Y * gas.mO2[-1]) ** 0.5)
        )

        gas.fo2.append(np.log(O2.Y * gas.mO2[-1] * sys.P))
        melt.fmq.append(cnvs.fo2_2fmq(gas.fo2[-1], sys.T, sys.P, sys.run.FMQ_MODEL))
        gas.get_fugacity(
            [H2O, H2, CO2, CO, CH4, S2, SO2, H2S, N2],
            [
                gas.mH2O[-1],
                gas.mH2[-1],
                gas.mCO2[-1],
                gas.mCO[-1],
                gas.mCH4[-1],
                gas.mS2[-1],
                gas.mSO2[-1],
                gas.mH2S[-1],
                gas.mN2[-1],
            ],
        )

        sys.WgT.append(gas.get_WgT(melt, system))
        melt.melt_composition(gas, system)
        sys.GvF.append(gas.get_vol_frac(melt))
        sys.rho.append(sys.rho_bulk(melt, gas))
        gas.M.append(
            cnvs.mean_mol_wt(
                H2O=gas.mH2O[-1],
                O2=gas.mO2[-1],
                H2=gas.mH2[-1],
                CO=gas.mCO[-1],
                CO2=gas.mCO2[-1],
                CH4=gas.mCH4[-1],
                S2=gas.mS2[-1],
                SO2=gas.mSO2[-1],
                H2S=gas.mH2S[-1],
                N2=gas.mN2[-1],
            )
        )

        # Check mass is being conserved

        if run.FE_SYSTEM is True:
            o = "o_tot"
        elif run.FE_SYSTEM is False:
            o = "o"

        for x in ["h", "c", o, "s", "n"]:
            if (
                abs(
                    cnvs.atomicM_calc(sys, melt, gas, x, -1, WgT=sys.WgT[-1])
                    - sys.atomicM[x]
                )
                / sys.atomicM[x]
            ) * 100 > 1e-5 and cnvs.atomicM_calc(
                sys, melt, gas, x, -1, WgT=sys.WgT[-1]
            ) != 0.0:
                sys.mass_conservation_reset(melt, gas)
                break

        # When running an open system,
        # remove a fraction of the gas phase for the next step.
        if run.RUN_TYPE == "open":
            gas.open_system(melt, run.LOSS_FRAC)

    else:
        print("There is no equation for this yet")
''',
"readin": r'''# readin.py

# ------------------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------------------

import numpy as np
import pandas as pd
import yaml
from numpy import exp, log10
from numpy import log as ln

import evo.conversions as cnvs
import evo.fixed_weights as fw
import evo.messages as msgs
import evo.sat_pressure as sat
import evo.solvgas as sg

# bundled scripts
from evo.dgs_classes import Gas, Melt, Molecule, Output, RunDef, ThermoSystem

# ------------------------------------------------------------------------
# FUNCTION DEFINTIONS
# ------------------------------------------------------------------------


def readin_env(df: pd.Series):
    """
    Instantiates the RunDef and ThermoSystem classes.

    Sets the parameters of the RunDef and ThermoSystem classes according
    to the setup parameters listed in the environment input file.

    Parameters
    ----------
    f : file object
        The open environment file

    Returns
    -------
    run : RunDef class
        The instantiated RunDef class
    sys : ThermoSystem class
        The instantiated ThermoSystem class
    """
    # setup run definitions
    run = RunDef(**df.to_dict())

    # use run definitions to initialise system state
    sys = ThermoSystem(run)

    if run.FO2_buffer_SET is True:
        if run.FO2_buffer == "FMQ":
            sys.FO2_buffer = run.FO2_buffer_START
            sys.FO2 = ln(
                10 ** cnvs.fmq2fo2(sys.FO2_buffer, sys.T, sys.P, run.FMQ_MODEL)
            )  # ln fO2.

        elif run.FO2_buffer == "IW":
            sys.FO2_buffer = run.FO2_buffer_START
            sys.FO2 = ln(10 ** cnvs.iw2fo2(sys.FO2_buffer, sys.T, sys.P, run.FMQ_MODEL))

        elif run.FO2_buffer == "NNO":
            sys.FO2_buffer = run.FO2_buffer_START
            sys.FO2 = ln(
                10 ** cnvs.nno2fo2(sys.FO2_buffer, sys.T, sys.P, run.FMQ_MODEL)
            )

        run.FO2_SET = True

    elif run.FO2_SET is True:
        sys.FO2 = np.log(run.FO2_START)
        sys.FO2_buffer = cnvs.fo2_2fmq(log10(exp(sys.FO2)), sys.T, sys.P, run.FMQ_MODEL)

    if run.GRAPHITE_SATURATED is True:
        if run.C_MODEL == "eguchi2018":
            pass
        else:
            exit(
                "Error: If melt is graphite saturated, "
                "please use the eguchi2018 model as the C_MODEL setting."
            )

    sys.gas_system(run)  # creates the molecule list (self.SC)

    return run, sys


#  -------------------------------------------------------------------------
def readin_chem(df, run, sys):
    """
    Instantiates the Melt class, reading in the chemistry file.

    Sets the parameters of the Melt class according to the chemistry
    input file. Checks that only recognised species are listed in the
    melt composition, and that all the information required to run the
    requested setup is provided.

    Parameters
    ----------
    df : pd.Series
        The chemistry data
    run : RunDef class
        Active instance of the RunDef class
    sys : ThermoSystem class
        Active instance of the ThermoSystem class

    Returns
    -------
    melt : Melt class
        The instantiated Melt class
    """

    ele_names = []
    chems = []

    # define allowed components
    allowed_names = [
        "sio2",
        "tio2",
        "al2o3",
        "feo",
        "fe2o3",
        "mno",
        "mgo",
        "cao",
        "na2o",
        "k2o",
        "p2o5",
        "nio",
        "cr2o3",
    ]

    for chem, frac in df.items():
        if chem.lower() in allowed_names:
            ele_names.append(chem.lower())
            chems.append(frac)
        else:
            exit(
                f"{chem} does not match an element in the allowed list "
                "(case insensitive):\n{allowed_names}"
            )

    # test for iron/ fO2 definition
    if "feo" in ele_names and "fe2o3" not in ele_names and run.FO2_SET is not True:
        if run.GAS_SYS == "OH" and (run.FH2_SET):
            pass
        elif run.FH2_SET and (run.FH2O_SET or run.WTH2O_SET):
            pass
        else:
            raise RuntimeError("Only FeO given without setting dgs_FO2 = True.")

    elif "fe2o3" in ele_names and "feo" not in ele_names and run.FO2_SET is not True:
        if run.GAS_SYS == "OH" and (run.FH2_SET):
            pass
        elif run.FH2_SET and (run.FH2O_SET or run.WTH2O_SET):
            pass
        else:
            raise RuntimeError("Only Fe2O3 given without setting dgs_FO2 = True.")

    elif "feo" not in ele_names and "fe2o3" not in ele_names:
        raise RuntimeError("No iron in system")

    elif run.FO2_SET is True and "feo" in ele_names and "fe2o3" in ele_names:
        raise RuntimeError("fO2 and iron proportions specified, only give one.")

    # populate melt object
    melt = Melt(run, sys)

    melt.chem_init(ele_names, chems)

    return melt


#  -------------------------------------------------------------------------
def readin_output(f):
    """
    Instantiates the Output class using the output file.

    Parameters
    ----------
    f : file object
        The open environment file

    Returns
    -------
    out : Output class
        The instantiated Output class
    """

    x = yaml.full_load(f)

    out = Output()
    out.set_outputs(x)

    return out


#  ------------------------------------------------------------------------
def run_melt_match(run, melt):
    """
    Asserts that the composition name given matches the melt SiO2 content.

    Parameters
    ----------
    run : RunDef class
        Active instance of the RunDef class
    melt : Melt class
        Active instance of the Melt class
    """

    if run.COMPOSITION == "basalt":
        sio2_lim = [45, 55]
    elif run.COMPOSITION == "phonolite":
        sio2_lim = [52, 63]
    if run.COMPOSITION == "rhyolite":
        sio2_lim = [65, 80]

    magma = melt.Cw()

    if magma["sio2"] > sio2_lim[0] and magma["sio2"] < sio2_lim[1]:
        pass
    else:
        # Creates a terminal message checking if they want to continue with their setup.
        msgs.run_chem_mismatch(run.COMPOSITION, sio2_lim, magma["sio2"])


#  -------------------------------------------------------------------------
def readin(df_chem: pd.DataFrame, df_env: pd.DataFrame, f_out=None):
    """
    Opens the input files and sets up major classes with the input data.

    Checks that correct input data has been provided for the requested
    run type.

    Parameters
    ----------
    f_chem : pd.Series
        Chemistry input file
    f_env : pd.Series
        Environment input file
    f_out : string or None
        optional path to the output options input file

    Returns
    -------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class
    out : Output class
        The active instance of the Output class
    """

    run, sys = readin_env(df_env)

    # check stepsize requirements
    if run.DP_MIN > run.DP_MAX:
        raise ValueError(
            "Please make the minimum stepsize DP_MIN <= maximum stepsize DP_MAX."
        )

    if run.RUN_TYPE == "open" and run.DP_MAX != run.DP_MIN:
        raise ValueError(
            "Open system degassing is path dependent.\n"
            "For internal consistency, please set the DP_MAX and DP_MIN "
            "pressure steps to be equal (we suggest <1 bar for the run to complete; "
            "0.5 bar is usually sufficient)"
        )

    # chemistry
    melt = readin_chem(df_chem, run, sys)

    # check the melt SiO2 content matches the run composition definition
    run_melt_match(run, melt)

    if run.FIND_SATURATION is False:
        msgs.vol_setup_standard(run, sys)
    elif run.FIND_SATURATION is True:
        msgs.vol_setup_saturation(run, sys)

    # outputs
    if f_out:
        with open(f_out) as f:
            out = readin_output(f)
    else:
        out = Output()

    return run, sys, melt, out


# -------------------------------------------------------------------------
def set_init_chem(run, sys, melt):
    """
    Wrapper function to setup initial conditions of the volcanic system.

    Instantiates the Molecule class for each of the gas phase species.
    Calculates the equilibrium constants for the system temperature, the
    initial gas gas composition, the starting pressure (if the saturation
    pressure is requested) and sets the atomic volatile masses for mass
    balance conservation. Normalises the melt oxide composition with the
    volatile content and records the mass of total Fe in the system.

    Parameters
    ----------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class

    Returns
    -------
    run : RunDef class
        The active instance of the RunDef class
    sys : ThermoSystem class
        The active instance of the ThermoSystem class
    melt : Melt class
        The active instance of the Melt class
    gas : Gas class
        The active instance of the Gas class
    molecules : tuple of Molecule classes
        A tuple of all the Molecule class instances. Setup to be passed
        through all functions, preserving ordering.
    """

    # Instantiates the Molecule class with each of the species that should be present
    # depending on the gas system which the user has specified.

    H2O = Molecule(run, sys, melt, "H2O")
    O2 = Molecule(run, sys, melt, "O2")
    H2 = Molecule(run, sys, melt, "H2")

    if run.GAS_SYS == "OH":
        molecules = H2O, O2, H2

    elif run.GAS_SYS == "SOH":
        S2 = Molecule(run, sys, melt, "S2")
        SO2 = Molecule(run, sys, melt, "SO2")
        H2S = Molecule(run, sys, melt, "H2S")

        molecules = H2O, O2, H2, S2, SO2, H2S

    else:
        if "C" in run.GAS_SYS:
            CO = Molecule(run, sys, melt, "CO")
            CO2 = Molecule(run, sys, melt, "CO2")
            CH4 = Molecule(run, sys, melt, "CH4")

            molecules = H2O, O2, H2, CO, CO2, CH4

            if "S" in run.GAS_SYS:
                S2 = Molecule(run, sys, melt, "S2")
                SO2 = Molecule(run, sys, melt, "SO2")
                H2S = Molecule(run, sys, melt, "H2S")

                molecules = H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S

                if "N" in run.GAS_SYS:
                    N2 = Molecule(run, sys, melt, "N2")

                    molecules = H2O, O2, H2, CO, CO2, CH4, S2, SO2, H2S, N2

    gas = Gas(sys)

    sys.K = sg.get_K(sys, molecules)  # Calculates and stores all relevant K values

    sg.set_Y(sys, molecules)  # Gets all relevant activity coefficients

    # Finds the saturation pressure for the melt composition given,
    # sets sys.P to this and WgT to 0.001%
    if run.FIND_SATURATION is True:
        sys.sat_conditions = sat.sat_pressure(run, sys, gas, melt, molecules)

    # Finds the saturation pressure for the melt composition given,
    # sets sys.P to this and WgT to 0.001%
    elif run.ATOMIC_MASS_SET is True:
        sys.sat_conditions = fw.sat_pressure(run, sys, gas, melt, molecules)

    else:
        sys.get_atomic_mass(
            run, gas, melt, molecules
        )  # Gets atomic mass values and initial partitioning

    # Normalises the wt % fractions of oxides in the melt with the volatile species
    # and FeO(t).
    sys.norm_with_gas(melt)

    # Gets the total WtO after the melt (and therefore Fe) has been normalised to
    # the gas content
    sys.get_wto_tot(melt)

    return run, sys, melt, gas, molecules
''',
"dgs": r'''"""
dgs.py  PL: UPDATE THESE DESCRIPTIONS

--- SPECIES LIST ---

C    CO   CO_2 CH_4    | carbon, carbon monoxide, carbon dioxide, methane
H_2  H_2O H_2S         | hydrogen, water, hydrogen sulfide
SO_2 S_2               | sulfur dioxide, sulfur
O_2  OCS               | oxygen, carbonyl sulfide
FeO  Fe_2O_3           | ferrous iron, ferric iron
N2                     | nitrogen

-- MODULES ---

density - calculate density of the melt
ferric - calculate fo2 from ferric iron proportions
fixed_weights - calculate the saturation point of a melt from it's volatile content
        given as atomic weights of species in the melt
init_cons - calculates the initial conditions in a standard run, given gas weight
             fraction and some subset of gas or melt properties
messages - stores warnings and error messages
multirun - a template for running parameters sweeps with EVo
sat_pressure - calculates the pressure of volatile saturation
solubility_laws - stores the solubility laws for all species
solver - calculates the final system composition at every pressure step
solvgas - calculate fugacity coefficients and equilibrium coefficients for gas phase
            equilibria

--- LITERATURE SOURCES ---

model description ---
Gaillard & Scaillet 2014, EPSL
Burgisser & Scaillet 2007
Moretti & Papale 2004
Holloway 1987
Burgisser 2015
Liggins et al 2020, 2021

solubility models ---
Iacono-marziano et al. 2012, EPSL
Gaillard & Scaillet 2009
Behrens and Gaillard 2006 (CO CH4 solubility)
Morizet et al. 2010 (CO CH4 solubility)
Hirschman 2012
Righter 2009


--- MODEL DESCRIPTION ---

Equilibrium constants and mass balance method
solve homogenoud and heterogeneous equilibria simultaneously at each step

"""

# ------------------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------------------

# python main [ ensure these are on your system ]
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import evo.conversions as cnvs
import evo.solver as solver

# bundled scripts
from evo.readin import readin, set_init_chem
from evo.writeout import writeout_figs, writeout_file

# ------------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------------


def run_evo(f_chem, f_env, f_out=None, folder="outputs", write_output=False):
    """Main function for EVo.

    Call to run the model.

    Parameters
    ----------
    f_chem : str | pd.Series
        Path to the chemistry input file, or chemistry series
    f_env : str | pd.Series
        Path to the environment input file, or env series
    f_out : str or NoneType
        Path to the file describing the required outputs, None if not used
    folder : str
        Path to the folder to write the results to
    write_output : bool
        If True, write the results CSV (and plots if f_out is provided) to
        `folder`. Defaults to False so that library callers receive only the
        returned DataFrame. The CLI sets this to True.
    """
    start = time.time()

    print("Reading in from:")
    if isinstance(f_chem, str):
        print("Chemistry file:", f_chem)
        with open(f_chem) as f_c:
            df_chem = pd.Series(yaml.full_load(f_c))
    elif isinstance(f_chem, pd.Series):
        df_chem = f_chem
    else:
        raise RuntimeError(
            f"Unrecognised input type for f_chem: {type(f_chem)}.\
            \nMust be string (filepath) or pandas Series."
        )

    if isinstance(f_env, str):
        print("Environment file:", f_env)
        with open(f_env) as f_e:
            df_env = pd.Series(yaml.full_load(f_e))
    elif isinstance(f_env, pd.Series):
        df_env = f_env
    else:
        raise RuntimeError(
            f"Unrecognised input type for f_env: {type(f_env)}.\
            \nMust be string (filepath) or pandas Series."
        )

    print("Output file:", f_out, "\n")

    # Instantiate the run, thermosystem, melt and output objects
    run, sys, melt, out = readin(df_chem, df_env, f_out)
    run.results_folder = Path(folder)

    print("Set parameters:")
    run.print_conditions()

    # Setup the initial gas + melt chemistry, calculate the atomic masses of elements
    run, sys, melt, gas, molecules = set_init_chem(run, sys, melt)

    print("\nSystem chemistry (wt%)")
    sm = 0.0
    for k, v in cnvs.mol2wt(melt.cm_dry).items():
        print(k, ":", f"{v * 100:0.3g}")
        sm += v * 100
    print("Total =", round(sm, 2), "\n")

    # print output specifications if made
    if f_out:
        out.print_conditions()

    # -------------------------------------------------------------------
    # CALCULATION LOOP --------------------------------------------------
    # -------------------------------------------------------------------

    if run.SINGLE_STEP is True:
        solver.decompress(run, sys, melt, gas, molecules)
        sys.P_track.append(sys.P)

        print(
            f"The total gas weight percent is {sys.WgT[-1] * 100:.2f}% and "
            f"the gas volume fraction is {sys.GvF[-1] * 100:.2f}%"
        )
        print(
            f"The gas phase at {sys.P:.0f} bar is partitioned according to molar "
            "fraction as: \n "
            f"O2: {gas.mO2} \n "
            f"H2: {gas.mH2} \n "
            f"H2O: {gas.mH2O} \n "
            f"CO2: {gas.mCO2} \n "
            f"CO: {gas.mCO} \n "
            f"CH4: {gas.mCH4} \n "
            f"S2: {gas.mS2} \n "
            f"SO2: {gas.mSO2} \n "
            f"H2S: {gas.mH2S} \n "
            f"N2: {gas.mN2} \n"
        )
        return writeout_file(sys, gas, melt, sys.P_track, write=write_output)

    elif run.SINGLE_STEP is False:
        dp = (
            int(abs(np.floor(np.log10(np.abs(run.P_STOP))))) + 1
            if run.P_STOP < 1e-4
            else 5
        )
        while np.round(sys.P, decimals=dp) >= run.P_STOP:
            sys.P = np.round(sys.P, decimals=dp)
            sys.P_track.append(sys.P)
            solver.decompress(run, sys, melt, gas, molecules)  # does one pressure step
            sys.pressure_step()

        if (
            run.FIND_SATURATION is True or run.ATOMIC_MASS_SET is True
        ) and sys.sat_conditions[0] < run.P_STOP:
            exit(
                f"Error: The saturation pressure ({sys.sat_conditions[0]:.3} bar) is "
                f"lower than P_STOP ({run.P_STOP:.3} bar).\n"
                "Please either lower P_STOP or change the melt initial fo2/melt "
                "volatile content.\n"
                "Exiting..."
            )

        print(
            f"The pressure is {run.P_STOP:.2f} bar, "
            f"the total gas weight percent is {sys.WgT[-1] * 100:.2f}% "
            f"and the gas volume fraction is {sys.GvF[-1] * 100:.2f}% "
            "\nThe run has finished."
        )

        print(
            "The gas is partitioned as (mol%): \n "
            f"O2: {gas.mO2[-1] * 100:.4e} \n "
            f"H2: {gas.mH2[-1] * 100:.4f} \n "
            f"H2O: {gas.mH2O[-1] * 100:.4f} \n "
            f"CO2: {gas.mCO2[-1] * 100:.4f} \n "
            f"CO: {gas.mCO[-1] * 100:.4f} \n "
            f"CH4: {gas.mCH4[-1] * 100:.4e} \n "
            f"S2: {gas.mS2[-1] * 100:.4f} \n "
            f"SO2: {gas.mSO2[-1] * 100:.4f} \n "
            f"H2S: {gas.mH2S[-1] * 100:.4f} \n "
            f"N2: {gas.mN2[-1] * 100:.4f} \n"
        )

        print(
            "The first, middle, last gas molar fractions are: \n "
            f"O2: {gas.mO2[0]}   {gas.mO2[(len(gas.mO2) // 2)]}   {gas.mO2[-1]}\n"
            f"H2: {gas.mH2[0]}   {gas.mH2[(len(gas.mH2) // 2)]}   {gas.mH2[-1]}\n"
            f"H2O: {gas.mH2O[0]}   {gas.mH2O[(len(gas.mH2O) // 2)]}   {gas.mH2O[-1]}\n"
            f"CO2: {gas.mCO2[0]}   {gas.mCO2[(len(gas.mCO2) // 2)]}   {gas.mCO2[-1]}\n"
            f"CO: {gas.mCO[0]}   {gas.mCO[(len(gas.mCO) // 2)]}   {gas.mCO[-1]}\n"
            f"CH4: {gas.mCH4[0]}   {gas.mCH4[(len(gas.mCH4) // 2)]}   {gas.mCH4[-1]}\n"
            f"S2: {gas.mS2[0]}   {gas.mS2[(len(gas.mS2) // 2)]}   {gas.mS2[-1]}\n"
            f"SO2: {gas.mSO2[0]}   {gas.mSO2[(len(gas.mSO2) // 2)]}   {gas.mSO2[-1]}\n"
            f"H2S: {gas.mH2S[0]}   {gas.mH2S[(len(gas.mH2S) // 2)]}   {gas.mH2S[-1]}\n"
            f"N2: {gas.mN2[0]}   {gas.mN2[(len(gas.mN2) // 2)]}   {gas.mN2[-1]}\n"
        )

        end = time.time()
        print("Run time is ", end - start)

        df = writeout_file(sys, gas, melt, sys.P_track, write=write_output)
        if write_output and f_out is not None:
            writeout_figs(df, run.results_folder, out=out)

        return df
''',
}
_EVO_SOURCES_BLANKS = {}   # blanks of whitespace-only lines, restored at load time

_EVO_DATA = {
"CH4.txt": r'''#Methane (CH4)	C1H4(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-10.024	-66.911	-66.911	INFINITE
100	33.258	149.500	216.485	-6.698	-69.644	-64.353	33.615
200	33.473	172.577	189.418	-3.368	-72.027	-58.161	15.190
250	34.216	180.113	186.829	-1.679	-73.426	-54.536	11.395
298.15	35.639	186.251	186.251	0.	-74.873	-50.768	8.894
300	35.708	186.472	186.252	0.066	-74.929	-50.618	8.813
350	37.874	192.131	186.694	1.903	-76.461	-46.445	6.932
400	40.500	197.356	187.704	3.861	-77.969	-42.054	5.492
450	43.374	202.291	189.053	5.957	-79.422	-37.476	4.350
500	46.342	207.014	190.614	8.200	-80.802	-32.741	3.420
600	52.227	215.987	194.103	13.130	-83.308	-22.887	1.993
700	57.794	224.461	197.840	18.635	-85.452	-12.643	0.943
800	62.932	232.518	201.675	24.675	-87.238	-2.115	0.138
900	67.601	240.205	205.532	31.205	-88.692	8.616	-0.500
1000	71.795	247.549	209.370	38.179	-89.849	19.492	-1.018
1100	75.529	254.570	213.162	45.549	-90.750	30.472	-1.447
1200	78.833	261.287	216.895	53.270	-91.437	41.524	-1.807
1300	81.744	267.714	220.558	61.302	-91.945	52.626	-2.115
1400	84.305	273.868	224.148	69.608	-92.308	63.761	-2.379
1500	86.556	279.763	227.660	78.153	-92.553	74.918	-2.609
1600	88.537	285.413	231.095	86.910	-92.703	86.088	-2.810
1700	90.283	290.834	234.450	95.853	-92.780	97.265	-2.989
1800	91.824	296.039	237.728	104.960	-92.797	108.445	-3.147
1900	93.188	301.041	240.930	114.212	-92.770	119.624	-3.289
2000	94.399	305.853	244.057	123.592	-92.709	130.802	-3.416
2100	95.477	310.485	247.110	133.087	-92.624	141.975	-3.531
2200	96.439	314.949	250.093	142.684	-92.521	153.144	-3.636
2300	97.301	319.255	253.007	152.371	-92.409	164.308	-3.732
2400	98.075	323.413	255.854	162.141	-92.291	175.467	-3.819
2500	98.772	327.431	258.638	171.984	-92.174	186.622	-3.899
2600	99.401	331.317	261.359	181.893	-92.060	197.771	-3.973
2700	99.971	335.080	264.020	191.862	-91.954	208.916	-4.042
2800	100.489	338.725	266.623	201.885	-91.857	220.058	-4.105
2900	100.960	342.260	269.171	211.958	-91.773	231.196	-4.164
3000	101.389	345.690	271.664	222.076	-91.705	242.332	-4.219
3100	101.782	349.021	274.106	232.235	-91.653	253.465	-4.271
3200	102.143	352.258	276.498	242.431	-91.621	264.598	-4.319
3300	102.474	355.406	278.842	252.662	-91.609	275.730	-4.364
3400	102.778	358.470	281.139	262.925	-91.619	286.861	-4.407
3500	103.060	361.453	283.391	273.217	-91.654	297.993	-4.447
3600	103.319	364.360	285.600	283.536	-91.713	309.127	-4.485
3700	103.560	367.194	287.767	293.881	-91.798	320.262	-4.521
3800	103.783	369.959	289.894	304.248	-91.911	331.401	-4.555
3900	103.990	372.658	291.982	314.637	-92.051	342.542	-4.588
4000	104.183	375.293	294.032	325.045	-92.222	353.687	-4.619
4100	104.363	377.868	296.045	335.473	-92.422	364.838	-4.648
4200	104.531	380.385	298.023	345.918	-92.652	375.993	-4.676
4300	104.688	382.846	299.967	356.379	-92.914	387.155	-4.703
4400	104.834	385.255	301.879	366.855	-93.208	398.322	-4.729
4500	104.972	387.612	303.758	377.345	-93.533	409.497	-4.753
4600	105.101	389.921	305.606	387.849	-93.891	420.679	-4.777
4700	105.223	392.182	307.424	398.365	-94.281	431.869	-4.800
4800	105.337	394.399	309.213	408.893	-94.702	443.069	-4.822
4900	105.445	396.572	310.973	419.432	-95.156	454.277	-4.843
5000	105.546	398.703	312.707	429.982	-95.641	465.495	-4.863
5100	105.642	400.794	314.414	440.541	-96.157	476.722	-4.883
5200	105.733	402.847	316.095	451.110	-96.703	487.961	-4.902
5300	105.818	404.861	317.750	461.688	-97.278	499.210	-4.920
5400	105.899	406.840	319.382	472.274	-97.882	510.470	-4.938
5500	105.976	408.784	320.990	482.867	-98.513	521.741	-4.955
5600	106.049	410.694	322.575	493.469	-99.170	533.025	-4.972
5700	106.118	412.572	324.137	504.077	-99.852	544.320	-4.988
5800	106.184	414.418	325.678	514.692	-100.557	555.628	-5.004
5900	106.247	416.234	327.197	525.314	-101.284	566.946	-5.019
6000	106.306	418.020	328.696	535.942	-102.032	578.279	-5.034
''',
"CO.txt": r'''#Carbon Monoxide (CO)	C1O1(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.000	0.000	INFINITE	-8.671	-113.805	-113.805	INFINITE
100	29.104	165.850	223.539	-5.769	-112.415	-120.239	62.807
200	29.108	186.025	200.317	-2.858	-111.286	-128.526	33.568
298.15	29.142	197.653	197.653	0.000	-110.527	-137.163	24.030
300	29.142	197.833	197.653	0.054	-110.516	-137.328	23.911
400	29.342	206.238	198.798	2.976	-110.102	-146.338	19.110
500	29.794	212.831	200.968	5.931	-110.003	-155.414	16.236
600	30.443	218.319	203.415	8.942	-110.150	-164.486	14.320
700	31.171	223.066	205.890	12.023	-110.469	-173.518	12.948
800	31.899	227.277	208.305	15.177	-110.905	-182.497	11.916
900	32.577	231.074	210.628	18.401	-111.418	-191.416	11.109
1000	33.183	234.538	212.848	21.690	-111.983	-200.275	10.461
1100	33.710	237.726	214.967	25.035	-112.586	-209.075	9.928
1200	34.175	240.679	216.988	28.430	-113.217	-217.819	9.481
1300	34.572	243.431	218.917	31.868	-113.870	-226.509	9.101
1400	34.920	246.006	220.761	35.343	-114.541	-235.149	8.774
1500	35.217	248.426	222.526	38.850	-115.229	-243.740	8.488
1600	35.480	250.707	224.216	42.385	-115.933	-252.284	8.236
1700	35.710	252.865	225.839	45.945	-116.651	-260.784	8.013
1800	35.911	254.912	227.398	49.526	-117.384	-269.242	7.813
1900	36.091	256.859	228.897	53.126	-118.133	-277.658	7.633
2000	36.250	258.714	230.342	56.744	-118.896	-286.034	7.470
2100	36.392	260.486	231.736	60.376	-119.675	-294.372	7.322
2200	36.518	262.182	233.081	64.021	-120.470	-302.672	7.186
2300	36.635	263.808	234.382	67.679	-121.282	-310.936	7.062
2400	36.740	265.369	235.641	71.348	-122.109	-319.165	6.946
2500	36.836	266.871	236.860	75.027	-122.953	-327.358	6.840
2600	36.924	268.318	238.043	78.715	-123.813	-335.517	6.741
2700	37.003	269.713	239.190	82.411	-124.689	-343.643	6.648
2800	37.083	271.060	240.304	86.116	-125.582	-351.736	6.562
2900	37.150	272.362	241.387	89.827	-126.490	-359.797	6.481
3000	37.217	273.623	242.441	93.546	-127.415	-367.826	6.404
3100	37.279	274.844	243.467	97.271	-128.356	-375.824	6.333
3200	37.338	276.029	244.466	101.001	-129.312	-383.792	6.265
3300	37.392	277.178	245.440	104.738	-130.283	-391.730	6.201
3400	37.443	278.295	246.390	108.480	-131.270	-399.638	6.140
3500	37.493	279.382	247.317	112.227	-132.271	-407.517	6.082
3600	37.543	280.438	248.222	115.978	-133.288	-415.366	6.027
3700	37.589	281.468	249.107	119.735	-134.318	-423.188	5.974
3800	37.631	282.471	249.972	123.496	-135.363	-430.981	5.924
3900	37.673	283.449	250.818	127.261	-136.422	-438.746	5.876
4000	37.715	284.403	251.645	131.031	-137.495	-446.485	5.830
4100	37.756	285.335	252.456	134.804	-138.581	-454.196	5.787
4200	37.794	286.245	253.250	138.582	-139.681	-461.881	5.744
4300	37.832	287.135	254.027	142.363	-140.794	-469.539	5.704
4400	37.869	288.005	254.790	146.148	-141.921	-477.171	5.665
4500	37.903	288.856	255.537	149.937	-143.062	-484.778	5.627
4600	37.941	289.690	256.271	153.729	-144.215	-492.359	5.591
4700	37.974	290.506	256.990	157.525	-145.383	-499.915	5.556
4800	38.007	291.306	257.697	161.324	-146.563	-507.446	5.522
4900	38.041	292.090	258.391	165.126	-147.758	-514.951	5.489
5000	38.074	292.859	259.073	168.932	-148.967	-522.433	5.458
5100	38.104	293.613	259.743	172.741	-150.190	-529.890	5.427
5200	38.137	294.354	260.401	176.553	-151.427	-537.323	5.397
5300	38.171	295.080	261.049	180.368	-152.679	-544.732	5.369
5400	38.200	295.794	261.685	184.187	-153.946	-552.118	5.341
5500	38.233	296.495	262.312	188.008	-155.228	-559.480	5.313
5600	38.263	297.184	262.929	191.833	-156.527	-566.818	5.287
5700	38.296	297.862	263.535	195.661	-157.841	-574.133	5.261
5800	38.325	298.528	264.133	199.492	-159.172	-581.424	5.236
5900	38.355	299.184	264.722	203.326	-160.521	-588.694	5.212
6000	38.388	299.829	265.301	207.163	-161.887	-595.939	5.188
''',
"CO2.txt": r'''#Carbon Dioxide (CO2)	C1O2(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-9.364	-393.151	-393.151	INFINITE
100	29.208	179.009	243.568	-6.456	-393.208	-393.683	205.639
200	32.359	199.975	217.046	-3.414	-393.404	-394.085	102.924
298.15	37.129	213.795	213.795	0.	-393.522	-394.389	69.095
300	37.221	214.025	213.795	0.069	-393.523	-394.394	68.670
400	41.325	225.314	215.307	4.003	-393.583	-394.675	51.539
500	44.627	234.901	218.290	8.305	-393.666	-394.939	41.259
600	47.321	243.283	221.772	12.907	-393.803	-395.182	34.404
700	49.564	250.750	225.388	17.754	-393.983	-395.398	29.505
800	51.434	257.494	228.986	22.806	-394.188	-395.586	25.829
900	52.999	263.645	232.500	28.030	-394.405	-395.748	22.969
1000	54.308	269.299	235.901	33.397	-394.623	-395.886	20.679
1100	55.409	274.528	239.178	38.884	-394.838	-396.001	18.805
1200	56.342	279.390	242.329	44.473	-395.050	-396.098	17.242
1300	57.137	283.932	245.356	50.148	-395.257	-396.177	15.919
1400	57.802	288.191	248.265	55.896	-395.462	-396.240	14.784
1500	58.379	292.199	251.062	61.705	-395.668	-396.288	13.800
1600	58.886	295.983	253.753	67.569	-395.876	-396.323	12.939
1700	59.317	299.566	256.343	73.480	-396.090	-396.344	12.178
1800	59.701	302.968	258.840	79.431	-396.311	-396.353	11.502
1900	60.049	306.205	261.248	85.419	-396.542	-396.349	10.896
2000	60.350	309.293	263.574	91.439	-396.784	-396.333	10.351
2100	60.622	312.244	265.822	97.488	-397.039	-396.304	9.858
2200	60.865	315.070	267.996	103.562	-397.309	-396.262	9.408
2300	61.086	317.781	270.102	109.660	-397.596	-396.209	8.998
2400	61.287	320.385	272.144	115.779	-397.900	-396.142	8.622
2500	61.471	322.890	274.124	121.917	-398.222	-396.062	8.275
2600	61.647	325.305	276.046	128.073	-398.562	-395.969	7.955
2700	61.802	327.634	277.914	134.246	-398.921	-395.862	7.658
2800	61.952	329.885	279.730	140.433	-399.299	-395.742	7.383
2900	62.095	332.061	281.497	146.636	-399.695	-395.609	7.126
3000	62.229	334.169	283.218	152.852	-400.111	-395.461	6.886
3100	62.347	336.211	284.895	159.081	-400.545	-395.298	6.661
3200	62.462	338.192	286.529	165.321	-400.998	-395.122	6.450
3300	62.573	340.116	288.124	171.573	-401.470	-394.932	6.251
3400	62.681	341.986	289.681	177.836	-401.960	-394.726	6.064
3500	62.785	343.804	291.202	184.109	-402.467	-394.506	5.888
3600	62.884	345.574	292.687	190.393	-402.991	-394.271	5.721
3700	62.980	347.299	294.140	196.686	-403.532	-394.022	5.563
3800	63.074	348.979	295.561	202.989	-404.089	-393.756	5.413
3900	63.166	350.619	296.952	209.301	-404.662	-393.477	5.270
4000	63.254	352.219	298.314	215.622	-405.251	-393.183	5.134
4100	63.341	353.782	299.648	221.951	-405.856	-392.874	5.005
4200	63.426	355.310	300.955	228.290	-406.475	-392.550	4.882
4300	63.509	356.803	302.236	234.637	-407.110	-392.210	4.764
4400	63.588	358.264	303.493	240.991	-407.760	-391.857	4.652
4500	63.667	359.694	304.726	247.354	-408.426	-391.488	4.544
4600	63.745	361.094	305.937	253.725	-409.106	-391.105	4.441
4700	63.823	362.466	307.125	260.103	-409.802	-390.706	4.342
4800	63.893	363.810	308.292	266.489	-410.514	-390.292	4.247
4900	63.968	365.128	309.438	272.882	-411.242	-389.862	4.156
5000	64.046	366.422	310.565	279.283	-411.986	-389.419	4.068
5100	64.128	367.691	311.673	285.691	-412.746	-388.959	3.984
5200	64.220	368.937	312.762	292.109	-413.522	-388.486	3.902
5300	64.312	370.161	313.833	298.535	-414.314	-387.996	3.824
5400	64.404	371.364	314.888	304.971	-415.123	-387.493	3.748
5500	64.496	372.547	315.925	311.416	-415.949	-386.974	3.675
5600	64.588	373.709	316.947	317.870	-416.794	-386.439	3.605
5700	64.680	374.853	317.953	324.334	-417.658	-385.890	3.536
5800	64.772	375.979	318.944	330.806	-418.541	-385.324	3.470
5900	64.865	377.087	319.920	337.288	-419.445	-384.745	3.406
6000	64.957	378.178	320.882	343.779	-420.372	-384.148	3.344
''',
"H2.txt": r'''#Hydrogen (H2)	H2(ref)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-8.467	0.	0.	0.
100	28.154	100.727	155.408	-5.468	0.	0.	0.
200	27.447	119.412	133.284	-2.774	0.	0.	0.
250	28.344	125.640	131.152	-1.378	0.	0.	0.
298.15	28.836	130.680	130.680	0.	0.	0.	0.
300	28.849	130.858	130.680	0.053	0.	0.	0.
350	29.081	135.325	131.032	1.502	0.	0.	0.
400	29.181	139.216	131.817	2.959	0.	0.	0.
450	29.229	142.656	132.834	4.420	0.	0.	0.
500	29.260	145.737	133.973	5.882	0.	0.	0.
600	29.327	151.077	136.392	8.811	0.	0.	0.
700	29.441	155.606	138.822	11.749	0.	0.	0.
800	29.624	159.548	141.171	14.702	0.	0.	0.
900	29.881	163.051	143.411	17.676	0.	0.	0.
1000	30.205	166.216	145.536	20.680	0.	0.	0.
1100	30.581	169.112	147.549	23.719	0.	0.	0.
1200	30.992	171.790	149.459	26.797	0.	0.	0.
1300	31.423	174.288	151.274	29.918	0.	0.	0.
1400	31.861	176.633	153.003	33.082	0.	0.	0.
1500	32.298	178.846	154.652	36.290	0.	0.	0.
1600	32.725	180.944	156.231	39.541	0.	0.	0.
1700	33.139	182.940	157.743	42.835	0.	0.	0.
1800	33.537	184.846	159.197	46.169	0.	0.	0.
1900	33.917	186.669	160.595	49.541	0.	0.	0.
2000	34.280	188.418	161.943	52.951	0.	0.	0.
2100	34.624	190.099	163.244	56.397	0.	0.	0.
2200	34.952	191.718	164.501	59.876	0.	0.	0.
2300	35.263	193.278	165.719	63.387	0.	0.	0.
2400	35.559	194.785	166.899	66.928	0.	0.	0.
2500	35.842	196.243	168.044	70.498	0.	0.	0.
2600	36.111	197.654	169.155	74.096	0.	0.	0.
2700	36.370	199.021	170.236	77.720	0.	0.	0.
2800	36.618	200.349	171.288	81.369	0.	0.	0.
2900	36.856	201.638	172.313	85.043	0.	0.	0.
3000	37.087	202.891	173.311	88.740	0.	0.	0.
3100	37.311	204.111	174.285	92.460	0.	0.	0.
3200	37.528	205.299	175.236	96.202	0.	0.	0.
3300	37.740	206.457	176.164	99.966	0.	0.	0.
3400	37.946	207.587	177.072	103.750	0.	0.	0.
3500	38.149	208.690	177.960	107.555	0.	0.	0.
3600	38.348	209.767	178.828	111.380	0.	0.	0.
3700	38.544	210.821	179.679	115.224	0.	0.	0.
3800	38.738	211.851	180.512	119.089	0.	0.	0.
3900	38.928	212.860	181.328	122.972	0.	0.	0.
4000	39.116	213.848	182.129	126.874	0.	0.	0.
4100	39.301	214.816	182.915	130.795	0.	0.	0.
4200	39.484	215.765	183.686	134.734	0.	0.	0.
4300	39.665	216.696	184.442	138.692	0.	0.	0.
4400	39.842	217.610	185.186	142.667	0.	0.	0.
4500	40.017	218.508	185.916	146.660	0.	0.	0.
4600	40.188	219.389	186.635	150.670	0.	0.	0.
4700	40.355	220.255	187.341	154.698	0.	0.	0.
4800	40.518	221.106	188.035	158.741	0.	0.	0.
4900	40.676	221.943	188.719	162.801	0.	0.	0.
5000	40.829	222.767	189.392	166.876	0.	0.	0.
5100	40.976	223.577	190.054	170.967	0.	0.	0.
5200	41.117	224.374	190.706	175.071	0.	0.	0.
5300	41.252	225.158	191.349	179.190	0.	0.	0.
5400	41.379	225.931	191.982	183.322	0.	0.	0.
5500	41.498	226.691	192.606	187.465	0.	0.	0.
5600	41.609	227.440	193.222	191.621	0.	0.	0.
5700	41.712	228.177	193.829	195.787	0.	0.	0.
5800	41.806	228.903	194.427	199.963	0.	0.	0.
5900	41.890	229.619	195.017	204.148	0.	0.	0.
6000	41.965	230.323	195.600	208.341	0.	0.	0.
''',
"H2O.txt": r'''#Water (H2O)	H2O1(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-9.904	-238.921	-238.921	INFINITE
100	33.299	152.388	218.534	-6.615	-240.083	-236.584	123.579
200	33.349	175.485	191.896	-3.282	-240.900	-232.766	60.792
298.15	33.590	188.834	188.834	0.	-241.826	-228.582	40.047
300	33.596	189.042	188.835	0.062	-241.844	-228.500	39.785
400	34.262	198.788	190.159	3.452	-242.846	-223.901	29.238
500	35.226	206.534	192.685	6.925	-243.826	-219.051	22.884
600	36.325	213.052	195.550	10.501	-244.758	-214.007	18.631
700	37.495	218.739	198.465	14.192	-245.632	-208.812	15.582
800	38.721	223.825	201.322	18.002	-246.443	-203.496	13.287
900	39.987	228.459	204.084	21.938	-247.185	-198.083	11.496
1000	41.268	232.738	206.738	26.000	-247.857	-192.590	10.060
1100	42.536	236.731	209.285	30.191	-248.460	-187.033	8.881
1200	43.768	240.485	211.730	34.506	-248.997	-181.425	7.897
1300	44.945	244.035	214.080	38.942	-249.473	-175.774	7.063
1400	46.054	247.407	216.341	43.493	-249.894	-170.089	6.346
1500	47.090	250.620	218.520	48.151	-250.265	-164.376	5.724
1600	48.050	253.690	220.623	52.908	-250.592	-158.639	5.179
1700	48.935	256.630	222.655	57.758	-250.881	-152.883	4.698
1800	49.749	259.451	224.621	62.693	-251.138	-147.111	4.269
1900	50.496	262.161	226.526	67.706	-251.368	-141.325	3.885
2000	51.180	264.769	228.374	72.790	-251.575	-135.528	3.540
2100	51.823	267.282	230.167	77.941	-251.762	-129.721	3.227
2200	52.408	269.706	231.909	83.153	-251.934	-123.905	2.942
2300	52.947	272.048	233.604	88.421	-252.092	-118.082	2.682
2400	53.444	274.312	235.253	93.741	-252.239	-112.252	2.443
2500	53.904	276.503	236.860	99.108	-252.379	-106.416	2.223
2600	54.329	278.625	238.425	104.520	-252.513	-100.575	2.021
2700	54.723	280.683	239.952	109.973	-252.643	-94.729	1.833
2800	55.089	282.680	241.443	115.464	-252.771	-88.878	1.658
2900	55.430	284.619	242.899	120.990	-252.897	-83.023	1.495
3000	55.748	286.504	244.321	126.549	-253.024	-77.163	1.344
3100	56.044	288.337	245.711	132.139	-253.152	-71.298	1.201
3200	56.323	290.120	247.071	137.757	-253.282	-65.430	1.068
3300	56.583	291.858	248.402	143.403	-253.416	-59.558	0.943
3400	56.828	293.550	249.705	149.073	-253.553	-53.681	0.825
3500	57.058	295.201	250.982	154.768	-253.696	-47.801	0.713
3600	57.276	296.812	252.233	160.485	-253.844	-41.916	0.608
3700	57.480	298.384	253.459	166.222	-253.997	-36.027	0.509
3800	57.675	299.919	254.661	171.980	-254.158	-30.133	0.414
3900	57.859	301.420	255.841	177.757	-254.326	-24.236	0.325
4000	58.033	302.887	256.999	183.552	-254.501	-18.334	0.239
4100	58.199	304.322	258.136	189.363	-254.684	-12.427	0.158
4200	58.357	305.726	259.252	195.191	-254.876	-6.516	0.081
4300	58.507	307.101	260.349	201.034	-255.078	-0.600	0.007
4400	58.650	308.448	261.427	206.892	-255.288	5.320	-0.063
4500	58.787	309.767	262.486	212.764	-255.508	11.245	-0.131
4600	58.918	311.061	263.528	218.650	-255.738	17.175	-0.195
4700	59.044	312.329	264.553	224.548	-255.978	23.111	-0.257
4800	59.164	313.574	265.562	230.458	-256.229	29.052	-0.316
4900	59.275	314.795	266.554	236.380	-256.491	34.998	-0.373
5000	59.390	315.993	267.531	242.313	-256.763	40.949	-0.428
5100	59.509	317.171	268.493	248.258	-257.046	46.906	-0.480
5200	59.628	318.327	269.440	254.215	-257.338	52.869	-0.531
5300	59.746	319.464	270.373	260.184	-257.639	58.838	-0.580
5400	59.864	320.582	271.293	266.164	-257.950	64.811	-0.627
5500	59.982	321.682	272.199	272.157	-258.268	70.791	-0.672
5600	60.100	322.764	273.092	278.161	-258.595	76.777	-0.716
5700	60.218	323.828	273.973	284.177	-258.930	82.769	-0.758
5800	60.335	324.877	274.841	290.204	-259.272	88.767	-0.799
5900	60.453	325.909	275.698	296.244	-259.621	94.770	-0.839
6000	60.571	326.926	276.544	302.295	-259.977	100.780	-0.877
''',
"H2S.txt": r'''#Hydrogen Sulfide (H2S)	H2S1(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-9.962	-17.584	-17.584	INFINITE
100	33.259	168.972	235.330	-6.636	-17.947	-23.519	12.285
200	33.380	192.038	208.586	-3.310	-18.957	-28.754	7.510
298.15	34.192	205.757	205.757	0.	-20.502	-33.329	5.839
300	34.208	205.969	205.758	0.063	-20.534	-33.408	5.817
400	35.581	215.992	207.115	3.551	-24.549	-37.343	4.876
500	37.192	224.102	209.726	7.188	-27.762	-40.179	4.197
600	38.936	231.037	212.713	10.994	-30.470	-42.399	3.691
700	40.740	237.174	215.777	14.978	-32.771	-44.201	3.298
800	42.518	242.731	218.804	19.142	-34.771	-45.694	2.984
900	44.212	247.838	221.750	23.479	-89.665	-45.870	2.662
1000	45.786	252.579	224.599	27.980	-90.024	-40.984	2.141
1100	47.200	257.011	227.346	32.631	-90.284	-36.066	1.713
1200	48.467	261.173	229.993	37.415	-90.465	-31.129	1.355
1300	49.593	265.098	232.544	42.320	-90.584	-26.179	1.052
1400	50.593	268.810	235.003	47.330	-90.656	-21.222	0.792
1500	51.476	272.331	237.375	52.434	-90.693	-16.261	0.566
1600	52.262	275.679	239.666	57.622	-90.705	-11.298	0.369
1700	52.961	278.869	241.879	62.883	-90.699	-6.336	0.195
1800	53.589	281.914	244.019	68.212	-90.681	-1.374	0.040
1900	54.145	284.827	246.091	73.599	-90.656	3.587	-0.099
2000	54.656	287.617	248.098	79.039	-90.627	8.547	-0.223
2100	55.107	290.295	250.044	84.528	-90.597	13.505	-0.336
2200	55.522	292.868	251.932	90.059	-90.566	18.461	-0.438
2300	55.898	295.345	253.766	95.631	-90.538	23.416	-0.532
2400	56.250	297.731	255.549	101.238	-90.512	28.370	-0.617
2500	56.568	300.034	257.282	106.879	-90.490	33.323	-0.696
2600	56.861	302.258	258.970	112.551	-90.472	38.275	-0.769
2700	57.133	304.409	260.613	118.251	-90.458	43.227	-0.836
2800	57.384	306.492	262.214	123.977	-90.450	48.178	-0.899
2900	57.618	308.510	263.776	129.727	-90.447	53.129	-0.957
3000	57.840	310.467	265.300	135.500	-90.450	58.080	-1.011
3100	58.049	312.367	266.788	141.294	-90.459	63.031	-1.062
3200	58.245	314.213	268.241	147.109	-90.473	67.982	-1.110
3300	58.430	316.008	269.662	152.943	-90.495	72.935	-1.154
3400	58.605	317.755	271.051	158.795	-90.524	77.887	-1.197
3500	58.773	319.456	272.409	164.664	-90.560	82.841	-1.236
3600	58.932	321.114	273.739	170.549	-90.604	87.796	-1.274
3700	59.086	322.731	275.042	176.450	-90.656	92.752	-1.309
3800	59.229	324.308	276.317	182.366	-90.717	97.710	-1.343
3900	59.371	325.849	277.568	188.296	-90.788	102.670	-1.375
4000	59.509	327.354	278.794	194.240	-90.868	107.631	-1.406
4100	59.639	328.825	279.996	200.197	-90.958	112.595	-1.434
4200	59.768	330.263	281.176	206.168	-91.058	117.561	-1.462
4300	59.890	331.671	282.334	212.151	-91.169	122.529	-1.488
4400	60.011	333.050	283.471	218.146	-91.290	127.500	-1.514
4500	60.128	334.399	284.588	224.153	-91.423	132.474	-1.538
4600	60.237	335.722	285.685	230.171	-91.568	137.451	-1.561
4700	60.346	337.019	286.763	236.200	-91.725	142.431	-1.583
4800	60.459	338.291	287.824	242.240	-91.893	147.416	-1.604
4900	60.563	339.538	288.866	248.292	-92.073	152.403	-1.625
5000	60.668	340.763	289.892	254.353	-92.265	157.394	-1.644
5100	60.764	341.965	290.901	260.425	-92.468	162.389	-1.663
5200	60.869	343.146	291.895	266.506	-92.682	167.388	-1.681
5300	60.965	344.306	292.873	272.598	-92.907	172.392	-1.699
5400	61.061	345.447	293.836	278.699	-93.143	177.400	-1.716
5500	61.153	346.568	294.785	284.810	-93.389	182.412	-1.732
5600	61.250	347.671	295.719	290.930	-93.644	187.429	-1.748
5700	61.342	348.756	296.640	297.060	-93.908	192.451	-1.764
5800	61.429	349.823	297.548	303.198	-94.180	197.477	-1.778
5900	61.526	350.874	298.443	309.346	-94.459	202.507	-1.793
6000	61.609	351.909	299.325	315.503	-94.745	207.543	-1.807
''',
"N2.txt": r'''#Nitrogen (N2)	N2(ref)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-8.670	0.	0.	0.
100	29.104	159.811	217.490	-5.768	0.	0.	0.
200	29.107	179.985	194.272	-2.857	0.	0.	0.
250	29.111	186.481	192.088	-1.402	0.	0.	0.
298.15	29.124	191.609	191.609	0.	0.	0.	0.
300	29.125	191.789	191.610	0.054	0.	0.	0.
350	29.165	196.281	191.964	1.511	0.	0.	0.
400	29.249	200.181	192.753	2.971	0.	0.	0.
450	29.387	203.633	193.774	4.437	0.	0.	0.
500	29.580	206.739	194.917	5.911	0.	0.	0.
600	30.110	212.176	197.353	8.894	0.	0.	0.
700	30.754	216.866	199.813	11.937	0.	0.	0.
800	31.433	221.017	202.209	15.046	0.	0.	0.
900	32.090	224.757	204.510	18.223	0.	0.	0.
1000	32.697	228.170	206.708	21.463	0.	0.	0.
1100	33.241	231.313	208.804	24.760	0.	0.	0.
1200	33.723	234.226	210.802	28.109	0.	0.	0.
1300	34.147	236.943	212.710	31.503	0.	0.	0.
1400	34.518	239.487	214.533	34.936	0.	0.	0.
1500	34.843	241.880	216.277	38.405	0.	0.	0.
1600	35.128	244.138	217.948	41.904	0.	0.	0.
1700	35.378	246.275	219.552	45.429	0.	0.	0.
1800	35.600	248.304	221.094	48.978	0.	0.	0.
1900	35.796	250.234	222.577	52.548	0.	0.	0.
2000	35.971	252.074	224.006	56.137	0.	0.	0.
2100	36.126	253.833	225.385	59.742	0.	0.	0.
2200	36.268	255.517	226.717	63.361	0.	0.	0.
2300	36.395	257.132	228.004	66.995	0.	0.	0.
2400	36.511	258.684	229.250	70.640	0.	0.	0.
2500	36.616	260.176	230.458	74.296	0.	0.	0.
2600	36.713	261.614	231.629	77.963	0.	0.	0.
2700	36.801	263.001	232.765	81.639	0.	0.	0.
2800	36.883	264.341	233.869	85.323	0.	0.	0.
2900	36.959	265.637	234.942	89.015	0.	0.	0.
3000	37.030	266.891	235.986	92.715	0.	0.	0.
3100	37.096	268.106	237.003	96.421	0.	0.	0.
3200	37.158	269.285	237.993	100.134	0.	0.	0.
3300	37.216	270.429	238.959	103.852	0.	0.	0.
3400	37.271	271.541	239.901	107.577	0.	0.	0.
3500	37.323	272.622	240.821	111.306	0.	0.	0.
3600	37.373	273.675	241.719	115.041	0.	0.	0.
3700	37.420	274.699	242.596	118.781	0.	0.	0.
3800	37.465	275.698	243.454	122.525	0.	0.	0.
3900	37.508	276.671	244.294	126.274	0.	0.	0.
4000	37.550	277.622	245.115	130.027	0.	0.	0.
4100	37.590	278.549	245.919	133.784	0.	0.	0.
4200	37.629	279.456	246.707	137.545	0.	0.	0.
4300	37.666	280.341	247.479	141.309	0.	0.	0.
4400	37.702	281.208	248.236	145.078	0.	0.	0.
4500	37.738	282.056	248.978	148.850	0.	0.	0.
4600	37.773	282.885	249.706	152.625	0.	0.	0.
4700	37.808	283.698	250.420	156.405	0.	0.	0.
4800	37.843	284.494	251.122	160.187	0.	0.	0.
4900	37.878	285.275	251.811	163.973	0.	0.	0.
5000	37.912	286.041	252.488	167.763	0.	0.	0.
5100	37.947	286.792	253.153	171.556	0.	0.	0.
5200	37.981	287.529	253.807	175.352	0.	0.	0.
5300	38.013	288.253	254.451	179.152	0.	0.	0.
5400	38.046	288.964	255.083	182.955	0.	0.	0.
5500	38.080	289.662	255.705	186.761	0.	0.	0.
5600	38.116	290.348	256.318	190.571	0.	0.	0.
5700	38.154	291.023	256.921	194.384	0.	0.	0.
5800	38.193	291.687	257.515	198.201	0.	0.	0.
5900	38.234	292.341	258.099	202.023	0.	0.	0.
6000	38.276	292.984	258.675	205.848	0.	0.	0.
''',
"O2.txt": r'''#Oxygen (O2)	O2(ref)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-8.683	0.	0.	0.
100	29.106	173.307	231.094	-5.779	0.	0.	0.
200	29.126	193.485	207.823	-2.868	0.	0.	0.
250	29.201	199.990	205.630	-1.410	0.	0.	0.
298.15	29.376	205.147	205.147	0.	0.	0.	0.
300	29.385	205.329	205.148	0.054	0.	0.	0.
350	29.694	209.880	205.506	1.531	0.	0.	0.
400	30.106	213.871	206.308	3.025	0.	0.	0.
450	30.584	217.445	207.350	4.543	0.	0.	0.
500	31.091	220.693	208.524	6.084	0.	0.	0.
600	32.090	226.451	211.044	9.244	0.	0.	0.
700	32.981	231.466	213.611	12.499	0.	0.	0.
800	33.733	235.921	216.126	15.835	0.	0.	0.
900	34.355	239.931	218.552	19.241	0.	0.	0.
1000	34.870	243.578	220.875	22.703	0.	0.	0.
1100	35.300	246.922	223.093	26.212	0.	0.	0.
1200	35.667	250.010	225.209	29.761	0.	0.	0.
1300	35.988	252.878	227.229	33.344	0.	0.	0.
1400	36.277	255.556	229.158	36.957	0.	0.	0.
1500	36.544	258.068	231.002	40.599	0.	0.	0.
1600	36.796	260.434	232.768	44.266	0.	0.	0.
1700	37.040	262.672	234.462	47.958	0.	0.	0.
1800	37.277	264.796	236.089	51.673	0.	0.	0.
1900	37.510	266.818	237.653	55.413	0.	0.	0.
2000	37.741	268.748	239.160	59.175	0.	0.	0.
2100	37.969	270.595	240.613	62.961	0.	0.	0.
2200	38.195	272.366	242.017	66.769	0.	0.	0.
2300	38.419	274.069	243.374	70.600	0.	0.	0.
2400	38.639	275.709	244.687	74.453	0.	0.	0.
2500	38.856	277.290	245.959	78.328	0.	0.	0.
2600	39.068	278.819	247.194	82.224	0.	0.	0.
2700	39.276	280.297	248.393	86.141	0.	0.	0.
2800	39.478	281.729	249.558	90.079	0.	0.	0.
2900	39.674	283.118	250.691	94.036	0.	0.	0.
3000	39.864	284.466	251.795	98.013	0.	0.	0.
3100	40.048	285.776	252.870	102.009	0.	0.	0.
3200	40.225	287.050	253.918	106.023	0.	0.	0.
3300	40.395	288.291	254.941	110.054	0.	0.	0.
3400	40.559	289.499	255.940	114.102	0.	0.	0.
3500	40.716	290.677	256.916	118.165	0.	0.	0.
3600	40.868	291.826	257.870	122.245	0.	0.	0.
3700	41.013	292.948	258.802	126.339	0.	0.	0.
3800	41.154	294.044	259.716	130.447	0.	0.	0.
3900	41.289	295.115	260.610	134.569	0.	0.	0.
4000	41.421	296.162	261.485	138.705	0.	0.	0.
4100	41.549	297.186	262.344	142.854	0.	0.	0.
4200	41.674	298.189	263.185	147.015	0.	0.	0.
4300	41.798	299.171	264.011	151.188	0.	0.	0.
4400	41.920	300.133	264.821	155.374	0.	0.	0.
4500	42.042	301.076	265.616	159.572	0.	0.	0.
4600	42.164	302.002	266.397	163.783	0.	0.	0.
4700	42.287	302.910	267.164	168.005	0.	0.	0.
4800	42.413	303.801	267.918	172.240	0.	0.	0.
4900	42.542	304.677	268.660	176.488	0.	0.	0.
5000	42.675	305.538	269.389	180.749	0.	0.	0.
5100	42.813	306.385	270.106	185.023	0.	0.	0.
5200	42.956	307.217	270.811	189.311	0.	0.	0.
5300	43.105	308.037	271.506	193.614	0.	0.	0.
5400	43.262	308.844	272.190	197.933	0.	0.	0.
5500	43.426	309.639	272.864	202.267	0.	0.	0.
5600	43.599	310.424	273.527	206.618	0.	0.	0.
5700	43.781	311.197	274.181	210.987	0.	0.	0.
5800	43.973	311.960	274.826	215.375	0.	0.	0.
5900	44.175	312.713	275.462	219.782	0.	0.	0.
6000	44.387	313.457	276.089	224.210	0.	0.	0.
''',
"OCS.txt": r'''#Carbon Oxide Sulfide (COS)	C1O1S1(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-9.929	-138.531	-138.531	INFINITE
100	29.610	194.193	264.316	-7.012	-137.816	-147.223	76.901
200	35.472	216.276	235.229	-3.791	-138.018	-156.581	40.895
298.15	41.497	231.581	231.581	0.	-138.407	-165.601	29.013
300	41.589	231.838	231.582	0.077	-138.415	-165.770	28.863
400	45.852	244.415	233.268	4.459	-141.138	-174.727	22.817
500	48.903	254.988	236.582	9.203	-143.179	-182.902	19.108
600	51.250	264.118	240.428	14.214	-144.910	-190.679	16.600
700	53.137	272.164	244.398	19.436	-146.435	-198.184	14.789
800	54.677	279.363	248.327	24.829	-147.842	-205.480	13.416
900	55.940	285.878	252.143	30.362	-202.304	-211.523	12.276
1000	56.990	291.828	255.818	36.010	-202.366	-212.544	11.102
1100	57.861	297.302	259.344	41.754	-202.442	-213.559	10.141
1200	58.597	302.369	262.721	47.577	-202.531	-214.565	9.340
1300	59.216	307.084	265.954	53.469	-202.633	-215.564	8.661
1400	59.748	311.492	269.051	59.418	-202.748	-216.555	8.080
1500	60.208	315.631	272.020	65.416	-202.878	-217.536	7.575
1600	60.605	319.529	274.869	71.457	-203.024	-218.509	7.134
1700	60.957	323.214	277.605	77.535	-203.187	-219.472	6.744
1800	61.266	326.707	280.237	83.647	-203.366	-220.425	6.397
1900	61.547	330.027	282.771	89.788	-203.563	-221.367	6.086
2000	61.798	333.191	285.213	95.955	-203.778	-222.299	5.806
2100	62.024	336.211	287.570	102.146	-204.011	-223.219	5.552
2200	62.233	339.102	289.847	108.359	-204.261	-224.128	5.321
2300	62.423	341.872	292.050	114.592	-204.529	-225.025	5.110
2400	62.601	344.533	294.181	120.843	-204.815	-225.910	4.917
2500	62.764	347.092	296.247	127.112	-205.118	-226.783	4.738
2600	62.919	349.556	298.250	133.396	-205.437	-227.643	4.573
2700	63.065	351.934	300.195	139.695	-205.773	-228.491	4.420
2800	63.203	354.230	302.084	146.009	-206.124	-229.326	4.278
2900	63.333	356.450	303.920	152.336	-206.491	-230.148	4.145
3000	63.459	358.599	305.707	158.675	-206.873	-230.958	4.021
3100	63.576	360.682	307.447	165.027	-207.270	-231.754	3.905
3200	63.691	362.702	309.143	171.390	-207.681	-232.537	3.796
3300	63.802	364.664	310.795	177.765	-208.107	-233.308	3.693
3400	63.910	366.570	312.408	184.151	-208.546	-234.064	3.596
3500	64.015	368.424	313.982	190.547	-208.998	-234.809	3.504
3600	64.114	370.229	315.520	196.954	-209.464	-235.539	3.418
3700	64.211	371.987	317.022	203.370	-209.944	-236.257	3.335
3800	64.306	373.701	318.491	209.796	-210.436	-236.962	3.257
3900	64.400	375.372	319.928	216.231	-210.942	-237.653	3.183
4000	64.492	377.004	321.335	222.676	-211.462	-238.331	3.112
4100	64.582	378.597	322.712	229.129	-211.994	-238.997	3.045
4200	64.670	380.155	324.061	235.592	-212.540	-239.649	2.980
4300	64.757	381.677	325.384	242.063	-213.100	-240.287	2.919
4400	64.843	383.167	326.680	248.543	-213.673	-240.913	2.860
4500	64.927	384.625	327.952	255.032	-214.261	-241.526	2.804
4600	65.009	386.053	329.199	261.529	-214.863	-242.125	2.749
4700	65.090	387.452	330.424	268.034	-215.479	-242.711	2.697
4800	65.170	388.823	331.626	274.547	-216.111	-243.284	2.647
4900	65.250	390.168	332.807	281.068	-216.759	-243.843	2.599
5000	65.329	391.487	333.968	287.597	-217.422	-244.389	2.553
5100	65.407	392.781	335.108	294.133	-218.101	-244.922	2.509
5200	65.484	394.052	336.230	300.678	-218.797	-245.441	2.465
5300	65.561	395.300	337.332	307.230	-219.511	-245.946	2.424
5400	65.638	396.527	338.417	313.790	-220.242	-246.439	2.384
5500	65.714	397.732	339.485	320.358	-220.991	-246.917	2.345
5600	65.789	398.916	340.536	326.933	-221.759	-247.381	2.307
5700	65.864	400.082	341.570	333.516	-222.546	-247.832	2.271
5800	65.939	401.228	342.589	340.106	-223.353	-248.268	2.236
5900	66.013	402.355	343.592	346.704	-224.180	-248.692	2.202
6000	66.086	403.466	344.581	353.308	-225.028	-249.099	2.169
''',
"S2.txt": r'''#Sulfur (S2)	S2(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.	0.	INFINITE	-9.124	128.300	128.300	INFINITE
100	29.367	195.067	255.684	-6.062	129.983	112.980	-59.015
200	30.452	215.621	231.072	-3.090	129.670	96.001	-25.073
250	31.513	222.529	228.695	-1.541	129.180	87.637	-18.311
298.15	32.490	228.165	228.165	0.	128.600	79.687	-13.961
300	32.525	228.366	228.166	0.060	128.576	79.384	-13.822
350	33.387	233.447	228.565	1.709	127.892	71.238	-10.632
400	34.090	237.953	229.462	3.396	122.718	63.371	-8.275
450	34.656	242.002	230.634	5.115	120.586	56.063	-6.508
500	35.111	245.678	231.958	6.860	118.326	49.019	-5.121
600	35.781	252.142	234.798	10.407	114.703	35.511	-3.092
700	36.260	257.695	237.681	14.009	111.612	22.562	-1.684
800	36.637	262.562	240.493	17.655	108.835	10.033	-0.655
882.117	36.908	266.155	242.717	20.674	FUGACITY = 1 bar
900	36.966	266.896	243.191	21.335	0. 0. 0.
1000	37.277	270.807	245.760	25.047	0.	0.	0.
1100	37.584	274.374	248.201	28.790	0.	0.	0.
1200	37.894	277.658	250.521	32.564	0.	0.	0.
1300	38.205	280.703	252.727	36.369	0.	0.	0.
1400	38.514	283.546	254.828	40.205	0.	0.	0.
1500	38.818	286.213	256.832	44.072	0.	0.	0.
1600	39.112	288.728	258.748	47.968	0.	0.	0.
1700	39.394	291.108	260.582	51.894	0.	0.	0.
1800	39.661	293.367	262.341	55.847	0.	0.	0.
1900	39.911	295.518	264.031	59.825	0.	0.	0.
2000	40.144	297.571	265.657	63.828	0.	0.	0.
2100	40.352	299.535	267.224	67.853	0.	0.	0.
2200	40.547	301.417	268.736	71.898	0.	0.	0.
2300	40.729	303.223	270.196	75.962	0.	0.	0.
2400	40.897	304.960	271.609	80.044	0.	0.	0.
2500	41.052	306.633	272.977	84.141	0.	0.	0.
2600	41.179	308.246	274.302	88.253	0.	0.	0.
2700	41.299	309.802	275.588	92.377	0.	0.	0.
2800	41.414	311.306	276.837	96.512	0.	0.	0.
2900	41.523	312.761	278.051	100.659	0.	0.	0.
3000	41.626	314.171	279.232	104.817	0.	0.	0.
3100	41.711	315.537	280.381	108.984	0.	0.	0.
3200	41.795	316.863	281.500	113.159	0.	0.	0.
3300	41.880	318.150	282.592	117.343	0.	0.	0.
3400	41.966	319.402	283.656	121.535	0.	0.	0.
3500	42.051	320.619	284.695	125.736	0.	0.	0.
3600	42.130	321.805	285.709	129.945	0.	0.	0.
3700	42.212	322.960	286.700	134.162	0.	0.	0.
3800	42.298	324.087	287.669	138.387	0.	0.	0.
3900	42.388	325.187	288.617	142.622	0.	0.	0.
4000	42.481	326.261	289.545	146.865	0.	0.	0.
4100	42.575	327.312	290.454	151.118	0.	0.	0.
4200	42.675	328.339	291.343	155.380	0.	0.	0.
4300	42.778	329.344	292.215	159.653	0.	0.	0.
4400	42.886	330.329	293.070	163.936	0.	0.	0.
4500	42.997	331.294	293.909	168.230	0.	0.	0.
4600	43.115	332.240	294.732	172.536	0.	0.	0.
4700	43.236	333.169	295.540	176.853	0.	0.	0.
4800	43.361	334.080	296.334	181.183	0.	0.	0.
4900	43.488	334.975	297.113	185.525	0.	0.	0.
5000	43.619	335.855	297.879	189.881	0.	0.	0.
5100	43.757	336.721	298.632	194.250	0.	0.	0.
5200	43.897	337.572	299.373	198.632	0.	0.	0.
5300	44.038	338.409	300.102	203.029	0.	0.	0.
5400	44.181	339.234	300.819	207.440	0.	0.	0.
5500	44.325	340.046	301.525	211.865	0.	0.	0.
5600	44.472	340.845	302.220	216.305	0.	0.	0.
5700	44.620	341.634	302.904	220.760	0.	0.	0.
5800	44.770	342.411	303.579	225.229	0.	0.	0.
5900	44.922	343.178	304.243	229.714	0.	0.	0.
6000	45.075	343.934	304.899	234.213	0.	0.	0.
''',
"SO2.txt": r'''#Sulfur Dioxide (SO2)	O2S1(g)
#T(K)	Cp	S	-[G-H(Tr)]/T	H-H(Tr)	delta-f H	delta-f G	log Kf
0	0.000	0.000	INFINITE	-10.552	-294.299	-294.299	INFINITE
100	33.526	209.025	281.199	-7.217	-294.559	-296.878	155.073
200	36.372	233.033	251.714	-3.736	-295.631	-298.813	78.042
298.15	39.878	248.212	248.212	0.000	-296.842	-300.125	52.581
300	39.945	248.459	248.213	0.074	-296.865	-300.145	52.260
400	43.493	260.448	249.824	4.250	-300.257	-300.971	39.303
500	46.576	270.495	252.979	8.758	-302.736	-300.871	31.432
600	49.049	279.214	256.641	13.544	-304.694	-300.305	26.144
700	50.961	286.924	260.427	18.548	-306.291	-299.444	22.345
800	52.434	293.829	264.178	23.721	-307.667	-298.370	19.482
900	53.580	300.073	267.825	29.023	-362.026	-296.051	17.182
1000	54.484	305.767	271.339	34.428	-361.940	-288.725	15.081
1100	55.204	310.995	274.710	39.914	-361.835	-281.409	13.363
1200	55.794	315.824	277.937	45.464	-361.720	-274.102	11.931
1300	56.279	320.310	281.026	51.069	-361.601	-266.806	10.720
1400	56.689	324.496	283.983	56.718	-361.484	-259.518	9.683
1500	57.036	328.419	286.816	62.404	-361.372	-252.239	8.784
1600	57.338	332.110	289.533	68.123	-361.268	-244.967	7.997
1700	57.601	335.594	292.141	73.870	-361.176	-237.701	7.304
1800	57.831	338.893	294.647	79.642	-361.096	-230.440	6.687
1900	58.040	342.026	297.059	85.436	-361.031	-223.183	6.136
2000	58.229	345.007	299.383	91.250	-360.981	-215.929	5.639
2100	58.400	347.853	301.624	97.081	-360.948	-208.678	5.191
2200	58.555	350.573	303.787	102.929	-360.931	-201.427	4.782
2300	58.702	353.179	305.878	108.792	-360.930	-194.177	4.410
2400	58.840	355.680	307.902	114.669	-360.947	-186.927	4.068
2500	58.965	358.085	309.861	120.559	-360.980	-179.675	3.754
2600	59.086	360.400	311.761	126.462	-361.030	-172.422	3.464
2700	59.199	362.632	313.604	132.376	-361.095	-165.166	3.195
2800	59.308	364.787	315.394	138.302	-361.175	-157.908	2.946
2900	59.413	366.870	317.133	144.238	-361.270	-150.648	2.713
3000	59.513	368.886	318.825	150.184	-361.379	-143.383	2.497
3100	59.609	370.839	320.471	156.140	-361.502	-136.114	2.294
3200	59.706	372.733	322.075	162.106	-361.638	-128.842	2.103
3300	59.794	374.572	323.638	168.081	-361.786	-121.565	1.924
3400	59.881	376.358	325.162	174.065	-361.946	-114.283	1.756
3500	59.969	378.095	326.650	180.057	-362.118	-106.996	1.597
3600	60.053	379.786	328.103	186.058	-362.300	-99.704	1.447
3700	60.137	381.432	329.522	192.068	-362.493	-92.408	1.305
3800	60.216	383.037	330.909	198.086	-362.697	-85.105	1.170
3900	60.296	384.602	332.266	204.111	-362.911	-77.796	1.042
4000	60.375	386.130	333.593	210.145	-363.134	-70.484	0.920
4100	60.450	387.621	334.893	216.186	-363.368	-63.164	0.805
4200	60.530	389.079	336.166	222.235	-363.611	-55.840	0.694
4300	60.605	390.504	337.413	228.292	-363.864	-48.508	0.589
4400	60.676	391.898	338.636	234.356	-364.128	-41.172	0.489
4500	60.752	393.263	339.834	240.427	-364.401	-33.829	0.393
4600	60.823	394.599	341.010	246.506	-364.686	-26.480	0.301
4700	60.894	395.908	342.165	252.592	-364.981	-19.125	0.213
4800	60.969	397.190	343.298	258.685	-365.288	-11.763	0.128
4900	61.036	398.448	344.410	264.785	-365.607	-4.394	0.047
5000	61.107	399.682	345.504	270.893	-365.938	2.981	-0.031
5100	61.178	400.893	346.578	277.007	-366.283	10.364	-0.106
5200	61.250	402.082	347.634	283.128	-366.641	17.751	-0.178
5300	61.317	403.249	348.672	289.257	-367.014	25.148	-0.248
5400	61.388	404.396	349.693	295.392	-367.403	32.550	-0.315
5500	61.455	405.523	350.698	301.534	-367.807	39.960	-0.380
5600	61.522	406.631	351.687	307.683	-368.230	47.379	-0.442
5700	61.588	407.720	352.661	313.838	-368.671	54.804	-0.502
5800	61.655	408.792	353.619	320.000	-369.131	62.238	-0.561
5900	61.727	409.846	354.563	326.169	-369.611	69.677	-0.617
6000	61.793	410.884	355.493	332.346	-370.113	77.128	-0.671
''',
}

_EVO_SHA256 = {'__init__': '151e369f7243d8d787d592eeae7562357544ce5a6131e2f2c8123fe1ab907b1f', 'constants': '840550a33d332eb0a1cd21a569b306c73069baae58cbd6df0aa7d4b4145c5a5f', 'ferric': '15d9bd330723af7c31b5107618a5ef77a0b59b12e34927d2d0c5091cbad0acb6', 'solubility_laws': 'b7cc944c480f4aeacf57545f08847067171d0c66c797adcfec76b678d45590c3', 'conversions': 'ed32b93bb3c1472c42b14108ccacdac5481cde8d000988cae2136c06efc098da', 'density': '868a2af39de223219048146daef8ef3fb379fc0156548da3f445ab7bd3c60bb6', 'init_cons': 'a1edb6a2ce2909502b55f80c04015d30c6dbfd46cda1c0509795b26e5a4af0eb', 'solvgas': '3d0283586db179a2d9fb901c0ec862a3e70bd021c18095b73655bed6bce4af61', 'plots': 'db115a281bebaa4d2fb2b1bb2b583f15310e2a5197fb6c695a28353984cfa8c4', 'sat_pressure': '3e14b41bc5b6688a8f67240bd04da880ef031112a7f4119163ba5dd37d9b949f', 'writeout': 'fce8ad6af9e94802d152e4b71a3e7645797a120758402920485390d87b3133b8', 'messages': '7f704a1e166e658bb0cd10e77d2a50aaa5f443d00f456590c180aea5ee411aab', 'fixed_weights': '861d71099dbb1b3a384cc3dcdbbc2f466d5aea41199b0524b8d7fb19a98260fe', 'dgs_classes': 'e66a61faf458bed8f8b2db56d9d14a4c05988e6291ce87f235e38f98632ca42b', 'solver': 'f58ffcc28528a0ffb09ac8fb9fde8ca1e7864a9dd0c19930100270248520dd61', 'readin': 'b4cdcfa025e061b958bb4da5e684bf4043b0c12c0208da81ff19e84c6d1b03a8', 'dgs': 'b4af83faf8a4268b0e4f5d1a7d6ff8a4d4efe53d5d9183a07d6bb38496159154', 'data/CH4.txt': '3ca2734db296a43852e2ed23df7731ce0186977f1e21739b7d35b1639ff84877', 'data/CO.txt': 'ed44d1046486a5c9f98423edd6bd42902f1b44991eb4318c4a2cdc73ea1998e5', 'data/CO2.txt': '51862359ba818bcf45e1e3bdf3c0078d15f270a1613ba982572d50971609b116', 'data/H2.txt': 'f7cd89824efc4d3092a3f4f0084d0086772382cc5ea9cc766d53846d5770f784', 'data/H2O.txt': 'd5e62f8cf0d2147240dd6c22c1cb8c0cc666619534b677e551ae56e72b7da460', 'data/H2S.txt': '6ed3ae97175c6b7aab4a8f36514b878af32fe12a816e02d103e3ad04d865f98d', 'data/N2.txt': 'c311a96298c88d8b3e1dbc73a73076a615e1c39c9decfd5ebb4a4407b6850f3c', 'data/O2.txt': '0e1d93e7145cb52c668bb478e23c0236b9122781aba79e6abee53fab0a797c61', 'data/OCS.txt': 'b50ae853bb5210ed9b0c325122e9201d8a4a6485faf3748c63dbedd155554671', 'data/S2.txt': 'f742aabb95216c8c2c184b39f8628e350d6b505f3a100d84cf2c3b8a7e4c9a43', 'data/SO2.txt': '78e0cee6e3f29b73fa2ba626f7c53f7d0d07c462aac303a05ae3685c702a8aa1'}


# =============================================================================
# D) Load the embedded EVo into memory as the package `evo_local`
# =============================================================================
if importlib.util.find_spec("yaml") is None:          # EVo imports yaml only to read files, which
    _yaml = types.ModuleType("yaml")                  # this notebook never does
    _yaml.full_load = lambda *a, **k: (_ for _ in ()).throw(RuntimeError("no YAML files are used"))
    sys.modules["yaml"] = _yaml

_EVO_SOURCES = restore_blank_lines(_EVO_SOURCES, _EVO_SOURCES_BLANKS)
_evo_audit = []
_evo_ready_sources = {}
for _name, _src in _EVO_SOURCES.items():
    _sha_ok = hashlib.sha256(_src.encode("utf-8")).hexdigest() == _EVO_SHA256[_name]
    _src2, _n_imp = _IMPORT_RE.subn(lambda m: f"{m.group(1)}{m.group(2)} evo_local", _src)
    for _mod, _old, _new, _why in EVO_PATCHES:
        if _mod == _name:
            assert _src2.count(_old) == 1, f"patch for {_mod} does not match the embedded source"
            _src2 = _src2.replace(_old, _new)
    _evo_ready_sources[_name] = _src2
    _evo_audit.append((_name, _src.count("\n"), _sha_ok, _n_imp))
for _fname, _txt in _EVO_DATA.items():
    _evo_audit.append((f"data/{_fname}", _txt.count("\n"),
                       hashlib.sha256(_txt.encode("utf-8")).hexdigest() == _EVO_SHA256[f"data/{_fname}"], 0))

evo_local = install_memory_package(
    "evo_local", _evo_ready_sources,
    inject_all={"exit": _evo_exit, "input": _evo_input},
    inject={"dgs_classes": {"_EVO_DATA": _EVO_DATA, "io": io}})
evo_local.messages.query_yes_no = _evo_yes              # never block on an interactive prompt


# =============================================================================
# E) evo_run(): EVo's own run sequence (dgs.run_evo), fed with Python dictionaries
# =============================================================================
def evo_run(chem, env):
    """Run EVo on a chemistry dict and an environment dict (the two input 'files').

    Executes exactly the calculation sequence of EVo's run_evo() - readin, set_init_chem
    (saturation pressure), decompress step by step down to P_STOP, writeout - but takes
    dictionaries, returns everything in memory and reports an undersaturated melt
    (P_sat < P_STOP) as a result instead of stopping.

    Returns a dict with
        saturated  : True if a vapour phase exists at P_STOP
        P_sat_bar  : volatile saturation pressure (bar), NaN if it was not computed
        df         : the complete EVo output table (all columns, every pressure step;
                     first row = saturation point), None if undersaturated
        sat_row    : the EVo output row at the saturation pressure
        log        : everything EVo printed;  prompts : questions answered 'yes'
    """
    from evo_local.readin import readin, set_init_chem
    from evo_local import solver, sat_pressure as _sat
    from evo_local.writeout import writeout_file

    n_prompts = len(EVO_PROMPTS)
    buf = io.StringIO()
    out = dict(saturated=False, P_sat_bar=np.nan, df=None, sat_row=None, log="", prompts=[])
    try:
        with contextlib.redirect_stdout(buf), warnings.catch_warnings():
            warnings.simplefilter("ignore")
            run, sys_, melt, _out = readin(pd.Series(dict(chem)), pd.Series(dict(env)), None)
            run, sys_, melt, gas, molecules = set_init_chem(run, sys_, melt)
            find_sat = run.FIND_SATURATION is True or run.ATOMIC_MASS_SET is True
            if find_sat:
                out["P_sat_bar"] = float(sys_.sat_conditions[0])

            if run.SINGLE_STEP is True:
                solver.decompress(run, sys_, melt, gas, molecules)
                sys_.P_track.append(sys_.P)
                out["df"] = writeout_file(sys_, gas, melt, sys_.P_track, write=False)
                out["saturated"] = True
            else:
                dp = (int(abs(np.floor(np.log10(np.abs(run.P_STOP))))) + 1
                      if run.P_STOP < 1e-4 else 5)
                while np.round(sys_.P, decimals=dp) >= run.P_STOP:
                    sys_.P = np.round(sys_.P, decimals=dp)
                    sys_.P_track.append(sys_.P)
                    solver.decompress(run, sys_, melt, gas, molecules)   # one pressure step
                    sys_.pressure_step()
                if find_sat and sys_.sat_conditions[0] < run.P_STOP:
                    # undersaturated at P_STOP: EVo itself would stop here with an error message.
                    # EVo writes the saturation-point row only after at least one pressure step,
                    # so it is taken from a run stopped at EVo's own first step below P_sat.
                    out["first_step_bar"] = float(run.P_START)
                else:
                    out["df"] = writeout_file(sys_, gas, melt, sys_.P_track, write=False)
                    out["saturated"] = True
                    if find_sat:
                        out["sat_row"] = out["df"].iloc[[0]].reset_index(drop=True)
    finally:
        out["log"] = buf.getvalue()
        out["prompts"] = EVO_PROMPTS[n_prompts:]
    p1 = out.pop("first_step_bar", None)
    if p1 is not None and p1 > 0:
        try:
            aux = evo_run(chem, dict(env, P_STOP=p1))
            if aux["df"] is not None:
                out["sat_row"] = aux["df"].iloc[[0]].reset_index(drop=True)
        except Exception:
            pass                     # the saturation-point row is informative only
    return out


# =============================================================================
# F) Self-test with fixed inputs (independent of the levers below)
# =============================================================================
_ST_CHEM = dict(SIO2=73.54, TIO2=0.13, AL2O3=13.10, FEO=2.21, MNO=0.07,
                MGO=0.34, CAO=1.02, NA2O=4.35, K2O=4.95, P2O5=0.04)
_ST_ENV = dict(COMPOSITION="rhyolite", RUN_TYPE="closed", SINGLE_STEP=False, FIND_SATURATION=True,
               ATOMIC_MASS_SET=False, GAS_SYS="COHS", FE_SYSTEM=True, OCS=False, S_SAT_WARN=False,
               T_START=1273.15, P_START=3000.0, P_STOP=147.0, DP_MIN=0.1, DP_MAX=50.0, MASS=100.0,
               WgT=1e-5, LOSS_FRAC=0.9999, DENSITY_MODEL="spera2000", FO2_MODEL="kc1991",
               FMQ_MODEL="frost1991", H2O_MODEL="burguisser2015", H2_MODEL="gaillard2003",
               C_MODEL="burguisser2015", CO_MODEL="None", CH4_MODEL="None",
               SULFIDE_CAPACITY="oneill2020", SULFATE_CAPACITY="nash2019", SCSS="liu2007",
               N_MODEL="libourel2003", FO2_buffer_SET=True, FO2_buffer="FMQ", FO2_buffer_START=1.0,
               FO2_SET=False, FO2_START=8.3e-12, FH2_SET=False, FH2_START=0.24, FH2O_SET=False,
               FH2O_START=1000.0, FCO2_SET=False, FCO2_START=1.0, WTH2O_SET=True, WTH2O_START=0.0125,
               WTCO2_SET=True, WTCO2_START=5e-05, SULFUR_SET=True, SULFUR_START=0.0002,
               NITROGEN_SET=False, NITROGEN_START=0.0001, GRAPHITE_SATURATED=False,
               GRAPHITE_START=0.0001)
_ST_REF = dict(phi_vol=15.541448468859048, P_sat_bar=297.85189291, n_rows=27)   # original EVo 1.1.0
_st = evo_run(_ST_CHEM, _ST_ENV)
_st_phi, _st_psat = float(_st["df"]["Exsol_vol%"].iloc[-1]), _st["P_sat_bar"]
EVO_SELFTEST_OK = (abs(_st_phi - _ST_REF["phi_vol"]) < 1e-9 * _ST_REF["phi_vol"]
                   and abs(_st_psat - _ST_REF["P_sat_bar"]) < 1e-9 * _ST_REF["P_sat_bar"]
                   and len(_st["df"]) == _ST_REF["n_rows"])

# ----------------------------------------------------------------------------- report
section("EVo 1.1.0 ready (package <code>evo_local</code>)")
_ok_all = all(a[2] for a in _evo_audit)
table(["File", "Lines", "Identical to the original EVo 1.1.0 file?", "Import lines renamed evo → evo_local"],
      [[f"<code>{n}</code>" if n.startswith("data/") else f"<code>{n}.py</code>", f"{l:,}",
        "yes (SHA-256 match)" if ok else "NO", str(ni) if not n.startswith("data/") else "—"]
       for n, l, ok, ni in _evo_audit],
      classes=lambda i, j: ("good" if _evo_audit[i][2] else "bad") if j == 2 else None,
      caption=f"{len(_EVO_SOURCES)} modules, {sum(a[1] for a in _evo_audit if not a[0].startswith('data/')):,} lines of "
              f"code and {len(_EVO_DATA)} thermochemical tables, embedded verbatim from {EVO_ORIGIN['repository']} "
              f"(v{EVO_ORIGIN['version']}, commit {EVO_ORIGIN['commit'][:7]}). Licence: {EVO_ORIGIN['licence']}.")
table(["Adaptation", "Where", "Why"],
      [["<code>import evo…</code> → <code>import evo_local…</code>", "all modules",
        "the copy in this cell runs as its own package, independent of any other EVo version"]] +
      [[f"source patch {i + 1}", f"<code>{m}.py</code>", why] for i, (m, _o, _n, why) in enumerate(EVO_PATCHES)] +
      [["<code>exit()</code>, <code>input()</code>, <code>query_yes_no()</code>", "run time",
        "EVo's stop messages become catchable exceptions (EVoStop); yes/no prompts are answered 'yes' "
        "and recorded, as in the original thesis wrapper; nothing waits for keyboard input"],
       ["<code>evo_run(chem, env)</code>", "this cell",
        "EVo's own run_evo() sequence fed with Python dictionaries instead of YAML files; an undersaturated "
        "melt is reported with its exact saturation pressure instead of stopping the run"]],
      left_cols=(0, 1, 2))
note(f"<b>Self-test {'passed' if EVO_SELFTEST_OK else 'FAILED'}.</b> Rhyolite (Scenario A budget) decompressed in a "
     f"closed C–O–H–S system to 147 bar: P<sub>sat</sub> = {_st_psat:.4f} bar (original EVo: {_ST_REF['P_sat_bar']:.4f}), "
     f"φ = {_st_phi:.6f} vol% (original EVo: {_ST_REF['phi_vol']:.6f}), {len(_st['df'])} output rows "
     f"(original: {_ST_REF['n_rows']}).", "ok" if (EVO_SELFTEST_OK and _ok_all) else "warn")

In [ ]:
#@title 3. dMODELS — analytical deformation sources (complete source code, dmodelspy 0.2)  { display-mode: "form" }
#@markdown The analytical deformation sources of step 7 come from **dMODELS** (Battaglia, Cervelli & Murray 2013),
#@markdown in the Python port `dmodelspy` 0.2 (W. Crawford, GPL-3.0): Yang et al. (1988) prolate spheroid with the
#@markdown Newman et al. (2006) corrections, and the Fialko et al. (2001) penny-shaped crack. The complete package
#@markdown (7 files) is stored verbatim below and loaded as the Python package `dmodels_local`.
import hashlib

DM_ORIGIN = dict(name="dmodelspy", version="0.2", authors="W. Crawford (Python port of dMODELS, "
                 "Battaglia, Cervelli & Murray 2013, USGS)", licence="GPL-3.0",
                 repository="pypi.org/project/dmodelspy/0.2")


# =============================================================================
# The embedded dmodelspy 0.2 source code (verbatim)
# =============================================================================
_DM_SOURCES = {
"__init__": r'''"""
Deformation forward modeling routines

Sill class (based on Fialko et al. 2001)
sphere2D function (based on McTigue 1987)
sphere3D_displ and sphere3D_strain functions (based on McTigue 1987)
spheroid and spheroid_displ functions (based on Yang et al. 1988)

==========================================================================
 USGS Software Disclaimer
 The software and related documentation were developed by the U.S.
 Geological Survey (USGS) for use by the USGS in fulfilling its mission.
 The software can be used, copied, modified, and distributed without any
 fee or cost. Use of appropriate credit is requested.

 The USGS provides no warranty, expressed or implied, as to the correctness
 of the furnished software or the suitability for any purpose. The software
 has been tested, but as with any complex software, there could be undetected
 errors. Users who find errors are requested to report them to the USGS.
 The USGS has limited resources to assist non-USGS users; however, we make
 an attempt to fix reported problems and help whenever possible.
==========================================================================
"""
from .mogi import Mogi
from .sill import Sill
from .sphere2D import Sphere2D
from .sphere3D import Sphere3D
from .spheroid import Spheroid

from .version import __version__

__bibtex__ = r"""
@Article{Fialko2001,
author = {Fialko, Yuri and Simons, Mark and Khazan, Yakov}, 
title = {Finite source modelling of magmatic unrest in Socorro, New Mexico, and Long Valley, California}, 
journal = {Geophys. J. Int.}, 
volume = {146}, 
number = {1}, 
pages = {191–200}, 
year = {2001}, 
abstract = {We investigate surface deformation associated with currently active crustal magma bodies in Socorro, New Mexico, and Long Valley, California, USA. We invert available geodetic data from these locations to constrain the overall geometry and dynamics of the inferred …}, 
location = {}, 
keywords = {}},

@Article{Yang1988,
author = {Yang, Xue‐Min and Davis, Paul M and Dieterich, James H}, 
title = {Deformation from inflation of a dipping finite prolate spheroid in an elastic half‐space as a model for volcanic stressing}, 
journal = {Journal of Geophysical Research: Solid Earth}, 
volume = {93}, 
number = {B5}, 
pages = {4249–4257}, 
year = {1988}, 
abstract = {Exact analytic expressions are given for the deformation field resulting from inflation of a finite prolate spheroidal cavity in an infinite elastic medium. The field is equivalent to that generated by a parabolic distribution of double forces and centers of dilatation along the …}, 
location = {}, 
keywords = {}},

@Article{McTigue1987,
author = {McTigue, DF}, 
title = {Elastic stress and deformation near a finite spherical magma body: resolution of the point source paradox}, 
journal = {Journal of Geophysical Research: Solid Earth}, 
volume = {92}, 
number = {B12}, 
pages = {12931–12940}, 
year = {1987}, 
abstract = {Approximate solutions are obtained for the stress and displacement fields due to a pressurized spherical cavity in an elastic half‐space. The solutions take the form of series expansions in powers of ε= a/d, where a is the cavity radius and d is the depth. The leading …}, 
location = {}, 
keywords = {}},

@Article{Battaglia2013,
author = {Battaglia, Maurizio and Cervelli, Peter F. and Murray, Jessica R.}, 
title = {dMODELS: A MATLAB software package for modeling crustal deformation near active faults and volcanic centers}, 
journal = {J. Volc. Geotherm. Res.}, 
volume = {254}, 
number = {}, 
pages = {1–4}, 
year = {2013}, 
abstract = {}, 
location = {}, 
keywords = {}},

@Article{Newman2006,
author = {Newman, Andrew V. and Dixon, Timothy H. and Gourmelen, Noel}, 
title = {A four-dimensional viscoelastic deformation model for Long Valley Caldera, California, between 1995 and 2000}, 
journal = {J. Volc. Geotherm. Res.}, 
volume = {150}, 
number = {1-3}, 
pages = {244--269}, 
year = {2006}, 
abstract = {}, 
location = {}, 
keywords = {}},

@Article{Mogi1958,
author = {Mogi, Kiyoo}, 
title = {Relations between the eruptions of various volcanoes and the deformatins of the ground surfaces around them}, 
journal = {Bull. Earthq Res Inst Univ Tokyo}, 
volume = {36}, 
number = {2}, 
pages = {99-134}, 
year = {1958}, 
abstract = {}, 
location = {}, 
keywords = {}}
"""
''',
"version": r'''__version__ = "0.2"
''',
"mogi": r'''from dataclasses import dataclass

from numpy import dot, multiply, sqrt, array, pi


@dataclass
class Mogi:

    """
    Mogi source

    All parameters are in SI (MKS) units
    :param x0,y0: coordinates of the center of the sphere
    :param z0: depth of the center of the sphere (positive downward and
        defined as distance below the reference surface)
    :param dV: injection (or removal) volume

    Poisson's ratio is assumed to be 0.25 (mu = lambda)

    :reference: Mogi1958
    """
    x0: float
    y0: float
    z0: float
    dV: float

    def calc_displ(self, x, y):
        """
        Calculate displacement for a Mogi source
        :param x,y: benchmark location
        :kind x,y: numpy.1darray()s, same shape
        :returns: u: horizontal (East component) deformation
                  v: horizontal (North component) deformation
                  w: vertical (Up component) deformation
        """
        x = array(x, ndmin=1)
        y = array(y, ndmin=1)
        assert x.shape == y.shape

        # translate the coordinates of the points where the displacement is
        # computed in the coordinates system centered in (x0,y0)
        xxn = x - self.x0
        yyn = y - self.y0
        # radial distance from source center to ur and uz points
        r = sqrt(xxn**2. + yyn**2.)

        # displacement (dimensional)
        u = 0.75 * (self.dV/pi) * r / (self.z0**2 + r**2)**1.5
        v = u.copy()
        w = 0.75 * (self.dV/pi) * self.z0 / (self.z0**2 + r**2)**1.5

        return u, v, w

    def calc_tilt(self, x, y):
        """
        Calculate tilt parameters

        :param x, y:      as in calc_displ()
        :returns:   dwdx, dwdy      ground tilts (East, North)
        """
        h = 0.001 * abs(max(x) - min(x))

        # East component
        _, _, wp = self.calc_displ(x + h, y)
        _, _, wm = self.calc_displ(x - h, y)
        dwdx = 0.5 * (wp - wm) / h
        # North component
        _, _, wp = self.calc_displ(x, y + h)
        _, _, wm = self.calc_displ(x, y - h)
        dwdy = 0.5 * (wp - wm) / h

        return dwdx, dwdy

    def calc_strain(self, x, y):
        """
        Calculate strain parameters

        :param x, y, z:      same as in calc()
        :returns: eea, gamma1, gamma2 (areal, shear1 & shear2)
        """
        h = 0.001 * abs(max(x) - min(x))

        up, _, _ = self.calc_displ(x + h, y)
        um, _, _ = self.calc_displ(x - h, y)
        dudx = 0.5 * (up - um) / h
        up, _, _ = self.calc_displ(x, y + h)
        um, _, _ = self.calc_displ(x, y - h)
        dudy = 0.5 * (up - um) / h
        _, vp, _ = self.calc_displ(x + h, y)
        _, vm, _ = self.calc_displ(x - h, y)
        dvdx = 0.5 * (vp - vm) / h
        _, vp, _ = self.calc_displ(x, y + h)
        _, vm, _ = self.calc_displ(x, y - h)
        dvdy = 0.5 * (vp - vm) / h

        eea = dudx + dvdy
        gamma1 = dudx - dvdy
        gamma2 = dudy + dvdx

        return eea, gamma1, gamma2

    def calc_all(self, x, y):
        """
        Calculate all 3D Green's function parameters

        :param x, y, z:      same as in calc()
        :returns:   u         horizontal (East component) deformation
                    v         horizontal (North component) deformation
                    w         vertical (Up component) deformation
                    dwdx      ground tilt (East component)
                    dwdy      ground tilt (North component)
                    eea       areal strain
                    gamma1    shear strain
                    gamma2    shear strain
        """
        u, v, w = self.calc_displ(x, y)
        dwdx, dwdy = self.calc_tilt(x, y)
        eea, gamma1, gamma2 = self.calc_strain(x, y)

        return u, v, w, dwdx, dwdy, eea, gamma1, gamma2
''',
"sill": r'''from dataclasses import dataclass, field

from numpy import (abs, arange, array, concatenate, cos, cosh, dot, exp, eye,
                   finfo, log, max, min, multiply, ones, pi, sqrt, sin, sinh,
                   zeros)
from numpy.linalg import solve
from scipy.special import jv as besselj


@dataclass
class Sill:
    """
    3D Green's function for sill-like source (Fialko et al., GJI, 2001)

    A pressurized sill-like source using the finite penny-crack model
    of Fialko et al (GJI, 2001)
    All parameters are in SI (MKS) units
    :param x0,y0:     coordinates of the center of the sphere
    :param z0:        depth of the center of the sill (positive downward and
                      defined as distance below the reference surface)
    :param P_G:       dimensionless excess pressure (pressure/shear modulus)
    :param a:         radius of the sphere
    :param nu:        Poisson's ratio

    """
    # There are two typos in the published paper
    # eq (12) and (13)
    #     (1) 2*Uz and 2*Ur must be replaced by Uz and Ur
    #     (2) dcsi/sinh(csi*h) must be replaced by dcsi
    # eq (24)
    #      (1-exp(-2*a)) must be replaced by exp(-a)
    x0: float
    y0: float
    z0: float
    P_G: float
    a: float
    nu: float
    # attributes affecting cached attributes
    _a: float = field(init=False, repr=False)
    _nu: float = field(init=False, repr=False)
    _P_G: float = field(init=False, repr=False)
    _z0: float = field(init=False, repr=False)
    # Cached variables
    _dV: float = field(init=False, repr=False, default=None)
    _csi_col: array = field(init=False, repr=False)
    _wcsi_col: array = field(init=False, repr=False)
    _Aarr: array = field(init=False, repr=False)
    _Barr: array = field(init=False, repr=False)

    @property
    def a(self) -> float:
        return self._a

    @a.setter
    def a(self, a: float):
        self._a = a
        self._dV = None

    @property
    def nu(self) -> float:
        return self._nu

    @nu.setter
    def nu(self, nu: float):
        self._nu = nu
        self._dV = None

    @property
    def P_G(self) -> float:
        return self._P_G

    @P_G.setter
    def P_G(self, P_G: float):
        self._P_G = P_G
        self._dV = None

    @property
    def z0(self) -> float:
        return self._z0

    @z0.setter
    def z0(self, z0: float):
        self._z0 = z0
        self._dV = None

    @property
    def dV(self) -> float:
        "Volume changeq"
        if self._dV is None:
            self._calc_base()
        return self._dV

    def _calc_base(self):
        """
        Calculate base (cached) parameters

        Generates self.(_dV, _csi_col, _wcsi_row, _Aarr, _Barr)
        """
        # General parameters
        eps = 1e-08
        h = self._z0 / self._a                      # dimensionless source depth

        # solve for PHI and PSI, Fialko et al. (2001), eq. (26)
        # ascissas and weights for Gauss-Legendre quadrature
        csi1, w1 = _gauleg(eps, 10, 41)
        csi2, w2 = _gauleg(10, 60, 41)
        csi = concatenate((csi1, csi2))
        wcsi = concatenate((w1, w2))

        self._csi_col = csi.reshape((-1, 1))
        self._wcsi_row = wcsi.reshape((1, -1))

        phi, psi, t, wt = _psi_phi(h)
        wt_col = wt.reshape((-1, 1))
        t_col = t.reshape((-1, 1))
        phi_col = phi.reshape((-1, 1))
        psi_col = psi.reshape((-1, 1))
        csi_t = self._csi_col @ t.reshape((1, -1))
        # Gauss-Legendre quadrature
        PHI = sin(csi_t) @ (wt_col * phi_col)
        PSI = (sin(csi_t) / (csi_t) - cos(csi_t)) @ (wt_col * psi_col)

        # compute A and B, Fialko et al. (2001), eq. (24)
        # NOTE error in eq (24), (1-exp(-2*a)) must be replaced by exp(-a)
        a = h * self._csi_col
        self._Aarr = exp(-a) * (a * PSI + (1 + a) * PHI)
        self._Barr = exp(-a) * ((1 - a) * PSI - a * PHI)

        # Volume change
        dV = -4*pi * (1-self._nu) * self._P_G * self._a**3. * dot(
                   t_col.T, wt_col * phi_col)
        self._dV = dV[0][0]

    def calc_displ(self, x, y, z):
        """
        Calculate displacement

        :param x,y:       benchmark location
        :param z:         depth within the crust (z=0 is the free surface)
        :kind x, y, z: 1d arrays
        :returns:   u, v, w:  E, N and Up deformation components
        """
        if self._dV is None:
            self._calc_base()

        # Coordinates transformation
        x = (array(x, ndmin=1) - self.x0) / self.a
        y = (array(y, ndmin=1) - self.y0) / self.a
        z = (array(z, ndmin=1) - self.z0) / self.a
        # print(f'{x.shape=}')
        # print(f'{x=}')
        # compute radial distance
        r = sqrt(x**2 + y**2)
        h = self.z0 / self.a                      # dimensionless source depth

        # compute Uz and Ur, Fialko et al. (2001), eq. (12) and (13)
        # NOTE there are two errors in eq (12) and (13)
        # (1) 2*Uz and 2*Ur must be replaced by Uz and Ur
        # (2) dcsi/sinh(csi*h) must be replaced by dcsi
        Uz = zeros(r.shape)
        Ur = zeros(r.shape)
        csi_z_h = (z+h) * self._csi_col
        for i in arange(r.size):
            J0 = besselj(0, r[i] * self._csi_col)
            Uzi = J0 * (((1 - 2*self.nu) * self._Barr - csi_z_h * self._Aarr) * sinh(csi_z_h)
                        + (2*(1-self.nu) * self._Aarr - csi_z_h * self._Barr) * cosh(csi_z_h))
            Uz[i] = dot(self._wcsi_row, Uzi)
            J1 = besselj(1, r[i] * self._csi_col)
            Uri = J1 * (((1 - 2*self.nu) * self._Aarr + csi_z_h * self._Barr) * sinh(csi_z_h)
                        + (2*(1-self.nu) * self._Barr + csi_z_h * self._Aarr) * cosh(csi_z_h))
            Ur[i] = dot(self._wcsi_row, Uri)

        # Deformation components
        u = self.a * self.P_G * Ur * x / r
        v = self.a * self.P_G * Ur * y / r
        w = -self.a * self.P_G * Uz

        for var in [u, v, w]:
            var = var.reshape(x.shape)
        return u, v, w

    def calc_tilt(self, x, y, z):
        """
        Calculate tilt parameters

        :param x, y, z:      same as in calc()
        :returns:   dwdx, dwdy      ground tilts (East, North)
        """
        h = 0.001 * abs(max(x) - min(x))

        # East component
        _, _, wp = self.calc_displ(x + h, y, z)
        _, _, wm = self.calc_displ(x - h, y, z)
        dwdx = 0.5 * (wp - wm) / h
        # North component
        _, _, wp = self.calc_displ(x, y + h, z)
        _, _, wm = self.calc_displ(x, y - h, z)
        dwdy = 0.5 * (wp - wm) / h

        return dwdx, dwdy

    def calc_strain(self, x, y, z):
        """
        Calculate strain parameters

        :param x, y, z:      same as in calc()
        :returns: eea, gamma1, gamma2 (areal, shear1 & shear2)
        """
        h = 0.001 * abs(max(x) - min(x))

        up, _, _ = self.calc_displ(x + h, y, z)
        um, _, _ = self.calc_displ(x - h, y, z)
        dudx = 0.5 * (up - um) / h
        up, _, _ = self.calc_displ(x, y + h, z)
        um, _, _ = self.calc_displ(x, y - h, z)
        dudy = 0.5 * (up - um) / h
        _, vp, _ = self.calc_displ(x + h, y, z)
        _, vm, _ = self.calc_displ(x - h, y, z)
        dvdx = 0.5 * (vp - vm) / h
        _, vp, _ = self.calc_displ(x, y + h, z)
        _, vm, _ = self.calc_displ(x, y - h, z)
        dvdy = 0.5 * (vp - vm) / h

        eea = dudx + dvdy
        gamma1 = dudx - dvdy
        gamma2 = dudy + dvdx

        return eea, gamma1, gamma2

    def calc_all(self, x, y, z):
        """
        Calculate all 3D Green's function parameters

        :param x, y, z:      same as in calc()
        :returns:   u         horizontal (East component) deformation
                    v         horizontal (North component) deformation
                    w         vertical (Up component) deformation
                    dwdx      ground tilt (East component)
                    dwdy      ground tilt (North component)
                    eea       areal strain
                    gamma1    shear strain
                    gamma2    shear strain
        """
        u, v, w = self.calc_displ(x, y, z)
        dwdx, dwdy = self.calc_tilt(x, y, z)
        eea, gamma1, gamma2 = self.calc_strain(x, y, z)

        return u, v, w, dwdx, dwdy, eea, gamma1, gamma2


def _legpol(x, N):
    """
    Legendre polynomial, Bonnetis recursion formula

    Reference: http://en.wikipedia.org/wiki/Legendre_polynomials
    :param x:
    :kind x: np.array of size=(1,M)
    :param N:
    :returns: pN, dpN

    Note
    index n in wiki goes from 0 to N, in MATLAB j goes from 1 to N+1. To
    obtain the right coefficients we introduced j = n+1 -> n = j-1
    Reversed for python implementation
    """
    P = ones((N+1, x.size))
    dP = zeros((N, x.size))
    P[1, :] = x
    for j in arange(1, N):
        P[j+1, :] = (multiply(dot((2*j+1), x), P[j, :]) - j*P[j-1, :]) / (j+1)
        dP[j, :] = dot(j, (multiply(x, P[j, :]) - P[j-1, :])) / (x ** 2 - 1)

    pN = P[N-1, :]
    dpN = dP[N-1, :]

    return pN, dpN


def _gauleg(x1, x2, N):
    """
    Return abcissas and weights of the Gaussian- Legendre quadrature

    Loosely based on SUBROUTINE gauleg (Numerical Recipes in Fortran 77, 4.5)
    :param x1: lower integration limit
    :param x2: upper integration limit
    :param N: length of arrays to return
    :returns: x, w: abcissas and weights arrays
    :rtype: (1,N) numpy.array

    Note:
    tested against the "High-precision Abscissas and Weights for Gaussian
    Quadrature of high order." table from http://www.holoborodko.com/pavel/
    click on Numerical Methods then Numerical Integration. The Table is at
    the end of the page
    >> _gauleg(-1, 1, 11)
    ****** for N = 11
     Abscissas x                      Weights w
     0                               0.2729250867779006307144835
    +-0.2695431559523449723315320 	0.2628045445102466621806889
    +-0.5190961292068118159257257 	0.2331937645919904799185237
    +-0.7301520055740493240934163 	0.1862902109277342514260976
    +-0.8870625997680952990751578 	0.1255803694649046246346943
    +-0.9782286581460569928039380 	0.0556685671161736664827537
    *************************************************************************
    """
    z = zeros((N))
    xm = dot(0.5, (x2 + x1))
    xl = dot(0.5, (x2 - x1))

    for n in arange(N):
        z[n] = cos(dot(pi, (n+1 - 0.25)) / (N+0.5))
        z1 = dot(100, z[n])
        while abs(z1 - z[n]) > finfo(float).eps:
            pN, dpN = _legpol(z[n], N+1)
            z1 = z[n]
            z[n] = z1 - pN / dpN

    _, dpN = _legpol(z, N+1)
    x = xm - dot(xl, z)
    w = dot(2, xl) / (multiply((1 - z**2), dpN**2))

    return x, w


def _psi_phi(h):
    """
    compute the functions phi(t) and psi(t)

    Using the Nystrom routine with the  N-point Gauss-Legendre rule
    (Numerical Recipes in Fortran 77 (1992), 18.1 Fredholm Equations of the
    Second Kind, p. 782.)

    :param h:     dimensionless source depth
    :returns: phi, psi, t, w
              phi and psi are expressions from Appendix A of Fialko et al
              (2001), equation (A1). phi and psi are used in the definition
              of the functions PHI and PSI. See also A_B.m and Fialko et al
              (2001), eq. (26), p. 184.
              t and w are the abscissas and weights from the Gauss-Legendre
              quadrature rule (t is a value between 0 and 1; length(t) = M)

    ###########################################################################
    # Press W. et al. (1992). Numerical Recipes in Fortran 77. 2nd Ed,
    #  Cambridge University Press. Available on-line at www.nrbook.com/fortran/
    ###########################################################################
    """
    t, w = _gauleg(0, 1, 41)

    # Solution at the quadrature points r(j) **********************************
    g = dot(-2, t) / pi
    d = concatenate((g, zeros(g.shape))).T
    T1, T2, T3, T4 = _calc_T(h, t, t)

    T1tilde = zeros(T1.shape)
    T2tilde = zeros(T1.shape)
    T3tilde = zeros(T1.shape)
    T4tilde = zeros(T1.shape)
    N = t.size
    for j in range(N):
        T1tilde[:, j] = dot(w[j], T1[:, j])
        T2tilde[:, j] = dot(w[j], T2[:, j])
        T3tilde[:, j] = dot(w[j], T3[:, j])
        T4tilde[:, j] = dot(w[j], T4[:, j])

    Ktilde = concatenate((concatenate((T1tilde, T3tilde)),
                          concatenate((T4tilde, T2tilde))),
                         axis=1)

    y = solve((eye(dot(2, N), dot(2, N)) - dot((2 / pi), Ktilde)), d)

    phi = y[: N]
    psi = y[N: 2*N]
    # phi=y[arange(1,N)]
    # psi=y[arange(N + 1,dot(2,N))]

    return phi, psi, t, w


def _calc_P(h, x):
    """
    :param h:     dimensionless source depth
    :param x:     dummy variable

    P1-P4 are expressions from Appendix A of Fialko et al (2001). Used in the
    definition of the functions T1 - T4 (see T.m)
    """
    outP = zeros((4, x.size))
    # P1(x), pg 189
    outP[0, :] = (dot(12, h**2) - x**2) / (dot(4, h**2) + x**2)**3
    # P2(x), pg 189
    outP[1, :] = log(4*h**2 + x**2)\
        + (8*h**4 + dot(2*x**2, h**2) - x**4) / (dot(4, h**2) + x**2)**2
    # P3(x), pg 189
    outP[2, :] = dot(2, (8*h**4 - dot(2*x**2, h**2) + x**4))\
        / (4*h**2 + x**2)**3
    # P4(x), pg 189
    outP[3, :] = (dot(4, h**2) - x**2) / (dot(4, h**2) + x**2)**2

    return outP


def _calc_T(h, t, r):
    """
    :param h:     dimensionless source depth
    :param r:     dummy variable (used to integrate along the sill
        dimenionless radius)
    :param t:     dummy variable

    :returns: T1-T4,  expressions from Appendix A of Fialko et al (2001),
        equations (A2) - (A5). T1-T4 are used in the definition of the
        functions phi and psi (see phi_psi.m)
    """
    M = t.size
    N = r.size
    T1 = zeros((M, N))
    T2 = T1.copy()
    T3 = T1.copy()

    for i in range(M):
        Pm = _calc_P(h, t[i] - r)
        Pp = _calc_P(h, t[i] + r)
        T1[i, :] = dot(dot(4, h**3), (Pm[0, :] - Pp[0, :]))
        T2[i, :] = multiply((h / (multiply(t[i], r))), (Pm[1, :] - Pp[1, :]))\
            + dot(h, (Pm[2, :] + Pp[2, :]))
        T3[i, :] = multiply((h**2. / r),
                            (Pm[3, :] - Pp[3, :]
                             - multiply(dot(2, r),
                                        (multiply((t[i] - r), Pm[0, :])
                                         + multiply((t[i] + r), Pp[0, :])))))

    T4 = T3.T
    return T1, T2, T3, T4
''',
"sphere2D": r'''from dataclasses import dataclass

from numpy import dot, multiply, sqrt, array


@dataclass
class Sphere2D:

    """
    Spherical source, forward model based on eq. (52) & (53) by McTigue (1988)

    All parameters are in SI (MKS) units
    :param x0,y0: coordinates of the center of the sphere
    :param z0: depth of the center of the sphere (positive downward and
        defined as distance below the reference surface)
    :param P_G: dimensionless excess pressure (pressure/shear modulus)
    :param a: radius of the sphere
    :param nu: Poisson's ratio
    :param x,y: benchmark location
    :kind x,y: numpy.1darray()s, same shape
    :returns: u: horizontal (East component) deformation
              v: horizontal (North component) deformation
              w: vertical (Up component) deformation
           dwdx: ground tilt (East component)
           dwdy: ground tilt (North component)

    Reference ***************************************************************
    McTigue, D.F. (1987). Elastic Stress and Deformation Near a Finite
    Spherical Magma Body: Resolution of the Point Source Paradox. J. Geophys.
    Res. 92, 12,931-12,940.
    """
    x0: float
    y0: float
    z0: float
    P_G: float
    a: float
    nu: float

    def _calc_base(self):
        """
        Parameters that don't depend on the station positions
        """
        # dimensionless coordinates
        e = self.a / self.z0
        # constant and expression used in the formulas
        c1 = e**3. / (7. - 5.*self.nu)
        return e, c1

    def _prep_arrays(self, x, y):
        """
        Prepare arrays
        """
        x = array(x, ndmin=1)
        y = array(y, ndmin=1)
        assert x.shape == y.shape

        # translate the coordinates of the points where the displacement is
        # computed in the coordinates system centered in (x0,y0)
        xxn = x - self.x0
        yyn = y - self.y0
        # radial distance from source center to ur and uz points
        r = sqrt(xxn**2. + yyn**2.)

        # dimensionless coordinates
        csi = xxn / self.z0
        psi = yyn / self.z0
        rho = r / self.z0
        f1 = 1. / (rho**2 + 1.)**1.5
        f2 = 1. / (rho**2 + 1.)**2.5

        return x, y, xxn, yyn, r, csi, psi, rho, f1, f2

    def calc_displ(self, x, y):
        x, y, xxn, yyn, r, csi, psi, rho, f1, _ = self._prep_arrays(x, y)
        e, c1 = self._calc_base()

        # displacement (dimensionless) [McTigue (1988), eq. (52) and (53)]
        uzbar = e**3. * (1.-self.nu) *\
            multiply(f1, (1. - dot(c1, ((0.5*(1.+self.nu))
                                        - 3.75*(2.-self.nu) / (rho**2 + 1.)))))
        urbar = multiply(rho, uzbar)
        # displacement (dimensional)
        u = self.P_G * self.z0 * urbar * xxn / r
        v = self.P_G * self.z0 * urbar * yyn / r
        w = self.P_G * self.z0 * uzbar

        return u, v, w

    def calc_tilt(self, x, y):
        x, y, xxn, yyn, r, csi, psi, rho, _, f2 = self._prep_arrays(x, y)
        e, c1 = self._calc_base()
        # see equation (5) in documentation
        dwmul = (3. - dot(c1, (1.5*(1.+self.nu)) - (18.75*(2.-self.nu))
                 / (rho**2. + 1.)))
        dwdx = -(1.-self.nu) * self.P_G * e**3. * csi * f2 * dwmul
        dwdy = -(1.-self.nu) * self.P_G * e**3. * psi * f2 * dwmul

        return dwdx, dwdy

    def calc_strain(self, x, y):
        """
        Calculate strain parameters

        :param x, y, z:      same as in calc()
        :returns: eea, gamma1, gamma2 (areal, shear1 & shear2)
        """
        h = 0.001 * abs(max(x) - min(x))

        up, _, _ = self.calc_displ(x + h, y)
        um, _, _ = self.calc_displ(x - h, y)
        dudx = 0.5 * (up - um) / h
        up, _, _ = self.calc_displ(x, y + h)
        um, _, _ = self.calc_displ(x, y - h)
        dudy = 0.5 * (up - um) / h
        _, vp, _ = self.calc_displ(x + h, y)
        _, vm, _ = self.calc_displ(x - h, y)
        dvdx = 0.5 * (vp - vm) / h
        _, vp, _ = self.calc_displ(x, y + h)
        _, vm, _ = self.calc_displ(x, y - h)
        dvdy = 0.5 * (vp - vm) / h

        eea = dudx + dvdy
        gamma1 = dudx - dvdy
        gamma2 = dudy + dvdx

        return eea, gamma1, gamma2

    def calc_all(self, x, y):
        """
        Calculate all 3D Green's function parameters

        :param x, y, z:      same as in calc()
        :returns:   u         horizontal (East component) deformation
                    v         horizontal (North component) deformation
                    w         vertical (Up component) deformation
                    dwdx      ground tilt (East component)
                    dwdy      ground tilt (North component)
                    eea       areal strain
                    gamma1    shear strain
                    gamma2    shear strain
        """
        u, v, w = self.calc_displ(x, y)
        dwdx, dwdy = self.calc_tilt(x, y)
        eea, gamma1, gamma2 = self.calc_strain(x, y)

        return u, v, w, dwdx, dwdy, eea, gamma1, gamma2
''',
"sphere3D": r'''from dataclasses import dataclass

from numpy import exp, sqrt, zeros, dot, multiply, shape, arange, array, pi
from scipy.integrate import quad as quadl
from scipy.special import jv as besselj


@dataclass
class Sphere3D:
    """
    3D Green's function for spherical source

    All parameters are in SI (MKS) units
    :param x0,y0: coordinates of the center of the sphere
    :param z0: depth of the center of the sphere (positive downward and
        defined as distance below the reference surface)
    :param P_G: dimensionless excess pressure (pressure/shear modulus)
    :param a: radius of the sphere
    :param nu: Poisson's ratio

    Reference ************************************************************
    McTigue, D.F. (1987). Elastic Stress and Deformation Near a Finite
    Spherical Magma Body: Resolution of the Point Source Paradox. J. Geophys.
    Res. 92, 12,931-12,940.
    """
    x0: float
    y0: float
    z0: float
    P_G: float
    a: float
    nu: float

    @property
    def dV(self) -> float:
        "Injection volume"
        # from between equations 1 and 2: dV = pi * p_0 * a**3 / G
        # and since P_G = p_0/G we have
        return pi * self.P_G * self.a**3.

    def calc_displ(self, x, y, z):
        """
        Calculate displacement

        :param x,y:       benchmark location
        :param z:         depth within the crust (z=0 is the free surface)
        :kind x, y, z: 1d arrays
        :returns:   u, v, w:  E, N and Up deformation components
        """
        x = array(x, ndmin=1)
        y = array(y, ndmin=1)
        z = array(z, ndmin=1)
        assert x.shape == y.shape
        # translate the coordinates of the points where the displacement is
        # computed in the coordinates system centered in (x0, y0)
        xxn = x - self.x0
        yyn = y - self.y0
        # radial distance from source center to points where we compute ur & uz
        r = sqrt(xxn**2 + yyn**2)
        # dimensionless parameters used in the formulas
        rho = r / self.z0
        e = self.a / self.z0
        zeta = z / self.z0

        # DIMENSIONLESS DISPLACEMENT *********************************
        # leading order solution for a pressurized sphere in an unbounded
        # region based on equation (11) of McTigue (1987)
        uz0 = (1 - zeta) * 0.25 * e**3 / (rho**2 + (1 - zeta)**2)**1.5
        # return equation (14) when zeta=0
        ur0 = dot(dot(e**3, 0.25), rho) / (rho**2 + (1 - zeta)**2)**1.5

        # first free surface correction, equations (A7), (A8), (A17) and (A18)
        # return equation (22) and (23) when csi = 0
        Auz1 = zeros((zeta.size, rho.size))
        Aur1 = zeros((zeta.size, rho.size))
        for i in arange(zeta.size):
            for j in arange(rho.size):
                args = (self.nu, float(rho[j]), float(zeta[i]))
                Auz1[i, j], _ = quadl(mctigueA7A17, 0, 50, args=args)
                Aur1[i, j], _ = quadl(mctigueA8A18, 0, 50, args=args)

        # higher order cavity correction, equations (38) and (39)
        R = sqrt(rho**2 + (1 - zeta)**2)
        sint = rho / R
        cost = (1 - zeta) / R

        ###########
        # The following do not depend on x, y, z and could be separated, but
        # but they're too quick to care about...
        C3 = [dot(e, (1 + self.nu)) / (dot(12, (1 - self.nu))),
              dot(dot(5, e**3), (2 - self.nu))
              / (dot(24, (7 - dot(5, self.nu))))]
        D3 = [dot(-e**3, (1 + self.nu)) / 12,
              dot(e**5, (2 - self.nu)) / (dot(4, (7 - self.nu)))]
        P0 = 1
        dP0 = 0
        ###########

        P2 = dot(0.5, (dot(3, cost**2) - 1))
        dP2 = dot(3, cost)
        ur38 = -0.5 * P0 * D3[0] / R**2\
            + multiply((C3[1] * (5 - 4*self.nu) - 1.5 * D3[1] / R**2), P2)\
            / R**2
        ut39 = dot(-(2 * C3[0] * (1 - self.nu) - 0.5 * D3[0] / R**2), dP0)\
            - multiply((C3[1] * (1 - (2*self.nu)) + (0.5*D3[1]) / R**2), dP2)\
            / R**2
        ut39 = multiply(ut39, sint)
        Auz3 = multiply(ur38, cost) - multiply(ut39, sint)
        Aur3 = multiply(ur38, sint) + multiply(ut39, cost)
        # Force Auz3 and Aur3 to be 2D
        Auz3 = Auz3.reshape(Auz3.shape[0], -1)
        Aur3 = Aur3.reshape(Aur3.shape[0], -1)

        # sixth order surface correction, return eq (50) and (51) when zeta=0
        Auz6 = zeros((zeta.size, rho.size))
        Aur6 = zeros((zeta.size, rho.size))
        for i in arange(zeta.size):
            for j in arange(rho.size):
                args = (self.nu, float(rho[j]), float(zeta[i]))
                Auz6[i, j], _ = quadl(mctigueA7A17a, 0, 50, args=args)
                Aur6[i, j], _ = quadl(mctigueA8A18a, 0, 50, args=args)

        # total surface displacement, return eq (52) and (53) when zeta = 0
        if shape(uz0)[0] == shape(Auz1)[1]:
            Auz1 = Auz1.T
            Aur1 = Aur1.T

        if shape(uz0)[0] == shape(Auz6)[1]:
            Auz6 = Auz6.T
            Aur6 = Aur6.T

        uz = uz0.reshape(uz0.shape[0], -1) + e**3 * (Auz1 + Auz3) + e**6 * Auz6
        # uz = e^3*Auz6;
        ur = ur0.reshape(ur0.shape[0], -1) + e**3 * (Aur1 + Aur3) + e**6 * Aur6

        # displacement components
        u = multiply(ur, xxn.reshape(xxn.shape[0], -1)) / r.reshape(r.shape[0],
                                                                    -1)
        v = multiply(ur, yyn.reshape(yyn.shape[0], -1)) / r.reshape(r.shape[0],
                                                                    -1)
        w = uz.copy()
        # print(f'{u.shape=}, {v.shape=}, {w.shape=}')

        # DIMENSIONAL DISPLACEMENT **********
        u = dot(dot(u, self.P_G), self.z0)
        v = dot(dot(v, self.P_G), self.z0)
        w = dot(dot(w, self.P_G), self.z0)

        u = u.reshape(x.shape)
        v = v.reshape(x.shape)
        w = w.reshape(x.shape)
        return u, v, w

    def calc_tilt(self, x, y, z):
        """
        Calculate tilt parameters

        :param x, y, z:      same as in calc()
        :returns:   dwdx, dwdy      ground tilts (East, North)
        """
        h = 0.001 * abs(max(x) - min(x))

        # East component
        _, _, wp = self.calc_displ(x + h, y, z)
        _, _, wm = self.calc_displ(x - h, y, z)
        dwdx = 0.5 * (wp - wm) / h
        # North component
        _, _, wp = self.calc_displ(x, y + h, z)
        _, _, wm = self.calc_displ(x, y - h, z)
        dwdy = 0.5 * (wp - wm) / h

        return dwdx, dwdy

    def calc_strain(self, x, y, z):
        """
        Calculate strain parameters

        :param x, y, z:      same as in calc()
        :returns: eea, gamma1, gamma2 (areal, shear1 & shear2)
        """
        h = 0.001 * abs(max(x) - min(x))

        up, _, _ = self.calc_displ(x + h, y, z)
        um, _, _ = self.calc_displ(x - h, y, z)
        dudx = 0.5 * (up - um) / h
        up, _, _ = self.calc_displ(x, y + h, z)
        um, _, _ = self.calc_displ(x, y - h, z)
        dudy = 0.5 * (up - um) / h
        _, vp, _ = self.calc_displ(x + h, y, z)
        _, vm, _ = self.calc_displ(x - h, y, z)
        dvdx = 0.5 * (vp - vm) / h
        _, vp, _ = self.calc_displ(x, y + h, z)
        _, vm, _ = self.calc_displ(x, y - h, z)
        dvdy = 0.5 * (vp - vm) / h

        eea = dudx + dvdy
        gamma1 = dudx - dvdy
        gamma2 = dudy + dvdx

        return eea, gamma1, gamma2

    def calc_all(self, x, y, z):
        """
        Calculate all 3D Green's function parameters

        :param x, y, z:      same as in calc()
        :returns:   u         horizontal (East component) deformation
                    v         horizontal (North component) deformation
                    w         vertical (Up component) deformation
                    dwdx      ground tilt (East component)
                    dwdy      ground tilt (North component)
                    eea       areal strain
                    gamma1    shear strain
                    gamma2    shear strain
        """
        u, v, w = self.calc_displ(x, y, z)
        dwdx, dwdy = self.calc_tilt(x, y, z)
        eea, gamma1, gamma2 = self.calc_strain(x, y, z)

        return u, v, w, dwdx, dwdy, eea, gamma1, gamma2


def mctigueA8A18a(tt,  nu,  rho,  zeta):
    """
    Sixth order free surface correction

    displacement functions (A8) and (A18) of McTigue (1987)
    Hankel transforms from Tellez et al (1997).
    Tables of Fourier, Laplace and Hankel transforms of n-dimensional
    generalized function. Acta Applicandae Mathematicae, 48, 235-284
    """
    # equation (48)
    sigma = dot(tt ** 2.0, exp(-tt)) / (7 - dot(5, nu))
    # missing Hankel transform of second part of equation (49)
    tau = dot(tt ** 2.0, exp(-tt)) / (7 - dot(5, nu))
    A8 = multiply(multiply(multiply(dot(0.5, sigma),
                                    ((1 - dot(2, nu))
                                    - multiply(tt, zeta))),
                           exp(multiply(-tt, zeta))),
                  besselj(1, multiply(tt, rho)))
    A18 = multiply(multiply(multiply(dot(0.5, tau),
                                     (dot(2, (1 - nu))
                                     - multiply(tt, zeta))),
                            exp(multiply(- tt, zeta))),
                   besselj(1, multiply(tt, rho)))

    return A8 + A18


def mctigueA8A18(tt, nu, rho, zzn):
    """
    First free surface correction

    Displacement functions (A8) and (A18) of McTigue (1987)
    """
    sigma = multiply(dot(0.5, tt), exp(-tt))
    tau1 = sigma.copy()
    A8 = multiply(multiply(multiply(dot(0.5, sigma),
                                    ((1 - dot(2, nu)) - multiply(tt, zzn))),
                           exp(multiply(-tt, zzn))),
                  besselj(1, multiply(tt, rho)))
    A18 = multiply(multiply(multiply(dot(0.5, tau1),
                                     (dot(2, (1 - nu)) - multiply(tt, zzn))),
                            exp(multiply(-tt, zzn))),
                   besselj(1, multiply(tt, rho)))
    return A8 + A18


def mctigueA7A17a(tt, nu, rho, zeta):
    """
    Sixth order free surface correction

    displacement functions (A7) and (A17) of McTigue (1987)
    Hankel transforms from Tellez et al (1997). Tables of Fourier, Laplace
    and Hankel transforms of n-dimensional generalized function. Acta
    Applicandae Mathematicae, 48, 235-284
    """
    R = sqrt(rho**2 + zeta**2)
    # equation (48)
    sigma = multiply(dot(1.5, (tt + tt**2)),
                     exp(-tt)) / (7 - dot(5, nu))
    # missing Hankel transform of second part of equation (49)
    tau = dot(tt**2.0, exp(-tt)) / (7 - dot(5, nu))
    A7 = multiply(multiply(multiply(dot(0.5, sigma),
                                    (dot(2, (1 - nu)) - multiply(tt, zeta))),
                           exp(multiply(tt, zeta))),
                  besselj(0, multiply(tt, R)))
    A17 = multiply(multiply(multiply(dot(0.5, tau),
                                     ((1 - dot(2, nu)) - multiply(tt, zeta))),
                            exp(multiply(tt, zeta))),
                   besselj(0, multiply(tt, R)))
    return A7 + A17


def mctigueA7A17(tt, nu, rho, zeta):
    """
    First free surface correction

    Displacement functions (A7) and (A17) of McTigue (1987)
    """
    R = sqrt(rho**2 + zeta**2)
    sigma = multiply(dot(0.5, tt), exp(-tt))
    tau1 = sigma.copy()
    A7 = multiply(multiply(multiply(dot(0.5, sigma),
                                    (dot(2, (1 - nu)) - multiply(tt, zeta))),
                           exp(multiply(tt, zeta))),
                  besselj(0, multiply(tt, R)))
    A17 = multiply(multiply(multiply(dot(0.5, tau1),
                                     ((1 - dot(2, nu)) - multiply(tt, zeta))),
                            exp(multiply(tt, zeta))),
                   besselj(0, multiply(tt, R)))
    return A7 + A17
''',
"spheroid": r'''from dataclasses import dataclass, field

from numpy import (arctan as atan, array, cos, dot, log, multiply, pi,
                   radians, sin, sqrt)


@dataclass
class Spheroid:
    """
    Compute displacement due to a pressurized ellipsoid

    :param x0,y0,z0:  coordinates of the center of the prolate spheroid
                      (positive downward) [m]
    :param a:         semimajor axis [m]
    :param asrat:     geometric aspect ratio [dimensionless]
    :param P_G:       dimensionless excess pressure (pressure/shear modulus)
    :param mu:        shear modulus [Pa]
    :param nu:        Poisson's ratio [dimensionless]
    :param theta:     plunge (dip) angle [deg] [90 = vertical spheroid]
    :param phi:       trend (strike) angle [deg] [0 = aligned to North]

    Uses the finite prolate spheroid model  from Yang et al (JGR,1988)
    with corrections by Newman et al (JVGR, 2006).
    """
    # The equations by Yang et al (1988) and Newman et al (2006) are valid for
    # a vertical prolate spheroid only. There is an additional typo at p4251
    # in Yang et al (1988), not reported in Newman et al. (2006), that gives
    # an error when the spheroid is tilted (plunge different from 90 deg):
    #           C0 = y0*cos(theta) + z0*sin(theta)
    # The correct equation is
    #           C0 = z0/sin(theta)
    # This error has been corrected in this script.
    x0: float
    y0: float
    z0: float
    a: float
    asrat: float
    P_G: float
    mu: float
    nu: float
    theta: float
    phi: float
    # attributes affecting cached attributes
    _z0: float = field(init=False, repr=False)
    _a: float = field(init=False, repr=False)
    _asrat: float = field(init=False, repr=False)
    _P_G: float = field(init=False, repr=False)
    _mu: float = field(init=False, repr=False)
    _nu: float = field(init=False, repr=False)
    _theta: float = field(init=False, repr=False)
    _phi: float = field(init=False, repr=False)
    # Cached variables
    _dV: float = field(init=False, repr=False, default=None)
    _a1: array = field(init=False, repr=False)
    _b1: array = field(init=False, repr=False)
    _c: array = field(init=False, repr=False)
    _Pdila: array = field(init=False, repr=False)
    _Pstar: array = field(init=False, repr=False)

    @property
    def a(self) -> float:
        return self._a

    @a.setter
    def a(self, a: float):
        self._a = a
        self._dV = None

    @property
    def asrat(self) -> float:
        return self._asrat

    @asrat.setter
    def asrat(self, asrat: float):
        asrat == min(asrat, 0.99)   # avoid singularity
        self._asrat = asrat
        self._dV = None

    @property
    def mu(self) -> float:
        return self._mu

    @mu.setter
    def mu(self, mu: float):
        self._mu = mu
        self._dV = None

    @property
    def nu(self) -> float:
        return self._nu

    @nu.setter
    def nu(self, nu: float):
        self._nu = nu
        self._dV = None

    @property
    def P_G(self) -> float:
        return self._P_G

    @P_G.setter
    def P_G(self, P_G: float):
        self._P_G = P_G
        self._dV = None

    @property
    def theta(self) -> float:
        return self._theta

    @theta.setter
    def theta(self, theta: float):
        assert theta > 0
        theta = min(theta, 89.99)
        self._theta = theta
        self._dV = None

    @property
    def phi(self) -> float:
        return self._phi

    @phi.setter
    def phi(self, phi: float):
        self._phi = phi
        self._dV = None

    @property
    def z0(self) -> float:
        return self._z0

    @z0.setter
    def z0(self, z0: float):
        self._z0 = z0
        self._dV = None

    @property
    def dV(self) -> float:
        "Volume changeq"
        if self._dV is None:
            self._calc_base()
        return self._dV

    @property
    def P(self) -> float:
        "Pressure [Pa]"
        return self.P_G * self.mu

    @property
    def lam(self) -> float:
        "Lame parameter lambda [Pa]"
        return 2 * self.mu * self.nu / (1 - 2 * self.nu)

    @property
    def b(self) -> float:
        "semiminor axis [m]"
        return self._a * self._asrat

    def _calc_base(self):
        """
        Calculate base (cached) parameters

        dV is from Newman2006, eq2
        """
        self.yangpar()
        self._dV = self.P_G * pi * self.a * self.b**2

    def calc_displ(self, x, y, z):
        """
        Calculate displacement

        :param x,y:       benchmark location
        :param z:         depth within the crust (z=0 is the free surface)
        :kind x, y, z: 1d arrays
        :returns:   u, v, w:  E, N and Up deformation components
        """
        # testing parameters ######################################
        # clear all; close all; clc;
        # a = 1000; b = 0.99*a;
        # lambda = 1; mu = lambda; nu = 0.25; P = 0.01;
        # theta = pi*89.99/180; phi = 0;
        # x = linspace(0,2E4,7);
        # y = linspace(0,1E4,7);
        # x0 = 0; y0 = 0; z0 = 5E3;
        ############################################################

        # compute the parameters for the spheroid model
        if self._dV is None:
            self._calc_base()
        # translate the coordinates of the points where the displacement is
        # computed in the coordinate systen centered in (x0,0)
        # Coordinates transformation
        x = array(x, ndmin=1)
        y = array(y, ndmin=1)
        z = array(z, ndmin=1)

        xxn = x - self.x0
        yyn = y - self.y0
        phi_rad = radians(self.phi)
        # rotate the coordinate system to be coherent with the model coordinate
        # system of Figure 3 (Yang et al., 1988)
        xxp = dot(cos(phi_rad), xxn) - dot(sin(phi_rad), yyn)
        yyp = dot(sin(phi_rad), xxn) + dot(cos(phi_rad), yyn)
        # compute displacement for a prolate ellipsoid at csi = c
        U1p, U2p, U3p = self.yangint(xxp, yyp, z, self._c)
        # compute displacement for a prolate ellipsoid at csi = -c
        U1m, U2m, U3m = self.yangint(xxp, yyp, z, -self._c)
        Upx = -U1p - U1m
        Upy = -U2p - U2m
        Upz = U3p + U3m
        # rotate horizontal displacement back (strike)
        u = dot(cos(phi_rad), Upx) + dot(sin(phi_rad), Upy)
        v = dot(-sin(phi_rad), Upx) + dot(cos(phi_rad), Upy)
        w = Upz.copy()

        u = u.reshape(x.shape)
        v = v.reshape(x.shape)
        w = w.reshape(x.shape)
        return u, v, w

    def calc_tilt(self, x, y, z):
        """
        Calculate tilt parameters

        :param x, y, z:      same as in calc()
        :returns:   dwdx, dwdy      ground tilts (East, North)
        """
        h = 0.001 * abs(max(x) - min(x))

        # East component
        _, _, wp = self.calc_displ(x + h, y, z)
        _, _, wm = self.calc_displ(x - h, y, z)
        dwdx = 0.5 * (wp - wm) / h
        # North component
        _, _, wp = self.calc_displ(x, y + h, z)
        _, _, wm = self.calc_displ(x, y - h, z)
        dwdy = 0.5 * (wp - wm) / h

        return dwdx, dwdy

    def calc_strain(self, x, y, z):
        """
        Calculate strain parameters

        :param x, y, z:      same as in calc()
        :returns: eea, gamma1, gamma2 (areal, shear1 & shear2)
        """
        h = 0.001 * abs(max(x) - min(x))

        up, _, _ = self.calc_displ(x + h, y, z)
        um, _, _ = self.calc_displ(x - h, y, z)
        dudx = 0.5 * (up - um) / h
        up, _, _ = self.calc_displ(x, y + h, z)
        um, _, _ = self.calc_displ(x, y - h, z)
        dudy = 0.5 * (up - um) / h
        _, vp, _ = self.calc_displ(x + h, y, z)
        _, vm, _ = self.calc_displ(x - h, y, z)
        dvdx = 0.5 * (vp - vm) / h
        _, vp, _ = self.calc_displ(x, y + h, z)
        _, vm, _ = self.calc_displ(x, y - h, z)
        dvdy = 0.5 * (vp - vm) / h

        eea = dudx + dvdy
        gamma1 = dudx - dvdy
        gamma2 = dudy + dvdx

        return eea, gamma1, gamma2

    def calc_all(self, x, y, z):
        """
        Calculate all 3D Green's function parameters

        :param x, y, z:      same as in calc()
        :returns:   u         horizontal (East component) deformation
                    v         horizontal (North component) deformation
                    w         vertical (Up component) deformation
                    dwdx      ground tilt (East component)
                    dwdy      ground tilt (North component)
                    eea       areal strain
                    gamma1    shear strain
                    gamma2    shear strain
        """
        u, v, w = self.calc_displ(x, y, z)
        dwdx, dwdy = self.calc_tilt(x, y, z)
        eea, gamma1, gamma2 = self.calc_strain(x, y, z)

        return u, v, w, dwdx, dwdy, eea, gamma1, gamma2

    def yangint(self, x, y, z, csi):
        """
        Compute the primitive of the displacement for a prolate ellipsoid

        Equation (1)-(8) from Yang et al (JGR, 1988)
        Corrections to some parameters from Newmann et al (JVGR, 2006)
        :param x,y,z:    coordinates of the point(s) where the displacement
            is computed [m]
        :param csi:      focus of the prolate spheroid [m]
        :returns: U1,U2,U3 : displacement in local coordinates [m] - see
            Figure 3 of Yang et al (1988)

        Notes:
        The location of the center of the prolate spheroid is (x0,y0,z0)
            with x0=0 and y0=0;
        The free surface is z=0;
        """
        # precalculate parameters that are used often
        sint = sin(radians(self.theta))
        cost = cos(radians(self.theta))

        # new coordinates and parameters from Yang et al (JGR, 1988), p. 4251
        # dimensions [m]
        csi2 = dot(csi, cost)
        csi3 = dot(csi, sint)

        x1 = x.copy()
        x2 = y.copy()
        x3 = z - self.z0
        xbar3 = z + self.z0

        y1 = x1.copy()
        y2 = x2 - csi2
        y3 = x3 - csi3
        ybar3 = xbar3 + csi3
        r2 = dot(x2, sint) - dot(x3, cost)
        q2 = dot(x2, sint) + dot(xbar3, cost)
        r3 = dot(x2, cost) + dot(x3, sint)
        q3 = dot(-x2, cost) + dot(xbar3, sint)
        rbar3 = r3 - csi
        qbar3 = q3 + csi
        R1 = sqrt(y1**2 + y2**2 + y3**2)
        R2 = sqrt(y1**2 + y2**2 + ybar3**2)
        #####################################################################
        # C0 = y0*cost + z0*sint;
        C0 = self.z0 / sint

        #####################################################################
        beta = (dot(q2, cost)
                + dot((1 + sint), (R2 + qbar3))) / (dot(cost, y1) + 1e-15)

        # precalculate parameters that are used often
        drbar3 = R1 + rbar3
        dqbar3 = R2 + qbar3
        dybar3 = R2 + ybar3
        # print(f'{R1=}, {rbar3=}')
        lrbar3 = log(R1 + rbar3)
        lqbar3 = log(R2 + qbar3)
        lybar3 = log(R2 + ybar3)
        atanb = atan(beta)
        # primitive parameters from Yang et al (1988), p. 4252
        Astar1 = (self._a1 / (multiply(R1, drbar3))
                  + multiply(self._b1, (lrbar3 + (r3 + csi) / drbar3)))
        Astarbar1 = (-self._a1 / (multiply(R2, dqbar3))
                     - multiply(self._b1, (lqbar3 + (q3 - csi) / dqbar3)))
        A1 = csi / R1 + lrbar3
        Abar1 = csi / R2 - lqbar3
        A2 = R1 - multiply(r3, lrbar3)
        Abar2 = R2 - multiply(q3, lqbar3)
        A3 = multiply(csi, rbar3) / R1 + R1
        Abar3 = multiply(csi, qbar3) / R2 - R2
        Bstar = ((self._a1 / R1 + multiply(2*self._b1, A2))
                 + dot((3 - 4*self.nu),
                       (self._a1 / R2 + multiply(2*self._b1, Abar2))))
        B = dot(csi, (csi + C0)) / R2 - Abar2 - dot(C0, lqbar3)
        # the 4 equations below have been changed to improve the fit to
        # internal deformation
        Fstar1 = 0
        Fstar2 = 0
        F1 = 0
        F2 = 0
        f1 = (dot(csi, y1) / dybar3
              + dot((3 / cost**2), (multiply(multiply(y1, sint), lybar3)
                                    - multiply(y1, lqbar3)
                                    + multiply(dot(2, q2), atanb)))
              + multiply(2*y1, lqbar3)
              - multiply(4*xbar3, atanb) / cost)
        f2 = (dot(csi, y2) / dybar3
              + dot((3 / cost**2), (multiply(multiply(q2, sint), lqbar3)
                                    - multiply(q2, lybar3)
                                    + multiply(dot(dot(2, y1), sint), atanb)
                                    + dot(cost, (R2 - ybar3))))
              - dot(2*cost, Abar2)
              + dot((2 / cost), (multiply(xbar3, lybar3)
                                 - multiply(q3, lqbar3))))
        f3 = (dot((1 / cost), (multiply(q2, lqbar3)
                               - multiply(multiply(q2, sint), lybar3)
                               + multiply(dot(2, y1), atanb)))
              + dot(2*sint, Abar2) + multiply(q3, lybar3) - csi)
        # precalculate coefficients that are used often
        cstar = (dot(self.a, self.b**2) / csi**3) / (dot(16*self.mu,
                                                     (1 - self.nu)))
        cdila = dot(dot(2, cstar), self._Pdila)
        # displacement components (2) to (7): primitive of eq(1) Yang1988
        Ustar1 = dot(cstar, (multiply(Astar1, y1)
                             + multiply(dot((3 - 4*self.nu), Astarbar1), y1)
                             + multiply(Fstar1, y1)))

        # U2star and U3star changed to improve fit to internal deformation
        Ustar2 = dot(cstar,
                     (dot(sint, (multiply(Astar1, r2)
                                 + multiply(dot((3 - 4*self.nu), Astarbar1),
                                            q2)
                                 + multiply(Fstar1, q2)))
                      + dot(cost, (Bstar - Fstar2))))

        # The formula used in the script by Fialko and Andy is different from
        # equation (4) of Yang et al (1988)
        # I use the same to continue to compare the results 2009 07 23
        # Ustar3 = cstar*(-cost*(Astarbar1.*r2
        #                        + (3-4*nu)*Astarbar1.*q2 - Fstar1.*q2)
        #                 + sint*(Bstar+Fstar2) + 2*cost^2*z.*Astarbar1)
        #####################################################################
        # The equation below is correct - follows eq4 from Yang et al (1988)
        Ustar3 = dot(cstar,
                     (dot(-cost,
                          (multiply(Astar1, r2)
                           + multiply(dot((3 - 4*self.nu), Astarbar1), q2)
                           - multiply(Fstar1, q2)))
                      + dot(sint, (Bstar + Fstar2))))

        Udila1 = dot(cdila,
                     ((multiply(A1, y1)
                       + multiply(dot((3 - 4*self.nu), Abar1), y1)
                       + multiply(F1, y1))
                      - dot(dot(4*(1-self.nu), (1-2*self.nu)), f1)))
        Udila2 = dot(cdila,
                     (dot(sint, (multiply(A1, r2)
                                 + multiply(dot((3 - 4*self.nu), Abar1), q2)
                                 + multiply(F1, q2)))
                      - dot(4*(1 - self.nu)*(1 - 2*self.nu), f2)
                      + dot(dot(4*(1 - self.nu), cost), (A2 + Abar2))
                      + dot(cost, (A3 - dot((3 - 4*self.nu), Abar3) - F2))))
        Udila3 = dot(cdila,
                     (dot(cost, (multiply(-A1, r2)
                                 + multiply(dot((3 - 4*self.nu), Abar1), q2)
                                 + multiply(F1, q2)))
                      + dot(4*(1 - self.nu)*(1 - 2*self.nu), f3)
                      + dot(dot(4*(1 - self.nu), sint), (A2 + Abar2))
                      + dot(sint, (A3 + dot((3 - 4*self.nu), Abar3)
                                   + F2 - dot(2*(3 - 4*self.nu), B)))))

        # displacement: equation (8) from Yang et al (1988) - see Figure 3
        U1 = Ustar1 + Udila1
        U2 = Ustar2 + Udila2
        U3 = Ustar3 + Udila3

        return U1, U2, U3

    def yangpar(self):
        """
        Compute the parameters for the spheroid model

        Formulas from [1] Yang et al (JGR,1988)
        Corrections from [2] Newmann et al (JVGR, 2006), Appendix

         self._a1, self._b1    pressure (stress) [units of P] from [1]
         self._c         prolate ellipsoid focus [m]
         self._Pdila     pressure (proportional to double couple forces)
                            [units of P] from [1]
         self._Pstar     pressure [units of P]

        Notes:
        [-]   : dimensionless
        """
        # epsn = 1e-10
        self._c = sqrt(self.a**2 - self.b**2)

        a2 = self.a**2
        a3 = self.a**3
        b2 = self.b**2
        c2 = self._c**2
        c3 = self._c**3
        c4 = self._c**4
        c5 = self._c**5
        ac = (self.a - self._c) / (self.a + self._c)

        coef1 = dot(dot(dot(2, pi), self.a), b2)
        den1 = dot(dot(8, pi), (1 - self.nu))

        Q = 3 / den1
        R = (1 - dot(2, self.nu)) / den1

        Ia = dot(-coef1, (2 / (dot(self.a, c2)) + log(ac) / c3))
        Iaa = dot(-coef1, (2 / (3*a3*c2) + 2 / (self.a*c4) + log(ac) / c5))

        a11 = dot(dot(2, R), (Ia - dot(4, pi)))
        a12 = dot(dot(-2, R), (Ia + dot(4, pi)))
        a21 = dot(dot(Q, a2), Iaa) + dot(R, Ia) - 1
        a22 = dot(dot(-Q, a2), Iaa) - dot(Ia, (dot(2, R) - Q))

        den2 = dot(3, self.lam) + dot(2, self.mu)
        num2 = dot(3, a22) - a12
        den3 = dot(a11, a22) - dot(a12, a21)
        num3 = a11 - dot(3, a21)

        self._Pdila = dot(dot(self.P, (dot(2, self.mu) / den2)),
                          (num2 - num3)) / den3
        self._Pstar = dot(dot(self.P, (1 / den2)),
                          (dot(num2, self.lam)
                           + dot(2*(self.lam + self.mu), num3))) / den3

        self._a1 = dot(dot(-2, b2), self._Pdila)
        self._b1 = dot(dot(3, (b2 / c2)), self._Pdila)\
            + dot(dot(2, (1 - dot(2, self.nu))), self._Pstar)

        # return a1, b1, c, Pdila, Pstar
''',
}
_DM_SOURCES_BLANKS = {'mogi': [(16, '    ')], 'sill': [(48, '    '), (52, '    '), (57, '    '), (61, '    '), (66, '    '), (70, '    '), (79, '    '), (84, '    '), (91, '    '), (108, '        ')], 'sphere3D': [(84, '        '), (96, '        ')]}   # blanks of whitespace-only lines, restored at load time

_DM_SHA256 = {'__init__': '80d84c744a60118b320cadb9796739d51940a818959906ff97b9a2dfcdef50a0', 'version': '332ba97e9d7c11565a704e13a7ba98a98a62984f1358926bfb3423ac1c082620', 'mogi': 'facff3583ae95be3a9e94afe5dcef58aa8c5274c8c80eebd747cd5bf7ac25b45', 'sill': '41c85638f1a2b9127b6a7653813c86142990dda8cf877e64d216035d09e62c3e', 'sphere2D': '551155818cef8cf4f81b1a8dd0b2ca61315a4144258321da4f209e77d79c5bc4', 'sphere3D': '6c693488bbfe7b6514841f128ad9d301eee6cab4a095c29629df01b326611904', 'spheroid': '55cfe54314a6557c07d0a9dd016a7c1a62a15134ace72f62f9f13bb42eb287f6'}


_DM_SOURCES = restore_blank_lines(_DM_SOURCES, _DM_SOURCES_BLANKS)
dmodels_local = install_memory_package("dmodels_local", _DM_SOURCES)
_dm_audit = [(n, s.count("\n"), hashlib.sha256(s.encode("utf-8")).hexdigest() == _DM_SHA256[n])
             for n, s in _DM_SOURCES.items()]
section("dMODELS deformation sources ready (package <code>dmodels_local</code>)")
table(["File", "Lines", "Identical to the original dmodelspy 0.2 file?", "Used for"],
      [[f"<code>{n}.py</code>", f"{l:,}", "yes (SHA-256 match)" if ok else "NO",
        {"sill": "Fialko et al. (2001) penny-shaped crack", "spheroid": "Yang et al. (1988) / Newman et al. (2006) spheroid",
         "mogi": "(Mogi point source — the notebook uses the closed-form expression, Eq. 3.23)"}.get(n, "package infrastructure")]
       for n, l, ok in _dm_audit],
      classes=lambda i, j: ("good" if _dm_audit[i][2] else "bad") if j == 2 else None, left_cols=(0, 3),
      caption=f"{DM_ORIGIN['authors']}; licence {DM_ORIGIN['licence']}. Two numpy-2 compatibility fixes of the crack "
              "solution (Gauss–Legendre nodes and the scalar extraction in the displacement sum) are applied in the "
              "calculation engine (cell 9), together with a finer quadrature of the crack displacement integral (identical within one source depth; removes the far-field aliasing of small, deep cracks).")

In [ ]:
#@title 4. Volcano and project settings  { display-mode: "form", run: "auto" }
#@markdown **Start values of the tables in cells 5, 6 and 8** — a ready-made volcano compiled from published data (the source of every value is listed under cell 10 and in the accompanying PDF), empty tables for your own volcano, or a copy you saved inside cells 5, 6 and 8. Changing the choice refills the tables; the *Undo* button of a table brings its previous values back.
START_FROM = "La Fossa, Vulcano (Italy) — thesis"  #@param ["La Fossa, Vulcano (Italy) — thesis", "Etna (Italy)", "Campi Flegrei (Italy)", "Kīlauea (Hawaiʻi)", "Empty tables (your own volcano)", "Saved copy (pasted into cells 5, 6 and 8)"]
#@markdown **Project name** shown on the results (empty = the name of the chosen volcano)
PROJECT_NAME = ""  #@param {type:"string"}

#@markdown **Crust** — lithostatic pressure P = ρ g z (a reservoir may give its own crustal density, depth for P, or pressure)
rho_crust = 2500          #@param {type:"number"}
g = 9.81                  #@param {type:"number"}

#@markdown **Default host-rock elasticity** — used where the reservoir table leaves μ or ν empty
default_mu_GPa = 10.0     #@param {type:"number"}
default_nu = 0.25         #@param {type:"number"}

#@markdown **Observation points** — distances from the source axis (km) at which the uplift is listed, separated by commas (empty = those of the chosen volcano)
obs_distances_km = ""  #@param {type:"string"}
#@markdown Azimuth of the profiles (° from North; matters only for dipping spheroids) and their length (km; 0 = automatic)
profile_azimuth = 0       #@param {type:"number"}
profile_max_km = 0        #@param {type:"number"}

#@markdown **Engine for step 2** (gas fraction, saturation pressure, speciation, fugacities)
step2_engine = "EVo (cell 2)"  #@param ["EVo (cell 2)", "analytical fallback"]

# ----------------------------------------------------------------------------- ready-made volcanoes
EXAMPLES = {
    'La Fossa, Vulcano (Italy) — thesis': {'project': 'La Fossa, Vulcano (Italy) — thesis, Scenario A', 'obs_km': [0.0, 1.0, 2.0], 'summary': 'The worked example of the thesis: rhyolite, trachyte, latite and shoshonite of the 1888–90 and older eruptions; storage levels at 2, 5 and 12 km tested as Mogi spheres and penny cracks, and the 2021 unrest source of Di Traglia et al. (2023).', 'frame': 'Depths below the free surface of the half-space (sea level); the 2021 source at 598 m b.s.l.', 'MAGMAS': 'property      | RHYOLITE   | TRACHYTE   | LATITE     | SHOSHONITE   # names are free\n# ── identity and physical properties ──────────────────────────────────────────────────────\nevo_class     | rhyolite   | phonolite  | phonolite  | basalt       # EVo solubility fit: basalt | phonolite | rhyolite\nT_C           | 1000       | 1075       | 1080       | 1100         # magma temperature, °C\nbeta_liquid   | 1.2e-10    | 1.1e-10    | 1.0e-10    | 0.8e-10      # melt compressibility, 1/Pa\nrho_melt      | 2300       | 2400       | 2450       | 2550         # melt density used when no vapour exists, kg/m3\n# ── how EVo describes the volatiles ───────────────────────────────────────────────────────\nevo_mode      | saturation | saturation | saturation | saturation   # saturation | atomic | gas_at_P\ngas_system    | auto       | auto       | auto       | auto         # auto | OH | COH | SOH | COHS | COHSN\n# ── volatile contents, wt%  (saturation: total budget;  gas_at_P: dissolved at the start pressure)\nH2O_wt        | 1.25       | 1.25       | 1.35       | 0.85\nCO2_wt        | 0.005      | 0.005      | 0.005      | 0.022\nS_wt          | 0.02       | 0.07       | 0.10       | 0.03\nN_wt          |            |            |            |              # nitrogen (needs gas_system COHSN)\ngraphite_wt   |            |            |            |              # graphite-saturated melt (instead of CO2_wt)\n# ── atomic mode: total element masses of the system, ppm ──────────────────────────────────\natomic_H_ppm  |            |            |            |\natomic_C_ppm  |            |            |            |\natomic_S_ppm  |            |            |            |\natomic_N_ppm  |            |            |            |\n# ── gas_at_P mode: vapour mass fraction (wt%) at the start pressure (MPa; empty = reservoir P)\ngas_wt        |            |            |            |\nP_start_MPa   |            |            |            |\n# ── redox: FMQ | NNO | IW (value = log units above the buffer), absolute (value = fO2 in bar),\n#           iron (fO2 from the FeO and Fe2O3 rows below; value ignored)\nfO2_buffer    | FMQ        | FMQ        | FMQ        | FMQ\nfO2_value     | 1.0        | 1.0        | 1.0        | 1.0\n# ── initial gas fugacities, bar (gas_at_P mode only) ──────────────────────────────────────\nfH2_bar       |            |            |            |\nfH2O_bar      |            |            |            |\nfCO2_bar      |            |            |            |\n# ── major elements, wt% volatile-free (Gioncada et al. 1998, Table 1; all Fe as FeO) ──────\nSIO2          | 73.54      | 61.60      | 57.45      | 53.79\nTIO2          | 0.13       | 0.61       | 0.61       | 0.70\nAL2O3         | 13.10      | 17.51      | 16.83      | 15.08\nFEO           | 2.21       | 4.30       | 6.60       | 8.16\nFE2O3         |            |            |            |              # only with fO2_buffer = iron\nMNO           | 0.07       | 0.10       | 0.14       | 0.16\nMGO           | 0.34       | 1.25       | 2.58       | 4.70\nCAO           | 1.02       | 2.25       | 5.17       | 7.67\nNA2O          | 4.35       | 4.42       | 3.81       | 3.55\nK2O           | 4.95       | 7.25       | 5.67       | 4.89\nP2O5          | 0.04       | 0.22       | 0.43       | 0.39\nNIO           |            |            |            |\nCR2O3         |            |            |            |\n# ── where the values come from (free text, shown on the page) ─────────────────────────────\nsource        | GS91-50c, 1888–90 eruption; Clocchiatti et al. 1994 | Palizzi pumices; Clocchiatti et al. 1994 | GS91-17 Pietre Cotte; Gioncada et al. 1998 | GS91-66 Vulcanello I; Fusillo et al. 2015', 'RESERVOIRS': 'name                     | shape    | depth_km | V0_m3  | Ve_m3           | resident | injected   | mu_GPa | nu   | radius_km | aspect | dip_deg | strike_deg | P_depth_km\nShallow · Mogi           | mogi     | 2        | 5e8    | 1e6; 5e6; 1e7   | RHYOLITE | TRACHYTE   | 10     | 0.25 |           |        |         |            |\nIntermediate · Mogi      | mogi     | 5        | 5e9    | 1e6; 5e6; 1e7   | TRACHYTE | LATITE     | 10     | 0.25 |           |        |         |            |\nDeep · Mogi              | mogi     | 12       | 5e10   | 1e6; 5e6; 1e7   | LATITE   | SHOSHONITE | 10     | 0.25 |           |        |         |            |\n2021 source · Yang       | spheroid | 0.598    | 8.83e6 | 1e5; 5e5; 1e6   | RHYOLITE | TRACHYTE   | 1      | 0.35 |           | 0.099  | 69      | 137        | 0.598\nShallow · penny 0.5 km   | penny    | 2        | 5e8    | 1e6; 5e6; 1e7   | RHYOLITE | TRACHYTE   | 10     | 0.25 | 0.5       |        |         |            |\nIntermediate · penny 0.5 | penny    | 5        | 5e9    | 1e6; 5e6; 1e7   | TRACHYTE | LATITE     | 10     | 0.25 | 0.5       |        |         |            |\nDeep · penny 0.5 km      | penny    | 12       | 5e10   | 1e6; 5e6; 1e7   | LATITE   | SHOSHONITE | 10     | 0.25 | 0.5       |        |         |            |\nShallow · penny 1 km     | penny    | 2        | 5e8    | 1e6; 5e6; 1e7   | RHYOLITE | TRACHYTE   | 10     | 0.25 | 1         |        |         |            |\nIntermediate · penny 1   | penny    | 5        | 5e9    | 1e6; 5e6; 1e7   | TRACHYTE | LATITE     | 10     | 0.25 | 1         |        |         |            |\nDeep · penny 1 km        | penny    | 12       | 5e10   | 1e6; 5e6; 1e7   | LATITE   | SHOSHONITE | 10     | 0.25 | 1         |        |         |            |\nShallow · penny 2 km     | penny    | 2        | 5e8    | 1e6; 5e6; 1e7   | RHYOLITE | TRACHYTE   | 10     | 0.25 | 2         |        |         |            |\nIntermediate · penny 2   | penny    | 5        | 5e9    | 1e6; 5e6; 1e7   | TRACHYTE | LATITE     | 10     | 0.25 | 2         |        |         |            |\nDeep · penny 2 km        | penny    | 12       | 5e10   | 1e6; 5e6; 1e7   | LATITE   | SHOSHONITE | 10     | 0.25 | 2         |        |         |            |', 'sources': None},
    'Etna (Italy)': {'project': 'Mount Etna (Italy) — preset from published data', 'obs_km': [0.0, 2.0, 5.0, 10.0], 'summary': 'Alkali basalt (trachybasalt) system of the 2002–03 flank eruption and the 2009–2021 recharge and lava-fountain cycles. Three magmas from the 2002 melt-inclusion record: the primitive volatile-rich basalt rising from ≥10 km b.s.l., the more evolved magma ponding at 4–6 km b.s.l., and the degassed shallow melt. Four sources: the GPS recharge and discharge centroids (McTigue spheres), the sill-like main chamber from fracture statistics, and the shallow lava-fountain spheroid.', 'frame': "The geodetic sources are published in km below sea level. For the flat half-space the free surface is placed 1.6 km above sea level (the free-surface elevation adopted for half-space event locations at Etna, Bonaccorso et al. 2021; an approximation of the mean elevation of the upper flanks), so depth_km = depth b.s.l. + 1.6 km. Pressures use P_depth_km = depth b.s.l. + 2.6 km with ρ = 2600 kg m⁻³, which reproduces Spilliaert et al. (2006)'s conversion of 170–220 MPa to 4–6 km b.s.l. (it accounts for the edifice above the sources).", 'MAGMAS': 'property      | ETNA_PRIMITIVE  | ETNA_PONDING    | ETNA_DEGASSED  \nevo_class     | basalt          | basalt          | basalt         \nT_C           | 1140            | 1125            | 1100           \nbeta_liquid   | 1.25e-10        | 1.25e-10        | 1.25e-10       \nrho_melt      | 2650            | 2650            | 2650           \nH2O_wt        | 3.4             | 2.5             | 1              \nCO2_wt        | 0.31            | 0.12            | 0.025          \nS_wt          | 0.32            | 0.31            | 0.13           \nfO2_buffer    | NNO             | NNO             | NNO            \nfO2_value     | 0               | 0               | -0.5           \nSIO2          | 47.15           | 48.16           | 50.04          \nTIO2          | 1.62            | 1.62            | 2.06           \nAL2O3         | 15.65           | 18.23           | 16.22          \nFEO           | 10.46           | 9.63            | 10.48          \nMNO           | 0.18            | 0.17            | 0.23           \nMGO           | 7.53            | 4.65            | 3.23           \nCAO           | 11.73           | 9.92            | 7.43           \nNA2O          | 2.97            | 4.03            | 4.41           \nK2O           | 1.75            | 2.19            | 3.5            \nP2O5          | 0.44            | 0.55            | 1.04           \nsource        | Primitive 2002 basalt (lapilli, 21 Nov 2002) | Evolved 2002 magma ponding at 4–6 km b.s.l. | Degassed shallow melt (2002 matrix glass)', 'RESERVOIRS': 'name                                 | shape    | depth_km | V0_m3   | Ve_m3           | resident       | injected       | mu_GPa | nu   | radius_km | aspect | dip_deg | strike_deg | P_depth_km | rho_crust\nRecharge source · McTigue            | sphere   | 8.1      | 4.19e9  | 5e6; 1.5e7; 3e7 | ETNA_PRIMITIVE | ETNA_PRIMITIVE | 20     | 0.25 | 1         |        |         |            | 9.1        | 2600\nDischarge / ponding source · McTigue | sphere   | 6.1      | 4.19e9  | 1e6; 5e6; 1e7   | ETNA_PONDING   | ETNA_PRIMITIVE | 20     | 0.25 | 1         |        |         |            | 7.1        | 2600\nMain chamber · sill                  | penny    | 6.1      | 1.03e11 | 1e7; 3e7; 1e8   | ETNA_PONDING   | ETNA_PRIMITIVE | 20     | 0.25 | 5.7       |        |         |            | 7.1        | 2600\nLava-fountain source · gas-rich      | spheroid | 2.35     | 3e8     | 1e6; 2e6; 5e6   | ETNA_PONDING   | ETNA_PRIMITIVE | 20     | 0.25 |           | 0.5    | 90      | 0          | 3.35       | 2600\nLava-fountain source · degassed      | spheroid | 2.35     | 3e8     | 1e6; 2e6; 5e6   | ETNA_DEGASSED  | ETNA_PRIMITIVE | 20     | 0.25 |           | 0.5    | 90      | 0          | 3.35       | 2600', 'sources': {'magmas': {'ETNA_PRIMITIVE': {'label': 'Primitive 2002 basalt (lapilli, 21 Nov 2002)', 'rows': [('major elements', 'Spilliaert2006', 'Table 1, 21 Nov 2002 lapilli (south vents); Fe2O3(t) 11.62 → FeO(t) = 0.8998 × Fe2O3'), ('T_C', 'Spilliaert2006', 'degassing model temperature 1140 °C (MELTS liquidus 1160 → 1125 °C)'), ('H2O_wt', 'Spilliaert2006', 'primitive inclusions 3.4 ± 0.2 wt% (max 3.6)'), ('CO2_wt', 'Spilliaert2006', 'maximum measured 3145 ppm; a minimum for the total budget'), ('S_wt', 'Spilliaert2006', 'primitive inclusions 0.27–0.35 wt% (max 0.347)'), ('fO2', 'Spilliaert2006', '≈ NNO (after Métrich & Clocchiatti 1996); Gennaro et al. (2020): NNO−0.9 to +0.4 for 2002–2013 inclusions'), ('beta_liquid', 'Scudero2019', 'βm ≈ 1.25 × 10⁻¹⁰ Pa⁻¹ for Etnean basaltic magma'), ('rho_melt', 'Spera2000', 'typical basaltic melt density (used only if undersaturated)')]}, 'ETNA_PONDING': {'label': 'Evolved 2002 magma ponding at 4–6 km b.s.l.', 'rows': [('major elements', 'Spilliaert2006', 'Table 1, 4 Nov 2002 lava (north vents), the most evolved 2002 bulk rock'), ('T_C', 'Spilliaert2006', 'lower end of the MELTS crystallisation interval (1125 °C)'), ('H2O_wt', 'Spilliaert2006', 'ponding-cluster inclusions at 170–220 MPa: 2.2–2.8 wt%'), ('CO2_wt', 'Spilliaert2006', 'ponding-cluster inclusions: 1000–1340 ppm'), ('S_wt', 'Spilliaert2006', 'ponding-cluster inclusions: 0.30–0.32 wt%'), ('fO2', 'Spilliaert2006', '≈ NNO, as for the primitive magma'), ('beta_liquid', 'Scudero2019', 'as above'), ('rho_melt', 'Spera2000', 'as above')]}, 'ETNA_DEGASSED': {'label': 'Degassed shallow melt (2002 matrix glass)', 'rows': [('major elements', 'Spilliaert2006', 'Table 2, matrix glass ET02-5 (29 Oct 2002), Fe as FeO'), ('T_C', '—', 'assumed, below the 1125 °C end of the MELTS interval'), ('H2O_wt', 'Spilliaert2006', 'evolved glass embayments trapped at 25–50 MPa: 0.3–1.3 wt%'), ('CO2_wt', 'Spilliaert2006', 'embayments: 140–270 ppm'), ('S_wt', 'Spilliaert2006', 'embayments: 0.11–0.14 wt%'), ('fO2', 'Gennaro2020', '2008 and 2013 inclusions NNO−0.9 to −0.2'), ('beta_liquid', 'Scudero2019', 'as above'), ('rho_melt', 'Spera2000', 'as above')]}}, 'reservoirs': {'Recharge source · McTigue': [('Aloisi2018', 'McTigue finite sphere; inflation centroid ≈ 6.5 km b.s.l. (phases 6.1–7.5 km); ΔV = 3×10⁶–2×10⁷ m³'), ('—', 'radius 1 km assumed (not reported); V0 = 4/3·π·(1 km)³'), ('Scudero2019', 'E = 50 GPa, ν = 0.25 → μ = 20 GPa (Aloisi et al. 2018 used μ = 30 GPa, called an overestimate)')], 'Discharge / ponding source · McTigue': [('Aloisi2018', 'deflation centroid ≈ 4.5 km b.s.l.; ΔV = −1.6×10⁶ to −8.2×10⁶ m³'), ('Spilliaert2006', 'ponding of the 2002 magma at 4–6 km b.s.l. (170–220 MPa)')], 'Main chamber · sill': [('Scudero2019', 'sill-like chamber 1 km thick, radius 5.7 km, most likely volume 103 km³ (range 69–206 km³), roof at 3–6 km b.s.l.; centre placed at 4.5 km b.s.l.')], 'Lava-fountain source · gas-rich': [('Bonaccorso2021', 'vertical prolate spheroid, aspect 0.5, 0.5–1 km b.s.l. (centre at 0.75 km), ΔV ≈ 2×10⁶ m³ per fountain, mean wall displacement ≈ 0.8 m'), ('Bonaccorso2013', '2011–2012 fountains: elongated source near sea level, ΔV ≈ 2×10⁶ m³'), ('—', 'V0 = 3×10⁸ m³ derived so that ΔV/(wall area) ≈ 0.8 m for this shape')], 'Lava-fountain source · degassed': [('—', 'same geometry, filled with the degassed melt: shows how much the gas content of the resident magma matters')]}, 'refs': {'Aloisi2018': 'Aloisi, M., Bonaccorso, A., Cannavò, F. & Currenti, G. (2018). Coupled short- and medium-term geophysical signals at Etna volcano: using deformation and strain to infer magmatic processes from 2009 to 2017. Front. Earth Sci. 6:109. doi:10.3389/feart.2018.00109', 'Bonaccorso2013': 'Bonaccorso, A., Currenti, G., Linde, A. & Sacks, S. (2013). New data from borehole strainmeters to infer lava fountain sources (Etna 2011–2012). Geophys. Res. Lett. 40(14). doi:10.1002/grl.50692', 'Bonaccorso2021': 'Bonaccorso, A., Carleo, L., Currenti, G. & Sicali, A. (2021). Magma migration at shallower levels and lava fountains sequence as revealed by borehole dilatometers on Etna volcano. Front. Earth Sci. 9:740505. doi:10.3389/feart.2021.740505', 'Gennaro2020': 'Gennaro, E., Paonita, A., Iacono-Marziano, G., Moussallam, Y., Pichavant, M., Peters, N. & Martel, C. (2020). Sulphur behaviour and redox conditions in Etnean magmas during magma differentiation and degassing. J. Petrol. 61(10), egaa095. doi:10.1093/petrology/egaa095', 'Scudero2019': 'Scudero, S., De Guidi, G. & Gudmundsson, A. (2019). Size distributions of fractures, dykes, and eruptions on Etna, Italy: implications for magma-chamber volume and eruption potential. Sci. Rep. 9, 4139. doi:10.1038/s41598-019-40563-0', 'Spera2000': 'Spera, F.J. (2000). Physical properties of magma. In: Sigurdsson, H. (ed.) Encyclopedia of Volcanoes, Academic Press, 171–190', 'Spilliaert2006': 'Spilliaert, N., Allard, P., Métrich, N. & Sobolev, A.V. (2006). Melt inclusion record of the conditions of ascent, degassing, and extrusion of volatile-rich alkali basalt during the powerful 2002 flank eruption of Mount Etna (Italy). J. Geophys. Res. 111, B04203. doi:10.1029/2005JB003934'}}},
    'Campi Flegrei (Italy)': {'project': 'Campi Flegrei caldera (Italy) — preset from published data', 'obs_km': [0.0, 1.0, 2.0, 5.0], 'summary': 'Potassic caldera system: a trachytic (Agnano–Monte Spina-type) magma in the shallow storage zone and a trachybasaltic (shoshonitic) parental magma in the deep sill-like reservoir. Three sources: the 1982–84 unrest spheroid, the 2018–2023 shallow source and the caldera-wide deep sill.', 'frame': 'Depths below the ground surface as published (Pozzuoli area, near sea level). Pressures ρgz with ρ = 2650 kg m⁻³ (the upper-crust density used by Arienzo et al. 2026 to convert pressures to depths).', 'MAGMAS': 'property      | CF_TRACHYTE      | CF_TRACHYBASALT \nevo_class     | phonolite        | basalt          \nT_C           | 950              | 1160            \nbeta_liquid   | 1.1e-10          | 8e-11           \nrho_melt      | 2400             | 2550            \nH2O_wt        | 3                | 2.8             \nCO2_wt        | 0.05             | 0.11            \nS_wt          | 0.02             | 0.1             \nfO2_buffer    | FMQ              | FMQ             \nfO2_value     | 1                | 1               \nSIO2          | 61.67            | 47.96           \nTIO2          | 0.49             | 0.78            \nAL2O3         | 18.63            | 14.61           \nFEO           | 3.38             | 7.73            \nMNO           | 0.01             | 0.08            \nMGO           | 0.67             | 9.37            \nCAO           | 2.48             | 8.94            \nNA2O          | 4.13             | 1.57            \nK2O           | 8.54             | 3.92            \nP2O5          |                  | 0.56            \nsource        | Trachyte, Agnano–Monte Spina type (4.1 ka) | Trachybasalt (shoshonitic parental melt, Minopoli 1)', 'RESERVOIRS': 'name                          | shape    | depth_km | V0_m3   | Ve_m3           | resident        | injected        | mu_GPa | nu   | radius_km | aspect | dip_deg | strike_deg | P_depth_km | rho_crust\n1982–84 source · spheroid     | spheroid | 2.9      | 1.16e10 | 1e7; 7.1e7; 1e8 | CF_TRACHYTE     | CF_TRACHYBASALT | 10     | 0.25 |           | 0.51   | 90      | 0          |            | 2650\n2018–23 shallow source · sill | penny    | 3.9      | 1.6e8   | 1e6; 1e7; 6e7   | CF_TRACHYTE     | CF_TRACHYBASALT | 10     | 0.25 | 2         |        |         |            |            | 2650\nDeep sill-like reservoir      | penny    | 8        | 1e11    | 1e7; 1e8; 1e9   | CF_TRACHYBASALT | CF_TRACHYBASALT | 10     | 0.25 | 5.64      |        |         |            |            | 2650', 'sources': {'magmas': {'CF_TRACHYTE': {'label': 'Trachyte, Agnano–Monte Spina type (4.1 ka)', 'rows': [('major elements', 'Pappalardo2012', 'Table 2, Agnano–Monte Spina unit A matrix glass (P2O5 below detection)'), ('T_C', 'Arienzo2026', '950 °C adopted for the trachytic melt inclusions'), ('H2O_wt', 'Petrelli2023', 'AMS melt inclusions 0.8–3.0 wt% (upper end used); Arienzo et al. (2026): 0.5–2.9 wt% for all CF inclusions'), ('CO2_wt', 'Petrelli2023', 'AMS inclusions 150–500 ppm (upper end used); Arienzo et al. (2026) report up to 2666 ppm in AMS-A inclusions'), ('S_wt', 'Arienzo2026', 'S 200–801 ppm, lowest in AMS'), ('fO2', 'Cannatelli2012', 'QFM to QFM+1 for Campi Flegrei magmas; assumed FMQ+1 (no direct measurement for the trachyte)'), ('beta_liquid', 'Spera2000', 'evolved alkaline melt (thesis value for trachyte)'), ('rho_melt', 'Spera2000', 'trachytic melt density (used only if undersaturated)')]}, 'CF_TRACHYBASALT': {'label': 'Trachybasalt (shoshonitic parental melt, Minopoli 1)', 'rows': [('major elements', 'Cannatelli2012', 'parental melt inclusion Mi1-C1 o6 M1 (Minopoli 1)'), ('T_C', 'Cannatelli2012', 'clinopyroxene saturation of the Minopoli 1 parental melt (liquidus 1300 °C)'), ('H2O_wt', 'Petrelli2023', 'Minopoli 2 / Fondo Riccio inclusions 0.2–2.84 wt% (upper end); Cannatelli et al.: 1–4 wt%'), ('CO2_wt', 'Petrelli2023', 'Minopoli 2 / Fondo Riccio inclusions 172–1100 ppm (upper end)'), ('S_wt', 'Cannatelli2012', 'parental melt SO2 0.21 wt% → S 0.105 wt%'), ('fO2', 'Cannatelli2012', 'QFM+1 for Minopoli 1'), ('beta_liquid', 'Spera2000', 'mafic melt (thesis value for shoshonite)'), ('rho_melt', 'Spera2000', 'mafic potassic melt (used only if undersaturated)')]}}, 'reservoirs': {'1982–84 source · spheroid': [('Gottsmann2006', 'vertical prolate spheroid 800 m E of Pozzuoli, depth 2.9 km (2.0–4.2), aspect 0.51, semi-major axis 2.2 km; ΔV = 71×10⁶ m³ and ΔP = 52 MPa for 1982–84; μ = 10 GPa, ν = 0.25'), ('—', 'V0 = 4/3·π·a·b² with a = 2.2 km, b/a = 0.51')], '2018–23 shallow source · sill': [('Astort2024', 'shallow source at 3.9 km (2018–2023), sill-like to oblate; volume ≈ 0.16 km³ in 2023; volume change 1.4–8.3×10⁶ m³/yr, > 6×10⁷ m³ in 2007–2023'), ('—', 'crack radius 2 km assumed (not reported): thin (w̄/2a ≈ 0.003) and buried (a/d ≈ 0.5)'), ('Gottsmann2006', 'μ = 10 GPa, ν = 0.25')], 'Deep sill-like reservoir': [('Astort2024', 'caldera-wide tabular source fixed at 8 km, area ≈ 100 km² (radius 5.64 km)'), ('Pappalardo2012', 'sill-like magma layer 1–2 km thick at 7–9 km; 1 km thickness used → V0 = 100 km³'), ('Petrelli2023', 'main reservoir ≈ 9 km')]}, 'refs': {'Arienzo2026': "Arienzo, I., Cariddi, B., Pelullo, C., D'Oriano, C., Deloule, E., D'Antonio, M. & Moretti, R. (2026). The source-to-surface journey of volatile-flooded magmas: the archetypal case of the Campi Flegrei unrest caldera. Geosci. Front. 17, 102262", 'Astort2024': 'Astort, A., Trasatti, E., Caricchi, L., Polcari, M., De Martino, P., Acocella, V., et al. (2024). Tracking the 2007–2023 magma-driven unrest at Campi Flegrei caldera (Italy). Commun. Earth Environ. 5, 506. doi:10.1038/s43247-024-01665-4', 'Cannatelli2012': "Cannatelli, C. (2012). Understanding magma evolution at Campi Flegrei (Campania, Italy) volcanic complex using melt inclusions and phase equilibria. Mineral. Petrol. 104, 29–42. doi:10.1007/s00710-011-0182-6 (values read from the authors' manuscript version: Cannatelli, Spera, Fedele & De Vivo, magma.geol.ucsb.edu)", 'Gottsmann2006': 'Gottsmann, J., Rymer, H. & Berrino, G. (2006). Unrest at the Campi Flegrei caldera (Italy): a critical evaluation of source parameters from geodetic data inversion. J. Volcanol. Geotherm. Res. 150, 132–145. doi:10.1016/j.jvolgeores.2005.07.002', 'Pappalardo2012': 'Pappalardo, L. & Mastrolorenzo, G. (2012). Rapid differentiation in a sill-like magma reservoir: a case study from the Campi Flegrei caldera. Sci. Rep. 2, 712. doi:10.1038/srep00712', 'Petrelli2023': 'Petrelli, M., Ágreda López, M., Pisello, A., et al. (2023). Pre-eruptive dynamics at the Campi Flegrei Caldera: from evidence of magma mixing to timescales estimates. Earth Planets Space 75, 19. doi:10.1186/s40623-023-01765-z', 'Spera2000': 'Spera, F.J. (2000). Physical properties of magma. In: Sigurdsson, H. (ed.) Encyclopedia of Volcanoes, Academic Press, 171–190'}}},
    'Kīlauea (Hawaiʻi)': {'project': 'Kīlauea (Hawaiʻi) — preset from published data', 'obs_km': [0.0, 1.0, 2.0, 5.0], 'summary': 'Tholeiitic basalt summit system with two reservoirs: Halemaʻumaʻu (HMM, ~2 km) and South Caldera (SC, ~3–4 km). Three volatile states of the same basalt: the CO₂-rich primary magma, the magma arriving at the summit reservoirs, and the degassed magma stored in HMM before 2018. Each reservoir is given in the two published geometries.', 'frame': 'Depths below the caldera floor as published. Pressures ρgz with ρ = 2500 kg m⁻³, which gives ≈ 49 MPa at 2 km, close to the ≈ 45 MPa magmastatic pressure of the HMM reservoir (Anderson et al. 2019).', 'MAGMAS': 'property      | KIL_STORED     | KIL_SUPPLY     | KIL_PRIMARY   \nevo_class     | basalt         | basalt         | basalt        \nT_C           | 1150           | 1200           | 1220          \nbeta_liquid   | 8e-11          | 8e-11          | 8e-11         \nrho_melt      | 2600           | 2600           | 2600          \nH2O_wt        | 0.27           | 0.32           | 0.32          \nCO2_wt        | 0.0265         | 0.32           | 0.7           \nS_wt          | 0.111          | 0.09           | 0.09          \nfO2_buffer    | FMQ            | FMQ            | FMQ           \nfO2_value     | 0.25           | 0.25           | 0.25          \nSIO2          | 49.3           | 49.3           | 49.3          \nTIO2          | 2.78           | 2.78           | 2.78          \nAL2O3         | 13.6           | 13.6           | 13.6          \nFEO           | 11.3           | 11.3           | 11.3          \nMNO           | 0.17           | 0.17           | 0.17          \nMGO           | 7.13           | 7.13           | 7.13          \nCAO           | 11.4           | 11.4           | 11.4          \nNA2O          | 2.4            | 2.4            | 2.4           \nK2O           | 0.51           | 0.51           | 0.51          \nP2O5          | 0.29           | 0.29           | 0.29          \nsource        | Degassed magma stored in HMM (2018) | Magma arriving at the summit reservoirs | CO₂-rich primary magma', 'RESERVOIRS': 'name                           | shape    | depth_km | V0_m3  | Ve_m3           | resident   | injected    | mu_GPa | nu   | radius_km | aspect | dip_deg | strike_deg | P_depth_km | rho_crust\nHMM · prolate spheroid         | spheroid | 2.2      | 4e9    | 3.5e5; 1e7; 1e8 | KIL_STORED | KIL_SUPPLY  | 3      | 0.25 |           | 0.56   | 90      | 0          |            | 2500\nHMM · McTigue sphere           | sphere   | 2        | 4e9    | 3.5e5; 1e7; 1e8 | KIL_STORED | KIL_SUPPLY  | 3      | 0.25 |           |        |         |            |            | 2500\nSouth Caldera · sill           | penny    | 3.6      | 2e9    | 3.5e5; 1e7; 1e8 | KIL_SUPPLY | KIL_PRIMARY | 3      | 0.25 | 1.5       |        |         |            |            | 2500\nSouth Caldera · McTigue sphere | sphere   | 3        | 2.14e9 | 3.5e5; 1e7; 1e8 | KIL_SUPPLY | KIL_PRIMARY | 3      | 0.25 | 0.8       |        |         |            |            | 2500', 'sources': {'magmas': {'KIL_STORED': {'label': 'Degassed magma stored in HMM (2018)', 'rows': [('major elements', 'USGS2022', 'BHVO-2(G), 1919 Halemaʻumaʻu overflow lava; K2O from K = 4270 ppm'), ('T_C', 'Gansecki2019', 'Halemaʻumaʻu lava lake 1155–1160 °C; Lerner et al. (2021): 1130–1145 °C on eruption'), ('H2O_wt', 'Lerner2021', 'best pre-eruptive estimate 0.27 ± 0.03 wt%'), ('CO2_wt', 'Lerner2021', '≈ 265 ppm (minimum pre-eruptive, interstitial melt 224 ppm)'), ('S_wt', 'Lerner2021', 'pre-eruptive 1110 ± 60 ppm'), ('fO2', 'Lerner2021', 'Fe³⁺/ΣFe = 0.18; FMQ+0.25'), ('beta_liquid', 'Spera2000', 'basaltic melt'), ('rho_melt', 'Anderson2019', 'reservoir magma 2550–2650 kg m⁻³')]}, 'KIL_SUPPLY': {'label': 'Magma arriving at the summit reservoirs', 'rows': [('major elements', 'USGS2022', 'BHVO-2(G), 1919 Halemaʻumaʻu overflow lava; K2O from K = 4270 ppm'), ('T_C', '—', 'assumed, between eruption (1140–1160 °C) and high-Fo inclusion (1220–1310 °C) temperatures'), ('H2O_wt', 'Greenland1985', 'magma arriving at the shallow summit reservoir: 0.32 wt% H2O'), ('CO2_wt', 'Greenland1985', '0.32 wt% CO2'), ('S_wt', 'Greenland1985', '0.09 wt% S'), ('fO2', 'Lerner2021', 'as above'), ('beta_liquid', 'Spera2000', 'basaltic melt'), ('rho_melt', 'Anderson2019', 'as above')]}, 'KIL_PRIMARY': {'label': 'CO₂-rich primary magma', 'rows': [('major elements', 'USGS2022', 'BHVO-2(G), 1919 Halemaʻumaʻu overflow lava; K2O from K = 4270 ppm (no primary-melt analysis used)'), ('T_C', 'Lerner2021', 'high-Fo melt-inclusion entrapment 1220–1310 °C (lower end)'), ('H2O_wt', 'Greenland1985', 'as for the supply magma'), ('CO2_wt', 'Gerlach2002', 'bulk CO2 of the primary magma ≈ 0.70 wt%'), ('S_wt', 'Greenland1985', 'as for the supply magma'), ('fO2', 'Lerner2021', 'as above'), ('beta_liquid', 'Spera2000', 'basaltic melt'), ('rho_melt', 'Anderson2019', 'as above')]}}, 'reservoirs': {'HMM · prolate spheroid': [('Wang2021', 'HMM prolate spheroid, vertical/horizontal aspect ≈ 1.8 (b/a = 0.56), depth ≈ 2.2 km'), ('Anderson2019', 'reservoir volume 2.5–7.2 km³ (68%); 4 km³ used; μ prior 2–10 GPa, 3 GPa in earlier work'), ('Anderson2020', 'deflation–inflation events ≈ 3.5×10⁵ m³')], 'HMM · McTigue sphere': [('Anderson2019', 'reservoir centred ≈ 2 km below the surface, roughly equant; 4 km³ (radius 0.98 km from V0)')], 'South Caldera · sill': [('Wang2021', 'SC oblate reservoir, aspect ≈ 0.14, depth ≈ 3.6 km'), ('—', 'radius 1.5 km assumed: an oblate body of this radius and c/a = 0.14 has V ≈ 2.0 km³, the volume of the 0.8-km sphere of Anderson et al. (2020)')], 'South Caldera · McTigue sphere': [('Anderson2020', 'SC reservoir as a McTigue sphere at 3 km, radius 800 m')]}, 'refs': {'Anderson2019': 'Anderson, K.R., Johanson, I.A., Patrick, M.R., Gu, M., Segall, P., Poland, M.P., Montgomery-Brown, E.K. & Miklius, A. (2019). Magma reservoir failure and the onset of caldera collapse at Kīlauea Volcano in 2018. Science 366, eaaz1822. doi:10.1126/science.aaz1822', 'Anderson2020': 'Anderson, A.N., Foster, J.H. & Frazer, N. (2020). Implications of deflation-inflation event models on Kīlauea Volcano, Hawaiʻi. J. Volcanol. Geotherm. Res. 397, 106832', 'Gansecki2019': "Gansecki, C., et al. (2019). The tangled tale of Kīlauea's 2018 eruption as told by geochemical monitoring. Science 366(6470), eaaz0147. doi:10.1126/science.aaz0147", 'Gerlach2002': 'Gerlach, T.M., McGee, K.A., Elias, T., Sutton, A.J. & Doukas, M.P. (2002). Carbon dioxide emission rate of Kīlauea Volcano: implications for primary magma and the summit reservoir. J. Geophys. Res. 107(B9). doi:10.1029/2001JB000407', 'Greenland1985': 'Greenland, L.P., Rose, W.I. & Stokes, J.B. (1985). An estimate of gas emissions and magmatic gas content from Kilauea volcano. Geochim. Cosmochim. Acta 49(1), 125–129. doi:10.1016/0016-7037(85)90196-6', 'Lerner2021': 'Lerner, A.H., Wallace, P.J., Shea, T., Mourey, A.J., Kelly, P.J., Nadeau, P.A., et al. (2021). The petrologic and degassing behavior of sulfur and other magmatic volatiles from the 2018 eruption of Kīlauea, Hawaiʻi. Bull. Volcanol. 83, 43. doi:10.1007/s00445-021-01459-y', 'Spera2000': 'Spera, F.J. (2000). Physical properties of magma. In: Sigurdsson, H. (ed.) Encyclopedia of Volcanoes, Academic Press, 171–190', 'USGS2022': 'U.S. Geological Survey (2022). Reference material information sheet BHVO-2G — basalt glass made from BHVO-2, pāhoehoe of the 1919 Halemaʻumaʻu overflow, Kīlauea; major-element estimates after Jochum et al. (2005)', 'Wang2021': "Wang, T., Zheng, Y., Pulvirenti, F. & Segall, P. (2021). Post-2018 caldera collapse re-inflation uniquely constrains Kīlauea's magmatic system. J. Geophys. Res. Solid Earth 126, e2021JB021803. doi:10.1029/2021JB021803"}}},
}


def start_table_text(table_key, choice, saved=""):
    """Start values of one table ('magmas', 'reservoirs' or 'step2') for the choice above."""
    if choice in EXAMPLES:
        return {"magmas": EXAMPLES[choice]["MAGMAS"], "reservoirs": EXAMPLES[choice]["RESERVOIRS"]}.get(table_key, "")
    return saved if choice == SAVED_COPY else ""


_ex = EXAMPLES.get(START_FROM)
PROJECT = str(PROJECT_NAME).strip() or (_ex["project"] if _ex else "My volcano")
if str(obs_distances_km).strip():
    try:
        OBS_KM = [float(x) for x in str(obs_distances_km).replace(";", ",").split(",") if x.strip()]
    except ValueError:
        raise ValueError(f"obs_distances_km must be numbers separated by commas, got '{obs_distances_km}'")
else:
    OBS_KM = list(_ex["obs_km"]) if _ex else [0.0, 1.0, 2.0, 5.0]
if not OBS_KM or any(d < 0 for d in OBS_KM):
    raise ValueError("obs_distances_km needs at least one distance ≥ 0")
OBS_KM = sorted(set(OBS_KM))
DISTS_M = [1000.0 * d for d in OBS_KM]
PHYS = dict(project=PROJECT, rho_crust=float(rho_crust), g=float(g),
            mu_GPa=float(default_mu_GPa), nu=float(default_nu),
            profile_azimuth=float(profile_azimuth), profile_max_km=float(profile_max_km),
            use_evo=(step2_engine == "EVo (cell 2)"))
TableEditor.start_all(START_FROM)          # refills the tables of cells 5, 6, 8 if they are open and the choice changed

section(f"Project: {html.escape(PROJECT)}")
table(["Setting", "Value", "Setting", "Value"],
      [["Start values of the tables", html.escape(START_FROM), "Observation distances", ", ".join(f"{d:g}" for d in OBS_KM) + " km"],
       ["Crustal density ρ<sub>c</sub>", f"{rho_crust:g} kg m<sup>−3</sup>", "Profile azimuth / length",
        f"{profile_azimuth:g}° / " + (f"{profile_max_km:g} km" if profile_max_km > 0 else "automatic")],
       ["Gravity g", f"{g:g} m s<sup>−2</sup>", "Step-2 engine", step2_engine],
       ["Default shear modulus μ / Poisson ratio ν", f"{default_mu_GPa:g} GPa / {default_nu:g}", "", ""]], left_cols=(0, 2))

## Guide to the tables — how to describe a volcano in cells 5 and 6

**Filling in the tables.** Run cells 5 and 6 to open them, then type into the boxes. An empty box means *not given*; its grey hint tells what is used instead (`cell 4` = the default set in cell 4). Numbers may be written `1e6`, `1.2e-10` or `5×10^6`; a box turns red when its text is not a number. Magma names are case-insensitive. Buttons: **+ Add** a magma (column) or a reservoir (row), **⧉** copy and **✕** remove one, **↶ Undo** the last change, **↺ Reset** to the start values of cell 4, **Show more rows / Show optional columns** for the less common entries, **Text view** to copy a whole table as text or to paste one from Excel or Google Sheets. Every row name and column title carries a short key in grey (e.g. `T_C`); the tables below and the text view use these keys.

### Cell 5 — Magmas (one column per magma)

| Row | Meaning | Needed |
|---|---|---|
| `evo_class` | EVo solubility fit (Burgisser et al. 2015): `basalt` (45–55 wt% SiO₂), `phonolite` (52–63), `rhyolite` (65–80) | always |
| `T_C` | magma temperature (°C) | always |
| `beta_liquid`, `rho_melt` | melt compressibility (Pa⁻¹, typically 0.5–2×10⁻¹⁰) and melt density used when there is no vapour (kg m⁻³) | always |
| `SIO2 … CR2O3` | 13 oxides, wt% volatile-free: SIO2 TIO2 AL2O3 FEO FE2O3 MNO MGO CAO NA2O K2O P2O5 NIO CR2O3 | SIO2 and iron |
| `evo_mode` | `saturation` (default), `atomic` or `gas_at_P` — see below | optional |
| `gas_system` | `auto` (from the volatiles given) or `OH`, `COH`, `SOH`, `COHS`, `COHSN` | optional |
| `H2O_wt`, `CO2_wt`, `S_wt`, `N_wt`, `graphite_wt` | volatile contents, wt% (N and graphite: *Show more rows*) | per mode |
| `atomic_H_ppm`, `atomic_C_ppm`, `atomic_S_ppm`, `atomic_N_ppm` | total element masses of the system, ppm (*Show more rows*) | atomic mode |
| `gas_wt`, `P_start_MPa` | vapour mass fraction (wt%) at the start pressure; start pressure (MPa, empty = reservoir pressure) (*Show more rows*) | gas_at_P mode |
| `fO2_buffer`, `fO2_value` | redox: `FMQ`, `NNO`, `IW` with value = log units above the buffer; `absolute` with value = fO₂ in bar; `iron` = fO₂ from the FEO and FE2O3 rows | optional (default cell 7) |
| `fH2_bar`, `fH2O_bar`, `fCO2_bar` | initial gas fugacities (bar) (*Show more rows*) | gas_at_P mode only |
| `source` | free text, shown on the page | optional |
| any EVo key (button *+ Add EVo setting row*) | e.g. `H2_MODEL`, `SULFIDE_CAPACITY`, `DP_MAX`, `RUN_TYPE`: changes the setting of cell 7 for this magma only | expert |

**The three EVo modes**

| `evo_mode` | You know… | EVo computes | Give |
|---|---|---|---|
| `saturation` (thesis) | the **total** volatile content of the magma (e.g. pre-eruptive melt inclusions) | the saturation pressure P<sub>sat</sub>, then closed-system decompression to the reservoir pressure | `H2O_wt` (+ `CO2_wt`, `S_wt`, `N_wt` or `graphite_wt`) and fO₂ |
| `atomic` | the total **mass of each element** (H, C, S, N) in magma + gas | as above | `atomic_H_ppm` (+ C, S, N) and fO₂ |
| `gas_at_P` | the **vapour fraction** at some pressure and the state of the gas (**fugacities**) or the melt | the equilibrium at that pressure (and decompression to the reservoir pressure if `P_start_MPa` is larger) | `gas_wt` and initial conditions: COHS/COHSN need **exactly 3** of {fO₂, fH₂, H₂O (`H2O_wt` or `fH2O_bar`), CO₂ (`CO2_wt`, `fCO2_bar` or `graphite_wt`), `S_wt`}; COH/SOH exactly 2 of {fO₂, fH₂, H₂O}; OH one of {fO₂, fH₂} |

Rules enforced by EVo: in `saturation` and `atomic` mode no gas fugacities can be given; H₂O and CO₂ are each given only once (as wt% or as fugacity); give fO₂ **or** the FeO/Fe₂O₃ split, not both. With `fO2_buffer = absolute` in saturation or atomic mode, EVo's saturation search does not converge with a fixed absolute fO₂, so the notebook converts it to the equivalent ΔFMQ at the reservoir T and P (EVo's own Frost 1991 conversion) and says so.

### Cell 6 — Reservoirs (one row per reservoir, or per geometry to test)

| Column | Meaning | Needed |
|---|---|---|
| `name` | unique label | always |
| `shape` | `mogi` point sphere · `sphere` finite sphere (McTigue 1987) · `spheroid` prolate spheroid (Yang et al. 1988) · `penny` horizontal crack (Fialko et al. 2001) | always |
| `depth_km`, `V0_m3` | depth of the source centre; reservoir volume | always |
| `Ve_m3` | injected volumes separated by `;` — any number | always |
| `resident`, `injected` | magma names from cell 5 (resident sets β<sub>m</sub>; injected is listed for reference) | resident |
| `mu_GPa`, `nu` | host-rock shear modulus and Poisson ratio (empty = cell 4) | optional |
| `radius_km` | penny: crack radius (required); sphere: radius (empty = from V0) | penny |
| `aspect`, `dip_deg`, `strike_deg` | spheroid: b/a (0 < aspect ≤ 1), dip of the long axis (default 90°), its azimuth (default 0°) | spheroid |
| `P_depth_km`, `P_MPa`, `rho_crust` | pressure datum depth; pressure given directly; overburden density (*Show optional columns*) | optional |
| `T_C`, `beta_c` | temperature of the resident magma in this reservoir (overrides cell 5); chamber compressibility given directly (Pa⁻¹), e.g. from a numerical model (*Show optional columns*) | optional |

The same storage level can appear in several rows with different shapes (as in the worked example) to compare geometries.

In [ ]:
#@title 5. Magmas — one column per magma  { display-mode: "form" }
#@markdown **Run this cell to open the table, then type into the boxes.** An empty box means *not given*; the grey hint in a box tells what is used instead, and a box turns red when its text is not a number. Hover over a row name for its meaning — the guide above explains every row.
#@markdown **+ Add magma** adds an empty column · **⧉** copies a column · **✕** removes it · **Show more rows** opens the rows of the atomic and gas_at_P modes, nitrogen, graphite and gas fugacities · **Text view** shows the table as text to copy, and takes a block pasted from Excel or Google Sheets.
#@markdown The table starts from the choice in cell 4 and keeps your entries while the Colab session is open (also through *Run all*). To keep them in the notebook file, copy the text of *Text view* between the quotes of `SAVED_MAGMAS` below (double-click the cell to see the code) and choose *Saved copy* in cell 4.
SAVED_MAGMAS = """
"""

_g = ["Magma type and physical properties",
      "How EVo describes the volatiles",
      "Volatile contents (wt%) — saturation mode: total content · gas_at_P mode: dissolved at the start pressure",
      "Redox state",
      "Atomic mode — total mass of each element in magma + gas (ppm)",
      "gas_at_P mode — vapour present at a start pressure, and initial gas fugacities",
      "Major elements (wt%, volatile-free)",
      "Analytical fallback (used only when EVo is switched off in cell 4 or cannot run)",
      "Where the values come from"]
MAGMA_FIELDS = [
    field("evo_class", "EVo solubility class", kind="choice", group=_g[0], choices=["basalt", "phonolite", "rhyolite"],
          help="Solubility fit of Burgisser et al. (2015) used by EVo: basalt 45–55, phonolite 52–63, rhyolite 65–80 wt% SiO2"),
    field("T_C", "Temperature", "°C", group=_g[0], hint="e.g. 1100"),
    field("beta_liquid", "Melt compressibility", "Pa⁻¹", group=_g[0], hint="e.g. 1e-10",
          help="Compressibility of the volatile-free melt, typically 0.5–2 × 10⁻¹⁰ Pa⁻¹"),
    field("rho_melt", "Melt density", "kg m⁻³", group=_g[0], hint="e.g. 2400",
          help="Melt density used when the magma holds no vapour"),
    field("evo_mode", "EVo mode", kind="choice", group=_g[1], choices=["saturation", "atomic", "gas_at_P"],
          help="Empty = saturation. saturation: total volatile contents known · atomic: total element masses known · "
               "gas_at_P: vapour fraction at a pressure known"),
    field("gas_system", "Gas system", kind="choice", group=_g[1], choices=["auto", "OH", "COH", "SOH", "COHS", "COHSN"],
          help="Empty = cell 7 (auto: chosen from the volatiles that are given)"),
    field("H2O_wt", "H₂O", group=_g[2]),
    field("CO2_wt", "CO₂", group=_g[2]),
    field("S_wt", "S", group=_g[2]),
    field("N_wt", "N", group=_g[2], optional=True, help="Nitrogen (needs gas system COHSN)"),
    field("graphite_wt", "Graphite", group=_g[2], optional=True, help="Graphite-saturated melt, instead of CO₂"),
    field("fO2_buffer", "Redox reference", kind="choice", group=_g[3], choices=["FMQ", "NNO", "IW", "absolute", "iron"],
          help="Empty = cell 7. FMQ, NNO, IW: the value is in log units above the buffer · absolute: the value is fO₂ "
               "in bar · iron: fO₂ follows from the FeO and Fe₂O₃ rows"),
    field("fO2_value", "Redox value", "log units or bar", group=_g[3], hint="cell 7"),
    field("atomic_H_ppm", "Total H", group=_g[4], optional=True),
    field("atomic_C_ppm", "Total C", group=_g[4], optional=True),
    field("atomic_S_ppm", "Total S", group=_g[4], optional=True),
    field("atomic_N_ppm", "Total N", group=_g[4], optional=True),
    field("gas_wt", "Vapour mass fraction", "wt%", group=_g[5], optional=True),
    field("P_start_MPa", "Start pressure", "MPa", group=_g[5], hint="reservoir P", optional=True,
          help="Pressure at which the vapour fraction is known (empty = the reservoir pressure)"),
    field("fH2_bar", "fH₂", "bar", group=_g[5], optional=True),
    field("fH2O_bar", "fH₂O", "bar", group=_g[5], optional=True),
    field("fCO2_bar", "fCO₂", "bar", group=_g[5], optional=True),
] + [field(ox, lab, group=_g[6], optional=ox in ("NIO", "CR2O3"),
           help="Only with redox reference = iron; otherwise give all iron as FeO" if ox == "FE2O3" else "")
     for ox, lab in (("SIO2", "SiO₂"), ("TIO2", "TiO₂"), ("AL2O3", "Al₂O₃"), ("FEO", "FeO"), ("FE2O3", "Fe₂O₃"),
                     ("MNO", "MnO"), ("MGO", "MgO"), ("CAO", "CaO"), ("NA2O", "Na₂O"), ("K2O", "K₂O"),
                     ("P2O5", "P₂O₅"), ("NIO", "NiO"), ("CR2O3", "Cr₂O₃"))] + [
    field("fallback_H2O_coeff", "H₂O solubility", "wt% / √MPa", group=_g[7], hint="by class", optional=True),
    field("fallback_CO2_coeff", "CO₂ solubility", "ppm / MPa", group=_g[7], hint="by class", optional=True),
    field("source", "Reference, sample", kind="long", group=_g[8], hint="free text"),
]
MAGMA_GRID = TableEditor.open("magmas", MAGMA_FIELDS, orient="cols", item="magma", start=START_FROM,
                              loader=lambda choice: start_table_text("magmas", choice, SAVED_MAGMAS),
                              label_width=290, item_width=132, extra_label="+ Add EVo setting row")
MAGMA_GRID.show()

In [ ]:
#@title 6. Reservoirs — one row per reservoir  { display-mode: "form" }
#@markdown **Run this cell to open the table.** One row per reservoir, or per geometry you want to test — the same storage level may appear in several rows with different shapes. An empty box means *not given* (the grey hint tells what is used instead); a red box is not a number. Hover over a column title for its meaning.
#@markdown **+ Add reservoir** adds an empty row · **⧉** copies a row · **✕** removes it · **Show optional columns** opens the columns that set the pressure, crustal density, temperature or chamber compressibility directly · **Text view** to copy the table or paste one from Excel. The magma boxes offer the names of cell 5; renaming a magma there renames it here too.
#@markdown To keep the table in the notebook file, copy the text of *Text view* between the quotes of `SAVED_RESERVOIRS` below (double-click the cell to see the code) and choose *Saved copy* in cell 4.
SAVED_RESERVOIRS = """
"""

_g = ["Reservoir", "Magmas (from cell 5)", "Host rock", "Shape details", "Optional (set directly)"]
RESERVOIR_FIELDS = [
    field("name", "Name", kind="text", group=_g[0], width=172, hint="unique name"),
    field("shape", "Shape", kind="choice", group=_g[0], width=122,
          choices=[("mogi · point", "mogi"), ("sphere · finite", "sphere"), ("spheroid · Yang", "spheroid"),
                   ("penny · crack", "penny")],
          help="mogi: point sphere (Mogi 1958) · sphere: finite sphere (McTigue 1987) · spheroid: prolate spheroid "
               "(Yang et al. 1988) · penny: horizontal penny-shaped crack (Fialko et al. 2001)"),
    field("depth_km", "Depth", "km", group=_g[0], width=58, help="Depth of the source centre below the surface"),
    field("V0_m3", "Volume V₀", "m³", group=_g[0], width=74, hint="e.g. 5e9", help="Reservoir volume"),
    field("Ve_m3", "Injected Vₑ", "m³, ; between", kind="list", group=_g[0], width=124, hint="1e6; 5e6",
          help="Injected volumes, any number, separated by ;"),
    field("resident", "Resident", kind="ref", ref="magmas", group=_g[1], width=114, hint="magma",
          help="Magma filling the reservoir: it sets the magma compressibility"),
    field("injected", "Injected", "optional", kind="ref", ref="magmas", group=_g[1], width=114,
          help="Recharge magma (listed for reference)"),
    field("mu_GPa", "μ", "GPa", group=_g[2], width=52, hint="cell 4", help="Shear modulus of the host rock (empty = cell 4)"),
    field("nu", "ν", group=_g[2], width=48, hint="cell 4", help="Poisson ratio of the host rock (empty = cell 4)"),
    field("radius_km", "Radius", "km", group=_g[3], width=64,
          help="Penny crack: crack radius (required) · finite sphere: radius (empty = from V₀)"),
    field("aspect", "b / a", "spheroid", group=_g[3], width=62,
          help="Spheroid: short / long semi-axis, 0 < b/a ≤ 1"),
    field("dip_deg", "Dip", "°", group=_g[3], width=56, hint="90", help="Spheroid: dip of the long axis (90 = vertical)"),
    field("strike_deg", "Azimuth", "°", group=_g[3], width=66, hint="0", help="Spheroid: azimuth of the long axis, ° from North"),
    field("P_depth_km", "Depth for P", "km", group=_g[4], width=72, hint="= depth", optional=True,
          help="Depth used for P = ρgz when it differs from the source depth (e.g. a different datum)"),
    field("P_MPa", "Pressure", "MPa", group=_g[4], width=64, hint="ρgz", optional=True,
          help="Reservoir pressure given directly (replaces ρgz)"),
    field("rho_crust", "ρ crust", "kg m⁻³", group=_g[4], width=70, hint="cell 4", optional=True,
          help="Mean density of the overburden for P = ρgz"),
    field("T_C", "T", "°C", group=_g[4], width=62, hint="cell 5", optional=True,
          help="Temperature of the resident magma in this reservoir (empty = cell 5)"),
    field("beta_c", "β_c", "Pa⁻¹", group=_g[4], width=72, hint="shape", optional=True,
          help="Chamber compressibility given directly, e.g. from a numerical model (empty = from the shape)"),
]
RESERVOIR_GRID = TableEditor.open("reservoirs", RESERVOIR_FIELDS, orient="rows", item="reservoir", start=START_FROM,
                                  loader=lambda choice: start_table_text("reservoirs", choice, SAVED_RESERVOIRS),
                                  item_width=70, more_label="Show optional columns")
RESERVOIR_GRID.show()

In [ ]:
#@title 7. EVo options shared by all magmas — run type, thermodynamic models, solver, default redox and gas system  { display-mode: "form", run: "auto" }
#@markdown These are the entries of EVo's environment file that are the same for every magma. Per magma (cell 5) the notebook sets `COMPOSITION`, `T_START`, the volatile and fugacity entries, the fO₂ setting and the run mode; per reservoir it sets `P_STOP` = reservoir pressure. Any key below can still be changed for one magma with *+ Add EVo setting row* in the magma table (cell 5). Defaults = thesis setup (Appendix B).

#@markdown --- **Run type and system**
RUN_TYPE = "closed"            #@param ["closed", "open"]
default_gas_system = "auto"    #@param ["auto", "OH", "COH", "SOH", "COHS", "COHSN"]
FE_SYSTEM = True               #@param {type:"boolean"}
S_SAT_WARN = False             #@param {type:"boolean"}

#@markdown --- **Default redox** (used where a magma leaves `fO2_buffer` / `fO2_value` empty)
default_fO2_buffer = "FMQ"     #@param ["FMQ", "NNO", "IW", "absolute", "iron"]
default_fO2_value = 1.0        #@param {type:"number"}

#@markdown --- **Pressure stepping and system** (bar, g). P_START is the upper bracket of the saturation search; WgT the trace vapour fraction used at saturation.
P_START = 3000                 #@param {type:"number"}
DP_MIN = 0.1                   #@param {type:"number"}
DP_MAX = 50                    #@param {type:"number"}
MASS = 100                     #@param {type:"number"}
WgT = 0.00001                  #@param {type:"number"}
LOSS_FRAC = 0.9999             #@param {type:"number"}

#@markdown --- **Thermodynamic models**
DENSITY_MODEL = "spera2000"    #@param ["spera2000"]
FO2_MODEL = "kc1991"           #@param ["kc1991", "r2013"]
FMQ_MODEL = "frost1991"        #@param ["frost1991"]
H2O_MODEL = "burguisser2015"   #@param ["burguisser2015"]
H2_MODEL = "gaillard2003"      #@param ["gaillard2003", "burguisser2015"]
C_MODEL = "burguisser2015"     #@param ["burguisser2015", "eguchi2018"]
CO_MODEL = "None"              #@param ["None", "armstrong2015"]
CH4_MODEL = "None"             #@param ["None", "ardia2013"]
SULFIDE_CAPACITY = "oneill2020"  #@param ["oneill2020", "oneill2002"]
SULFATE_CAPACITY = "nash2019"  #@param ["nash2019"]
SCSS = "liu2007"               #@param ["liu2007"]
N_MODEL = "libourel2003"       #@param ["libourel2003"]

#@markdown --- **Notebook wrapper.** Trace floors (ppm) raise a volatile content given as 0 or very small in saturation mode, because the equilibrium is undefined for an absent element (thesis: 10 / 1 / 1 ppm). P_STOP is rounded to whole bar to stay on EVo's pressure grid; on a solver failure P_STOP is nudged by the listed amounts (bar) before the analytical fallback is used.
H2O_floor_ppm = 10             #@param {type:"number"}
CO2_floor_ppm = 1              #@param {type:"number"}
S_floor_ppm = 1                #@param {type:"number"}
P_STOP_integer_bar = True      #@param {type:"boolean"}
P_STOP_retry_bumps_bar = "1, 2, 5"  #@param {type:"string"}

# Global part of the environment file. Entries that are switched per magma start "off" here, with
# the neutral values of the thesis template, so that an unused option never changes a result.
EVO_ENV = dict(
    RUN_TYPE=RUN_TYPE, SINGLE_STEP=False, FIND_SATURATION=True, ATOMIC_MASS_SET=False,
    GAS_SYS="COHS", FE_SYSTEM=bool(FE_SYSTEM), OCS=False, S_SAT_WARN=bool(S_SAT_WARN),
    P_START=float(P_START), DP_MIN=float(DP_MIN), DP_MAX=float(DP_MAX), MASS=float(MASS),
    WgT=float(WgT), LOSS_FRAC=float(LOSS_FRAC),
    DENSITY_MODEL=DENSITY_MODEL, FO2_MODEL=FO2_MODEL, FMQ_MODEL=FMQ_MODEL, H2O_MODEL=H2O_MODEL,
    H2_MODEL=H2_MODEL, C_MODEL=C_MODEL, CO_MODEL=CO_MODEL, CH4_MODEL=CH4_MODEL,
    SULFIDE_CAPACITY=SULFIDE_CAPACITY, SULFATE_CAPACITY=SULFATE_CAPACITY, SCSS=SCSS, N_MODEL=N_MODEL,
    FO2_buffer_SET=True, FO2_buffer="FMQ", FO2_buffer_START=1.0, FO2_SET=False, FO2_START=8.3e-12,
    FH2_SET=False, FH2_START=0.24, FH2O_SET=False, FH2O_START=1000.0, FCO2_SET=False, FCO2_START=1.0,
    ATOMIC_H=550.0, ATOMIC_C=200.0, ATOMIC_S=4000.0, ATOMIC_N=10.0,
    WTH2O_SET=False, WTH2O_START=0.03, WTCO2_SET=False, WTCO2_START=0.01,
    SULFUR_SET=False, SULFUR_START=0.001, NITROGEN_SET=False, NITROGEN_START=0.0001,
    GRAPHITE_SATURATED=False, GRAPHITE_START=0.0001)
EVO_DEFAULTS = dict(gas_system=default_gas_system, fO2_buffer=default_fO2_buffer,
                    fO2_value=float(default_fO2_value))
EVO_WRAP = dict(H2O_floor_wt=float(H2O_floor_ppm) * 1e-4, CO2_floor_wt=float(CO2_floor_ppm) * 1e-4,
                S_floor_wt=float(S_floor_ppm) * 1e-4, P_STOP_integer_bar=bool(P_STOP_integer_bar),
                retry_bumps=[float(x) for x in str(P_STOP_retry_bumps_bar).replace(";", ",").split(",") if x.strip()])

section("EVo settings shared by all magmas")
_shared = ["RUN_TYPE", "FE_SYSTEM", "S_SAT_WARN", "P_START", "DP_MIN", "DP_MAX", "MASS", "WgT", "LOSS_FRAC",
           "DENSITY_MODEL", "FO2_MODEL", "FMQ_MODEL", "H2O_MODEL", "H2_MODEL", "C_MODEL", "CO_MODEL",
           "CH4_MODEL", "SULFIDE_CAPACITY", "SULFATE_CAPACITY", "SCSS", "N_MODEL"]
_half = (len(_shared) + 1) // 2
table(["Key", "Value", "Key", "Value"],
      [[f"<code>{a}</code>", html.escape(str(EVO_ENV[a])),
        f"<code>{_shared[i + _half]}</code>" if i + _half < len(_shared) else "",
        html.escape(str(EVO_ENV[_shared[i + _half]])) if i + _half < len(_shared) else ""]
       for i, a in enumerate(_shared[:_half])], left_cols=(0, 2),
      caption=f"Defaults for the magma table: gas system <b>{default_gas_system}</b>, redox <b>{default_fO2_buffer} "
              f"{default_fO2_value:+g}</b>. Trace floors {H2O_floor_ppm:g} / {CO2_floor_ppm:g} / {S_floor_ppm:g} ppm "
              "(H<sub>2</sub>O / CO<sub>2</sub> / S). <code>OCS</code> is fixed to False (not released in EVo 1.1.0).")
if RUN_TYPE == "open" and DP_MIN != DP_MAX:
    note("Open-system runs need DP_MIN = DP_MAX (EVo suggests ≤ 1 bar); EVo will refuse otherwise.", "warn")

In [ ]:
#@title 8. Optional: step-2 values entered by hand  { display-mode: "form" }
#@markdown Leave this table empty to compute step 2 with EVo for every reservoir (the normal case). Add a row to replace EVo for the resident magma of one reservoir — for example with the result of a separate EVo run, of another thermodynamic code, or a value for a sensitivity test. Only the vapour fraction φ enters the chain (β_m = φ·β_gas + (1 − φ)·β_liquid); the other columns are reported next to it. Empty box = not available.
SAVED_STEP2 = """
"""

_g = ["Reservoir (from cell 6)", "Used", "Reported only"]
STEP2_FIELDS = [
    field("reservoir", "Reservoir", kind="ref", ref="reservoirs", group=_g[0], width=180, hint="reservoir name"),
    field("phi_vol%", "Vapour φ", "vol%", group=_g[1], width=76),
    field("P_sat_MPa", "P_sat", "MPa", group=_g[2], width=64, help="Volatile saturation pressure"),
    field("gas_wt%", "Vapour", "wt%", group=_g[2], width=64, help="Vapour mass fraction"),
    field("rho_melt", "ρ melt", "kg m⁻³", group=_g[2], width=64),
    field("beta_m_eff", "β_m eff", "Pa⁻¹", group=_g[2], width=74, help="Equilibrium (effective) magma compressibility"),
    field("fH2O_bar", "fH₂O", "bar", group=_g[2], width=60),
    field("fCO2_bar", "fCO₂", "bar", group=_g[2], width=60),
    field("fSO2_bar", "fSO₂", "bar", group=_g[2], width=60),
    field("fH2S_bar", "fH₂S", "bar", group=_g[2], width=60),
]
STEP2_GRID = TableEditor.open("step2", STEP2_FIELDS, orient="rows", item="row", start=START_FROM,
                              loader=lambda choice: start_table_text("step2", choice, SAVED_STEP2), min_items=0)
STEP2_GRID.show()

In [ ]:
#@title 9. Calculation engine — reads the tables, runs EVo (step 2) and the compressibility and deformation formulas (steps 1–7)  { display-mode: "form" }
import io, re, json, difflib, warnings
import numpy as np
import pandas as pd

R_GAS = 8.314  # J/mol/K


# =============================================================================
# READING THE INPUT TABLES
# =============================================================================
class InputError(Exception):
    """A problem in the input tables, shown as one short message instead of a traceback."""
    def _render_traceback_(self):
        return ["\x1b[1;31mInput error\x1b[0m — " + str(self)]


def fail(msg):
    note("<b>Input error — please correct the table and run again.</b><br>" +
         html.escape(msg).replace("\n", "<br>"), "warn")
    raise InputError(msg)


_BLANK = {"", "-", "—", "–"}


def read_table(text, what):
    """Header and rows of an input table: (header, rows) from a fill-in table, or a text table
    (| -, tab- or comma-separated; text after # is a comment)."""
    if isinstance(text, tuple):
        return text
    try:
        return split_table_text(text)
    except ValueError as e:
        fail(f"{what}: {e}")


def to_num(x, where, positive=False, allow_zero=True):
    s = (str(x).strip().replace("−", "-").replace("×10^", "e").replace("x10^", "e")
         .replace("·10^", "e").replace("*10^", "e").replace(" ", ""))
    try:
        v = float(s)
    except ValueError:
        fail(f"{where}: '{x}' is not a number.")
    if not np.isfinite(v):
        fail(f"{where}: '{x}' is not a finite number.")
    if positive and (v < 0 or (v == 0 and not allow_zero)):
        fail(f"{where}: must be {'≥ 0' if allow_zero else '> 0'}, got {x}.")
    return v


def _suggest(word, options):
    m = difflib.get_close_matches(str(word), list(options), n=1, cutoff=0.6)
    return f" Did you mean '{m[0]}'?" if m else ""


# ----------------------------------------------------------------------------- magmas
OXIDES = ["SIO2", "TIO2", "AL2O3", "FEO", "FE2O3", "MNO", "MGO", "CAO", "NA2O", "K2O", "P2O5", "NIO", "CR2O3"]
_CHOICES = {
    "evo_class": ["basalt", "phonolite", "rhyolite"],
    "evo_mode": ["saturation", "atomic", "gas_at_P"],
    "gas_system": ["auto", "OH", "COH", "SOH", "COHS", "COHSN"],
    "fO2_buffer": ["FMQ", "NNO", "IW", "absolute", "iron"],
}
MAGMA_ROWS = {   # property: (kind, description)
    "evo_class": ("choice", "EVo solubility fit"), "T_C": ("num", "temperature (°C)"),
    "beta_liquid": ("num", "melt compressibility (1/Pa)"), "rho_melt": ("num", "melt density (kg/m³)"),
    "evo_mode": ("choice", "EVo mode"), "gas_system": ("choice", "gas system"),
    "H2O_wt": ("num", "H₂O (wt%)"), "CO2_wt": ("num", "CO₂ (wt%)"), "S_wt": ("num", "S (wt%)"),
    "N_wt": ("num", "N (wt%)"), "graphite_wt": ("num", "graphite (wt%)"),
    "atomic_H_ppm": ("num", "total H (ppm)"), "atomic_C_ppm": ("num", "total C (ppm)"),
    "atomic_S_ppm": ("num", "total S (ppm)"), "atomic_N_ppm": ("num", "total N (ppm)"),
    "gas_wt": ("num", "vapour (wt%)"), "P_start_MPa": ("num", "start pressure (MPa)"),
    "fO2_buffer": ("choice", "redox reference"), "fO2_value": ("num", "redox value"),
    "fH2_bar": ("num", "fH₂ (bar)"), "fH2O_bar": ("num", "fH₂O (bar)"), "fCO2_bar": ("num", "fCO₂ (bar)"),
    "fallback_H2O_coeff": ("num", "fallback H₂O solubility (wt%/√MPa)"),
    "fallback_CO2_coeff": ("num", "fallback CO₂ solubility (ppm/MPa)"),
    "source": ("text", "source of the values"),
}
_NOTEBOOK_SET = {"COMPOSITION", "T_START", "P_STOP"}   # set from evo_class, T_C and the reservoir pressure


def _evo_key_types():
    from evo_local.dgs_classes import RunDef
    out = {}
    for k, f in RunDef.model_fields.items():
        if k == "results_folder":
            continue
        ann = str(f.annotation)
        out[k] = "bool" if f.annotation is bool else ("num" if f.annotation in (float, int) else "text")
    return out


EVO_KEY_TYPES = _evo_key_types()


def _to_bool(x, where):
    s = str(x).strip().lower()
    if s in ("true", "yes", "1", "on", "y"):
        return True
    if s in ("false", "no", "0", "off", "n"):
        return False
    fail(f"{where}: '{x}' should be True or False.")


def parse_magmas(text):
    header, rows = read_table(text, "Magma table (cell 5)")
    if len(header) < 2:
        fail("Magma table (cell 5): the first line must be 'property | NAME1 | NAME2 | …' with at least one magma.")
    names = header[1:]
    if any(n == "" for n in names):
        fail("Magma table (cell 5): every magma column needs a name in the header line.")
    keys = [n.upper() for n in names]
    if len(set(keys)) != len(keys):
        fail("Magma table (cell 5): two magma columns have the same name.")
    canon = {k.lower(): k for k in EVO_KEY_TYPES}         # expert EVo keys first, so that the
    canon.update({o.lower(): o for o in OXIDES})            # friendly rows (e.g. fO2_buffer) win
    canon.update({k.lower(): k for k in MAGMA_ROWS})        # over EVo keys of the same spelling
    mags = {k: dict(name=n, chem={}, evo_overrides={}, given=set()) for k, n in zip(keys, names)}
    seen = set()
    for r in rows:
        prop_raw = r[0]
        prop = canon.get(prop_raw.lower())
        if prop is None:
            fail(f"Magma table (cell 5): unknown row '{prop_raw}'." +
                 _suggest(prop_raw, list(MAGMA_ROWS) + OXIDES + list(EVO_KEY_TYPES)) +
                 " The allowed rows are listed in the guide above cell 5.")
        if prop in seen:
            fail(f"Magma table (cell 5): the row '{prop}' appears twice.")
        seen.add(prop)
        if prop in _NOTEBOOK_SET:
            fail(f"Magma table (cell 5): '{prop}' is set by the notebook — use the rows evo_class / T_C "
                 "and the reservoir depth or P_MPa instead.")
        for k, cell in zip(keys, r[1:]):
            if cell in _BLANK:
                continue
            where = f"Magma table (cell 5), row '{prop}', magma '{mags[k]['name']}'"
            m = mags[k]
            if prop in OXIDES:
                m["chem"][prop] = to_num(cell, where, positive=True)
            elif prop in MAGMA_ROWS:
                kind = MAGMA_ROWS[prop][0]
                if kind == "num":
                    m[prop] = to_num(cell, where, positive=(prop not in ("fO2_value", "T_C")))
                elif kind == "choice":
                    opts = _CHOICES[prop]
                    hit = [o for o in opts if o.lower() == cell.lower()]
                    if not hit:
                        fail(f"{where}: '{cell}' is not one of {', '.join(opts)}." + _suggest(cell, opts))
                    m[prop] = hit[0]
                else:
                    m[prop] = cell
                m["given"].add(prop)
            else:                                     # an EVo environment key, for this magma only
                t = EVO_KEY_TYPES[prop]
                m["evo_overrides"][prop] = (_to_bool(cell, where) if t == "bool" else
                                            to_num(cell, where) if t == "num" else cell)
    # defaults and checks
    for k, m in mags.items():
        n = m["name"]
        for req in ("evo_class", "T_C", "beta_liquid", "rho_melt"):
            if req not in m:
                fail(f"Magma table (cell 5): magma '{n}' has no value in row '{req}'.")
        m.setdefault("evo_mode", "saturation")
        m.setdefault("gas_system", EVO_DEFAULTS["gas_system"])
        m.setdefault("fO2_buffer", EVO_DEFAULTS["fO2_buffer"])
        if "fO2_value" not in m:
            m["fO2_value"] = EVO_DEFAULTS["fO2_value"]
        if m.get("chem", {}).get("SIO2", 0) <= 0:
            fail(f"Magma table (cell 5): magma '{n}' needs SIO2.")
        fe, fe3 = m["chem"].get("FEO", 0) > 0, m["chem"].get("FE2O3", 0) > 0
        if not (fe or fe3):
            fail(f"Magma table (cell 5): magma '{n}' needs iron (FEO, and FE2O3 if fO2_buffer = iron).")
        if m["fO2_buffer"] == "iron" and not (fe and fe3):
            fail(f"Magma '{n}': fO2_buffer = iron derives fO2 from Fe³⁺/Fe²⁺, so both FEO and FE2O3 are needed.")
        if m["fO2_buffer"] != "iron" and fe3:
            fail(f"Magma '{n}': EVo accepts fO2 or the FeO/Fe2O3 split, not both. Either set fO2_buffer = iron "
                 "or leave FE2O3 empty (give all iron as FEO).")
        if m["fO2_buffer"] == "absolute" and m["fO2_value"] <= 0:
            fail(f"Magma '{n}': with fO2_buffer = absolute, fO2_value is the fO2 in bar and must be > 0.")
        mode = m["evo_mode"]
        fug = [f for f in ("fH2_bar", "fH2O_bar", "fCO2_bar") if f in m]
        if mode == "saturation":
            if "H2O_wt" not in m:
                fail(f"Magma '{n}' (saturation mode): give the total H2O_wt (0 is allowed).")
            if fug:
                fail(f"Magma '{n}': gas fugacities ({', '.join(fug)}) can only be used with evo_mode = gas_at_P; "
                     "the saturation search of EVo works from melt contents and fO2 only.")
        elif mode == "atomic":
            if "atomic_H_ppm" not in m:
                fail(f"Magma '{n}' (atomic mode): give atomic_H_ppm (and atomic_C_ppm / atomic_S_ppm / atomic_N_ppm).")
            if fug:
                fail(f"Magma '{n}': gas fugacities can only be used with evo_mode = gas_at_P.")
        elif mode == "gas_at_P":
            if m.get("gas_wt", 0) <= 0:
                fail(f"Magma '{n}' (gas_at_P mode): give gas_wt > 0, the vapour mass fraction (wt%) at the start pressure.")
            if "H2O_wt" in m and "fH2O_bar" in m:
                fail(f"Magma '{n}': give H₂O either as H2O_wt or as fH2O_bar, not both.")
            if "CO2_wt" in m and "fCO2_bar" in m:
                fail(f"Magma '{n}': give CO₂ either as CO2_wt or as fCO2_bar, not both.")
        if "graphite_wt" in m and ("CO2_wt" in m or "fCO2_bar" in m):
            fail(f"Magma '{n}': give carbon either as graphite_wt or as CO₂, not both.")
        m.setdefault("fallback_H2O_coeff", {"basalt": 0.36, "phonolite": 0.39, "rhyolite": 0.41}[m["evo_class"]])
        m.setdefault("fallback_CO2_coeff", {"basalt": 5.5, "phonolite": 4.5, "rhyolite": 3.8}[m["evo_class"]])
        m["gas_sys_resolved"] = resolve_gas_system(m)
    return mags


def resolve_gas_system(m):
    gs = m.get("gas_system", "auto")
    if gs != "auto":
        return gs
    mode = m["evo_mode"]
    if mode == "gas_at_P":
        return "COHSN" if "N_wt" in m else "COHS"
    if mode == "atomic":
        C, S, N = ("atomic_C_ppm" in m), ("atomic_S_ppm" in m), ("atomic_N_ppm" in m)
    else:
        C, S, N = ("CO2_wt" in m or "graphite_wt" in m), ("S_wt" in m), ("N_wt" in m)
    if N:
        return "COHSN"
    return {(False, False): "OH", (True, False): "COH", (False, True): "SOH", (True, True): "COHS"}[(C, S)]


# ----------------------------------------------------------------------------- reservoirs
SHAPES = {"mogi": "mogi", "point": "mogi", "sphere": "sphere", "mctigue": "sphere", "finite sphere": "sphere",
          "spheroid": "spheroid", "yang": "spheroid", "prolate": "spheroid", "prolate spheroid": "spheroid",
          "penny": "penny", "crack": "penny", "sill": "penny", "fialko": "penny", "penny crack": "penny"}
SHAPE_LABEL = {"mogi": "Mogi point sphere", "sphere": "finite sphere (McTigue)",
               "spheroid": "prolate spheroid (Yang)", "penny": "penny-shaped crack (Fialko)"}
RES_COLS = {"name": "text", "shape": "shape", "depth_km": "num", "V0_m3": "num", "Ve_m3": "list",
            "resident": "magma", "injected": "magma", "mu_GPa": "num", "nu": "num", "radius_km": "num",
            "aspect": "num", "dip_deg": "num", "strike_deg": "num", "P_depth_km": "num", "P_MPa": "num",
            "rho_crust": "num", "T_C": "num", "beta_c": "num"}
RES_REQUIRED = ["name", "shape", "depth_km", "V0_m3", "Ve_m3", "resident"]


def parse_reservoirs(text, mags):
    header, rows = read_table(text, "Reservoir table (cell 6)")
    canon = {k.lower(): k for k in RES_COLS}
    cols = []
    for h in header:
        c = canon.get(h.lower())
        if c is None:
            fail(f"Reservoir table (cell 6): unknown column '{h}'." + _suggest(h, RES_COLS) +
                 " Allowed columns: " + ", ".join(RES_COLS) + ".")
        cols.append(c)
    miss = [c for c in RES_REQUIRED if c not in cols]
    if miss:
        fail("Reservoir table (cell 6): missing column(s) " + ", ".join(miss) + ".")
    if not rows:
        fail("Reservoir table (cell 6): add at least one reservoir row.")
    out, names = [], set()
    for i, r in enumerate(rows, start=1):
        d = {}
        for c, cell in zip(cols, r):
            if cell in _BLANK:
                continue
            where = f"Reservoir table (cell 6), row {i} ('{r[cols.index('name')]}'), column '{c}'"
            kind = RES_COLS[c]
            if kind == "num":
                d[c] = to_num(cell, where, positive=(c not in ("strike_deg",)))
            elif kind == "list":
                vals = [to_num(v, where, positive=True, allow_zero=False) for v in re.split(r"[;\s]+", cell) if v]
                if not vals:
                    fail(f"{where}: give at least one injected volume.")
                d[c] = vals
            elif kind == "shape":
                s = SHAPES.get(cell.lower())
                if s is None:
                    fail(f"{where}: unknown shape '{cell}'. Use mogi, sphere, spheroid or penny.")
                d[c] = s
            elif kind == "magma":
                if cell.upper() not in mags:
                    fail(f"{where}: no magma named '{cell}' in the magma table." + _suggest(cell.upper(), mags))
                d[c] = cell.upper()
            else:
                d[c] = cell
        for c in RES_REQUIRED:
            if c not in d:
                fail(f"Reservoir table (cell 6), row {i}: the column '{c}' is empty.")
        n = d["name"]
        if n in names:
            fail(f"Reservoir table (cell 6): the name '{n}' is used twice; names must be unique.")
        names.add(n)
        if d["depth_km"] <= 0 or d["V0_m3"] <= 0:
            fail(f"Reservoir '{n}': depth_km and V0_m3 must be > 0.")
        d.setdefault("mu_GPa", PHYS["mu_GPa"])
        d.setdefault("nu", PHYS["nu"])
        if not (0 < d["nu"] < 0.5) or d["mu_GPa"] <= 0:
            fail(f"Reservoir '{n}': need mu_GPa > 0 and 0 < nu < 0.5.")
        sh = d["shape"]
        if sh == "penny" and d.get("radius_km", 0) <= 0:
            fail(f"Reservoir '{n}': a penny-shaped crack needs radius_km.")
        if sh == "sphere" and "radius_km" not in d:
            d["radius_km"] = (3 * d["V0_m3"] / (4 * np.pi)) ** (1 / 3) / 1e3
        if sh == "spheroid":
            if not (0 < d.get("aspect", 0) <= 1):
                fail(f"Reservoir '{n}': a prolate spheroid needs aspect = b/a with 0 < aspect ≤ 1.")
            d.setdefault("dip_deg", 90.0)
            d.setdefault("strike_deg", 0.0)
        d["row"] = i
        out.append(d)
    return out


def parse_step2_manual(text, reservoirs):
    header, rows = read_table(text, "Step-2 table (cell 8)")
    if not rows:
        return {}
    allowed = ["reservoir", "phi_vol%", "P_sat_MPa", "gas_wt%", "rho_melt", "beta_m_eff",
               "fH2O_bar", "fCO2_bar", "fSO2_bar", "fH2S_bar"]
    canon = {a.lower(): a for a in allowed}
    cols = []
    for h in header:
        if h.lower() not in canon:
            fail(f"Step-2 table (cell 8): unknown column '{h}'." + _suggest(h, allowed))
        cols.append(canon[h.lower()])
    if "reservoir" not in cols or "phi_vol%" not in cols:
        fail("Step-2 table (cell 8): the columns 'reservoir' and 'phi_vol%' are required.")
    names = {r["name"] for r in reservoirs}
    out = {}
    for r in rows:
        d = dict(zip(cols, r))
        n = d["reservoir"]
        if n not in names:
            fail(f"Step-2 table (cell 8): no reservoir named '{n}' in cell 6." + _suggest(n, names))
        rec = {}
        for c in cols[1:]:
            v = d.get(c, "")
            rec[c] = np.nan if v in _BLANK else to_num(v, f"Step-2 table (cell 8), '{n}', '{c}'", positive=True)
        if not np.isfinite(rec.get("phi_vol%", np.nan)) or rec["phi_vol%"] >= 100:
            fail(f"Step-2 table (cell 8), '{n}': phi_vol% must be a number between 0 and 100.")
        out[n] = rec
    return out


# =============================================================================
# STEP 1 · PRESSURE
# =============================================================================
def reservoir_pressure_MPa(row):
    if "P_MPa" in row:
        return row["P_MPa"]
    z = row.get("P_depth_km", row["depth_km"])
    return row.get("rho_crust", PHYS["rho_crust"]) * PHYS["g"] * z * 1e3 / 1e6


# =============================================================================
# STEP 2 · EVo (embedded, cell 2)
# =============================================================================
_EVO_CACHE = {}
_GAS_SPECIES = ("mH2O", "mH2", "mCO2", "mCO", "mCH4", "mSO2", "mH2S", "mS2", "mO2", "mN2")


def evo_inputs(m, P_MPa, T_C=None):
    """The EVo chemistry dict and environment dict for magma record `m` at P_MPa (and T_C).
    Returns chem, env and a list of notes about conversions made."""
    notes = []
    chem = {ox: m["chem"][ox] for ox in OXIDES if m["chem"].get(ox, 0) > 0}
    env = dict(EVO_ENV)
    T = m["T_C"] if T_C is None else T_C
    P_stop = (max(int(round(P_MPa * 10.0)), 1) if EVO_WRAP["P_STOP_integer_bar"] else max(P_MPa * 10.0, 1e-3))
    env.update(COMPOSITION=m["evo_class"], T_START=T + 273.15, P_STOP=float(P_stop),
               GAS_SYS=m["gas_sys_resolved"])
    mode = m["evo_mode"]
    # -- redox
    buf, val = m["fO2_buffer"], float(m["fO2_value"])
    if buf in ("FMQ", "NNO", "IW"):
        env.update(FO2_buffer_SET=True, FO2_buffer=buf, FO2_buffer_START=val, FO2_SET=False)
    elif buf == "absolute":
        if mode == "gas_at_P":
            env.update(FO2_buffer_SET=False, FO2_SET=True, FO2_START=val)
        else:          # EVo's saturation search does not converge with a fixed absolute fO2
            import evo_local.conversions as _cv
            d = float(_cv.fo2_2fmq(np.log10(val), T + 273.15, P_stop, env["FMQ_MODEL"]))
            env.update(FO2_buffer_SET=True, FO2_buffer="FMQ", FO2_buffer_START=d, FO2_SET=False)
            notes.append(f"absolute fO₂ {val:.3g} bar converted to FMQ{d:+.3f} at {T:g} °C and {P_stop / 10:g} MPa")
    else:              # iron: fO2 from FeO/Fe2O3
        env.update(FO2_buffer_SET=False, FO2_SET=False)
    # -- volatiles
    have = lambda k: k in m
    if mode == "saturation":
        env.update(FIND_SATURATION=True, ATOMIC_MASS_SET=False, SINGLE_STEP=False)
        env.update(WTH2O_SET=True, WTH2O_START=max(m["H2O_wt"], EVO_WRAP["H2O_floor_wt"]) / 100.0)
        if have("CO2_wt"):
            env.update(WTCO2_SET=True, WTCO2_START=max(m["CO2_wt"], EVO_WRAP["CO2_floor_wt"]) / 100.0)
        if have("S_wt"):
            env.update(SULFUR_SET=True, SULFUR_START=max(m["S_wt"], EVO_WRAP["S_floor_wt"]) / 100.0)
    elif mode == "atomic":
        env.update(FIND_SATURATION=False, ATOMIC_MASS_SET=True, SINGLE_STEP=False)
        for el in "HCSN":
            if have(f"atomic_{el}_ppm"):
                env[f"ATOMIC_{el}"] = float(m[f"atomic_{el}_ppm"])
    else:              # gas_at_P: known vapour fraction and initial conditions at P_START
        P0 = float(m["P_start_MPa"]) * 10.0 if have("P_start_MPa") else float(P_stop)
        if P0 < P_stop:
            notes.append(f"P_start ({P0 / 10:g} MPa) is below the reservoir pressure; EVo is run at the reservoir pressure")
            P0 = float(P_stop)
        env.update(FIND_SATURATION=False, ATOMIC_MASS_SET=False, SINGLE_STEP=(P0 == P_stop),
                   P_START=P0, WgT=float(m["gas_wt"]) / 100.0)
        if have("H2O_wt"):
            env.update(WTH2O_SET=True, WTH2O_START=m["H2O_wt"] / 100.0)
        if have("CO2_wt"):
            env.update(WTCO2_SET=True, WTCO2_START=m["CO2_wt"] / 100.0)
        if have("S_wt"):
            env.update(SULFUR_SET=True, SULFUR_START=m["S_wt"] / 100.0)
        for f, key in (("fH2_bar", "FH2"), ("fH2O_bar", "FH2O"), ("fCO2_bar", "FCO2")):
            if have(f):
                env.update({f"{key}_SET": True, f"{key}_START": float(m[f])})
    if have("N_wt"):
        env.update(NITROGEN_SET=True, NITROGEN_START=m["N_wt"] / 100.0)
    if have("graphite_wt"):
        env.update(GRAPHITE_SATURATED=True, GRAPHITE_START=m["graphite_wt"] / 100.0)
    env.update(m["evo_overrides"])            # expert rows of the magma table, last
    return chem, env, notes


def _evo_extras(last):
    sp = {k[1:]: float(last[k]) for k in _GAS_SPECIES if k in last.index}
    out = dict(gas_species=sp)
    for key, col in (("melt_H2O_wt", "H2O_melt"), ("melt_CO2_wt", "CO2_melt"), ("melt_S_wt", "Stot_melt")):
        if col in last.index:
            out[key] = float(last[col])
    return out


def run_evo_state(mkey, P_MPa, T_C=None, record=None):
    """Magmatic state of magma `mkey` (or of the given magma `record`) at P_MPa from the embedded EVo:
    phi, P_sat, fugacities, densities, plus the complete EVo output and the inputs EVo received."""
    m = MAGMA[mkey] if record is None else record
    T = m["T_C"] if T_C is None else T_C
    chem, env, conv = evo_inputs(m, P_MPa, T)
    key = json.dumps([m["name"], round(P_MPa, 6), chem, env, m["rho_melt"]], sort_keys=True, default=str)
    if key in _EVO_CACHE:
        return _EVO_CACHE[key]
    buf_col = env["FO2_buffer"]
    res = dict(comp=m["name"], P_MPa=P_MPa, T_C=T, phi=0.0, saturated=False, P_sat_MPa=np.nan,
               gas_molmass=np.nan, rho_melt=m["rho_melt"], rho_gas=np.nan, fO2_dFMQ=np.nan,
               fH2O=np.nan, fCO2=np.nan, fSO2=np.nan, fH2S=np.nan, fS2=np.nan, gas_wt=0.0,
               XH2O_gas=np.nan, XCO2_gas=np.nan, beta_m_evo=np.nan, engine="EVo", gas_species={},
               evo_chem=chem, evo_env=env, evo_df=None, evo_sat_row=None, evo_log="",
               evo_prompts=[], evo_error=None, evo_notes=conv)
    err = None
    for bump in [0.0] + list(EVO_WRAP["retry_bumps"]):
        e = dict(env, P_STOP=env["P_STOP"] + bump)
        if e.get("SINGLE_STEP"):
            e["P_START"] = e["P_STOP"]
        try:
            out = evo_run(chem, e)
        except (EVoStop, Exception) as ex:
            err = f"{type(ex).__name__}: {ex}"
            if isinstance(ex, EVoStop) or "validation error" in str(ex):
                break                       # an input problem: retrying cannot help
            continue
        res.update(evo_env=e, evo_df=out["df"], evo_sat_row=out["sat_row"], evo_log=out["log"],
                   evo_prompts=out["prompts"], P_sat_MPa=out["P_sat_bar"] / 10.0)
        if bump:
            res["engine"] = f"EVo (P_STOP+{bump:g} bar)"
        if out["saturated"]:
            df = out["df"]
            last = df.iloc[-1]
            prev = df.iloc[-2] if len(df) > 1 else last
            res.update(phi=float(last["Exsol_vol%"]) / 100.0, saturated=float(last["Exsol_vol%"]) > 0,
                       gas_molmass=float(last["mol_mass"]), rho_melt=float(last["rho_melt"]),
                       fO2_dFMQ=float(last[buf_col]), fH2O=float(last["fH2O"]), fCO2=float(last["fCO2"]),
                       fSO2=float(last["fSO2"]), fH2S=float(last["fH2S"]), fS2=float(last["fS2"]),
                       gas_wt=float(last["Gas_wt"]) / 100.0,
                       XH2O_gas=float(last["mH2O"]), XCO2_gas=float(last["mCO2"]))
            if e["FIND_SATURATION"] or e["ATOMIC_MASS_SET"]:
                res["P_sat_MPa"] = float(df["P"].iloc[0]) / 10.0
            res.update(_evo_extras(last))
            wg = res["gas_wt"]
            if wg > 0:
                inv_ = 1.0 / float(last["rho_bulk"]) - (1 - wg) / res["rho_melt"]
                if inv_ > 0:
                    res["rho_gas"] = wg / inv_
            if len(df) > 1 and last["P"] != prev["P"]:
                dP = (float(prev["P"]) - float(last["P"])) * 1e5
                res["beta_m_evo"] = (np.log(float(prev["rho_bulk"])) - np.log(float(last["rho_bulk"]))) / dP
        err = None
        break
    if err is not None:
        fb = simple_phi_model(m, P_MPa, T)
        fb.update(engine="fallback (EVo failed)", evo_chem=chem, evo_env=env, evo_error=err,
                  evo_df=None, evo_sat_row=None, evo_log="", evo_prompts=[], evo_notes=conv)
        res = fb
    _EVO_CACHE[key] = res
    return res


def simple_phi_model(m, P_MPa, T_C=None):
    """Analytical fallback: sqrt-law H2O solubility + Henry CO2, ideal-gas mixture (H2O, CO2 only)."""
    T_K = (m["T_C"] if T_C is None else T_C) + 273.15
    sw, kc = m["fallback_H2O_coeff"], m["fallback_CO2_coeff"]
    h2o, co2 = m.get("H2O_wt", 0.0), m.get("CO2_wt", 0.0)
    ex_h2o = max(h2o - sw * np.sqrt(P_MPa), 0.0) / 100.0
    ex_co2 = max(co2 - kc * P_MPa * 1e-4, 0.0) / 100.0
    wg = ex_h2o + ex_co2
    res = dict(comp=m["name"], P_MPa=P_MPa, T_C=T_K - 273.15, phi=0.0, saturated=wg > 0, P_sat_MPa=np.nan,
               gas_molmass=np.nan, rho_melt=m["rho_melt"], rho_gas=np.nan, fO2_dFMQ=np.nan, fH2O=np.nan,
               fCO2=np.nan, fSO2=np.nan, fH2S=np.nan, fS2=np.nan, gas_wt=wg, XH2O_gas=np.nan,
               XCO2_gas=np.nan, beta_m_evo=np.nan, engine="fallback", gas_species={})
    if wg <= 0:
        return res
    n_h2o, n_co2 = ex_h2o / 18.015e-3, ex_co2 / 44.01e-3
    M = (ex_h2o + ex_co2) / (n_h2o + n_co2)
    rho_g = P_MPa * 1e6 * M / (R_GAS * T_K)
    Vg, Vl = wg / rho_g, (1 - wg) / m["rho_melt"]
    res.update(phi=Vg / (Vg + Vl), gas_molmass=M, rho_gas=rho_g,
               XH2O_gas=n_h2o / (n_h2o + n_co2), XCO2_gas=n_co2 / (n_h2o + n_co2))
    return res


def manual_state(resname, mkey, P_MPa, T_C):
    r = STEP2_TABLE[resname]
    phi = r["phi_vol%"] / 100.0
    wg = r.get("gas_wt%", np.nan) / 100.0 if np.isfinite(r.get("gas_wt%", np.nan)) else np.nan
    rl = r.get("rho_melt", np.nan)
    rl = rl if np.isfinite(rl) else MAGMA[mkey]["rho_melt"]
    rho_g = (wg * (1 - phi) * rl / (phi * (1 - wg))) if (0 < phi < 1 and np.isfinite(wg) and 0 < wg < 1) else np.nan
    g_ = lambda k: r.get(k, np.nan)
    return dict(comp=MAGMA[mkey]["name"], P_MPa=P_MPa, T_C=T_C, phi=phi, saturated=phi > 0,
                P_sat_MPa=g_("P_sat_MPa"), gas_molmass=np.nan, rho_melt=rl, rho_gas=rho_g, fO2_dFMQ=np.nan,
                fH2O=g_("fH2O_bar"), fCO2=g_("fCO2_bar"), fSO2=g_("fSO2_bar"), fH2S=g_("fH2S_bar"), fS2=np.nan,
                gas_wt=wg if np.isfinite(wg) else 0.0, XH2O_gas=np.nan, XCO2_gas=np.nan,
                beta_m_evo=g_("beta_m_eff"), engine="manual (cell 8)", gas_species={})


def engine_state(mkey, P_MPa, T_C=None):
    if PHYS["use_evo"]:
        return run_evo_state(mkey, P_MPa, T_C)
    return simple_phi_model(MAGMA[mkey], P_MPa, T_C)


def step2_state(row, P_MPa):
    """Step 2 for the resident magma of a reservoir row (manual values of cell 8 if given)."""
    T = row.get("T_C", MAGMA[row["resident"]]["T_C"])
    if row["name"] in STEP2_TABLE:
        return manual_state(row["name"], row["resident"], P_MPa, T)
    return engine_state(row["resident"], P_MPa, T)


# =============================================================================
# STEPS 3-7 · COMPRESSIBILITIES AND DEFORMATION SOURCES (embedded dMODELS, cell 3)
# =============================================================================
def beta_gas(P_Pa):
    """Ideal-gas isothermal compressibility, 1/P (Rivalta & Segall 2008)."""
    return 1.0 / P_Pa


def beta_magma(phi, beta_vol, beta_liq):
    """beta_m = phi*beta_volatile + (1-phi)*beta_liquid  (step 3)."""
    return phi * beta_vol + (1.0 - phi) * beta_liq


import dmodels_local.sill as _sillmod


def _gauleg_fixed(x1, x2, N):
    z, w = np.polynomial.legendre.leggauss(int(N))
    return 0.5 * (x2 + x1) + 0.5 * (x2 - x1) * z, 0.5 * (x2 - x1) * w
_sillmod._gauleg = _gauleg_fixed          # numpy>=2 compatibility patch

from dmodels_local import Sill as _Sill, Spheroid as _Spheroid, Sphere2D as _Sphere2D


class SillFixed(_Sill):
    """Fialko et al. (2001) penny-shaped crack, numpy-2-safe displacement.

    The Hankel integral of the displacement is evaluated with composite Gauss-Legendre
    quadrature (300 panels x 8 nodes on 0 < csi < min(60, max(10, 45/h)), h = depth/radius)
    instead of dMODELS' fixed 82 nodes on 0 < csi < 60. Within one source depth both agree to
    ~1e-9; beyond it the 82-node rule aliases the Bessel kernel of small, deep cracks and
    produces spurious far-field oscillations. The crack volume change (hence beta_c) does not
    depend on these nodes."""
    PANELS = 300
    _GL8 = np.polynomial.legendre.leggauss(8)

    def _calc_base(self):
        from numpy import exp, sin, cos, pi, dot
        h = self._z0 / self._a
        cmax = min(60.0, max(10.0, 45.0 / h))
        edges = np.linspace(1e-08, cmax, self.PANELS + 1)
        mid, half = 0.5 * (edges[1:] + edges[:-1]), 0.5 * (edges[1:] - edges[:-1])
        z8, w8 = self._GL8
        csi = (mid[:, None] + half[:, None] * z8[None, :]).ravel()
        wcsi = (half[:, None] * w8[None, :]).ravel()
        self._csi_col = csi.reshape((-1, 1))
        self._wcsi_row = wcsi.reshape((1, -1))
        phi, psi, t, wt = _sillmod._psi_phi(h)                 # Fialko et al. (2001), eq. (26)
        wt_col, t_col = wt.reshape((-1, 1)), t.reshape((-1, 1))
        phi_col, psi_col = phi.reshape((-1, 1)), psi.reshape((-1, 1))
        csi_t = self._csi_col @ t.reshape((1, -1))
        PHI = sin(csi_t) @ (wt_col * phi_col)
        PSI = (sin(csi_t) / csi_t - cos(csi_t)) @ (wt_col * psi_col)
        a = h * self._csi_col                                  # eq. (24), as corrected in dMODELS
        self._Aarr = exp(-a) * (a * PSI + (1 + a) * PHI)
        self._Barr = exp(-a) * ((1 - a) * PSI - a * PHI)
        dV = -4 * pi * (1 - self._nu) * self._P_G * self._a ** 3. * dot(t_col.T, wt_col * phi_col)
        self._dV = dV[0][0]

    def calc_displ(self, x, y, z):
        """Displacement (u, v, w) at points (x, y, z); vectorised over the points."""
        from scipy.special import j0, j1
        if self._dV is None:
            self._calc_base()
        x = (np.array(x, ndmin=1, dtype=float) - self.x0) / self.a
        y = (np.array(y, ndmin=1, dtype=float) - self.y0) / self.a
        z = (np.array(z, ndmin=1, dtype=float) - self.z0) / self.a
        r = np.sqrt(x ** 2 + y ** 2)
        h = self.z0 / self.a
        csi, w = self._csi_col.ravel(), self._wcsi_row.ravel()
        A, B, nu_ = self._Aarr.ravel(), self._Barr.ravel(), self.nu
        Uz, Ur = np.zeros(r.size), np.zeros(r.size)
        for s0 in range(0, r.size, 500):                     # blocks of points, to bound memory
            sl = slice(s0, s0 + 500)
            czh = (z[sl, None] + h) * csi[None, :]
            rc = r[sl, None] * csi[None, :]
            sh, ch = np.sinh(czh), np.cosh(czh)
            Uz[sl] = (j0(rc) * (((1 - 2 * nu_) * B - czh * A) * sh
                                        + (2 * (1 - nu_) * A - czh * B) * ch)) @ w
            Ur[sl] = (j1(rc) * (((1 - 2 * nu_) * A + czh * B) * sh
                                        + (2 * (1 - nu_) * B + czh * A) * ch)) @ w
        rs = np.where(r == 0, 1.0, r)
        return (self.a * self.P_G * Ur * x / rs,
                self.a * self.P_G * Ur * y / rs,
                -self.a * self.P_G * Uz)


def mogi_uz(dVc, z0, r, nu):
    """Mogi (1958) point source, volume form: uz=(1-nu)dV z0 / (pi R^3)."""
    r = np.atleast_1d(np.asarray(r, dtype=float))
    return (1 - nu) * dVc / np.pi * z0 / (z0 ** 2 + r ** 2) ** 1.5


def beta_c_sphere(mu):
    """Chamber compressibility of a sphere: 3/(4 mu) (Segall 2010, eq. 8.25)."""
    return 3.0 / (4.0 * mu)


def spheroid_dV(a, aspect, dP, mu):
    """Cavity volume change of a pressurized prolate spheroid (Amoruso & Crescentini 2009):
        dV = pi a b^2 (dP/mu) [A^2/3 - 0.7A + 1.37],  A = b/a (0 < A <= 1)"""
    A = aspect
    return np.pi * a * (A * a) ** 2 * dP / mu * (A ** 2 / 3.0 - 0.7 * A + 1.37)


def yang_source(V0, aspect, z0, mu, nu, dip, strike, dP_ref=None):
    """Yang (1988)/Newman (2006) spheroid with volume V0 (m3)."""
    a = (3.0 * V0 / (4.0 * np.pi * aspect ** 2)) ** (1.0 / 3.0)
    dP = 1e-4 * mu if dP_ref is None else dP_ref
    sp = _Spheroid(0., 0., z0, a=a, asrat=aspect, P_G=dP / mu, mu=mu, nu=nu,
                   theta=min(dip, 89.99), phi=strike)
    return sp, a, dP


def beta_c_yang(V0, aspect, z0, mu, nu, dip, strike):
    """beta_c = (3/4mu) [A^2/3 - 0.7A + 1.37]  (sphere: 3/4mu at A=1)."""
    a = (3.0 * V0 / (4.0 * np.pi * aspect ** 2)) ** (1.0 / 3.0)
    return spheroid_dV(a, aspect, 1.0, mu) / V0


def penny_source(a, z0, mu, nu, dP_ref=None):
    dP = 1e-4 * mu if dP_ref is None else dP_ref
    return SillFixed(0., 0., z0, P_G=dP / mu, a=a, nu=nu), dP


def beta_c_penny(a, V0, z0, mu, nu):
    """dV/dP of a Fialko half-space crack, normalised by reservoir volume V0."""
    s, dP = penny_source(a, z0, mu, nu)
    return s.dV / (V0 * dP)


def rV_factor(beta_m, beta_c):
    return 1.0 + beta_m / beta_c


def uplift_mogi(dVc, z0, dists, nu):
    return mogi_uz(dVc, z0, np.asarray(dists), nu)


def uplift_sphere(dVc, a, z0, mu, nu, dists):
    """McTigue (1987) finite sphere (dMODELS mctigue2D), scaled to the cavity volume change."""
    dP = 1e-4 * mu
    s = _Sphere2D(0., 0., z0, P_G=dP / mu, a=a, nu=nu)
    d = np.asarray(dists, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        w = np.ravel(s.calc_displ(d, np.zeros_like(d))[2])
    return w * dVc / (np.pi * dP / mu * a ** 3)


def uplift_yang(dVc, V0, aspect, z0, mu, nu, dip, strike, dists, azimuth=0.0):
    sp, a, dP = yang_source(V0, aspect, z0, mu, nu, dip, strike)
    scale = dVc / spheroid_dV(a, aspect, dP, mu)   # true cavity dV (A&C 2009)
    az = np.deg2rad(azimuth)
    d = np.asarray(dists, dtype=float)
    x, y = d * np.sin(az), d * np.cos(az)     # az measured from North
    u, v, w = sp.calc_displ(x, y, np.zeros_like(d))
    return w * scale


def uplift_penny(dVc, a, z0, mu, nu, dists):
    s, dP = penny_source(a, z0, mu, nu)
    scale = dVc / s.dV
    d = np.asarray(dists, dtype=float)
    u, v, w = s.calc_displ(d, np.zeros_like(d), np.zeros_like(d))
    return w * scale


def chamber_beta(row):
    """beta_c of a reservoir row from its shape (or the beta_c column if given)."""
    if "beta_c" in row:
        return row["beta_c"]
    mu, nu, d_m = row["mu_GPa"] * 1e9, row["nu"], row["depth_km"] * 1e3
    sh = row["shape"]
    if sh in ("mogi", "sphere"):
        return beta_c_sphere(mu)
    if sh == "spheroid":
        return beta_c_yang(row["V0_m3"], row["aspect"], d_m, mu, nu, row["dip_deg"], row["strike_deg"])
    return beta_c_penny(row["radius_km"] * 1e3, row["V0_m3"], d_m, mu, nu)


def uplift_row(row, dVc, dists, azimuth=None):
    """Vertical surface displacement (m) of reservoir row `row` for a cavity volume change dVc."""
    az = PHYS["profile_azimuth"] if azimuth is None else azimuth
    mu, nu, d_m = row["mu_GPa"] * 1e9, row["nu"], row["depth_km"] * 1e3
    sh = row["shape"]
    if sh == "mogi":
        return uplift_mogi(dVc, d_m, dists, nu)
    if sh == "sphere":
        return uplift_sphere(dVc, row["radius_km"] * 1e3, d_m, mu, nu, dists)
    if sh == "spheroid":
        return uplift_yang(dVc, row["V0_m3"], row["aspect"], d_m, mu, nu, row["dip_deg"],
                           row["strike_deg"], dists, az)
    return uplift_penny(dVc, row["radius_km"] * 1e3, d_m, mu, nu, dists)


def uplift_map(row, dVc, X, Y):
    """Uplift (m) on a map grid (metres east/north of the source centre)."""
    mu, nu, d_m = row["mu_GPa"] * 1e9, row["nu"], row["depth_km"] * 1e3
    sh = row["shape"]
    if sh == "spheroid":
        sp, a, dP = yang_source(row["V0_m3"], row["aspect"], d_m, mu, nu, row["dip_deg"], row["strike_deg"])
        w = sp.calc_displ(X.ravel(), Y.ravel(), np.zeros(X.size))[2] * dVc / spheroid_dV(a, row["aspect"], dP, mu)
        return np.asarray(w).reshape(X.shape)
    R = np.hypot(X, Y)                                      # axisymmetric sources
    rr = np.linspace(0, R.max(), 400)
    return np.interp(R, rr, uplift_row(row, dVc, rr))


section("Calculation engine ready")
note("Step 2 runs EVo (cell 2) with each magma's column of cell 5 and the shared settings of cell 7; "
     "steps 3–7 use closed-form compressibilities and the dMODELS sources (cell 3): Mogi point sphere, "
     "McTigue finite sphere, Yang prolate spheroid and Fialko penny-shaped crack." +
     ("" if PHYS["use_evo"] else " <b>The analytical fallback is selected in cell 4.</b>"),
     "ok" if PHYS["use_evo"] else "warn")

In [ ]:
#@title 10. Input check — every magma and reservoir as it will be calculated  { display-mode: "form" }
#@markdown Reads the tables of cells 5, 6 and 8. A missing or wrong entry stops here with a message naming the table, the magma or reservoir and the row or column.
MAGMA = parse_magmas(MAGMA_GRID.as_table())
RES_ROWS = parse_reservoirs(RESERVOIR_GRID.as_table(), MAGMA)
STEP2_TABLE = parse_step2_manual(STEP2_GRID.as_table(), RES_ROWS)
_EVO_CACHE.clear()


def _refkey(k):
    m = re.match(r"^(\D+)(\d{4})$", k)
    return f"{m.group(1)} {m.group(2)}" if m else k


_start = MAGMA_GRID.synced_choice
_ex = EXAMPLES.get(_start) if RESERVOIR_GRID.synced_choice == _start else None
if _ex is not None:
    _edited = MAGMA_GRID.edited or RESERVOIR_GRID.edited
    section(f"Start values: {html.escape(_start)}")
    note(html.escape(_ex["summary"]) + "<br><br><b>Depths and pressures.</b> " + html.escape(_ex["frame"]) +
         ("<br><br><b>You have changed some values in the tables</b>; everything below uses your values." if _edited else ""))
    if _ex["sources"]:
        _src = _ex["sources"]
        _parts = [table_html(["Quantity", "Source", "Note"], [[q, _refkey(r), html.escape(n)] for q, r, n in _d["rows"]],
                             title=f"Magma {html.escape(_m)} — {html.escape(_d['label'])}", left_cols=(0, 1, 2))
                  for _m, _d in _src["magmas"].items()]
        _rows = [[html.escape(_r) if i == 0 else "", _refkey(r), html.escape(n)]
                 for _r, _items in _src["reservoirs"].items() for i, (r, n) in enumerate(_items)]
        _parts.append(table_html(["Reservoir", "Source", "Basis"], _rows, title="Reservoirs", left_cols=(0, 1, 2)))
        _parts.append(table_html(["Key", "Reference"], [[k, html.escape(v)] for k, v in _src["refs"].items()],
                                 title="References", left_cols=(0, 1),
                                 caption="“—” marks an assumption or a value derived from the cited ones (see the note)."))
        _show("<details><summary style='cursor:pointer;font-weight:700;color:#1F4E79;font-size:13px'>"
              "Where every start value comes from (click to open)</summary>" + "".join(_parts) + "</details>")
    else:
        note("These are the inputs of the thesis (Scenario A); their sources are given in the thesis (Chapter 4) and in "
             "the last row of the magma table.")
    if _start.startswith("Kīlauea"):
        note("CO₂-rich magmas take EVo longer (long decompression paths): this volcano needs about one minute.")

# ----------------------------------------------------------------------------- magmas
section(f"Magmas ({len(MAGMA)})")
_mk = list(MAGMA)
_lab = {"SIO2": "SiO<sub>2</sub>", "TIO2": "TiO<sub>2</sub>", "AL2O3": "Al<sub>2</sub>O<sub>3</sub>", "FEO": "FeO",
        "FE2O3": "Fe<sub>2</sub>O<sub>3</sub>", "MNO": "MnO", "MGO": "MgO", "CAO": "CaO", "NA2O": "Na<sub>2</sub>O",
        "K2O": "K<sub>2</sub>O", "P2O5": "P<sub>2</sub>O<sub>5</sub>", "NIO": "NiO", "CR2O3": "Cr<sub>2</sub>O<sub>3</sub>"}

def _redox(m):
    b, v = m["fO2_buffer"], m["fO2_value"]
    return {"absolute": f"fO<sub>2</sub> = {v:.3g} bar", "iron": "from FeO / Fe<sub>2</sub>O<sub>3</sub>"}.get(b, f"{b} {v:+g}")

def _vols(m):
    mode = m["evo_mode"]
    if mode == "atomic":
        return ", ".join(f"{el} {m[f'atomic_{el}_ppm']:g} ppm" for el in "HCSN" if f"atomic_{el}_ppm" in m)
    parts = [f"{lab} {m[k]:g} wt%" for k, lab in (("H2O_wt", "H<sub>2</sub>O"), ("CO2_wt", "CO<sub>2</sub>"),
             ("S_wt", "S"), ("N_wt", "N"), ("graphite_wt", "graphite")) if k in m]
    parts += [f"{lab} {m[k]:g} bar" for k, lab in (("fH2_bar", "fH<sub>2</sub>"), ("fH2O_bar", "fH<sub>2</sub>O"),
              ("fCO2_bar", "fCO<sub>2</sub>")) if k in m]
    if mode == "gas_at_P":
        parts.insert(0, f"vapour {m['gas_wt']:g} wt%" + (f" at {m['P_start_MPa']:g} MPa" if "P_start_MPa" in m else
                                                           " at the reservoir pressure"))
    return ", ".join(parts)

_rows = [["EVo class (solubility fit)"] + [MAGMA[k]["evo_class"] for k in _mk],
         ["Temperature (°C)"] + [f"{MAGMA[k]['T_C']:g}" for k in _mk],
         ["β<sub>liquid</sub> (Pa<sup>−1</sup>) / ρ<sub>melt</sub> (kg m<sup>−3</sup>)"] +
         [f"{sci(MAGMA[k]['beta_liquid'])} / {MAGMA[k]['rho_melt']:g}" for k in _mk],
         ["EVo mode"] + [MAGMA[k]["evo_mode"] for k in _mk],
         ["Gas system"] + [MAGMA[k]["gas_sys_resolved"] + (" (auto)" if MAGMA[k]["gas_system"] == "auto" else "") for k in _mk],
         ["Volatiles / fugacities"] + [_vols(MAGMA[k]) for k in _mk],
         ["Redox"] + [_redox(MAGMA[k]) for k in _mk],
         ["Expert EVo overrides"] + [", ".join(f"{a}={b}" for a, b in MAGMA[k]["evo_overrides"].items()) or "—" for k in _mk]]
_grp = len(_rows)
_rows.append(["Major elements (wt%)"])
_rows += [[_lab[o]] + [f"{MAGMA[k]['chem'][o]:.2f}" if MAGMA[k]["chem"].get(o) else "—" for k in _mk]
          for o in OXIDES if any(MAGMA[k]["chem"].get(o) for k in _mk)]
_rows.append(["Total"] + [f"{sum(MAGMA[k]['chem'].values()):.2f}" for k in _mk])
_rows.append(["Source"] + [html.escape(MAGMA[k].get("source", "—")) for k in _mk])
_RANGE = {"basalt": (45, 55), "phonolite": (52, 63), "rhyolite": (65, 80)}
_mis = [k for k in _mk if not (_RANGE[MAGMA[k]["evo_class"]][0] < MAGMA[k]["chem"]["SIO2"] < _RANGE[MAGMA[k]["evo_class"]][1])]
table(["Property"] + [MAGMA[k]["name"] for k in _mk], _rows, group_rows={_grp},
      caption="Everything EVo will receive for each magma (cell 5), after the defaults of cells 4 and 7 are filled in. "
              "The EVo input files actually used for every reservoir state are listed in full under step 2.")
if _mis:
    note("SiO<sub>2</sub> lies outside the range of the selected EVo class for: " +
         ", ".join(MAGMA[k]["name"] for k in _mis) +
         " (basalt 45–55, phonolite 52–63, rhyolite 65–80 wt%). EVo warns and continues; the warning is listed under step 2.", "warn")

# ----------------------------------------------------------------------------- reservoirs
section(f"Reservoirs ({len(RES_ROWS)})")
_warn = []
_rows = []
for r in RES_ROWS:
    P = reservoir_pressure_MPa(r)
    geo = {"mogi": lambda: "—",
           "sphere": lambda: f"radius {r['radius_km']:.3g} km",
           "spheroid": lambda: f"b/a {r['aspect']:g}, dip {r['dip_deg']:g}°, azimuth {r['strike_deg']:g}°",
           "penny": lambda: f"radius {r['radius_km']:g} km"}[r["shape"]]()
    Psrc = ("given" if "P_MPa" in r else
            f"ρgz, z = {r.get('P_depth_km', r['depth_km']):g} km" +
            (f", ρ = {r['rho_crust']:g}" if "rho_crust" in r else ""))
    _rows.append([html.escape(r["name"]), SHAPE_LABEL[r["shape"]], f"{r['depth_km']:g}", geo, sci(r["V0_m3"]),
                  "; ".join(sci(v, 1) for v in r["Ve_m3"]), MAGMA[r["resident"]]["name"],
                  MAGMA[r["injected"]]["name"] if "injected" in r else "—",
                  f"{r['mu_GPa']:g} / {r['nu']:g}", f"{P:.2f} ({Psrc})",
                  f"{r.get('T_C', MAGMA[r['resident']]['T_C']):g}",
                  "manual (cell 8)" if r["name"] in STEP2_TABLE else ("EVo" if PHYS["use_evo"] else "fallback")])
    d = r["depth_km"]
    if r["shape"] == "sphere" and r["radius_km"] / d > 0.5:
        _warn.append(f"{r['name']}: sphere radius/depth = {r['radius_km'] / d:.2f} &gt; 0.5 — beyond the accuracy of McTigue's solution.")
    if r["shape"] == "spheroid":
        a = (3 * r["V0_m3"] / (4 * np.pi * r["aspect"] ** 2)) ** (1 / 3) / 1e3
        top = d - a * np.sin(np.deg2rad(min(r["dip_deg"], 89.99)))
        if top <= 0:
            _warn.append(f"{r['name']}: the spheroid (semi-major axis {a:.2f} km) reaches the surface (top at {top:.2f} km).")
    if r["shape"] == "penny" and r["radius_km"] / d >= 1:
        _warn.append(f"{r['name']}: crack radius ≥ depth — the limit of validity of the half-space crack solution.")
    if "beta_c" in r:
        _warn.append(f"{r['name']}: β<sub>c</sub> = {sci(r['beta_c'])} Pa<sup>−1</sup> is taken from the table instead of the shape.")
table(["Reservoir", "Shape", "Depth (km)", "Geometry", "V<sub>0</sub> (m³)", "V<sub>e</sub> (m³)", "Resident",
       "Injected", "μ (GPa) / ν", "P (MPa)", "T (°C)", "Step 2"], _rows, left_cols=(0, 1, 3, 9))
if _warn:
    note("<b>Geometry notes</b><br>" + "<br>".join(_warn), "warn")
if STEP2_TABLE:
    note("Step 2 is taken from cell 8 (typed by hand) for: " + ", ".join(html.escape(n) for n in STEP2_TABLE) + ".", "warn")
note(f"{len(RES_ROWS)} reservoir configurations, {sum(len(r['Ve_m3']) for r in RES_ROWS)} injected volumes, "
     f"{len(MAGMA)} magmas. All inputs read without errors.", "ok")

In [ ]:
#@title 11. Steps 1–6 — pressure, vapour fraction, compressibilities, r_V and chamber volume change  { display-mode: "form" }

def compute_row(row):
    """Apply the 7-step workflow to one reservoir row; return a result dict."""
    d_m = row["depth_km"] * 1e3
    P_MPa = reservoir_pressure_MPa(row)                              # (1)
    P_Pa = P_MPa * 1e6
    st = step2_state(row, P_MPa)                                     # (2)
    phi = st["phi"]
    b_liq = MAGMA[row["resident"]]["beta_liquid"]
    b_gas = beta_gas(P_Pa)
    b_m = beta_magma(phi, b_gas, b_liq)                              # (3)
    b_c = chamber_beta(row)                                          # (4)
    rV = rV_factor(b_m, b_c)                                         # (5)
    mu, nu = row["mu_GPa"] * 1e9, row["nu"]
    out = dict(name=row["name"], shape=row["shape"], radius_km=row.get("radius_km", np.nan),
               aspect=row.get("aspect", np.nan), depth_km=row["depth_km"],
               chamber=MAGMA[row["resident"]]["name"],
               intrusion=MAGMA[row["injected"]]["name"] if "injected" in row else "—",
               V0_m3=row["V0_m3"], P_MPa=P_MPa, T_C=st["T_C"], engine=st["engine"],
               saturated=st["saturated"], P_sat_MPa=st["P_sat_MPa"],
               phi=phi, XH2O_gas=st["XH2O_gas"], XCO2_gas=st["XCO2_gas"],
               fO2_dFMQ=st["fO2_dFMQ"], fH2O_bar=st["fH2O"],
               fCO2_bar=st["fCO2"], fSO2_bar=st["fSO2"],
               rho_melt=st["rho_melt"],
               beta_liquid=b_liq, beta_gas=b_gas, beta_m=b_m,
               beta_m_evo_effective=st["beta_m_evo"],
               beta_c=b_c, rV=rV, mu_Pa=mu, nu=nu,
               uz0_per_dVc_mm_per_m3=float(uplift_row(row, 1.0, [0.0])[0]) * 1e3,
               n_Ve=len(row["Ve_m3"]))
    for k, Ve in enumerate(row["Ve_m3"], start=1):
        dVc = Ve / rV                                                # (6)
        uz = uplift_row(row, dVc, DISTS_M)                           # (7)
        out[f"Ve{k}_m3"] = Ve
        out[f"dVc{k}_m3"] = dVc
        for dkm, w in zip(OBS_KM, uz):
            out[f"uz{k}_{dkm:g}km_mm"] = w * 1e3
    return out

ROW_BY_NAME = {r["name"]: r for r in RES_ROWS}
res = pd.DataFrame([compute_row(r) for r in RES_ROWS])

def ve_list(r):
    return [r[f"Ve{k}_m3"] for k in range(1, int(r["n_Ve"]) + 1)]

# =========================================================== STEPS 1-2
section("Steps 1–2 · Pressure and magmatic state of every reservoir")
STEP2_STATES = {}      # unique states (role, magma, P, T) -> {labels, role, comp, P, T, st}
_rows = []
for row in RES_ROWS:
    P = reservoir_pressure_MPa(row)
    st_r = step2_state(row, P)
    T_r = st_r["T_C"]
    k = ("resident", row["resident"], round(P, 6), T_r, st_r["engine"])
    STEP2_STATES.setdefault(k, dict(labels=[], role="resident", comp=MAGMA[row["resident"]]["name"],
                                    P=P, T=T_r, st=st_r))["labels"].append(row["name"])
    cells = [html.escape(row["name"]), f"{row['depth_km']:g}", f"{P:.2f}", MAGMA[row["resident"]]["name"], f"{T_r:g}",
             num(st_r["P_sat_MPa"], 2), "yes" if st_r["saturated"] else "no", num(100 * st_r["phi"], 3),
             num(100 * st_r.get("gas_wt", 0.0), 3), st_r["engine"]]
    if "injected" in row:
        st_i = engine_state(row["injected"], P)
        k = ("injected", row["injected"], round(P, 6), st_i["T_C"], st_i["engine"])
        STEP2_STATES.setdefault(k, dict(labels=[], role="injected", comp=MAGMA[row["injected"]]["name"],
                                        P=P, T=st_i["T_C"], st=st_i))["labels"].append(row["name"])
        cells += [MAGMA[row["injected"]]["name"], num(st_i["P_sat_MPa"], 2), "yes" if st_i["saturated"] else "no",
                  num(100 * st_i["phi"], 3)]
    else:
        cells += ["—", "—", "—", "—"]
    _rows.append(cells)
table(["Reservoir", "Depth (km)", "P (MPa)", "Resident", "T (°C)", "P<sub>sat</sub> (MPa)", "Saturated?",
       "φ (vol%)", "Gas (wt%)", "Step-2 source", "Injected", "P<sub>sat</sub> (MPa)", "Saturated?", "φ (vol%)"], _rows,
      classes=lambda i, j: (("good" if _rows[i][j] == "yes" else None) if j in (6, 12) else
                            ("hi" if (j == 9 and not _rows[i][9].startswith("EVo")) else None)),
      caption="P = ρ<sub>c</sub> g z (Eq. 3.1) unless given. P<sub>sat</sub> is the pressure at which the magma's "
              "volatile budget just saturates; below it a vapour phase exists. β<sub>m</sub> uses the <b>resident</b> "
              "magma; the injected magma is listed for reference. The complete EVo output of every state is in cell 12.")

for k, s in STEP2_STATES.items():
    st = s["st"]
    if s["role"] != "resident" or not st["saturated"] or not st.get("gas_species"):
        continue
    sp = st["gas_species"]
    order = [("H2O", "H<sub>2</sub>O", "fH2O"), ("CO2", "CO<sub>2</sub>", "fCO2"), ("SO2", "SO<sub>2</sub>", "fSO2"),
             ("H2S", "H<sub>2</sub>S", "fH2S"), ("H2", "H<sub>2</sub>", None), ("S2", "S<sub>2</sub>", "fS2"),
             ("CO", "CO", None), ("CH4", "CH<sub>4</sub>", None), ("N2", "N<sub>2</sub>", None)]
    rows = [[name, num(100 * sp[kk], 3) if kk in sp else "—",
             num(st.get(f), 4) if f and np.isfinite(st.get(f, np.nan)) else "—"]
            for kk, name, f in order if (kk in sp and sp[kk] > 0) or (f and np.isfinite(st.get(f, np.nan)))]
    table(["Gas species", "mol%", "Fugacity (bar)"], rows,
          title=f"Equilibrium vapour of {s['comp']} at {s['P']:.2f} MPa — " + ", ".join(html.escape(x) for x in s["labels"]),
          caption=(f"Mean molar mass {num(1000 * st['gas_molmass'], 1)} g mol<sup>−1</sup>; "
                   f"ρ<sub>gas</sub> = {num(st['rho_gas'], 1)} kg m<sup>−3</sup>; ρ<sub>melt</sub> = {num(st['rho_melt'], 0)} kg m<sup>−3</sup>; "
                   f"log fO<sub>2</sub> = {st.get('evo_env', {}).get('FO2_buffer', 'FMQ')} {num(st['fO2_dFMQ'], 2)}. Residual dissolved: "
                   f"H<sub>2</sub>O {num(st.get('melt_H2O_wt', np.nan), 3)} wt%, "
                   f"CO<sub>2</sub> {num(1e4 * st.get('melt_CO2_wt', np.nan), 1)} ppm, "
                   f"S {num(1e4 * st.get('melt_S_wt', np.nan), 0)} ppm."))

_und = sorted({lab for s in STEP2_STATES.values() if s["role"] == "resident" and not s["st"]["saturated"]
               for lab in s["labels"]}, key=[r["name"] for r in RES_ROWS].index)
if _und:
    note("Resident magma undersaturated (φ = 0, so β<sub>m</sub> = β<sub>liquid</sub>) in: " +
         ", ".join(html.escape(x) for x in _und) + ".")
_failed = [f"{', '.join(s['labels'])} / {s['comp']}: {html.escape(str(s['st'].get('evo_error')))}"
           for s in STEP2_STATES.values() if s["st"].get("evo_error")]
if _failed:
    note("<b>EVo could not run these states</b> (analytical fallback used — check the magma's EVo settings):<br>" +
         "<br>".join(_failed), "warn")
_conv = sorted({f"{s['comp']}: {c}" for s in STEP2_STATES.values() for c in s["st"].get("evo_notes", [])})
if _conv:
    note("<b>Conversions made for EVo:</b><br>" + "<br>".join(_conv))

# =========================================================== STEPS 3-5
section("Steps 3–5 · Magma compressibility, chamber compressibility and r<sub>V</sub>")
rows = []
for _, r in res.iterrows():
    rows.append([html.escape(r["name"]), SHAPE_LABEL[r["shape"]], f"{r.depth_km:g}", f"{r.P_MPa:.2f}",
                 num(100 * r.phi, 2), sci(r.beta_liquid), sci(r.beta_gas), sci(r.beta_m), sci(r.beta_c),
                 f"{r.beta_c * r.mu_Pa:.3f}", f"{r.rV:.3f}", f"{100 / r.rV:.1f}%"])
table(["Reservoir", "Shape", "z (km)", "P (MPa)", "φ (vol%)", "β<sub>liquid</sub>", "β<sub>gas</sub> = 1/P",
       "β<sub>m</sub>", "β<sub>c</sub>", "β<sub>c</sub>·μ", "r<sub>V</sub>", "Expressed 1/r<sub>V</sub>"], rows,
      classes=lambda i, j: "hi" if (j in (7, 10) and res.iloc[i].phi > 0) else None, left_cols=(0, 1),
      caption="β<sub>m</sub> = φβ<sub>gas</sub> + (1−φ)β<sub>liquid</sub> (Eq. 3.12); β<sub>c</sub> from the reservoir shape "
              "(sphere 3/4μ; prolate spheroid, Amoruso & Crescentini 2009; penny crack, Fialko et al. 2001, half-space); "
              "r<sub>V</sub> = 1 + β<sub>m</sub>/β<sub>c</sub> (Eq. 3.22). All compressibilities in Pa<sup>−1</sup>. "
              "Highlighted: gas-bearing resident magma.")
for _, r in res[res.phi > 0].iterrows():
    if np.isfinite(r.beta_m_evo_effective):
        rv_eq = 1 + r.beta_m_evo_effective / r.beta_c
        note(f"<b>Frozen versus equilibrium compressibility ({html.escape(r['name'])}).</b> The frozen-phase mixture gives "
             f"β<sub>m</sub> = {sci(r.beta_m)} Pa<sup>−1</sup>; differencing the bulk density along the EVo equilibrium path "
             f"gives β<sub>m</sub><sup>eff</sup> = {sci(r.beta_m_evo_effective)} Pa<sup>−1</sup> "
             f"({r.beta_m_evo_effective / r.beta_m:.1f}×). With it r<sub>V</sub> would be {rv_eq:.1f} instead of "
             f"{r.rV:.1f}: an upper bound for slow pressurisation.")

# =========================================================== STEP 6
section("Step 6 · Reservoir volume change ΔV<sub>c</sub> = V<sub>e</sub> / r<sub>V</sub>")
rows = []
for _, r in res.iterrows():
    rows.append([html.escape(r["name"]), f"{r.depth_km:g}", f"{r.rV:.3f}",
                 "<br>".join(f"V{k}: {sci(r[f'Ve{k}_m3'], 1)} → {sci(r[f'dVc{k}_m3'], 2)}"
                             for k in range(1, int(r.n_Ve) + 1)), f"{100 / r.rV:.1f}%"])
table(["Reservoir", "z (km)", "r<sub>V</sub>", "V<sub>e</sub> → ΔV<sub>c</sub> (m³)", "Fraction expressed"], rows,
      left_cols=(0, 3),
      caption="Each line: injected volume → resulting cavity volume change (the part of the injection seen by geodesy).")

In [ ]:
#@title 12. Step 2 in detail — the complete EVo input and output of every magma state  { display-mode: "form" }
#@markdown Everything EVo computes is shown here: the chemistry and environment it received, all 58 output quantities at the reservoir pressure and at the saturation pressure, and optionally the full decompression path.
show_decompression_paths = False  #@param {type:"boolean"}
show_evo_messages = False         #@param {type:"boolean"}

EVO_OUTPUT_GROUPS = [
    ("System state", [
        ("P", "pressure", "bar"), ("Δbuffer", "log fO<sub>2</sub> relative to the run's buffer (FMQ, NNO or IW)", "log units"),
        ("fo2", "absolute oxygen fugacity", "bar"), ("F", "Fe<sub>2</sub>O<sub>3</sub>/FeO mole ratio of the melt", "—"),
        ("rho_bulk", "bulk density, melt + vapour", "kg m<sup>−3</sup>"), ("rho_melt", "melt density", "kg m<sup>−3</sup>"),
        ("Exsol_vol%", "vapour volume fraction φ", "vol%"), ("Gas_wt", "vapour mass fraction", "wt%"),
        ("mol_mass", "mean molar mass of the vapour", "kg mol<sup>−1</sup>")]),
    ("Vapour composition — mole fractions", [(f"m{s}", f"{s} in the vapour", "mol frac.") for s in
        ("H2O", "H2", "O2", "CO2", "CO", "CH4", "SO2", "H2S", "S2", "N2")]),
    ("Vapour composition — mass fractions", [(f"w{s}", f"{s} in the vapour", "mass frac.") for s in
        ("H2O", "H2", "O2", "CO2", "CO", "CH4", "SO2", "H2S", "S2", "N2")]),
    ("Fugacities", [(f"f{s}", f"fugacity of {s}", "bar") for s in
        ("H2O", "H2", "CO2", "CO", "CH4", "SO2", "H2S", "S2", "N2")]),
    ("Molar ratios in the vapour", [(c, c, "—") for c in ("mCO2/CO", "mCO2/H2O", "mCO2/SO2", "mH2S/SO2")]),
    ("Dissolved in the melt", [("H2O_melt", "H<sub>2</sub>O", "wt%"), ("H2_melt", "H<sub>2</sub>", "wt%"),
        ("CO2_melt", "CO<sub>2</sub> (as carbonate)", "wt%"), ("CO_melt", "CO", "wt%"), ("CH4_melt", "CH<sub>4</sub>", "wt%"),
        ("graph_melt", "graphite", "wt%"), ("S2-_melt", "sulfide S<sup>2−</sup>", "wt%"),
        ("S6+_melt", "sulfate S<sup>6+</sup>", "wt%"), ("Stot_melt", "total S", "wt%"), ("N_melt", "N", "wt%")]),
    ("Element totals of the system (mass balance)", [
        ("tot_H", "H", "ppm"), ("tot_C", "C", "ppm"), ("tot_O_gas", "O in volatile species", "ppm"),
        ("tot_O", "O including melt Fe oxides", "ppm"), ("tot_S", "S", "ppm"), ("tot_N", "N", "ppm")]),
]

def _short(labels, role, comp):
    lab = labels[0] + (f" +{len(labels) - 1}" if len(labels) > 1 else "")
    return f"{html.escape(lab)}<br><i>{role}</i> {html.escape(comp)}"

def evo_output_table(cols, title, caption):
    """cols = [(header, pandas Series or None)] -> one wide table with every EVo output quantity."""
    rows, groups = [], []
    for gname, items in EVO_OUTPUT_GROUPS:
        groups.append(len(rows))
        rows.append([gname])
        for key, desc, unit in items:
            vals = []
            for _, s in cols:
                k = next((b for b in ("FMQ", "NNO", "IW") if b in s.index), None) if key == "Δbuffer" else key
                vals.append((f"{gnum(s[k])} ({k})" if key == "Δbuffer" else gnum(s[k]))
                            if (s is not None and k is not None and k in s.index) else "—")
            rows.append([f"<code>{html.escape(key)}</code>", desc, unit] + vals)
    table(["EVo column", "Quantity", "Unit"] + [h for h, _ in cols], rows, title=title, caption=caption,
          left_cols=(0, 1), group_rows=set(groups))

_ran = lambda st: bool(st.get("evo_chem")) and not st.get("evo_error")
_states = [(s["labels"], s["role"], s["comp"], s["P"], s["st"]) for s in STEP2_STATES.values()]
_evo_states = [x for x in _states if _ran(x[4])]
_other = [(l, r, c, st["engine"]) for l, r, c, P, st in _states if not _ran(st)]
section("Step 2 in full — the complete EVo calculation of every reservoir state")
if not _evo_states:
    note("No state was computed with EVo (analytical fallback or manual values selected in cells 4 and 8).", "warn")
else:
    # ---------------------------------------------------------------- inputs
    _in_rows = [["Chemistry file (wt%)"]]
    _grp = {0}
    for ox in OXIDES:
        if any(ox in st["evo_chem"] for *_x, st in _evo_states):
            _in_rows.append([f"<code>{ox}</code>"] + [gnum(st["evo_chem"].get(ox), 4) for *_x, st in _evo_states])
    _grp.add(len(_in_rows)); _in_rows.append(["Environment file — entries that differ from the shared settings of cell 7"])
    _base = {k: v for k, v in EVO_ENV.items()}
    _keys = ["COMPOSITION", "GAS_SYS", "T_START", "P_STOP"] + sorted(
        {k for *_x, st in _evo_states for k, v in st["evo_env"].items()
         if k not in ("COMPOSITION", "GAS_SYS", "T_START", "P_STOP") and v != _base.get(k)})
    for k in _keys:
        _in_rows.append([f"<code>{k}</code>"] + [html.escape(str(st["evo_env"].get(k))) if isinstance(st["evo_env"].get(k), (str, bool))
                                                   else gnum(st["evo_env"].get(k), 6) for *_x, st in _evo_states])
    table(["Input"] + [_short(l, r, c) for l, r, c, _p, _s in _evo_states], _in_rows, group_rows=_grp,
          title="A · What EVo received",
          caption="All other environment entries are the shared settings of cell 7. "
                  "T_START in K, P_STOP and P_START in bar, volatile contents as mass fractions, fugacities in bar.")

    # ---------------------------------------------------------------- outputs at reservoir P
    _sat_states = [(l, r, c, p, s) for l, r, c, p, s in _evo_states if s.get("evo_df") is not None]
    if _sat_states:
        evo_output_table([(_short(l, r, c) + f"<br>{p:.2f} MPa", s["evo_df"].iloc[-1]) for l, r, c, p, s in _sat_states],
                         "B · Complete EVo output at the reservoir pressure (vapour-saturated states)",
                         "The last row of EVo's output table, i.e. the equilibrium state at P_STOP after closed-system "
                         "decompression from P<sub>sat</sub>. φ = <code>Exsol_vol%</code>/100 enters step 3.")
    _uns = [(l, r, c, p, s) for l, r, c, p, s in _evo_states if s.get("evo_df") is None]
    if _uns:
        note("Undersaturated at the reservoir pressure (no vapour, φ = 0; EVo output exists only at P<sub>sat</sub>, table C): " +
             "; ".join(f"{html.escape(', '.join(l))} — {r} {html.escape(c)} (P = {p:.1f} MPa &gt; P<sub>sat</sub> = {num(s['P_sat_MPa'], 2)} MPa)"
                       for l, r, c, p, s in _uns) + ".")

    # ---------------------------------------------------------------- outputs at P_sat
    _sr = [(l, r, c, p, s) for l, r, c, p, s in _evo_states if s.get("evo_sat_row") is not None]
    if _sr:
        evo_output_table([(_short(l, r, c) + f"<br>P<sub>sat</sub> = {num(s['P_sat_MPa'], 2)} MPa", s["evo_sat_row"].iloc[0])
                          for l, r, c, p, s in _sr],
                         "C · EVo state at the volatile saturation pressure P<sub>sat</sub> (first vapour)",
                         "EVo's saturation-point row: composition of the first vapour bubble and the melt at P<sub>sat</sub>. "
                         "As in EVo itself, <code>Exsol_vol%</code> and <code>Gas_wt</code> here describe the trace vapour "
                         "fraction <code>WgT</code> of cell 7, and the melt density (hence <code>rho_bulk</code>) and "
                         "<code>H2_melt</code> are evaluated with the melt composition of the last computed pressure step — for "
                         "an undersaturated magma, EVo's first step just below P<sub>sat</sub>.")

    # ---------------------------------------------------------------- messages
    _msgs = [(l, r, c, s) for l, r, c, p, s in _evo_states if s.get("evo_prompts")]
    if _msgs:
        note("<b>EVo asked and was answered 'yes' (continue):</b><br>" + "<br>".join(
            f"{html.escape(', '.join(l))} / {html.escape(c)}: " + "; ".join(html.escape(q) for q in s["evo_prompts"]) for l, r, c, s in _msgs), "warn")
    if show_evo_messages:
        for l, r, c, p, s in _evo_states:
            _show(f'<div class="lft">EVo console output — {html.escape(", ".join(l))}, {r} {html.escape(c)}</div>'
                  f'<pre style="font-size:11px;max-height:260px;overflow:auto;background:#f7f7f7;padding:6px">'
                  f'{html.escape(s.get("evo_log", ""))}</pre>')

    # ---------------------------------------------------------------- full paths
    if show_decompression_paths:
        for l, r, c, p, s in _sat_states:
            _df = s["evo_df"]
            _show(f'<div class="lft">EVo path — {html.escape(", ".join(l))}, {r} {html.escape(c)} ({len(_df)} rows, '
                  f'P<sub>sat</sub> → {p:.2f} MPa)</div><div class="lfs">' +
                  _df.to_html(float_format=lambda v: f"{v:.6g}", classes="lf", border=0) + "</div>")
    else:
        note("Tick <code>show_decompression_paths</code> above to list every pressure step of EVo's output "
             "(all columns), and <code>show_evo_messages</code> to see EVo's console output.")
if _other:
    note("Not computed with EVo: " + "; ".join(f"{html.escape(', '.join(l))} — {r} {html.escape(c)} ({e})" for l, r, c, e in _other) + ".")

In [ ]:
#@title 13. Step 7 — surface uplift for every injected volume  { display-mode: "form" }
#@markdown In each UPLIFT cell, V1, V2, … are the uplifts produced by the injected volumes V1, V2, … of the same row (vertical displacement of the surface, above the source centre and at the observation distances of cell 4).

def _sci_plain(v):
    e = int(np.floor(np.log10(v)))
    m = v / 10 ** e
    return (f"{m:g}×10<sup>{e}</sup>").replace("-", "−")

def _uplift_cell(r, dkm):
    return "<br>".join(f"V{k}:&nbsp;{mm(r[f'uz{k}_{dkm:g}km_mm'])}".replace(" ", "&nbsp;")
                       for k in range(1, int(r["n_Ve"]) + 1))

section(f"Step 7 · Predicted surface uplift — {html.escape(PHYS['project'])}")
_HDR = (["RESERVOIR", "DEPTH", "CHAMBER COMPOSITION", "INTRUSION COMPOSITION", "INITIAL VOLUME (m³)",
         "ADDED VOLUMES (m³)"] + [("UPLIFT AT CENTRE" if d == 0 else f"UPLIFT AT {d:g} KM") for d in OBS_KM])
for _sh in dict.fromkeys(res["shape"]):                       # shapes in order of first appearance
    _sub = res[res["shape"] == _sh]
    _rows = []
    for _, r in _sub.iterrows():
        geo = (f" (a = {r.radius_km:g} km)" if _sh == "penny" else
               f" (a = {r.radius_km:.3g} km)" if _sh == "sphere" else
               f" (b/a = {r.aspect:g})" if _sh == "spheroid" else "")
        _rows.append([html.escape(r["name"]) + geo, f"{r.depth_km:g} km", r.chamber, r.intrusion, _sci_plain(r.V0_m3),
                      "<br>".join(f"V{k}:&nbsp;{_sci_plain(r[f'Ve{k}_m3'])}" for k in range(1, int(r.n_Ve) + 1))] +
                     [_uplift_cell(r, d) for d in OBS_KM])
    table(_HDR, _rows, title=SHAPE_LABEL[_sh].upper(), left_cols=tuple(range(0, 1)) + tuple(range(5, 6 + len(OBS_KM))),
          caption="r<sub>V</sub> of the rows (top to bottom): " + ", ".join(f"{v:.3g}" for v in _sub.rV) + ".")

# ------------------------------------------------------------------ shape diagnostic
_far = [d for d in OBS_KM if d > 0]
if 0 in OBS_KM and _far:
    section("Shape of the uplift field — a depth indicator independent of volume and r<sub>V</sub>")
    _rows = []
    for _, r in res.iterrows():
        u0 = r["uz1_0km_mm"]
        _rows.append([html.escape(r["name"]), f"{r.depth_km:g}"] + [f"{r[f'uz1_{d:g}km_mm'] / u0:.2f}" for d in _far])
    table(["Reservoir", "Depth (km)"] + [f"u<sub>z</sub>({d:g} km) / u<sub>z</sub>(0)" for d in _far], _rows,
          caption="Every source is linear in its strength, so these ratios do not depend on the injected volume, on "
                  "r<sub>V</sub> or on the shear modulus: a signal that decays within 1–2 km points to a shallow source, "
                  "one still near its peak at several km to a deep one.")

_p = PHYS
note("<b>How these numbers were obtained.</b> P = ρgz (cell 4, or per reservoir); vapour fraction, speciation and "
     "fugacities from EVo (cell 2; Liggins et al. 2020, 2022) with the magma columns of cell 5; "
     "β<sub>m</sub> = φβ<sub>gas</sub> + (1−φ)β<sub>liquid</sub> with β<sub>gas</sub> = 1/P; β<sub>c</sub> from the "
     "reservoir shape; r<sub>V</sub> = 1 + β<sub>m</sub>/β<sub>c</sub>; ΔV<sub>c</sub> = V<sub>e</sub>/r<sub>V</sub>; "
     "uplift from Mogi (1958), McTigue (1987), Yang et al. (1988)/Newman et al. (2006) or Fialko et al. (2001) as "
     "implemented in dMODELS (Battaglia et al. 2013), in an elastic half-space.")

In [ ]:
#@title 14. Detectability — the volume needed for a target uplift, and validity checks  { display-mode: "form" }
target_uplift_mm = 10.0        #@param {type:"number"}
max_admissible_dP_MPa = 10.0   #@param {type:"number"}

# ============================================================ inverse analysis
section(f"Inverse analysis · what is needed for {target_uplift_mm:g} mm of uplift above the source?")
inv = []
for _, r in res.iterrows():
    uz_per_Ve = r.uz0_per_dVc_mm_per_m3 / r.rV                 # mm of central uplift per m³ injected
    Ve = target_uplift_mm / uz_per_Ve                            # the chain is linear in Ve
    dVc = Ve / r.rV
    dP = dVc / (r.V0_m3 * r.beta_c) / 1e6                        # Eq. 3.21
    inv.append(dict(cfg=r["name"], depth=r.depth_km, rV=r.rV, Ve=Ve, dVc=dVc, dP=dP,
                    ok=dP <= max_admissible_dP_MPa))
inv = sorted(inv, key=lambda d: d["Ve"])
_rows = [[html.escape(d["cfg"]), f"{d['depth']:g}", f"{d['rV']:.2f}", sci(d["Ve"]), sci(d["dVc"]),
          f"{d['dP']:.3g}", "yes" if d["ok"] else "NO"] for d in inv]
table(["Reservoir", "Depth (km)", "r<sub>V</sub>", "V<sub>e</sub> needed (m³)",
       "ΔV<sub>c</sub> (m³)", "ΔP needed (MPa)", "Admissible?"], _rows,
      classes=lambda i, j: (("good" if inv[i]["ok"] else "bad") if j in (5, 6) else None),
      caption=f"Ordered from the cheapest to the most expensive signal. ΔP = ΔV<sub>c</sub>/(β<sub>c</sub>V<sub>0</sub>) "
              f"(Eq. 3.21). A configuration is judged inadmissible when ΔP exceeds {max_admissible_dP_MPa:g} MPa "
              "(cell parameter): the reservoir walls would likely fail before such a signal could form.")
_bad = [d["cfg"] for d in inv if not d["ok"]]
note(f"<b>Reading the table.</b> {len(inv) - len(_bad)} of {len(inv)} configurations reach {target_uplift_mm:g} mm at an "
     "admissible overpressure." + (f" Inadmissible: {html.escape(', '.join(_bad))}." if _bad else ""),
     "warn" if _bad else "info")

# ============================================================ geometric consistency
section("Geometric consistency of the reservoirs")
_rows, _flags = [], []
for _, r in res.iterrows():
    sh = r["shape"]
    if sh == "penny":
        a = r.radius_km * 1e3
        wbar = r.V0_m3 / (np.pi * a ** 2)
        ratio = wbar / (2 * a)
        ok = ratio <= 0.1 and r.radius_km / r.depth_km < 1
        _rows.append([html.escape(r["name"]), SHAPE_LABEL[sh],
                      f"mean opening w̄ = {wbar:,.0f} m; w̄/2a = {ratio:.3g}; a/d = {r.radius_km / r.depth_km:.2f}",
                      "thin crack, a &lt; d" if ok else
                      ("w̄/2a &gt; 0.1: not a thin crack" if ratio > 0.1 else "") +
                      ("; " if ratio > 0.1 and r.radius_km / r.depth_km >= 1 else "") +
                      ("a ≥ d: rim reaches the surface" if r.radius_km / r.depth_km >= 1 else "")])
    elif sh == "sphere":
        ok = r.radius_km / r.depth_km <= 0.5
        _rows.append([html.escape(r["name"]), SHAPE_LABEL[sh], f"radius/depth = {r.radius_km / r.depth_km:.2f}",
                      "a/d ≤ 0.5" if ok else "a/d &gt; 0.5: beyond McTigue's accuracy"])
    elif sh == "spheroid":
        a = (3 * r.V0_m3 / (4 * np.pi * r.aspect ** 2)) ** (1 / 3)
        row = ROW_BY_NAME[r["name"]]
        top = r.depth_km * 1e3 - a * np.sin(np.deg2rad(min(row["dip_deg"], 89.99)))
        ok = top > 0
        _rows.append([html.escape(r["name"]), SHAPE_LABEL[sh],
                      f"semi-axes {a:,.0f} / {a * r.aspect:,.0f} m; top of the spheroid {top:,.0f} m below the surface",
                      "buried" if ok else "reaches the surface"])
    else:
        ok = True
        _rows.append([html.escape(r["name"]), SHAPE_LABEL[sh], f"equivalent radius {(3 * r.V0_m3 / 4 / np.pi) ** (1 / 3):,.0f} m, "
                      f"radius/depth = {(3 * r.V0_m3 / 4 / np.pi) ** (1 / 3) / (1e3 * r.depth_km):.2f}",
                      "point-source approximation"])
    _flags.append(ok)
table(["Reservoir", "Shape", "Geometry", "Check"], _rows, left_cols=(0, 1, 2),
      classes=lambda i, j: ("good" if _flags[i] else "bad") if j == 3 else None,
      caption="Penny cracks: the solution assumes w̄/(2a) ≪ 1 (threshold 0.1 here); rows that violate it describe a "
              "reservoir of the given volume with the wall stiffness of a much thinner crack. Spheres: McTigue's expansion "
              "is accurate for radius/depth ≲ 0.5. Spheroids must lie entirely below the surface.")
if not all(_flags):
    note(f"{len(_flags) - sum(_flags)} of {len(_flags)} configurations fall outside the validity of their source model "
         "— read their results as idealisations.", "warn")

In [ ]:
#@title 15. Figures and the complete results table  { display-mode: "form" }

_depths = sorted(res.depth_km.unique())
_pal = ["#2e8b57", "#c0392b", "#e08b1e", "#2c6fbb", "#7d3c98", "#7f8c8d", "#16a085", "#8e5b3c"]
_col = {d: _pal[i % len(_pal)] for i, d in enumerate(_depths)}
def _extent_km(depth_km):
    """Profile length: the value of cell 4, or ~4 source depths (at least 2 km)."""
    return PHYS["profile_max_km"] if PHYS["profile_max_km"] > 0 else float(np.clip(4.0 * depth_km, 2.0, 60.0))

def _vlab(v):
    e = int(np.floor(np.log10(v))); m = v / 10 ** e
    return (f"{m:g}×" if abs(m - 1) > 1e-9 else "") + f"10$^{{{e}}}$"

_rmax_km = _extent_km(max(_depths))
_r = np.linspace(0, _rmax_km * 1e3, 241)

# ------------------------------------------------------------ Figure 1: radial profiles
section("Figure 1 · Radial profiles of vertical uplift, every reservoir and injected volume")
_n = len(res) + 1
_MAXP = 24
_show_rows = res.iloc[:_MAXP - 1]
_ncol = 3
_nrow = int(np.ceil(min(_n, _MAXP) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(14, 3.3 * _nrow), squeeze=False)
_vcol = plt.cm.viridis(np.linspace(0.15, 0.85, max(int(res.n_Ve.max()), 2)))
for ax, (_, r) in zip(axes.flat, _show_rows.iterrows()):
    row = ROW_BY_NAME[r["name"]]
    _ri = np.linspace(0, _extent_km(r.depth_km) * 1e3, 241)
    for k, Ve in enumerate(ve_list(r), start=1):
        ax.plot(_ri / 1e3, uplift_row(row, Ve / r.rV, _ri) * 1e3, color=_vcol[k - 1], lw=1.5,
                label=f"V$_e$ = {_vlab(Ve)} m$^3$")
    ax.set_title(f"{r['name']}  ({r.depth_km:g} km, r$_V$ = {r.rV:.3g})", fontsize=9)
    ax.grid(alpha=.25, lw=.5); ax.set_xlim(0, _ri[-1] / 1e3); ax.legend(fontsize=7, frameon=False)
    ax.set_xlabel("radial distance [km]", fontsize=8); ax.set_ylabel("uplift [mm]", fontsize=8)
ax = axes.flat[len(_show_rows)]
for _, r in _show_rows.iterrows():
    u = uplift_row(ROW_BY_NAME[r["name"]], 1.0, _r)
    ax.plot(_r / 1e3, u / u[0], color=_col[r.depth_km], lw=1.2, alpha=.85)
for d in _depths:
    ax.plot([], [], color=_col[d], label=f"{d:g} km")
ax.set_title("Normalised shape u$_z$(r)/u$_z$(0) — depth indicator", fontsize=9)
ax.set_xlim(0, _rmax_km); ax.set_ylim(0, 1.05); ax.grid(alpha=.25, lw=.5); ax.legend(fontsize=7, frameon=False)
ax.set_xlabel("radial distance [km]", fontsize=8)
for ax in list(axes.flat)[len(_show_rows) + 1:]:
    ax.axis("off")
fig.tight_layout(); plt.show()
if len(res) > _MAXP - 1:
    note(f"Figure 1 shows the first {_MAXP - 1} of {len(res)} reservoirs; all are in the tables.")

# ------------------------------------------------------------ Figure 2: r_V and detectability
section("Figure 2 · Volume-partitioning factor and detectability threshold")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, max(5.2, 0.32 * len(res) + 2)))
a1.bar(range(len(res)), res.rV, color=[_col[d] for d in res.depth_km], edgecolor="k", lw=.4)
a1.set_yscale("log"); a1.axhline(1, color="k", ls=":", lw=.8)
a1.set_xticks(range(len(res))); a1.set_xticklabels([f"{n}" for n in res["name"]], rotation=90, fontsize=7)
a1.set_ylabel(r"$r_V = 1+\beta_m/\beta_c$"); a1.grid(axis="y", alpha=.25, lw=.5)
a1.set_title("(a) Volume-partitioning factor (dotted: r$_V$ = 1, full expression)", fontsize=10)
yy = np.arange(len(inv))
bars = a2.barh(yy, [d["Ve"] for d in inv], color=[_col[d["depth"]] for d in inv], edgecolor="k", lw=.4)
for b, d in zip(bars, inv):
    if not d["ok"]:
        b.set_hatch("///"); b.set_alpha(.55)
a2.set_yticks(yy); a2.set_yticklabels([f"{d['cfg']} | {d['depth']:g} km" for d in inv], fontsize=7.5)
a2.set_xscale("log"); a2.grid(axis="x", alpha=.25, lw=.5)
a2.set_xlabel(f"injected volume needed for {target_uplift_mm:g} mm uplift above the source [m$^3$]")
a2.set_title(f"(b) Detectability threshold (hatched: ΔP > {max_admissible_dP_MPa:g} MPa)", fontsize=10)
from matplotlib.patches import Patch
a2.legend(handles=[Patch(facecolor=_col[d], edgecolor="k", lw=.4, label=f"{d:g} km") for d in _depths],
          fontsize=7.5, loc="lower right", title="depth", title_fontsize=7.5)
fig.tight_layout(); plt.show()

# ------------------------------------------------------------ complete results
section("Complete numerical results (every quantity computed for every reservoir)")
_full = res.copy()
_full["phi"] = 100 * _full["phi"]
_full = _full.rename(columns={"phi": "phi_vol%", "depth_km": "depth (km)", "P_MPa": "P (MPa)",
                              "P_sat_MPa": "P_sat (MPa)", "V0_m3": "V0 (m3)", "fO2_dFMQ": "fO2_dbuffer"})
pd.set_option("display.max_columns", 300); pd.set_option("display.width", 250)
try:
    _sty = (_full.style.format(precision=4, thousands=",", na_rep="—")
            .format({c: "{:.3e}" for c in _full.columns if c.startswith(("beta", "V", "dVc", "uz0_per"))}, na_rep="—")
            .set_properties(**{"font-size": "11px"})
            .set_table_styles([{"selector": "th", "props": [("font-size", "11px"), ("background", "#2E5D8A"),
                                                         ("color", "white")]}]))
    display(_sty)
except Exception:                       # pandas Styler needs jinja2; plain table otherwise
    display(_full.round(6))
note("To try other values, change the tables (cells 5, 6, 8) or the settings (cells 4, 7, 14), click cell 9 and "
     "choose <i>Runtime ▸ Run after</i>.", "ok")

In [ ]:
#@title 16. Uplift map of one reservoir  { display-mode: "form", run: "auto" }
#@markdown Type the name of a reservoir from cell 6 (empty = the first one) and choose which injected volume (1 = V1, …; 0 = the largest). Half-width of the map in km (0 = automatic).
MAP_reservoir = ""        #@param {type:"string"}
MAP_volume_index = 0      #@param {type:"integer"}
MAP_half_width_km = 0     #@param {type:"number"}

_name = MAP_reservoir.strip() or res["name"].iloc[0]
if _name not in ROW_BY_NAME:
    _cand = difflib.get_close_matches(_name, list(ROW_BY_NAME), n=1, cutoff=0.4)
    note(f"No reservoir named '{html.escape(_name)}'" + (f"; showing '{html.escape(_cand[0])}' instead." if _cand else
         "; showing the first reservoir.") + " Names: " + ", ".join(html.escape(n) for n in ROW_BY_NAME), "warn")
    _name = _cand[0] if _cand else res["name"].iloc[0]
_row = ROW_BY_NAME[_name]
_rr = res[res["name"] == _name].iloc[0]
_ves = ve_list(_rr)
_k = len(_ves) if MAP_volume_index <= 0 or MAP_volume_index > len(_ves) else int(MAP_volume_index)
_Ve = _ves[_k - 1]
_dVc = _Ve / _rr.rV
_hw = MAP_half_width_km * 1e3 if MAP_half_width_km > 0 else max(1.5 * _row["depth_km"] * 1e3, 800.0)
_g = np.linspace(-_hw, _hw, 121)
_X, _Y = np.meshgrid(_g, _g)
_W = uplift_map(_row, _dVc, _X, _Y) * 1e3
_imax = int(np.argmax(_W))
_xm, _ym = _X.ravel()[_imax], _Y.ravel()[_imax]

section(f"Map view · {html.escape(_name)} — V<sub>e</sub> = {sci(_Ve)} m³ → ΔV<sub>c</sub> = {sci(_dVc)} m³")
fig, ax = plt.subplots(figsize=(6.6, 5.6))
cs = ax.contourf(_X / 1e3, _Y / 1e3, _W, levels=18, cmap="viridis")
_lv = [l for l in np.unique(np.round(np.nanmax(_W) * np.array([0.1, 0.25, 0.5, 0.75]), 3)) if l > 0]
if _lv:
    ax.contour(_X / 1e3, _Y / 1e3, _W, levels=_lv, colors="w", linewidths=.6)
ax.plot(0, 0, "k+", ms=10, mew=1.5, label="source centre")
ax.plot(_xm / 1e3, _ym / 1e3, "r*", ms=11, label=f"maximum {mm(_W.ravel()[_imax])}")
ax.set_aspect("equal"); ax.set_xlabel("east [km]"); ax.set_ylabel("north [km]")
ax.legend(fontsize=8, loc="lower right"); plt.colorbar(cs, ax=ax, label="vertical uplift [mm]")
ax.set_title(f"{_name}: {SHAPE_LABEL[_row['shape']]}, depth {_row['depth_km']:g} km", fontsize=9.5)
fig.tight_layout(); plt.show()
note(f"Uplift above the centre {mm(float(uplift_row(_row, _dVc, [0.0])[0]) * 1e3)}; maximum {mm(_W.ravel()[_imax])} "
     f"at {np.hypot(_xm, _ym):.0f} m from the centre" +
     (f", towards azimuth {(np.degrees(np.arctan2(_xm, _ym)) + 360) % 360:.0f}°" if np.hypot(_xm, _ym) > 1 else "") +
     f". White contours: 10, 25, 50 and 75% of the maximum.")

In [ ]:
#@title 17. EVo calculator — one magma at any pressure, with optional changes  { display-mode: "form", run: "auto" }
#@markdown Type the name of a magma from cell 5 (empty = the first one) and a pressure; tick a box to change that input for this calculation only. Every EVo output is shown.
CALC_magma = ""                #@param {type:"string"}
CALC_P_MPa = 14.66595          #@param {type:"number"}
CALC_change_T = False          #@param {type:"boolean"}
CALC_T_C = 1000                #@param {type:"number"}
CALC_change_volatiles = False  #@param {type:"boolean"}
CALC_H2O_wt = 1.25             #@param {type:"number"}
CALC_CO2_wt = 0.005            #@param {type:"number"}
CALC_S_wt = 0.02               #@param {type:"number"}
CALC_change_redox = False      #@param {type:"boolean"}
CALC_fO2_buffer = "FMQ"        #@param ["FMQ", "NNO", "IW", "absolute", "iron"]
CALC_fO2_value = 1.0           #@param {type:"number"}

import copy
_key = CALC_magma.strip().upper() or next(iter(MAGMA))
_calc_msg = None
if _key not in MAGMA:
    _alt = difflib.get_close_matches(_key, list(MAGMA), n=1, cutoff=0.6)
    _key2 = _alt[0] if _alt else next(iter(MAGMA))
    _calc_msg = (f"No magma named '{html.escape(CALC_magma)}' in the current tables; showing "
                 f"<b>{html.escape(MAGMA[_key2]['name'])}</b> instead. Magmas available: " +
                 ", ".join(html.escape(m["name"]) for m in MAGMA.values()) + ".")
    _key = _key2
_rec = copy.deepcopy(MAGMA[_key])
_changes = []
if CALC_change_volatiles:
    for k, v in (("H2O_wt", CALC_H2O_wt), ("CO2_wt", CALC_CO2_wt), ("S_wt", CALC_S_wt)):
        _rec[k] = float(v)
    _changes.append(f"H<sub>2</sub>O {CALC_H2O_wt:g}, CO<sub>2</sub> {CALC_CO2_wt:g}, S {CALC_S_wt:g} wt%")
if CALC_change_redox:
    _rec["fO2_buffer"], _rec["fO2_value"] = CALC_fO2_buffer, float(CALC_fO2_value)
    _changes.append(f"redox {CALC_fO2_buffer} {CALC_fO2_value:g}")
_rec["gas_sys_resolved"] = resolve_gas_system(_rec)
_T = float(CALC_T_C) if CALC_change_T else _rec["T_C"]
if CALC_change_T:
    _changes.append(f"T {CALC_T_C:g} °C")
calc = run_evo_state(None, float(CALC_P_MPa), _T, record=_rec)

section(f"EVo calculator · {html.escape(_rec['name'])} at {CALC_P_MPa:g} MPa and {_T:g} °C")
if _calc_msg:
    note(_calc_msg)
if _changes:
    note("Changed for this calculation only: " + "; ".join(_changes) + ".")
if calc.get("evo_error"):
    note("EVo could not run these inputs: " + html.escape(str(calc["evo_error"])) +
         ". Values below come from the analytical fallback.", "warn")
for c in calc.get("evo_notes", []):
    note(html.escape(c))
table(["Quantity", "Value", "Quantity", "Value"],
      [["Volatile saturation pressure P<sub>sat</sub>", f"{num(calc['P_sat_MPa'], 3)} MPa",
        "Vapour present at P?", "yes" if calc["saturated"] else "no (φ = 0)"],
       ["Vapour volume fraction φ", f"{num(100 * calc['phi'], 4)} vol%",
        "Vapour mass fraction", f"{num(100 * calc['gas_wt'], 4)} wt%"],
       ["Melt density", f"{num(calc['rho_melt'], 1)} kg m<sup>−3</sup>",
        "Vapour density", f"{num(calc['rho_gas'], 2)} kg m<sup>−3</sup>"],
       [f"log fO<sub>2</sub> − {calc.get('evo_env', {}).get('FO2_buffer', 'FMQ')}", num(calc["fO2_dFMQ"], 3),
        "β<sub>m</sub><sup>eff</sup> (equilibrium path)", f"{sci(calc['beta_m_evo'])} Pa<sup>−1</sup>"],
       ["fH<sub>2</sub>O / fCO<sub>2</sub>", f"{num(calc['fH2O'], 3)} / {num(calc['fCO2'], 4)} bar",
        "fSO<sub>2</sub> / fH<sub>2</sub>S", f"{num(calc['fSO2'], 4)} / {num(calc['fH2S'], 4)} bar"]],
      left_cols=(0, 2))
_cols = []
if calc.get("evo_df") is not None:
    _cols.append((f"at P = {CALC_P_MPa:g} MPa", calc["evo_df"].iloc[-1]))
if calc.get("evo_sat_row") is not None:
    _cols.append((f"at P<sub>sat</sub> = {num(calc['P_sat_MPa'], 3)} MPa", calc["evo_sat_row"].iloc[0]))
if _cols:
    evo_output_table(_cols, "Complete EVo output", "All quantities EVo returns (definitions in cell 12).")

if calc.get("evo_df") is not None and len(calc["evo_df"]) > 1:
    _d = calc["evo_df"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    a1.plot(_d["P"] / 10, _d["Exsol_vol%"], "o-", ms=3, color="#2c6fbb")
    a1b = a1.twinx(); a1b.plot(_d["P"] / 10, _d["Gas_wt"], "s--", ms=3, color="#c0392b")
    a1.set_xlabel("pressure [MPa]"); a1.set_ylabel("vapour volume fraction φ [vol%]", color="#2c6fbb")
    a1b.set_ylabel("vapour mass fraction [wt%]", color="#c0392b"); a1.invert_xaxis(); a1.grid(alpha=.25, lw=.5)
    a1.set_title("(a) Exsolution during decompression", fontsize=10)
    for s, col in (("H2O", "#2c6fbb"), ("CO2", "#7f8c8d"), ("SO2", "#e08b1e"), ("H2S", "#2e8b57"),
                   ("H2", "#7d3c98"), ("S2", "#c0392b"), ("CO", "#555555"), ("N2", "#16a085")):
        if f"m{s}" in _d and (_d[f"m{s}"] > 0).any():
            a2.plot(_d["P"] / 10, _d[f"m{s}"], lw=1.6, color=col, label=s)
    a2.set_yscale("log"); a2.invert_xaxis(); a2.grid(alpha=.25, lw=.5); a2.legend(fontsize=8, frameon=False, ncol=2)
    a2.set_xlabel("pressure [MPa]"); a2.set_ylabel("mole fraction in the vapour")
    a2.set_title("(b) Vapour speciation", fontsize=10)
    fig.tight_layout(); plt.show()
elif not calc["saturated"]:
    note(f"No vapour at {CALC_P_MPa:g} MPa (P<sub>sat</sub> = {num(calc['P_sat_MPa'], 3)} MPa): there is no "
         "decompression path to plot. Lower the pressure or raise the volatile budget.")

In [ ]:
#@title 18. Benchmark — the deformation code checked against a published inversion (independent of your inputs)  { display-mode: "form" }
#@markdown A fixed test, not part of your scenario: the prolate-spheroid mechanics of this notebook are compared with the inversion of the 2021 La Fossa unrest by Di Traglia et al. (2023, Supporting Information, Table S1). Untick to skip.
run_benchmark = True   #@param {type:"boolean"}
if run_benchmark:
    # ============================================================ validation vs Di Traglia 2023
    section("Benchmark · the spheroid source against Di Traglia et al. (2023, Supporting Information)")
    # published values, Table S1 (nu = 0.35) — used as fixed validation targets, independent of the levers
    _A, _a_pub, _b_pub, _V0, _k = 0.099, 595.0, 59.0, 8.83e6, 11.6e-8
    _mu, _nu, _dip, _az, _z = 1e9, 0.35, 69.0, 137.0, 598.0
    _dV_pub, _dP_pub = 73108.0, 8.48e6

    bcmu_mine = beta_c_yang(_V0, _A, _z, _mu, _nu, _dip, _az) * _mu
    bcmu_pub = 1.0 / (_k * _V0)
    dV_mine = bcmu_mine / _mu * _V0 * _dP_pub
    dP_V0 = _dV_pub / (_V0 * bcmu_mine / _mu)
    dP_axes = _dV_pub / spheroid_dV(_a_pub, _b_pub / _a_pub, 1.0, _mu)
    uz_c = uplift_yang(_dV_pub, _V0, _A, _z, _mu, _nu, _dip, _az, [0.0, 1000.0, 2000.0])

    # maximum uplift on a map grid and its position
    _g = np.linspace(-800, 800, 161)   # 10 m spacing
    _X, _Y = np.meshgrid(_g, _g)
    _sp, _asp, _dPref = yang_source(_V0, _A, _z, _mu, _nu, _dip, _az)
    _W = _sp.calc_displ(_X.ravel(), _Y.ravel(), np.zeros(_X.size))[2] * _dV_pub / spheroid_dV(_asp, _A, _dPref, _mu)
    _imax = int(np.argmax(_W))
    _xm, _ym = _X.ravel()[_imax], _Y.ravel()[_imax]
    _az_max = (np.degrees(np.arctan2(_xm, _ym)) + 360) % 360

    def _pct(a, b):
        return f"{100 * (a / b - 1):+.1f}%"

    _rows = [
        ["β<sub>c</sub>·μ of the spheroid (Eq. 3.17)", f"{bcmu_mine:.3f}", f"{bcmu_pub:.3f} (from k = 11.6×10<sup>−8</sup> m<sup>−3</sup>)", _pct(bcmu_mine, bcmu_pub)],
        ["ΔV<sub>c</sub> at ΔP = 8.48 MPa", f"{dV_mine:,.0f} m³", "73,108 ± 6,900 m³", _pct(dV_mine, _dV_pub)],
        ["ΔP for ΔV<sub>c</sub> = 73,108 m³ (using V<sub>0</sub>)", f"{dP_V0 / 1e6:.2f} MPa", "8.48 MPa", _pct(dP_V0, _dP_pub)],
        ["ΔP for ΔV<sub>c</sub> = 73,108 m³ (using semi-axes 595/59 m)", f"{dP_axes / 1e6:.2f} MPa", "8.48 MPa", _pct(dP_axes, _dP_pub)],
        ["Vertical uplift above the centroid", mm(uz_c[0] * 1e3), "cm-scale (LOS, longer window)", "consistent"],
        ["Vertical uplift at 1 km / 2 km", f"{mm(uz_c[1] * 1e3)} / {mm(uz_c[2] * 1e3)}", "confined to the crater area", "consistent"],
        ["Maximum uplift and its position", f"{mm(_W[_imax] * 1e3)}, {np.hypot(_xm, _ym):.0f} m at azimuth {_az_max:.0f}°",
         "expected towards the up-dip tip (azimuth ≈ 317°)", "consistent"],
    ]
    table(["Quantity", "This notebook", "Published", "Difference"], _rows,
          classes=lambda i, j: "good" if j == 3 else None,
          caption="Targets are fixed at the published Table S1 values (ν = 0.35, μ = 1 GPa), so this check does not depend "
                  "on the levers above. Because Di Traglia et al. used dMODELS, which adopts the same Amoruso & Crescentini "
                  "(2009) expression, the agreement demonstrates that the two implementations are equivalent; the physical "
                  "accuracy of the expression itself is set by its exact sphere and needle limits (0.3% and 2.8%).")

    # four-nu cross-check of Table S1
    _S1 = {0.25: (855, 85, 2.62e7, 3.91e-8), 0.30: (703, 70, 1.46e7, 7.02e-8),
           0.35: (595, 59, 8.83e6, 11.6e-8), 0.40: (706, 71, 1.47e7, 6.94e-8), 0.45: (656, 66, 1.18e7, 8.66e-8)}
    _rows = []
    for _nu_i, (a_i, b_i, V_i, k_i) in _S1.items():
        m_i = beta_c_yang(V_i, b_i / a_i, 598.0, 1.0, _nu_i, _dip, _az)
        p_i = 1.0 / (k_i * V_i)
        _rows.append([f"{_nu_i:.2f}", f"{a_i} / {b_i}", f"{b_i / a_i:.4f}", sci(V_i), f"{m_i:.3f}", f"{p_i:.3f}",
                      _pct(m_i, p_i) + (" *" if _nu_i == 0.25 else "")])
    table(["ν", "a / b (m)", "A", "V<sub>0</sub> (m³)", "β<sub>c</sub>μ this notebook", "β<sub>c</sub>μ from published k", "Difference"],
          _rows, caption="All five Poisson-ratio inversions of Table S1. * The k printed for ν = 0.25 (3.91×10<sup>−3</sup>) "
                         "is inconsistent with its neighbours by three orders of magnitude; it is read here as "
                         "3.91×10<sup>−8</sup> m<sup>−3</sup>, a probable misprint.")
    _bench_ok = abs(bcmu_mine / bcmu_pub - 1) < 0.01 and abs(dV_mine / _dV_pub - 1) < 0.01 and abs(dP_V0 / _dP_pub - 1) < 0.01
    note(("<b>Benchmark passed.</b> The code reproduces the published pressure–volume mechanics of the 2021 source "
          "to within a few tenths of a per cent." if _bench_ok else
          "<b>Benchmark FAILED</b> — the spheroid mechanics differ from the published values by more than 1%."),
         "ok" if _bench_ok else "warn")
else:
    note("Benchmark skipped.")

### References — software, worked-example data and methods
* **EVo** (thermodynamic degassing model, step 2): Liggins, P., Shorttle, O. & Rimmer, P. B. (2020), *Earth Planet. Sci. Lett.* 550, 116546; Liggins, P. et al. (2022), *J. Geophys. Res. Planets* 127, e2021JE007123. Source: github.com/pipliggins/EVo, v1.1.0, commit `2487939`. **Licence GPL-3.0-or-later.** Embedded verbatim in cell 2 (17 modules + 11 data tables); the adaptations are listed under that cell.
* **dmodelspy** 0.2 (W. Crawford; Python port of dMODELS, Battaglia, Cervelli & Murray 2013, *J. Volcanol. Geotherm. Res.* 254, 1–4). **Licence GPL-3.0.** Embedded verbatim in cell 3.
* **Deformation sources:** Mogi (1958); McTigue (1987, *JGR* 92); Yang et al. (1988, *JGR* 93) with Newman et al. (2006); Fialko et al. (2001, *GJI* 146)
* **Compressibility framework:** Rivalta & Segall (2008, *GRL* 35, L04306); Kilbride et al. (2016, *Nat. Commun.* 7, 13744); Segall (2010); prolate spheroid compliance: Amoruso & Crescentini (2009, *JGR* 114, B02210)
* **Solubility and redox models in EVo:** Burgisser et al. (2015); Gaillard et al. (2003); Eguchi & Dasgupta (2018); Kress & Carmichael (1991); Righter et al. (2013); Frost (1991); O'Neill (2002, 2020); Nash et al. (2019); Liu et al. (2007); Libourel et al. (2003); Spera (2000)
* **Worked example (La Fossa, Vulcano):** melt compositions Gioncada et al. (1998, *Bull. Volcanol.* 60, Table 1); temperatures and volatiles Clocchiatti et al. (1994), Gioncada et al. (1998), Fusillo et al. (2015), Paonita et al. (2013); 2021 source Di Traglia et al. (2023, *GRL* 50, e2023GL104952); host-rock moduli Heap et al. (2020)

**Assumptions.** Homogeneous, isotropic elastic half-space with a flat surface; static end-state displacements; ideal-gas β<sub>gas</sub> = 1/P with frozen phases during pressurisation (the equilibrium β<sub>m</sub><sup>eff</sup> from EVo is reported as an upper bound); the spheroid compliance is a full-space expression, the crack compliance a half-space one; each reservoir is treated on its own (no interaction between sources).

This notebook combines GPL-3.0 code and is itself distributed under GPL-3.0-or-later; keep the credits above when sharing it.